# Pathology Hub — Complete Textbook Workflow Notebook

**Workstream:** Textbook + Tag RAG / Source Normalization

This is the combined workflow:

```text
GCS source PDFs
→ Colab PDF extraction to raw *_UNIFIED.json only
→ CHATGPT INTERMISSION: upload UNIFIED + cleaner/tagging prompt
→ ChatGPT returns LEAN ZIP packages
→ Colab integrates LEAN packages, builds SQLite FTS, uploads final artifacts
```

Guardrails:

- Raw `*_UNIFIED.json` files are **staged only**.
- ChatGPT is where LEAN cleaning + controlled candidate/semantic tag assignment happens.
- Colab resumes only for integration, SQLite FTS, validation, and GCS upload.
- No vectorization or API exposure is claimed.


In [ ]:
# ============================================================
# 00 CONFIG — EDIT HERE ONLY
# ============================================================

PROJECT_ID = "pathology-annotation-project"
SOURCE_GCS_PDF_URI = "gs://pathology-hub-0/source_pdfs"
DEST_BUCKET = "pathology_hub"  # canonical artifact bucket, no gs:// prefix

# Process these source PDFs only. Default is Charlie's selected 39.
SELECTED_PDF_NAMES = [
    "BST_Horvai.pdf",
    "Bone_Atlas.pdf",
    "Bone_Dorfman.pdf",
    "Bone_Pattern.pdf",
    "Breast_Atlas.pdf",
    "Breast_Biopsy.pdf",
    "Breast_FAQ.pdf",
    "Breast_Pattern.pdf",
    "Cyto_Breast_Yokohama.pdf",
    "Cyto_Cibas.pdf",
    "Cyto_Comprehensive_Part_One.pdf",
    "Cyto_Comprehensive_Part_Two.pdf",
    "Cyto_Fluids_Ali.pdf",
    "Cyto_GU_Paris.pdf",
    "Cyto_Gyn_Bethesda.pdf",
    "Cyto_Milan.pdf",
    "Cyto_PSC_Lung.pdf",
    "Cyto_Pattern.pdf",
    "Cyto_Serous_Fluids.pdf",
    "Cyto_Thyroid_Bethesda.pdf",
    "Derm_Elston.pdf",
    "Derm_Levers.pdf",
    "Derm_McKee.pdf",
    "Endo_Atlas.pdf",
    "GI_Atlas.pdf",
    "GI_Biopsy_Interpretation_(Neoplastic).pdf",
    "GI_Biopsy_Interpretation_(Non_Neoplastic).pdf",
    "GI_Intestinal_Atlas1.pdf",
    "GI_Liver_Macsween.pdf",
    "GU_Bladder.pdf",
    "GU_Practical.pdf",
    "GU_Prostate.pdf",
    "Gyn_Atlas_Part_One.pdf",
    "Gyn_Atlas_Part_Two.pdf",
    "Gyn_Essentials.pdf",
    "Molecular_FAQ.pdf",
    "Molecular_Vasef.pdf",
    "SoftTissue_Enzinger.pdf",
    "SoftTissue_Pattern.pdf"
]
EXPECTED_SELECTED_PDF_COUNT = 39

# Parallel PDF extraction limit.
MAX_WORKERS = 5

# Extraction behavior.
EXTRACT_EMBEDDED_FIGURES = True
RENDER_PAGE_IMAGES = False        # set True only if you want full-page PNG/thumbnail assets; slower/larger
PAGE_RENDER_DPI = 120
THUMB_MAX_WIDTH = 400

# Safety.
CLEAR_STAGED_PDFS_BEFORE_DOWNLOAD = True
CLEAR_STAGE1_UNIFIED_OUTPUTS_BEFORE_RUN = False  # set True only when intentionally rebuilding all UNIFIED files

# Upload behavior.
SYNC_STAGE1_TO_GCS_AFTER_BATCH = True   # uploads raw UNIFIED + extracted assets to 01_staged only
SYNC_CHATGPT_JOBS_TO_GCS = True         # uploads prompt/job ZIPs to 01_staged/textbooks/chatgpt_jobs
SYNC_LEAN_OUTPUTS_TO_GCS = True         # uploads LEAN normalized/index/audits after ChatGPT returns packages

# ChatGPT return package discovery.
RUN_UPLOAD_PICKER_FOR_LEAN_ZIPS = True  # after ChatGPT intermission, run the upload cell and select returned LEAN ZIPs

print("Selected PDFs:", len(SELECTED_PDF_NAMES))
assert len(SELECTED_PDF_NAMES) == EXPECTED_SELECTED_PDF_COUNT


## 01 Install dependencies

In [ ]:
!pip -q install pymupdf pillow tqdm

## 02 Imports, local folders, and helpers

In [ ]:
# ============================================================
# 02 IMPORTS + LOCAL PATHS + HELPERS
# ============================================================

import os, re, io, json, time, shutil, zipfile, sqlite3, hashlib, datetime, subprocess, traceback
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Any, Dict, List, Optional

import fitz  # PyMuPDF
from PIL import Image
from tqdm.auto import tqdm

BASE = Path("pathology_hub")
RUNTIME_DIR = BASE / "00_runtime"
SOURCE_PDF_DIR = RUNTIME_DIR / "selected_pdfs"

# STAGE 1 raw extraction outputs. These are staged/provenance only, not canonical LEAN normalized artifacts.
STAGED_TEXTBOOK_DIR = BASE / "01_staged" / "textbooks"
STAGED_UNIFIED_DIR = STAGED_TEXTBOOK_DIR / "unified_raw"
STAGED_RAW_FIGURE_JSONL_DIR = STAGED_TEXTBOOK_DIR / "raw_figure_jsonl"
STAGED_ASSET_DIR = STAGED_TEXTBOOK_DIR / "assets"
STAGED_PAGE_IMAGE_DIR = STAGED_ASSET_DIR / "page_images"
STAGED_THUMB_DIR = STAGED_ASSET_DIR / "page_thumbnails"
STAGED_FIGURE_IMAGE_DIR = STAGED_ASSET_DIR / "figure_images"
CHATGPT_JOB_DIR = STAGED_TEXTBOOK_DIR / "chatgpt_jobs"
PDF_INGEST_AUDIT_DIR = BASE / "06_audits" / "textbooks" / "pdf_ingest"

# STAGE 3 LEAN package integration outputs. These are canonical normalized/index artifacts after ChatGPT cleaning.
LEAN_RETURN_DIR = Path("/content/lean_returns")
STAGED_LEAN_PACKAGE_DIR = STAGED_TEXTBOOK_DIR / "lean_packages"
LEAN_NORM_DIR = BASE / "02_normalized" / "textbooks" / "lean"
LEAN_INDEX_DIR = BASE / "03_indexes" / "textbooks" / "lean"
LEAN_AUDIT_DIR = BASE / "06_audits" / "textbooks" / "lean"

for d in [
    SOURCE_PDF_DIR, STAGED_UNIFIED_DIR, STAGED_RAW_FIGURE_JSONL_DIR, STAGED_PAGE_IMAGE_DIR,
    STAGED_THUMB_DIR, STAGED_FIGURE_IMAGE_DIR, CHATGPT_JOB_DIR, PDF_INGEST_AUDIT_DIR,
    LEAN_RETURN_DIR, STAGED_LEAN_PACKAGE_DIR, LEAN_NORM_DIR, LEAN_INDEX_DIR, LEAN_AUDIT_DIR
]:
    d.mkdir(parents=True, exist_ok=True)


def now_utc() -> str:
    return datetime.datetime.now(datetime.timezone.utc).isoformat()


def slugify(value: str, max_len: int = 140) -> str:
    value = str(value or "").strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    if len(value) > max_len:
        h = hashlib.sha1(value.encode("utf-8", errors="ignore")).hexdigest()[:10]
        value = value[:max_len-11].strip("_") + "_" + h
    return value or "unknown"


def infer_source_id(path_or_name: str) -> str:
    stem = Path(path_or_name).stem
    stem = re.sub(r"_(UNIFIED|UNIFIED_LEAN|LEAN|MASTER|CONTENT|FIGURES|WITH_PAGE_IMAGES)$", "", stem, flags=re.I)
    return slugify(stem)


def clean_text(x) -> str:
    if x is None:
        return ""
    if isinstance(x, (list, tuple)):
        x = "\n\n".join(str(v) for v in x)
    elif isinstance(x, dict):
        x = json.dumps(x, ensure_ascii=False)
    else:
        x = str(x)
    x = x.replace("\x00", " ")
    x = re.sub(r"\r\n?", "\n", x)
    x = re.sub(r"[ \t]+", " ", x)
    x = re.sub(r"\n{3,}", "\n\n", x)
    return x.strip()


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def save_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def read_jsonl(path):
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                try:
                    out.append(json.loads(line))
                except Exception as e:
                    raise ValueError(f"JSONL parse failure in {path} line {line_no}: {e}")
    return out


def count_jsonl(path) -> int:
    path = Path(path)
    if not path.exists():
        return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def sha256_file(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def local_to_gcs(local_path) -> str:
    local_path = Path(local_path)
    rel = local_path.as_posix()
    base_prefix = BASE.as_posix() + "/"
    if rel.startswith(base_prefix):
        rel = rel[len(base_prefix):]
    return f"gs://{DEST_BUCKET}/{rel}"


def gs_to_https(uri):
    if not uri or not str(uri).startswith("gs://"):
        return uri
    rest = str(uri)[5:]
    bucket, _, blob = rest.partition("/")
    return f"https://storage.googleapis.com/{bucket}/{blob}"


def run_cmd(cmd, allow_fail=False):
    print("RUN:", " ".join(map(str, cmd)))
    r = subprocess.run([str(x) for x in cmd], capture_output=True, text=True)
    if r.stdout:
        print(r.stdout[-4000:])
    if r.stderr:
        print(r.stderr[-4000:])
    if r.returncode != 0 and not allow_fail:
        raise RuntimeError(r.stderr)
    return r


def authenticate_gcp_colab():
    try:
        from google.colab import auth
        auth.authenticate_user()
        run_cmd(["gcloud", "config", "set", "project", PROJECT_ID], allow_fail=True)
        print("GCP authenticated.")
        return True
    except Exception as e:
        print("GCP auth skipped/failed:", e)
        return False

print("Local root:", BASE.resolve())
print("Stage 1 raw UNIFIED local:", STAGED_UNIFIED_DIR)
print("Stage 3 LEAN normalized local:", LEAN_NORM_DIR)


## 03 GCS selection, download, and sync helpers

In [ ]:
# ============================================================
# 03 GCS HELPERS
# ============================================================

SOURCE_PDF_GCS_MAP = {}


def register_source_pdf_gcs_uri(local_path, gcs_uri: str):
    p = Path(local_path)
    SOURCE_PDF_GCS_MAP[str(p)] = gcs_uri
    SOURCE_PDF_GCS_MAP[p.name] = gcs_uri
    try:
        SOURCE_PDF_GCS_MAP[str(p.resolve())] = gcs_uri
    except Exception:
        pass


def get_source_pdf_gcs_uri(local_path):
    p = Path(local_path)
    for key in [str(p), p.name]:
        if key in SOURCE_PDF_GCS_MAP:
            return SOURCE_PDF_GCS_MAP[key]
    try:
        key = str(p.resolve())
        if key in SOURCE_PDF_GCS_MAP:
            return SOURCE_PDF_GCS_MAP[key]
    except Exception:
        pass
    return None


def list_source_pdfs_from_gcs(gcs_prefix=SOURCE_GCS_PDF_URI):
    prefix = gcs_prefix.rstrip("/")
    r = run_cmd(["gsutil", "ls", f"{prefix}/*.pdf"], allow_fail=False)
    pdfs = sorted([line.strip() for line in r.stdout.splitlines() if line.strip().lower().endswith(".pdf")], key=lambda x: x.lower())
    print(f"Found {len(pdfs)} PDFs under {gcs_prefix}")
    for idx, uri in enumerate(pdfs, start=1):
        print(f"{idx:>3}. {Path(uri).name} | {uri}")
    return pdfs


def resolve_selected_pdf_uris(all_gcs_pdfs, selected_names=SELECTED_PDF_NAMES):
    by_name = {Path(u).name: u for u in all_gcs_pdfs}
    missing = [name for name in selected_names if name not in by_name]
    if missing:
        raise RuntimeError("Selected PDFs missing from source bucket: " + json.dumps(missing, indent=2))
    selected = [by_name[name] for name in selected_names]
    if EXPECTED_SELECTED_PDF_COUNT is not None and len(selected) != EXPECTED_SELECTED_PDF_COUNT:
        raise RuntimeError(f"Expected {EXPECTED_SELECTED_PDF_COUNT} PDFs, got {len(selected)}")
    return selected


def download_selected_pdfs(selected_uris):
    SOURCE_PDF_DIR.mkdir(parents=True, exist_ok=True)
    if CLEAR_STAGED_PDFS_BEFORE_DOWNLOAD:
        for p in SOURCE_PDF_DIR.glob("*.pdf"):
            p.unlink()
    local_paths = []
    for i, uri in enumerate(selected_uris, start=1):
        name = Path(uri).name
        dest = SOURCE_PDF_DIR / name
        print(f"[{i:02d}/{len(selected_uris):02d}] {name}")
        if not dest.exists():
            run_cmd(["gsutil", "cp", uri, str(dest)], allow_fail=False)
        register_source_pdf_gcs_uri(dest, uri)
        local_paths.append(dest)
    print("Downloaded/staged PDFs:", len(local_paths))
    return local_paths


def sync_dir_to_gcs(local_dir, gcs_dest):
    local_dir = Path(local_dir)
    if not local_dir.exists():
        print("Skipping missing dir:", local_dir)
        return {"local_dir": str(local_dir), "gcs_dest": gcs_dest, "status": "missing_local"}
    # Prefer gcloud storage rsync; fallback to gsutil rsync.
    r = run_cmd(["gcloud", "storage", "rsync", "--recursive", str(local_dir), gcs_dest], allow_fail=True)
    if r.returncode != 0:
        r = run_cmd(["gsutil", "-m", "rsync", "-r", str(local_dir), gcs_dest], allow_fail=False)
    return {"local_dir": str(local_dir), "gcs_dest": gcs_dest, "status": "ok"}


## 04 PDF → raw UNIFIED extraction functions

This stage intentionally creates **raw page-based `*_UNIFIED.json`** only. LEAN cleaning and tagging happen in ChatGPT.

In [ ]:
# ============================================================
# 04 PDF EXTRACTION TO RAW UNIFIED JSON
# ============================================================

FIGURE_PATTERN = re.compile(r"\b(?:fig(?:ure)?\.?\s*)\d+(?:[\.\-]\d+)?[a-z]?", flags=re.I)


def find_figure_ids_in_text(page_text: str) -> List[str]:
    seen = []
    for m in FIGURE_PATTERN.finditer(page_text or ""):
        fig_id = re.sub(r"\s+", " ", m.group(0).strip())
        fig_id = re.sub(r"^fig\b", "Fig", fig_id, flags=re.I)
        fig_id = re.sub(r"^figure\b", "Figure", fig_id, flags=re.I)
        if fig_id not in seen:
            seen.append(fig_id)
    return seen


def caption_for_figure(page_text: str, fig_id: str, max_chars: int = 700) -> str:
    if not page_text or not fig_id or fig_id == "Unidentified":
        return ""
    text = clean_text(page_text)
    idx = text.lower().find(fig_id.lower())
    if idx < 0:
        return ""
    cap = text[idx: idx + max_chars]
    next_match = FIGURE_PATTERN.search(cap, pos=max(5, len(fig_id)))
    if next_match:
        cap = cap[:next_match.start()]
    return cap.strip()


def render_page_assets(page, source_id, page_number, dpi=PAGE_RENDER_DPI):
    if not RENDER_PAGE_IMAGES:
        return None
    src_page_dir = STAGED_PAGE_IMAGE_DIR / source_id
    src_thumb_dir = STAGED_THUMB_DIR / source_id
    src_page_dir.mkdir(parents=True, exist_ok=True)
    src_thumb_dir.mkdir(parents=True, exist_ok=True)
    matrix = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=matrix, alpha=False)
    page_image_path = src_page_dir / f"{source_id}_p{page_number:04d}.png"
    pix.save(str(page_image_path))
    thumb_path = src_thumb_dir / f"{source_id}_p{page_number:04d}.webp"
    try:
        img = Image.open(page_image_path)
        w, h = img.size
        if w > THUMB_MAX_WIDTH:
            new_h = int(h * THUMB_MAX_WIDTH / w)
            img = img.resize((THUMB_MAX_WIDTH, new_h))
        img.save(thumb_path, "WEBP", quality=70)
    except Exception:
        thumb_path = None
    return {
        "page_image_path": local_to_gcs(page_image_path),
        "page_thumbnail_path": local_to_gcs(thumb_path) if thumb_path else None,
        "local_page_image_path": str(page_image_path),
        "local_page_thumbnail_path": str(thumb_path) if thumb_path else None,
        "render_status": "ok",
    }


def ingest_pdf_to_unified(pdf_path, overwrite=False, show_page_progress=False):
    pdf_path = Path(pdf_path)
    source_id = infer_source_id(pdf_path.name)
    source_title = pdf_path.stem.replace("_", " ")
    source_pdf_gcs_uri = get_source_pdf_gcs_uri(pdf_path)
    pdf_sha = sha256_file(pdf_path)

    src_fig_dir = STAGED_FIGURE_IMAGE_DIR / source_id
    src_fig_dir.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(str(pdf_path))
    unified_pages = []
    all_figures = []

    page_iter = tqdm(range(len(doc)), desc=f"UNIFIED {source_id}", leave=False) if show_page_progress else range(len(doc))
    for page_idx in page_iter:
        page_number = page_idx + 1
        page = doc[page_idx]
        try:
            page_text = page.get_text("text", sort=True) or ""
        except TypeError:
            page_text = page.get_text("text") or ""
        page_text = clean_text(page_text)
        figure_ids_in_text = find_figure_ids_in_text(page_text)

        page_image_record = render_page_assets(page, source_id, page_number)

        page_figures = []
        if EXTRACT_EMBEDDED_FIGURES:
            seen_xrefs = set()
            for img_idx, img_info in enumerate(page.get_images(full=True), start=1):
                xref = img_info[0]
                if xref in seen_xrefs:
                    continue
                seen_xrefs.add(xref)
                try:
                    extracted = doc.extract_image(xref)
                except Exception:
                    continue
                image_bytes = extracted.get("image")
                ext = extracted.get("ext", "png")
                if not image_bytes:
                    continue
                fig_id = figure_ids_in_text[img_idx - 1] if img_idx - 1 < len(figure_ids_in_text) else "Unidentified"
                fig_path = src_fig_dir / f"{source_id}_p{page_number:04d}_fig{img_idx:02d}_{slugify(fig_id)}.{ext}"
                if overwrite or not fig_path.exists():
                    with open(fig_path, "wb") as f:
                        f.write(image_bytes)
                legend = caption_for_figure(page_text, fig_id)
                fig_rec = {
                    "schema_version": "textbook_raw_figure.v1",
                    "figure_record_id": f"rawfig:{source_id}:p{page_number:04d}:{img_idx:03d}",
                    "source_id": source_id,
                    "source_title": source_title,
                    "source_document": pdf_path.name,
                    "source_pdf_gcs_uri": source_pdf_gcs_uri,
                    "page": page_number,
                    "source_page": page_number,
                    "figure_index": img_idx,
                    "figure_id": fig_id,
                    "legend": legend,
                    "caption": legend or None,
                    "path": local_to_gcs(fig_path),
                    "image_path": local_to_gcs(fig_path),
                    "local_image_path": str(fig_path),
                    "xref": int(xref),
                    "width_px": extracted.get("width"),
                    "height_px": extracted.get("height"),
                    "extension": ext,
                    "checksum_sha256": sha256_file(fig_path),
                    "method": "pymupdf_extract_image",
                }
                page_figures.append(fig_rec)
                all_figures.append(fig_rec)

        unified_pages.append({
            "schema_version": "textbook_unified_page.raw.v1",
            "source_id": source_id,
            "source_title": source_title,
            "source_document": pdf_path.name,
            "source_pdf_gcs_uri": source_pdf_gcs_uri,
            "page": page_number,
            "content": page_text,
            "figures": page_figures,
            "page_image": page_image_record,
            "provenance": {
                "stage": "colab_pdf_to_unified_raw_only",
                "extractor": "pymupdf",
                "created_at_utc": now_utc(),
                "raw_pdf_sha256": pdf_sha,
            },
        })

    unified_path = STAGED_UNIFIED_DIR / f"{source_id}_UNIFIED.json"
    figures_path = STAGED_RAW_FIGURE_JSONL_DIR / f"{source_id}_raw_figures.jsonl"
    audit_path = PDF_INGEST_AUDIT_DIR / f"{source_id}_pdf_to_unified_audit.json"

    save_json(unified_pages, unified_path)
    save_jsonl(all_figures, figures_path)

    audit = {
        "schema_version": "textbook_pdf_to_unified_audit.v1",
        "source_id": source_id,
        "source_title": source_title,
        "source_document": pdf_path.name,
        "source_pdf_gcs_uri": source_pdf_gcs_uri,
        "created_at_utc": now_utc(),
        "stage": "pdf_to_unified_only",
        "record_counts": {
            "pages": len(unified_pages),
            "pages_with_text": sum(1 for p in unified_pages if p.get("content")),
            "figures_extracted": len(all_figures),
            "pages_with_figures": sum(1 for p in unified_pages if p.get("figures")),
        },
        "local_paths": {
            "unified_json": str(unified_path),
            "raw_figures_jsonl": str(figures_path),
            "figure_images_dir": str(src_fig_dir),
        },
        "gcs_stage_paths": {
            "unified_json": local_to_gcs(unified_path),
            "raw_figures_jsonl": local_to_gcs(figures_path),
            "figure_images_dir": f"gs://{DEST_BUCKET}/01_staged/textbooks/assets/figure_images/{source_id}/",
        },
        "not_canonical_normalized": True,
        "known_limitations": [
            "Raw UNIFIED output is staged only and must be cleaned in ChatGPT before becoming canonical normalized textbook data.",
            "Caption extraction at this stage is heuristic; final caption cleanup happens in ChatGPT.",
            "No vector index and no API exposure are created by this stage."
        ]
    }
    save_json(audit, audit_path)

    doc.close()
    return {
        "status": "ok",
        "source_id": source_id,
        "source_title": source_title,
        "pdf": str(pdf_path),
        "source_pdf_gcs_uri": source_pdf_gcs_uri,
        "unified_path": str(unified_path),
        "figures_path": str(figures_path),
        "audit_path": str(audit_path),
        "pages": len(unified_pages),
        "figures": len(all_figures),
    }


def process_pdf_to_unified_safe(pdf_path):
    try:
        return ingest_pdf_to_unified(pdf_path, overwrite=True, show_page_progress=False)
    except Exception as e:
        return {
            "status": "error",
            "pdf": str(pdf_path),
            "error": repr(e),
            "traceback": traceback.format_exc()[-4000:],
        }


def run_parallel_pdf_to_unified(pdf_paths, max_workers=MAX_WORKERS):
    pdf_paths = [Path(p) for p in pdf_paths]
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(process_pdf_to_unified_safe, p): p for p in pdf_paths}
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"PDF → UNIFIED x{max_workers}"):
            res = fut.result()
            results.append(res)
            print(res.get("status"), Path(res.get("pdf", "")).name, "pages=", res.get("pages"), "figures=", res.get("figures"), "error=", res.get("error"))
    results = sorted(results, key=lambda r: r.get("pdf", ""))
    batch_audit = {
        "schema_version": "textbook_pdf_to_unified_batch_audit.v1",
        "created_at_utc": now_utc(),
        "workstream": "Textbook + Tag RAG / Source Normalization",
        "stage": "colab_pdf_to_unified_raw_only",
        "source_gcs_pdf_uri": SOURCE_GCS_PDF_URI,
        "selected_pdf_count": len(pdf_paths),
        "max_workers": max_workers,
        "result_counts": Counter(r.get("status") for r in results),
        "results": results,
        "outputs_are_staged_only": True,
        "vectorized": False,
        "api_exposed": False,
    }
    audit_path = PDF_INGEST_AUDIT_DIR / "textbook_pdf_to_unified_batch_audit.json"
    save_json(batch_audit, audit_path)
    return results, audit_path


# STAGE 1 — Run PDF → raw UNIFIED batch

This downloads the selected PDFs from the legacy bucket and processes **up to 5 PDFs simultaneously**. It produces staged raw `*_UNIFIED.json` files, not LEAN outputs.

In [ ]:
# ============================================================
# STAGE 1 RUN CELL — GCS PDFs → RAW *_UNIFIED.json ONLY
# ============================================================

authenticate_gcp_colab()

if CLEAR_STAGE1_UNIFIED_OUTPUTS_BEFORE_RUN:
    for d in [STAGED_UNIFIED_DIR, STAGED_RAW_FIGURE_JSONL_DIR, STAGED_PAGE_IMAGE_DIR, STAGED_THUMB_DIR, STAGED_FIGURE_IMAGE_DIR, PDF_INGEST_AUDIT_DIR]:
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)

all_gcs_pdfs = list_source_pdfs_from_gcs(SOURCE_GCS_PDF_URI)
selected_gcs_uris = resolve_selected_pdf_uris(all_gcs_pdfs, SELECTED_PDF_NAMES)

print("\nSelected PDFs for this run:", len(selected_gcs_uris))
for u in selected_gcs_uris:
    print(" -", Path(u).name, "| source:", u)

local_pdf_paths = download_selected_pdfs(selected_gcs_uris)
assert len(local_pdf_paths) == EXPECTED_SELECTED_PDF_COUNT

STAGE1_RESULTS, STAGE1_BATCH_AUDIT_PATH = run_parallel_pdf_to_unified(local_pdf_paths, max_workers=MAX_WORKERS)

ok = [r for r in STAGE1_RESULTS if r.get("status") == "ok"]
failed = [r for r in STAGE1_RESULTS if r.get("status") != "ok"]
print("\nSTAGE 1 COMPLETE")
print("OK:", len(ok), "FAILED:", len(failed))
print("Batch audit:", STAGE1_BATCH_AUDIT_PATH)

if failed:
    print("\nFailures:")
    print(json.dumps(failed, indent=2)[:8000])

if SYNC_STAGE1_TO_GCS_AFTER_BATCH:
    print("\nUploading/staging raw UNIFIED + assets to canonical GCS staging paths...")
    sync_status = []
    sync_status.append(sync_dir_to_gcs(STAGED_UNIFIED_DIR, f"gs://{DEST_BUCKET}/01_staged/textbooks/unified_raw/"))
    sync_status.append(sync_dir_to_gcs(STAGED_RAW_FIGURE_JSONL_DIR, f"gs://{DEST_BUCKET}/01_staged/textbooks/raw_figure_jsonl/"))
    sync_status.append(sync_dir_to_gcs(STAGED_ASSET_DIR, f"gs://{DEST_BUCKET}/01_staged/textbooks/assets/"))
    sync_status.append(sync_dir_to_gcs(PDF_INGEST_AUDIT_DIR, f"gs://{DEST_BUCKET}/06_audits/textbooks/pdf_ingest/"))
    save_json(sync_status, PDF_INGEST_AUDIT_DIR / "stage1_gcs_sync_status.json")
    print("Stage 1 sync status:")
    print(json.dumps(sync_status, indent=2))

print("\nNEXT: run the ChatGPT intermission job-packaging cell below.")


In [ ]:
# ============================================================
# PATCH EXISTING CHATGPT JOB ZIPS TO INCLUDE ACTUAL TAG FILES
#
# Adds:
#   Tags/*.txt
#   TAG_URLS.txt
#
# to every:
#   *_CHATGPT_LEAN_TAGGING_JOB.zip
#
# in:
#   /content/drive/MyDrive/4-Archives/Pathology_Hub_Intermission/pending_chatgpt_jobs
# ============================================================

from pathlib import Path
import zipfile, urllib.request, json, datetime, os

DRIVE_ROOT = Path("/content/drive/MyDrive/4-Archives/Pathology_Hub_Intermission")
DRIVE_PENDING = DRIVE_ROOT / "pending_chatgpt_jobs"
DRIVE_MANIFESTS = DRIVE_ROOT / "manifests"
LOCAL_TAG_DIR = Path("pathology_hub/00_runtime/controlled_tags")

DRIVE_MANIFESTS.mkdir(parents=True, exist_ok=True)
LOCAL_TAG_DIR.mkdir(parents=True, exist_ok=True)

TAG_URLS = [
    "https://storage.googleapis.com/pathology-hub-0/Tags/BST_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Breast_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Adrenal_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Bone_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Breast_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Derm_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Fluids_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_GI_Tract_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_GYN_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Head_Neck_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Heme_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Kidney_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Liver_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Management_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Mediastinum_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Neuro_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Pancreatobiliary_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Retroperitoneum_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Salivary_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Soft_Tissue_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Thoracic_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Thyroid_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Urinary_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Endo_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Eye_Orbit_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/GI_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/GU_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/GYN_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/HN_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Heme_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Molecular_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Neuro_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Peds_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Skin_Tags.txt",
    "https://storage.googleapis.com/pathology-hub-0/Tags/Thorax_Mediastinum_Tags.txt",
]

# Download tag files locally
downloaded = []
failed = []

for url in TAG_URLS:
    fname = url.split("/")[-1]
    dest = LOCAL_TAG_DIR / fname
    try:
        if not dest.exists() or dest.stat().st_size == 0:
            print("Downloading:", fname)
            urllib.request.urlretrieve(url, dest)
        downloaded.append(dest)
    except Exception as e:
        failed.append({"url": url, "error": repr(e)})
        print("FAILED:", url, repr(e))

assert not failed, "Some tag files failed to download:\n" + json.dumps(failed, indent=2)

tag_urls_txt = "\n".join(TAG_URLS) + "\n"

job_zips = sorted(DRIVE_PENDING.glob("*_CHATGPT_LEAN_TAGGING_JOB.zip"))
print("\nPending job ZIPs found:", len(job_zips))

patched = []

for zp in job_zips:
    print("Patching:", zp.name)

    # Append/overwrite behavior: zipfile can create duplicate names if same arcname exists.
    # So rebuild cleanly to avoid duplicate old entries.
    temp_zip = zp.with_suffix(".tmp.zip")

    with zipfile.ZipFile(zp, "r") as zin, zipfile.ZipFile(temp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zout:
        existing_names = set()

        for item in zin.infolist():
            # Drop old Tags/ and TAG_URLS if present, then rewrite fresh below.
            if item.filename.startswith("Tags/") or item.filename == "TAG_URLS.txt":
                continue
            data = zin.read(item.filename)
            zout.writestr(item, data)
            existing_names.add(item.filename)

        zout.writestr("TAG_URLS.txt", tag_urls_txt)

        for tag_file in downloaded:
            zout.write(tag_file, arcname=f"Tags/{tag_file.name}")

    temp_zip.replace(zp)
    patched.append(str(zp))

manifest = {
    "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "pending_folder": str(DRIVE_PENDING),
    "patched_job_count": len(patched),
    "tag_file_count": len(downloaded),
    "tag_files": [p.name for p in downloaded],
    "patched_jobs": patched,
    "failed_downloads": failed,
}

manifest_path = DRIVE_MANIFESTS / "patched_chatgpt_jobs_with_tag_files_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\n✅ Patched job ZIPs with actual tag files.")
print("Patched jobs:", len(patched))
print("Tag files included per ZIP:", len(downloaded))
print("Manifest:", manifest_path)

# Quick verify one ZIP
if job_zips:
    with zipfile.ZipFile(job_zips[0], "r") as z:
        names = z.namelist()
    print("\nVerify first ZIP:", job_zips[0].name)
    print("Has CHATGPT_PROMPT.txt:", "CHATGPT_PROMPT.txt" in names)
    print("Has README_FOR_CHATGPT.txt:", "README_FOR_CHATGPT.txt" in names)
    print("Tag files in ZIP:", len([n for n in names if n.startswith("Tags/") and n.endswith(".txt")]))

In [ ]:
# ============================================================
# CREATE MISSING HN/NEURO CHATGPT JOB ZIPS ONLY
# Uses /content/gdrive mount.
#
# Creates:
#   hn_atlas_CHATGPT_LEAN_TAGGING_JOB.zip
#   hn_biopsy_interpretation_CHATGPT_LEAN_TAGGING_JOB.zip
#   hn_cardesa_CHATGPT_LEAN_TAGGING_JOB.zip
#   hn_faq_CHATGPT_LEAN_TAGGING_JOB.zip
#   hn_gnepp_CHATGPT_LEAN_TAGGING_JOB.zip
#   hn_oral_CHATGPT_LEAN_TAGGING_JOB.zip
#   hn_thompson_CHATGPT_LEAN_TAGGING_JOB.zip
#   neuro_kleinschmidt_CHATGPT_LEAN_TAGGING_JOB.zip
#
# Does NOT rerun PDF extraction.
# ============================================================

from pathlib import Path
from google.colab import drive
import zipfile, json, datetime, subprocess, shutil

# ----------------------------
# Mount/check Drive at /content/gdrive
# ----------------------------

MOUNTPOINT = Path("/content/gdrive")

if not (MOUNTPOINT / "MyDrive").exists():
    if MOUNTPOINT.exists() and any(MOUNTPOINT.iterdir()):
        backup = Path(f"/content/gdrive_mountpoint_junk_{datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')}")
        print("Moving non-mounted /content/gdrive contents to:", backup)
        shutil.move(str(MOUNTPOINT), str(backup))
    MOUNTPOINT.mkdir(parents=True, exist_ok=True)
    drive.mount(str(MOUNTPOINT), force_remount=True)

assert (MOUNTPOINT / "MyDrive").exists(), "Drive not mounted at /content/gdrive/MyDrive"

DRIVE_ROOT = MOUNTPOINT / "MyDrive/4-Archives/Pathology_Hub_Intermission"
DRIVE_PENDING = DRIVE_ROOT / "pending_chatgpt_jobs"
DRIVE_COMPLETED = DRIVE_ROOT / "completed_lean_zips"
DRIVE_MANIFESTS = DRIVE_ROOT / "manifests"
DRIVE_PROMPTS = DRIVE_ROOT / "prompts"

for d in [DRIVE_PENDING, DRIVE_COMPLETED, DRIVE_MANIFESTS, DRIVE_PROMPTS]:
    d.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Target source IDs already extracted
# ----------------------------

TARGET_SOURCE_IDS = [
    "hn_atlas",
    "hn_biopsy_interpretation",
    "hn_cardesa",
    "hn_faq",
    "hn_gnepp",
    "hn_oral",
    "hn_thompson",
    "neuro_kleinschmidt",
]

ROOT = Path("pathology_hub")
STAGED_UNIFIED_DIR = ROOT / "01_staged" / "textbooks" / "unified_raw"
STAGED_UNIFIED_DIR.mkdir(parents=True, exist_ok=True)

DEST_BUCKET = "pathology_hub"

def run_cmd(cmd, allow_fail=False):
    cmd = [str(x) for x in cmd]
    print("RUN:", " ".join(cmd))
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if p.stdout:
        print(p.stdout[-3000:])
    if p.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return p

# ----------------------------
# Restore missing UNIFIEDs from GCS staging if local missing
# ----------------------------

unified_files = []
missing = []

for sid in TARGET_SOURCE_IDS:
    local_p = STAGED_UNIFIED_DIR / f"{sid}_UNIFIED.json"
    gcs_p = f"gs://{DEST_BUCKET}/01_staged/textbooks/unified_raw/{sid}_UNIFIED.json"

    if not local_p.exists():
        print("Local missing, restoring from GCS:", sid)
        p = run_cmd(["gsutil", "cp", gcs_p, str(local_p)], allow_fail=True)
        if p.returncode != 0:
            missing.append({"source_id": sid, "expected_local": str(local_p), "gcs": gcs_p})
            continue

    unified_files.append(local_p)

assert not missing, "Missing expected UNIFIED files:\n" + json.dumps(missing, indent=2)

print("\nUNIFIED files for HN/Neuro jobs:", len(unified_files))
for p in unified_files:
    print(" -", p)

# ----------------------------
# Get prompt text
# Prefer global, then Drive prompt file, then existing job ZIP
# ----------------------------

prompt_text = None

if "CHATGPT_INTERMISSION_PROMPT" in globals():
    prompt_text = CHATGPT_INTERMISSION_PROMPT
    print("Using global CHATGPT_INTERMISSION_PROMPT.")
else:
    prompt_file = DRIVE_PROMPTS / "CHATGPT_INTERMISSION_PROMPT_vFINAL.txt"
    if prompt_file.exists():
        prompt_text = prompt_file.read_text(encoding="utf-8")
        print("Using Drive prompt file:", prompt_file)

if not prompt_text:
    existing_jobs = sorted(DRIVE_PENDING.glob("*_CHATGPT_LEAN_TAGGING_JOB.zip"))
    assert existing_jobs, "No existing jobs found to copy CHATGPT_PROMPT.txt from."
    with zipfile.ZipFile(existing_jobs[0], "r") as z:
        prompt_text = z.read("CHATGPT_PROMPT.txt").decode("utf-8")
    print("Copied prompt from existing job:", existing_jobs[0].name)

assert prompt_text and len(prompt_text.strip()) > 1000, "Prompt text missing or too short."

# ----------------------------
# Copy tag payload from any already-patched job ZIP
# If none has tags yet, new jobs still get prompt + TAG_URLS if present.
# ----------------------------

tag_payload = {}
tag_urls_text = ""

for job in sorted(DRIVE_PENDING.glob("*_CHATGPT_LEAN_TAGGING_JOB.zip")):
    try:
        with zipfile.ZipFile(job, "r") as z:
            names = z.namelist()
            tag_names = [n for n in names if n.startswith("Tags/") and n.endswith(".txt")]
            if tag_names:
                print("Copying tag files from already-patched job:", job.name)
                for n in tag_names:
                    tag_payload[n] = z.read(n)
                if "TAG_URLS.txt" in names:
                    tag_urls_text = z.read("TAG_URLS.txt").decode("utf-8")
                break
    except Exception:
        pass

print("Tag files copied for new HN/Neuro jobs:", len(tag_payload))

# If no actual tag files copied, include a short warning file.
if not tag_payload:
    print("WARNING: No patched job with Tags/*.txt found. New jobs will contain prompt only, not bundled tag txt files.")
    tag_urls_text = "Tag files were not bundled because no patched source job was available. Use TAG_URLS in CHATGPT_PROMPT.txt.\n"

# ----------------------------
# Create HN/Neuro job ZIPs
# ----------------------------

created = []
overwritten = []
skipped_completed = []

for p in unified_files:
    source_id = p.name.replace("_UNIFIED.json", "")
    job_zip = DRIVE_PENDING / f"{source_id}_CHATGPT_LEAN_TAGGING_JOB.zip"
    completed_zip = DRIVE_COMPLETED / f"{source_id}_LEAN_PACKAGE.zip"

    if completed_zip.exists():
        print("Skipping; completed LEAN package already exists:", completed_zip)
        skipped_completed.append(str(completed_zip))
        continue

    if job_zip.exists():
        overwritten.append(str(job_zip))
        print("Overwriting existing pending job:", job_zip.name)

    with zipfile.ZipFile(job_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(p, arcname=p.name)
        z.writestr("CHATGPT_PROMPT.txt", prompt_text)

        if tag_urls_text:
            z.writestr("TAG_URLS.txt", tag_urls_text)

        for arcname, data in tag_payload.items():
            z.writestr(arcname, data)

        z.writestr(
            "README_FOR_CHATGPT.txt",
            f"""Pathology Hub ChatGPT intermission job.

Input file:
{p.name}

Task:
Run CHATGPT_PROMPT.txt on this *_UNIFIED.json.

After creating the completed LEAN package ZIP, save/upload it to:

/MyDrive/4-Archives/Pathology_Hub_Intermission/completed_lean_zips/

Required output filename:

{source_id}_LEAN_PACKAGE.zip

If direct Google Drive upload is not available, return the ZIP as a downloadable file and state clearly that manual placement into completed_lean_zips is required.

Return ZIP containing:
- {source_id}_UNIFIED_LEAN.json
- {source_id}_pages_LEAN.jsonl
- {source_id}_chunks_LEAN.jsonl
- {source_id}_figures_LEAN.jsonl
- {source_id}_fts_LEAN.sqlite
- {source_id}_LEAN_cleaning_audit.json
- {source_id}_tag_catalog.jsonl, if tag files are used
- {source_id}_candidate_tagging_audit.json, if tag files are used
- {source_id}_semantic_tagging_audit.json, only if actual semantic LLM/API tagging is performed

Do not claim vectorization.
Do not claim API exposure.
Do not claim image interpretation.
"""
        )

    created.append(str(job_zip))
    print("Created:", job_zip)

# ----------------------------
# Verify + manifest
# ----------------------------

verify_rows = []

for zp in [Path(x) for x in created]:
    with zipfile.ZipFile(zp, "r") as z:
        names = z.namelist()
    verify_rows.append({
        "zip": zp.name,
        "has_prompt": "CHATGPT_PROMPT.txt" in names,
        "has_readme": "README_FOR_CHATGPT.txt" in names,
        "has_tag_urls": "TAG_URLS.txt" in names,
        "tag_txt_count": len([n for n in names if n.startswith("Tags/") and n.endswith(".txt")]),
        "has_unified_json": any(n.endswith("_UNIFIED.json") for n in names),
    })

manifest = {
    "schema_version": "pathology_hub_chatgpt_intermission_drive_manifest.v1",
    "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "stage": "HN_NEURO_MISSING_JOB_CREATION",
    "target_source_ids": TARGET_SOURCE_IDS,
    "pending_folder": str(DRIVE_PENDING),
    "completed_folder": str(DRIVE_COMPLETED),
    "created_count": len(created),
    "created_jobs": created,
    "overwritten": overwritten,
    "skipped_completed": skipped_completed,
    "tag_files_included_per_zip": len(tag_payload),
    "verify": verify_rows,
}

manifest_path = DRIVE_MANIFESTS / "hn_neuro_missing_jobs_created_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\n✅ HN/Neuro pending jobs created.")
print("Created:", len(created))
print("Overwritten:", len(overwritten))
print("Skipped completed:", len(skipped_completed))
print("Tag files included per new ZIP:", len(tag_payload))
print("Pending folder:", DRIVE_PENDING)
print("Manifest:", manifest_path)

print("\nVerification:")
print(json.dumps(verify_rows, indent=2))

# Final count
all_pending = sorted(DRIVE_PENDING.glob("*_CHATGPT_LEAN_TAGGING_JOB.zip"))
print("\nTotal pending jobs now:", len(all_pending))
for p in all_pending:
    if p.name.startswith(("hn_", "neuro_kleinschmidt")):
        print("HN/Neuro job:", p.name)

In [ ]:
# ============================================================
# FIX v04.5 JOURNAL HYBRID BUG
#
# Error:
#   TypeError("unhashable type: 'list'")
#
# Cause:
#   merge_nonempty() used: dst.get(k) in {None, "", []}
#   [] cannot be inside a Python set.
#
# This patches app.py, rebuilds/deploys, and smoke-tests journals.
# ============================================================

from pathlib import Path
import subprocess, json, requests, datetime, re

PROJECT_ID = "pathology-annotation-project"
REGION = "us-central1"
SERVICE_NAME = "pathology-hub-v04"
AR_REPO = "pathology-hub"
IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/{SERVICE_NAME}:latest"

APP_DIR = Path("/content/pathology_hub_v04_textbook_api")
APP_PATH = APP_DIR / "app.py"
assert APP_PATH.exists(), f"Missing {APP_PATH}"

OPENAI_SECRET_NAME = "OPEN_AI_KEY_01"

TEXTBOOK_SQLITE_GCS = "gs://pathology_hub/03_indexes/textbooks/lean/textbook_lean_fts.sqlite"
TEXTBOOK_MANIFEST_GCS = "gs://pathology_hub/03_indexes/textbooks/lean/textbook_lean_index_manifest.json"
TEXTBOOK_FAISS_GCS = "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_faiss.index"
TEXTBOOK_DOCSTORE_GCS = "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_docstore.jsonl"
TEXTBOOK_VECTOR_MANIFEST_GCS = "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_manifest.json"
TEXTBOOK_FIGURES_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_lean_figures.jsonl"
TEXTBOOK_WEB_MAP_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_figure_web_map_v1_FILTERED_NO_MCKEE_DORFMAN.jsonl"

JOURNAL_FAISS_GCS = "gs://pathology_hub/03_indexes/journals/vector/journal_faiss.index"
JOURNAL_DOCSTORE_GCS = "gs://pathology_hub/03_indexes/journals/vector/journal_vector_docstore.jsonl"
JOURNAL_VECTOR_MANIFEST_GCS = "gs://pathology_hub/03_indexes/journals/vector/journal_vector_manifest.json"

UPSTREAM_EVIDENCE_URL = "https://pathology-hub-830130787988.us-central1.run.app/evidence/search"

def run(cmd, cwd=None, check=True):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout:
        print(p.stdout[-12000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Colab authenticated.")
except Exception as e:
    print("Auth note:", repr(e))

run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

# Backup
backup = APP_DIR / f"app_v045_pre_unhashable_fix_{datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')}.py"
backup.write_text(APP_PATH.read_text(encoding="utf-8"), encoding="utf-8")
print("Backup:", backup)

txt = APP_PATH.read_text(encoding="utf-8")

bad = 'elif k not in dst or dst.get(k) in {None, "", []}:'
good = 'elif k not in dst or dst.get(k) is None or dst.get(k) == "" or dst.get(k) == []:'

if bad in txt:
    txt = txt.replace(bad, good)
else:
    # Regex fallback if spacing changed
    txt2 = re.sub(
        r'elif\s+k\s+not\s+in\s+dst\s+or\s+dst\.get\(k\)\s+in\s+\{None,\s*"",\s*\[\]\}:',
        good,
        txt,
    )
    if txt2 == txt:
        print("WARNING: exact bad line not found. Searching nearby merge_nonempty block:")
        m = re.search(r"def merge_nonempty\(.*?def row_to_journal_result", txt, flags=re.S)
        print(m.group(0)[:2000] if m else "merge_nonempty block not found")
        raise RuntimeError("Could not patch merge_nonempty.")
    txt = txt2

APP_PATH.write_text(txt, encoding="utf-8")
print("Patched merge_nonempty bug.")

run(["python", "-m", "py_compile", str(APP_PATH)], check=True)

# Rebuild/redeploy same v04.5 image
run(["gcloud", "auth", "configure-docker", f"{REGION}-docker.pkg.dev", "--quiet"], check=True)

print("\nBuilding fixed v04.5 image...")
run(["gcloud", "builds", "submit", "--tag", IMAGE, "."], cwd=APP_DIR, check=True)

env_vars = ",".join([
    f"TEXTBOOK_SQLITE_GCS={TEXTBOOK_SQLITE_GCS}",
    f"TEXTBOOK_MANIFEST_GCS={TEXTBOOK_MANIFEST_GCS}",
    f"TEXTBOOK_FAISS_GCS={TEXTBOOK_FAISS_GCS}",
    f"TEXTBOOK_DOCSTORE_GCS={TEXTBOOK_DOCSTORE_GCS}",
    f"TEXTBOOK_VECTOR_MANIFEST_GCS={TEXTBOOK_VECTOR_MANIFEST_GCS}",
    f"TEXTBOOK_FIGURES_GCS={TEXTBOOK_FIGURES_GCS}",
    f"TEXTBOOK_WEB_MAP_GCS={TEXTBOOK_WEB_MAP_GCS}",
    f"JOURNAL_FAISS_GCS={JOURNAL_FAISS_GCS}",
    f"JOURNAL_DOCSTORE_GCS={JOURNAL_DOCSTORE_GCS}",
    f"JOURNAL_VECTOR_MANIFEST_GCS={JOURNAL_VECTOR_MANIFEST_GCS}",
    f"UPSTREAM_EVIDENCE_URL={UPSTREAM_EVIDENCE_URL}",
    "EMBEDDING_MODEL=text-embedding-3-small",
    "JOURNAL_VECTOR_POOL=25",
    "JOURNAL_FTS_POOL=10",
])

print("\nDeploying fixed v04.5...")
run([
    "gcloud", "run", "deploy", SERVICE_NAME,
    "--image", IMAGE,
    "--region", REGION,
    "--platform", "managed",
    "--allow-unauthenticated",
    "--memory", "12Gi",
    "--cpu", "4",
    "--timeout", "300",
    "--min-instances", "1",
    "--set-env-vars", env_vars,
    "--set-secrets",
    f"PATHOLOGY_HUB_API_KEY=pathology-hub-api-key:latest,OPENAI_API_KEY={OPENAI_SECRET_NAME}:latest,FIGURE_PROXY_SECRET=pathology-hub-api-key:latest",
    "--quiet",
], check=True)

service_url = subprocess.check_output([
    "gcloud", "run", "services", "describe", SERVICE_NAME,
    "--region", REGION,
    "--format", "value(status.url)"
], text=True).strip()

api_key = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", "pathology-hub-api-key"
], text=True).strip()

print("\nSERVICE URL:", service_url)

# Health
r = requests.get(f"{service_url}/health", timeout=300)
print("\nhealth:", r.status_code)
health = r.json()
print(json.dumps({
    "version": health.get("version"),
    "journal_search_mode": health.get("journal_search_mode"),
    "journal_vectorized": health.get("journal_vectorized"),
    "journal_vector_records": health.get("journal_vector_records"),
    "textbook_search_mode": health.get("textbook_search_mode"),
    "public_figure_map_records_loaded": health.get("public_figure_map_records_loaded"),
}, indent=2))
assert r.status_code == 200
assert health.get("journal_vectorized") is True
assert int(health.get("journal_vector_records") or 0) == 103830

def post_search(payload):
    rr = requests.post(
        f"{service_url}/evidence/search",
        headers={"X-API-Key": api_key, "Content-Type": "application/json"},
        json=payload,
        timeout=300,
    )
    print("\nQUERY:", payload["query"])
    print("SOURCES:", payload["sources"])
    print("status:", rr.status_code)
    data = rr.json()
    print("source_status:", data.get("source_status"))
    print("search_mode:", data.get("search_mode"))
    print("counts:", {
        "who": len(data.get("who_results", [])),
        "journals": len(data.get("journal_results", [])),
        "pathout": len(data.get("pathout_results", [])),
        "textbooks": len(data.get("textbook_results", [])),
        "figures": len(data.get("figures", [])),
    })
    print(json.dumps(data, indent=2)[:9000])
    assert rr.status_code == 200
    return data

# Journal smoke
journal_data = post_search({
    "query": "intestinal type adenocarcinoma sinonasal occupational wood dust",
    "sources": ["journals"],
    "max_results": 5,
    "include_figures": False,
    "max_figures": 0,
    "compact": True,
    "excerpt_char_limit": 1000,
})
assert journal_data["source_status"]["journals"] == "ok"
assert len(journal_data.get("journal_results", [])) >= 1
assert any(
    h.get("retrieval_mode") in {"hybrid_fts_vector", "vector_only", "fts_only"}
    for h in journal_data.get("journal_results", [])
)

# Combined smoke
combined_data = post_search({
    "query": "bladder CIS p53 CK20",
    "sources": ["textbooks", "pathout", "journals"],
    "max_results": 2,
    "include_figures": False,
    "max_figures": 0,
    "compact": True,
    "excerpt_char_limit": 800,
})
assert combined_data["source_status"]["textbooks"] == "ok"
assert combined_data["source_status"]["journals"] == "ok"

# Textbook figure smoke
fig_data = post_search({
    "query": "fallopian tube precursor lesion abnormal p53 increased proliferation",
    "sources": ["textbooks"],
    "max_results": 3,
    "include_figures": True,
    "max_figures": 3,
    "compact": True,
    "excerpt_char_limit": 900,
})
assert len(fig_data.get("figures", [])) >= 1
first_url = fig_data["figures"][0]["figure_url"]
img = requests.get(first_url, timeout=120)
print("first figure:", img.status_code, img.headers.get("content-type"), len(img.content))
assert img.status_code == 200
assert img.headers.get("content-type", "").startswith("image/")

print("\n✅ FIXED v04.5 JOURNAL HYBRID API DEPLOYED AND SMOKE TESTED")
print("Service:", service_url)

In [ ]:
# ============================================================
# FIX COLAB DRIVE MOUNTPOINT ERROR:
# ValueError: Mountpoint must not already contain files
# ============================================================

from pathlib import Path
from google.colab import drive
import shutil, datetime, os, time

MOUNTPOINT = Path("/content/drive")

# If Drive is already correctly mounted, do nothing.
if (MOUNTPOINT / "MyDrive").exists():
    print("✅ Drive already mounted correctly:", MOUNTPOINT / "MyDrive")

else:
    # Try normal cleanup first.
    try:
        drive.flush_and_unmount()
        print("Flushed existing Drive mount.")
        time.sleep(2)
    except Exception as e:
        print("flush_and_unmount skipped:", repr(e))

    # If /content/drive contains junk but is not mounted, move it aside.
    if MOUNTPOINT.exists() and any(MOUNTPOINT.iterdir()):
        ts = datetime.datetime.now(datetime.UTC).strftime("%Y%m%dT%H%M%SZ")
        backup = Path(f"/content/drive_mountpoint_junk_{ts}")
        print("Moving non-mounted /content/drive contents to:", backup)
        shutil.move(str(MOUNTPOINT), str(backup))

    # Recreate empty mountpoint.
    MOUNTPOINT.mkdir(parents=True, exist_ok=True)

    # Mount fresh.
    drive.mount(str(MOUNTPOINT), force_remount=True)

assert (MOUNTPOINT / "MyDrive").exists(), "Drive mount failed: /content/drive/MyDrive not found."

print("✅ Drive mounted successfully.")
print("MyDrive:", MOUNTPOINT / "MyDrive")

# Quick check pending folder
DRIVE_PENDING = Path("/content/drive/MyDrive/4-Archives/Pathology_Hub_Intermission/pending_chatgpt_jobs")
print("Pending folder exists:", DRIVE_PENDING.exists())
if DRIVE_PENDING.exists():
    jobs = sorted(DRIVE_PENDING.glob("*_CHATGPT_LEAN_TAGGING_JOB.zip"))
    print("Pending job ZIPs:", len(jobs))
    for j in jobs[:50]:
        print(" -", j.name)

In [ ]:
# ============================================================
# CHECKPOINT RAW PDF→UNIFIED PROGRESS TO GCS
# Run this immediately after PDF→UNIFIED extraction finishes.
#
# Saves:
#   - raw *_UNIFIED.json
#   - extracted figure/image assets
#   - audits/manifests
#
# Does NOT upload source PDFs again.
# Does NOT claim LEAN, vectorization, or API exposure.
# ============================================================

from pathlib import Path
import json, subprocess, datetime, os, re, hashlib, shutil

PROJECT_ID = "pathology-annotation-project"
CANONICAL_GCS_ROOT = "gs://pathology_hub"

RUN_LABEL = "selected39_pdf_to_unified"
TS = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

# Local project root used by the notebook.
ROOT_CANDIDATES = [
    Path("pathology_hub"),
    Path("/content/pathology_hub"),
]
ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), None)
assert ROOT is not None, "Could not find local pathology_hub folder."

STAGED_TEXTBOOKS_DIR = ROOT / "01_staged" / "textbooks"
AUDIT_DIR = ROOT / "06_audits" / "textbooks"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

GCS_STAGED_TEXTBOOKS = f"{CANONICAL_GCS_ROOT}/01_staged/textbooks"
GCS_AUDIT_CHECKPOINTS = f"{CANONICAL_GCS_ROOT}/06_audits/textbooks/checkpoints"

print("ROOT:", ROOT)
print("STAGED_TEXTBOOKS_DIR:", STAGED_TEXTBOOKS_DIR)
print("AUDIT_DIR:", AUDIT_DIR)
print("GCS_STAGED_TEXTBOOKS:", GCS_STAGED_TEXTBOOKS)
print("GCS_AUDIT_CHECKPOINTS:", GCS_AUDIT_CHECKPOINTS)

def run(cmd, check=True):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout:
        print(p.stdout[-5000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {' '.join(map(str, cmd))}")
    return p

# Make sure GCP project is set.
run(["gcloud", "config", "set", "project", PROJECT_ID], check=False)

# ------------------------------------------------------------
# Detect expected selected PDFs from notebook globals if present,
# otherwise infer from staged selected_pdfs folder.
# ------------------------------------------------------------

expected_pdf_names = []

for varname in ["SELECTED_PDF_URIS", "SELECTED_PDFS", "selected_pdf_uris", "selected_pdfs"]:
    if varname in globals():
        try:
            vals = globals()[varname]
            expected_pdf_names = [str(x).split("/")[-1] for x in vals if str(x).lower().endswith(".pdf")]
            if expected_pdf_names:
                print(f"Detected expected PDFs from variable {varname}: {len(expected_pdf_names)}")
                break
        except Exception:
            pass

if not expected_pdf_names:
    local_pdf_dir = ROOT / "00_runtime" / "selected_pdfs"
    if local_pdf_dir.exists():
        expected_pdf_names = [p.name for p in sorted(local_pdf_dir.glob("*.pdf"))]
        print(f"Detected expected PDFs from local selected_pdfs folder: {len(expected_pdf_names)}")

expected_stems = sorted({Path(x).stem for x in expected_pdf_names})

# ------------------------------------------------------------
# Detect raw UNIFIED JSONs.
# Exclude LEAN outputs and ChatGPT-returned artifacts.
# ------------------------------------------------------------

unified_files = []
for p in ROOT.rglob("*_UNIFIED.json"):
    sp = str(p)
    name = p.name
    if "_UNIFIED_LEAN" in name:
        continue
    if "LEAN" in name:
        continue
    if ".ipynb_checkpoints" in sp:
        continue
    unified_files.append(p)

unified_files = sorted(set(unified_files))
print("\nDetected raw *_UNIFIED.json files:", len(unified_files))
for p in unified_files[:10]:
    print(" -", p)
if len(unified_files) > 10:
    print(" ...")

# ------------------------------------------------------------
# Quick parse/count audit.
# ------------------------------------------------------------

records = []
bad_json = []

for p in unified_files:
    rel = str(p.relative_to(ROOT)) if str(p).startswith(str(ROOT)) else str(p)
    rec = {
        "path": str(p),
        "relative_path": rel,
        "filename": p.name,
        "source_stem": p.name.replace("_UNIFIED.json", ""),
        "json_parse_ok": False,
        "pages": None,
        "figures": None,
        "size_bytes": p.stat().st_size if p.exists() else None,
    }
    try:
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        rec["json_parse_ok"] = True
        if isinstance(data, list):
            rec["pages"] = len(data)
            fig_count = 0
            for page in data:
                if isinstance(page, dict):
                    figs = page.get("figures") or []
                    if isinstance(figs, list):
                        fig_count += len(figs)
            rec["figures"] = fig_count
        else:
            rec["pages"] = None
            rec["figures"] = None
    except Exception as e:
        rec["error"] = repr(e)
        bad_json.append(str(p))
    records.append(rec)

# Try to estimate missing expected books by stem matching.
unified_stems_lower = {r["source_stem"].lower() for r in records}
missing_expected = []
for stem in expected_stems:
    s = stem.lower()
    # tolerate minor naming transforms: spaces/parentheses/underscores
    normalized_s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    found = False
    for u in unified_stems_lower:
        normalized_u = re.sub(r"[^a-z0-9]+", "_", u).strip("_")
        if normalized_s == normalized_u or normalized_s in normalized_u or normalized_u in normalized_s:
            found = True
            break
    if not found:
        missing_expected.append(stem)

# Count local extracted assets without hashing every huge file.
image_exts = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"}
asset_files = [
    p for p in STAGED_TEXTBOOKS_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in image_exts
] if STAGED_TEXTBOOKS_DIR.exists() else []

manifest = {
    "schema_version": "pathology_hub_textbook_pdf_to_unified_checkpoint.v1",
    "workstream": "Textbook + Tag RAG / Source Normalization",
    "stage": "PDF_to_UNIFIED_raw_extraction_checkpoint",
    "created_at_utc": TS,
    "run_label": RUN_LABEL,
    "project_id": PROJECT_ID,
    "local_root": str(ROOT),
    "source_pdf_gcs_root": "gs://pathology-hub-0/source_pdfs",
    "canonical_gcs_root": CANONICAL_GCS_ROOT,
    "gcs_staged_textbooks_target": GCS_STAGED_TEXTBOOKS,
    "expected_pdf_count": len(expected_pdf_names),
    "expected_pdf_names": expected_pdf_names,
    "raw_unified_json_count": len(unified_files),
    "raw_unified_records": records,
    "bad_json_files": bad_json,
    "missing_expected_stems_best_effort": missing_expected,
    "extracted_asset_file_count_best_effort": len(asset_files),
    "notes": [
        "This checkpoint saves raw *_UNIFIED.json and extracted assets only.",
        "LEAN normalization is still performed later by ChatGPT intermission.",
        "No vector index was built.",
        "No API exposure is claimed.",
        "Source PDFs are not reuploaded because they already exist in gs://pathology-hub-0/source_pdfs."
    ],
}

manifest_path = AUDIT_DIR / f"{RUN_LABEL}_checkpoint_manifest_{TS}.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

latest_manifest_path = AUDIT_DIR / f"{RUN_LABEL}_checkpoint_manifest_LATEST.json"
latest_manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\nCHECKPOINT MANIFEST WRITTEN:")
print(" -", manifest_path)
print(" -", latest_manifest_path)

print("\nSUMMARY")
print("Expected PDFs:", len(expected_pdf_names))
print("Raw UNIFIED JSON files:", len(unified_files))
print("Bad JSON files:", len(bad_json))
print("Missing expected stems best-effort:", len(missing_expected))
if missing_expected:
    print("Missing/unfinished candidates:")
    for x in missing_expected:
        print(" -", x)
print("Extracted asset files best-effort:", len(asset_files))

# ------------------------------------------------------------
# Upload/sync staged raw UNIFIED + extracted assets.
# IMPORTANT: exclude source PDFs so we don't reupload huge files.
# ------------------------------------------------------------

if STAGED_TEXTBOOKS_DIR.exists():
    print("\nUploading staged raw UNIFIED + extracted assets to canonical GCS...")
    run([
        "gsutil", "-m", "rsync", "-r",
        "-x", r".*\.pdf$",
        str(STAGED_TEXTBOOKS_DIR),
        GCS_STAGED_TEXTBOOKS
    ])
else:
    print("\nWARNING: STAGED_TEXTBOOKS_DIR not found. Uploading raw UNIFIED JSONs individually.")
    fallback_gcs = f"{GCS_STAGED_TEXTBOOKS}/unified_raw"
    for p in unified_files:
        run(["gsutil", "cp", str(p), f"{fallback_gcs}/{p.name}"])

# Upload audit/checkpoint manifests.
print("\nUploading checkpoint audit manifest...")
run(["gsutil", "cp", str(manifest_path), f"{GCS_AUDIT_CHECKPOINTS}/{manifest_path.name}"])
run(["gsutil", "cp", str(latest_manifest_path), f"{GCS_AUDIT_CHECKPOINTS}/{latest_manifest_path.name}"])

# Also sync all local textbook audit files, if present.
if AUDIT_DIR.exists():
    run([
        "gsutil", "-m", "rsync", "-r",
        str(AUDIT_DIR),
        f"{CANONICAL_GCS_ROOT}/06_audits/textbooks"
    ])

print("\n✅ RAW PDF→UNIFIED CHECKPOINT SAVED")
print("Raw/staged outputs:", GCS_STAGED_TEXTBOOKS)
print("Checkpoint manifest:", f"{GCS_AUDIT_CHECKPOINTS}/{manifest_path.name}")
print("Latest manifest:", f"{GCS_AUDIT_CHECKPOINTS}/{latest_manifest_path.name}")
print("\nNow it is safe to run the ChatGPT intermission/job ZIP cell.")

In [ ]:
# ============================================================
# EXPORT CHATGPT INTERMISSION JOBS TO GOOGLE DRIVE
# Revised Drive root:
# /content/drive/MyDrive/4-Archives/Pathology_Hub_Intermission
#
# Run after raw *_UNIFIED.json checkpoint is saved.
# ============================================================

from google.colab import drive
from pathlib import Path
import shutil, zipfile, json, datetime

drive.mount("/content/drive")

# Revised location
DRIVE_ROOT = Path("/content/drive/MyDrive/4-Archives/Pathology_Hub_Intermission")
DRIVE_PENDING = DRIVE_ROOT / "pending_chatgpt_jobs"
DRIVE_COMPLETED = DRIVE_ROOT / "completed_lean_zips"
DRIVE_MANIFESTS = DRIVE_ROOT / "manifests"

DRIVE_PENDING.mkdir(parents=True, exist_ok=True)
DRIVE_COMPLETED.mkdir(parents=True, exist_ok=True)
DRIVE_MANIFESTS.mkdir(parents=True, exist_ok=True)

ROOT = Path("pathology_hub")

unified_files = sorted(ROOT.rglob("*_UNIFIED.json"))
unified_files = [
    p for p in unified_files
    if "_UNIFIED_LEAN" not in p.name
    and "LEAN" not in p.name
    and ".ipynb_checkpoints" not in str(p)
]

print("UNIFIED files to package:", len(unified_files))
for p in unified_files[:20]:
    print(" -", p)
if len(unified_files) > 20:
    print(" ...")

# Require real prompt variable from notebook.
assert "CHATGPT_INTERMISSION_PROMPT" in globals(), (
    "Missing CHATGPT_INTERMISSION_PROMPT. Run the prompt-definition cell first."
)
assert len(CHATGPT_INTERMISSION_PROMPT.strip()) > 1000, (
    "CHATGPT_INTERMISSION_PROMPT looks too short. Make sure the full cleaner/tagging prompt is loaded."
)

created = []

for p in unified_files:
    source_id = p.name.replace("_UNIFIED.json", "")
    job_zip = DRIVE_PENDING / f"{source_id}_CHATGPT_LEAN_TAGGING_JOB.zip"

    with zipfile.ZipFile(job_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(p, arcname=p.name)
        z.writestr("CHATGPT_PROMPT.txt", CHATGPT_INTERMISSION_PROMPT)
        z.writestr(
            "README_FOR_CHATGPT.txt",
            f"""Pathology Hub ChatGPT intermission job.

Input file:
{p.name}

Task:
Run CHATGPT_PROMPT.txt on this *_UNIFIED.json.

Return ZIP containing:
- {source_id}_UNIFIED_LEAN.json
- {source_id}_pages_LEAN.jsonl
- {source_id}_chunks_LEAN.jsonl
- {source_id}_figures_LEAN.jsonl
- {source_id}_fts_LEAN.sqlite
- {source_id}_LEAN_cleaning_audit.json
- {source_id}_tag_catalog.jsonl, if tag files are used
- {source_id}_candidate_tagging_audit.json, if tag files are used
- {source_id}_semantic_tagging_audit.json, only if actual semantic LLM/API tagging is performed

Do not claim vectorization.
Do not claim API exposure.
Do not claim image interpretation.
"""
        )

    created.append(str(job_zip))
    print("Created:", job_zip)

manifest = {
    "schema_version": "pathology_hub_chatgpt_intermission_drive_manifest.v1",
    "created_at_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "workstream": "Textbook + Tag RAG / Source Normalization",
    "stage": "CHATGPT_INTERMISSION_LEAN_TAGGING_JOBS",
    "drive_root": str(DRIVE_ROOT),
    "pending_folder": str(DRIVE_PENDING),
    "completed_folder": str(DRIVE_COMPLETED),
    "job_count": len(created),
    "jobs": created,
    "notes": [
        "Google Drive is used as a handoff mailbox only.",
        "GCS remains canonical storage.",
        "Each job ZIP contains one raw *_UNIFIED.json plus the full ChatGPT cleaner/tagging prompt.",
        "Completed LEAN ZIPs should be placed in completed_lean_zips before Colab resumes validation/upload."
    ],
}

manifest_path = DRIVE_MANIFESTS / "chatgpt_intermission_drive_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\n✅ Drive handoff ready.")
print("Drive root:", DRIVE_ROOT)
print("Pending ChatGPT jobs:", DRIVE_PENDING)
print("Put completed LEAN ZIPs here:", DRIVE_COMPLETED)
print("Manifest:", manifest_path)

## Stage 1 diagnostic

In [ ]:
# ============================================================
# STAGE 1 DIAGNOSTIC
# ============================================================

checks = {
    "staged_pdfs": len(list(SOURCE_PDF_DIR.glob("*.pdf"))),
    "raw_unified_json_count": len(list(STAGED_UNIFIED_DIR.glob("*_UNIFIED.json"))),
    "raw_figure_jsonl_count": len(list(STAGED_RAW_FIGURE_JSONL_DIR.glob("*_raw_figures.jsonl"))),
    "individual_pdf_ingest_audits": len(list(PDF_INGEST_AUDIT_DIR.glob("*_pdf_to_unified_audit.json"))),
    "batch_audit_exists": (PDF_INGEST_AUDIT_DIR / "textbook_pdf_to_unified_batch_audit.json").exists(),
}
print(json.dumps(checks, indent=2))
print("\nRaw UNIFIED files:")
for p in sorted(STAGED_UNIFIED_DIR.glob("*_UNIFIED.json")):
    print(" -", p.name, "=>", local_to_gcs(p))


# STAGE 2 — CHATGPT INTERMISSION

This is the required manual step.

1. Run the next cell to create one ChatGPT job ZIP per raw `*_UNIFIED.json`.
2. Download a job ZIP or use the GCS job path.
3. In ChatGPT, upload the job ZIP and paste/use the included prompt.
4. ChatGPT returns a LEAN package ZIP.
5. Come back to this notebook and upload those returned LEAN ZIPs in Stage 3.

The prompt includes the controlled tag URL list and the uploaded v4 cleaner/tagging rules.

In [ ]:
# ============================================================
# STAGE 2 — BUILD CHATGPT INTERMISSION JOB PACKAGES
# ============================================================

TAG_URLS = [
  "https://storage.googleapis.com/pathology-hub-0/Tags/BST_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Breast_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Adrenal_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Bone_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Breast_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Derm_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Fluids_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_GI_Tract_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_GYN_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Head_Neck_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Heme_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Kidney_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Liver_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Management_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Mediastinum_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Neuro_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Pancreatobiliary_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Retroperitoneum_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Salivary_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Soft_Tissue_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Thoracic_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Thyroid_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Urinary_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Endo_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Eye_Orbit_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/GI_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/GU_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/GYN_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/HN_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Heme_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Molecular_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Neuro_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Peds_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Skin_Tags.txt",
  "https://storage.googleapis.com/pathology-hub-0/Tags/Thorax_Mediastinum_Tags.txt"
]

CLEANER_PROMPT_TEXT = "# CONTROLLED TAG URL ADDENDUM \u2014 use with this prompt\n\nThe controlled Pathology Hub tag lists for this job are available at these authoritative URLs. Use exact tag strings from these files only. Do not invent tags. If your ChatGPT environment cannot fetch these URLs, ask the user to upload the tag files or continue with tagging_status=not_attempted / candidate_tags_only_no_llm as appropriate.\n\n- https://storage.googleapis.com/pathology-hub-0/Tags/BST_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Breast_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Adrenal_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Bone_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Breast_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Derm_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Fluids_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_GI_Tract_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_GYN_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Head_Neck_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Heme_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Kidney_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Liver_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Management_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Mediastinum_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Neuro_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Pancreatobiliary_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Retroperitoneum_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Salivary_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Soft_Tissue_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Thoracic_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Thyroid_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Cyto_Urinary_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Endo_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Eye_Orbit_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/GI_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/GU_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/GYN_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/HN_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Heme_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Molecular_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Neuro_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Peds_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Skin_Tags.txt\n- https://storage.googleapis.com/pathology-hub-0/Tags/Thorax_Mediastinum_Tags.txt\n\n---\n\n# Pathology Hub Textbook UNIFIED JSON Cleaner \u2014 Instructions for Another AI Conversation\n\nYou are working inside the Pathology Hub project.\n\n## Workstream\n\nTextbook + Tag RAG / Source Normalization.\n\nThis is not a new isolated system. The output must plug into the shared Pathology Hub architecture.\n\nThe goal is to clean a page-based textbook `*_UNIFIED.json` file into a lean, whole-book, RAG-ready resource.\n\nDo not build or claim an API. Do not claim the source is indexed/vectorized/API-exposed unless you actually create and audit an index file. This task is source cleanup and normalization only.\n\n---\n\n# 1. Input\n\nThe user will upload one textbook `*_UNIFIED.json`.\n\nExpected structure is usually a list of page records like:\n\n```json\n[\n  {\n    \"page\": 1,\n    \"content\": \"page text...\",\n    \"figures\": [\n      {\n        \"figure_id\": \"Fig 1.1\",\n        \"legend\": \"\",\n        \"path\": \"gs://pathology-hub-0/_asset_library/textbooks/...jpg\",\n        \"source_page\": 1,\n        \"method\": \"error_fallback\"\n      }\n    ]\n  }\n]\n```\n\nIf the uploaded file is not page-based and instead is WHO/entity-level JSON, stop and say it needs the WHO/entity cleaner, not the textbook page cleaner.\n\n---\n\n# 2. Main output philosophy\n\nKeep it simple.\n\nThe canonical cleaned artifact should be one whole-book JSON file:\n\n```text\n<SOURCE_ID>_UNIFIED_LEAN.json\n```\n\nDo not make chapter-specific JSON files unless explicitly requested.\n\nChapter information should be stored as metadata fields inside each page/chunk, not as separate canonical files.\n\nGlobal JSONL files are allowed and useful for indexing/API later:\n\n```text\n<SOURCE_ID>_pages_LEAN.jsonl\n<SOURCE_ID>_chunks_LEAN.jsonl\n<SOURCE_ID>_figures_LEAN.jsonl\n<SOURCE_ID>_fts_LEAN.sqlite\n<SOURCE_ID>_LEAN_cleaning_audit.json\n```\n\nDo not produce excessive documentation, migration manifests, or verbose provenance unless the user asks.\n\n---\n\n# 3. Required final ZIP contents\n\nReturn one ZIP containing the lean normalization outputs.\n\nCore required files:\n\n```text\n<SOURCE_ID>_UNIFIED_LEAN.json\n<SOURCE_ID>_pages_LEAN.jsonl\n<SOURCE_ID>_chunks_LEAN.jsonl\n<SOURCE_ID>_figures_LEAN.jsonl\n<SOURCE_ID>_fts_LEAN.sqlite\n<SOURCE_ID>_LEAN_cleaning_audit.json\n```\n\nIf controlled tag files are provided, also include candidate-tag staging artifacts:\n\n```text\n<SOURCE_ID>_tag_catalog.jsonl\n<SOURCE_ID>_candidate_tagging_audit.json\n```\n\nIf actual LLM/API semantic tagging is performed as an optional second pass, also include:\n\n```text\n<SOURCE_ID>_semantic_tagging_audit.json\n```\n\nOptional only if genuinely helpful:\n\n```text\nREADME_SHORT.md\n```\n\nDo not include chapter folders by default.\n\nDo not include long methodology documents.\n\nDo not include raw extracted duplicate files unless requested.\n\nDo not include raw LLM prompt/response logs in the ZIP unless the user explicitly asks for debugging artifacts. The audit should contain counts and limitations, not full prompt transcripts.\n\n---\n\n# 4. Cleaned UNIFIED JSON schema\n\nThe cleaned unified JSON should remain a list of page records.\n\nUse this lean page shape:\n\n```json\n{\n  \"page\": 19,\n  \"chapter_number\": \"1\",\n  \"chapter_title\": \"Precursor Lesions for Squamous Carcinoma in the Upper Aerodigestive Tract\",\n  \"section_heading\": \"Normal Oral Anatomy\",\n  \"content\": \"Cleaned page prose/table text only. No figure captions. No references section.\",\n  \"figures\": [\n    {\n      \"figure_id\": \"Fig 1.1\",\n      \"caption\": \"Normal mucosa of the mouth...\",\n      \"image_path\": \"gs://pathology-hub-0/_asset_library/textbooks/HN_Gnepp/figure_images/HN_Gnepp_p19_fig2.jpg\",\n      \"page\": 19,\n      \"figure_index\": 1\n    }\n  ]\n}\n```\n\nRequired page fields:\n\n```text\npage\ncontent\nfigures\n```\n\nPreferred page fields:\n\n```text\nchapter_number\nchapter_title\nsection_heading\n```\n\nRequired figure fields:\n\n```text\nfigure_id\ncaption\nimage_path\npage\nfigure_index\n```\n\nAvoid these fields in the lean output unless explicitly needed:\n\n```text\nlegend\ncaption_full\nimage_gcs_uri_legacy\nimage_gcs_uri_canonical\nraw_figure\nprovenance\nmigration_required\nfigshift_policy\nlegacy_figure_record_id\ncanonical_asset_migration_required\n```\n\nUse **caption**, not both `legend` and `caption`.\n\nUse **one image path**, not legacy/canonical duplicates.\n\nDo not include canonical GCS image targets unless the user explicitly asks for image migration.\n\n---\n\n# 5. Cleaning priorities\n\nPerform cleanup in this order.\n\n## Step 1 \u2014 Parse and audit raw structure\n\nLoad the JSON with Python.\n\nRecord:\n\n```text\nraw page count\nnonempty page count\nraw figure count\npages with figures\ninput field names\n```\n\nIf JSON parse fails, stop and report the parse error.\n\n## Step 2 \u2014 Infer `source_id`\n\nInfer from file name.\n\nExamples:\n\n```text\nHN_Gnepp_UNIFIED.json -> hn_gnepp\nHN_FAQ_UNIFIED.json -> hn_faq\nHN_Cardesa_UNIFIED.json -> hn_cardesa\nGU_Practical_UNIFIED.json -> gu_practical\n```\n\nUse this for output filenames.\n\n## Step 3 \u2014 Remove obvious boilerplate/front matter/back matter\n\nRemove from RAG content:\n\n```text\nblank pages\n\"this page intentionally left blank\"\ncopyright pages\nlicense pages\nebook activation pages\npublisher notices\ndedications\npreface pages\nacknowledgments\ncontributors\nabbreviations list\ntable of contents pages\nindex pages\n```\n\nBut do not necessarily delete the page record from the unified JSON. Preferred handling:\n\n```text\nIf a page is pure boilerplate, omit it from pages/chunks JSONL.\nIn UNIFIED_LEAN, either omit the page or keep it with content=\"\" and figures=[].\n```\n\nFor simplicity, it is acceptable to omit pure boilerplate pages entirely from `UNIFIED_LEAN`.\n\nDo not overfit to one publisher. Use keyword and structure heuristics.\n\nCommon boilerplate markers include:\n\n```text\ncopyright\nall rights reserved\nISBN\nLibrary of Congress\nPrinted in\npublisher\npreface\nacknowledgments\ncontributors\ncontents\nindex\nabbreviations used in text\nthis page intentionally left blank\nexpertconsult\nredeem\nterms of use\n```\n\n## Step 4 \u2014 Clean page text\n\nNormalize text:\n\n```text\nremove repeated running headers\nremove isolated page numbers\nremove excessive whitespace\nremove control characters\nfix line-break hyphenation when safe\ncollapse repeated spaces\nnormalize unicode punctuation only if it improves readability\n```\n\nDo not rewrite medical content.\n\nDo not summarize content.\n\nDo not invent headings.\n\n## Step 5 \u2014 Detect chapters, but do not split canonical output by chapter\n\nUse table of contents if it is available and parseable.\n\nGoal: assign these metadata fields:\n\n```text\nchapter_number\nchapter_title\nsection_heading\n```\n\nDo not create separate chapter JSON files by default.\n\nIf TOC parsing is unreliable, use heading heuristics:\n\n```text\nchapter number at start of page\nlarge title-like line near top of page\nrepeated chapter title in running header\n```\n\nIf chapter cannot be confidently assigned, leave fields null.\n\nGeneral rule:\n\n```text\nChapter metadata is useful.\nChapter files are optional and usually unnecessary.\n```\n\n## Step 6 \u2014 Remove chapter-end references\n\nRemove references from RAG page/chunk text.\n\nReference sections usually start with headings like:\n\n```text\nReferences\nREFERENCES\nBibliography\nSuggested Reading\nFurther Reading\nSelected References\n```\n\nThe main target is **chapter-end reference lists**.\n\nDo not remove inline citation numbers in ordinary prose.\n\nDo not remove text merely because it has citation numbers like `1\u20135`.\n\nRemove the reference section from:\n\n```text\nUNIFIED_LEAN page content\npages JSONL\nchunks JSONL\nFTS index\n```\n\nIt is optional to record only a count of removed reference blocks in the audit. Do not preserve full reference text unless the user asks.\n\n## Step 7 \u2014 Extract figure captions from page text\n\nCaptions may appear in page content as:\n\n```text\nFig. 1.1 Normal mucosa...\nFigure 1.1 Normal mucosa...\nFIGURE 1.1 Normal mucosa...\n```\n\nExtract these captions and attach them to the corresponding figure records.\n\nAfter extraction, remove the caption text from the page `content`.\n\nImportant rule:\n\n```text\nFigure caption text should NOT remain inside page content.\nCaption text should live only in:\n1. figure records\n2. figure_caption chunks\n```\n\nThis prevents duplicate search hits and makes page text cleaner.\n\n## Step 8 \u2014 Clean figure records\n\nFor each figure record, output only:\n\n```json\n{\n  \"figure_id\": \"Fig 1.1\",\n  \"caption\": \"caption text or null\",\n  \"image_path\": \"gs://...\",\n  \"page\": 19,\n  \"figure_index\": 1\n}\n```\n\nIf the original uses `legend`, convert it to `caption`.\n\nIf both page-caption extraction and original legend exist, prefer the cleaner/full caption.\n\nIf no caption is found, use:\n\n```json\n\"caption\": null\n```\n\nDo not fabricate captions.\n\nDo not use AI imagination to describe the image unless the source text provides it.\n\n## Step 9 \u2014 Handle known page-top `fig1` bug only when appropriate\n\nSome source extractions have a known bug where every page-level `_fig1.jpg` is just a top-of-page capture, and the true figures start at `_fig2.jpg`.\n\nOnly apply this rule if the user explicitly says the book has this bug, or if audit clearly proves `_fig1` is consistently a non-figure top crop.\n\nFor Gnepp, apply:\n\n```text\ndrop every image path ending in _fig1.jpg\nrenumber remaining figures logically on the page\nold _fig2.jpg becomes figure_index 1\nold _fig3.jpg becomes figure_index 2\n```\n\nBut keep `image_path` pointing to the existing real image object:\n\n```text\nimage_path = old working path, e.g. HN_Gnepp_p84_fig2.jpg\n```\n\nDo not add canonical migration paths.\n\nDo not include long figshift explanations.\n\nFor books without this confirmed bug, do not drop `_fig1`.\n\n## Step 10 \u2014 Build page JSONL\n\nEach page JSONL record should mirror the cleaned unified page:\n\n```json\n{\n  \"source_id\": \"hn_gnepp\",\n  \"page\": 19,\n  \"chapter_number\": \"1\",\n  \"chapter_title\": \"Precursor Lesions...\",\n  \"section_heading\": \"Normal Oral Anatomy\",\n  \"content\": \"cleaned page text\",\n  \"figure_ids\": [\"Fig 1.1\"]\n}\n```\n\nKeep it lean.\n\n## Step 11 \u2014 Build figure JSONL\n\nEach figure JSONL record:\n\n```json\n{\n  \"source_id\": \"hn_gnepp\",\n  \"page\": 19,\n  \"chapter_number\": \"1\",\n  \"chapter_title\": \"Precursor Lesions...\",\n  \"figure_id\": \"Fig 1.1\",\n  \"caption\": \"Normal mucosa of the mouth...\",\n  \"image_path\": \"gs://pathology-hub-0/...\"\n}\n```\n\nNo duplicate `legend`.\n\nNo canonical/legacy split.\n\nNo raw nested figure object.\n\n## Step 12 \u2014 Build chunk JSONL\n\nCreate two chunk types.\n\n### Page text chunks\n\n```json\n{\n  \"chunk_id\": \"hn_gnepp:p0019:c001\",\n  \"source_id\": \"hn_gnepp\",\n  \"chunk_type\": \"page_text\",\n  \"page\": 19,\n  \"chapter_number\": \"1\",\n  \"chapter_title\": \"Precursor Lesions...\",\n  \"section_heading\": \"Normal Oral Anatomy\",\n  \"text\": \"cleaned chunk text\"\n}\n```\n\n### Figure caption chunks\n\n```json\n{\n  \"chunk_id\": \"hn_gnepp:p0019:fig001:caption\",\n  \"source_id\": \"hn_gnepp\",\n  \"chunk_type\": \"figure_caption\",\n  \"page\": 19,\n  \"chapter_number\": \"1\",\n  \"chapter_title\": \"Precursor Lesions...\",\n  \"figure_id\": \"Fig 1.1\",\n  \"image_path\": \"gs://...\",\n  \"text\": \"Normal mucosa of the mouth...\"\n}\n```\n\nChunking rules:\n\n```text\nDo not chunk references.\nDo not include figure captions in page_text chunks.\nDo include figure captions as figure_caption chunks.\nKeep chunks reasonably small, usually 800\u20131800 characters.\nUse overlap only for long continuous prose, not tables/captions.\nDo not split tiny pages unnecessarily.\n```\n\nImportant sequencing rule:\n\nAfter Step 12 creates page_text and figure_caption chunks, perform controlled tag candidate staging before building the SQLite FTS index if tag files are provided. Candidate tags are useful and worth keeping, but they are not semantic AI tags. Actual LLM/API semantic tagging is optional and should be selective, not a required blocker for creating the lean package.\n\nThe final SQLite table should store tag metadata columns, but the FTS index itself should remain based on source text, chapter title, section heading, and caption text.\n---\n\n# 13. Controlled tag staging: candidate tags by default, semantic LLM tags only when actually performed\n\nThis section replaces the prior requirement to semantically tag every record immediately.\n\nThe lean normalized file is the primary deliverable. Tagging is useful, but it should not block creation of the lean package, chunks, figures, captions, audit, or SQLite FTS.\n\nUse a tiered tag strategy:\n\n```text\nTier 0 \u2014 No tags supplied:\nBuild the lean source, chunks, figures, audit, and FTS only.\n\nTier 1 \u2014 Controlled candidate tags supplied:\nParse the tag registry and assign candidate_tags for pages, chunks, and figure captions using deterministic candidate narrowing.\nThis is the recommended default.\n\nTier 2 \u2014 Actual semantic LLM/API tagging available:\nRun selective semantic LLM tagging only on high-yield records with plausible candidates.\nStore results as ai_tags only when real LLM/API calls were actually performed.\n\nTier 3 \u2014 Human-reviewed tags:\nStore reviewed_tags separately after manual review or correction.\n```\n\nDo not pretend candidate tags are semantic tags.\n\nDo not mark anything as `ai_assigned_unreviewed` unless an actual LLM/API call assigned the tag.\n\nIf no LLM/API calls are made, the correct status is:\n\n```text\ncandidate_tags_only_no_llm\n```\n\nnot:\n\n```text\nai_assigned_unreviewed\n```\n\n## 13.1 Why candidate tags are still worth doing\n\nCandidate tags are useful for:\n\n```text\nfuture LLM batch narrowing\ntag-family routing\nQA/audit\nfinding missing tags\nprioritizing entity-dense chunks\nfuture retrieval boosting after review\n```\n\nCandidate tags are not final semantic labels. They should not be used as strong filters.\n\n## 13.2 Controlled tag assignment goal\n\nAssign controlled tag metadata from the user-provided controlled tag set to cleaned pages, chunks, and figure captions.\n\nTags are metadata for routing, filtering, boosting, and QA. Tags do not replace source text retrieval and must not be treated as diagnostic truth.\n\nDo not invent tags. Only use exact tags from the provided tag files.\n\nThe default output should distinguish:\n\n```text\ncandidate_tags = deterministic candidate-narrowing layer\nai_tags = actual semantic LLM/API-assigned layer\nreviewed_tags = human-reviewed/corrected layer\n```\n\n## 13.3 Controlled tag registry input\n\nThe user may provide a ZIP or folder containing tag files such as:\n\n```text\nHN_Tags.txt\nEndo_Tags.txt\nGU_Tags.txt\nGYN_Tags.txt\nBreast_Tags.txt\nGI_Tags.txt\nHeme_Tags.txt\nSkin_Tags.txt\nEye_Orbit_Tags.txt\nBST_Tags.txt\nPeds_Tags.txt\nThorax_Mediastinum_Tags.txt\nMolecular_Tags.txt\nCyto_*.txt\n```\n\nEach line is one allowed tag, usually hierarchical:\n\n```text\nHN::Sinonasal::Papilloma::Schneiderian_Papilloma\nEndo::Thyroid::Neoplastic::Epithelial::Malignant::Cribriform_Morular_Thyroid_Carcinoma\nGU::Bladder::Urothelial::Urothelial_Carcinoma_In_Situ\n```\n\nBuild a `tag_catalog` from these files.\n\nEach tag catalog record should include:\n\n```json\n{\n  \"tag\": \"Endo::Thyroid::Neoplastic::Epithelial::Malignant::Cribriform_Morular_Thyroid_Carcinoma\",\n  \"tag_family\": \"Endo\",\n  \"tag_file\": \"Endo_Tags.txt\",\n  \"tag_tokens\": [\"endo\", \"thyroid\", \"neoplastic\", \"epithelial\", \"malignant\", \"cribriform\", \"morular\", \"thyroid\", \"carcinoma\"],\n  \"tag_display\": \"Cribriform Morular Thyroid Carcinoma\"\n}\n```\n\nDo not modify tag strings.\n\nWrite the catalog to:\n\n```text\n<SOURCE_ID>_tag_catalog.jsonl\n```\n\nif controlled tags were provided.\n\n## 13.4 Tag family selection\n\nBefore candidate staging, decide which tag file(s) are eligible.\n\nUse source ID, source title, chapter title, section heading, chunk text, figure caption, and immediate page context.\n\nDefault source-based routing:\n\n```text\nsource_id starts with hn_       \u2192 HN_Tags.txt\nsource_id starts with endo_     \u2192 Endo_Tags.txt\nsource_id starts with gu_       \u2192 GU_Tags.txt\nsource_id starts with gyn_      \u2192 GYN_Tags.txt\nsource_id contains breast       \u2192 Breast_Tags.txt\nsource_id contains gi           \u2192 GI_Tags.txt\nsource_id contains skin/derm    \u2192 Skin_Tags.txt\nsource_id contains heme         \u2192 Heme_Tags.txt\nsource_id contains neuro        \u2192 Neuro_Tags.txt\nsource_id contains peds         \u2192 Peds_Tags.txt\nsource_id contains thorax/lung  \u2192 Thorax_Mediastinum_Tags.txt\nsource_id contains molecular    \u2192 Molecular_Tags.txt\nsource_id contains cyto         \u2192 relevant Cyto_*.txt first\n```\n\nChapter/context overrides:\n\n```text\nHN book + chapter_title contains thyroid/parathyroid/adrenal/pituitary/endocrine\n\u2192 include Endo_Tags.txt\n\nHN book + chapter_title contains hematopoietic/lymphoma/leukemia/plasma cell\n\u2192 include Heme_Tags.txt\n\nHN book + chapter_title contains skin/cutaneous/adnexal/melanocytic\n\u2192 include Skin_Tags.txt\n\nHN book + chapter_title contains conjunctiva/orbit/lacrimal/ocular/eye\n\u2192 include Eye_Orbit_Tags.txt\n\nHN book + chapter_title contains soft tissue/sarcoma/peripheral nerve sheath\n\u2192 include HN_Tags.txt first, BST_Tags.txt second\n\nAny source + section/chunk contains molecular testing, sequencing, fusion, mutation, copy number, MSI, LOH, NGS, FISH, PCR\n\u2192 include disease/source tag family first, Molecular_Tags.txt second\n\nAny source + FNA/cytology/smear/Bethesda/Milan/Paris/Yokohama terminology\n\u2192 include relevant Cyto_*.txt second, unless the book is a cytology source, then Cyto_*.txt first\n```\n\nNever choose a candidate tag solely because it belongs to the same book. A candidate tag must have some support from:\n\n```text\nchunk text\nsection heading\nchapter title\nfigure caption\nimmediate page context\nadjacent section/page context when included\n```\n\n## 13.5 Candidate tag staging before optional AI\n\nDo not give any future LLM the entire tag universe.\n\nFor each page/chunk/figure caption:\n\n1. Build context text:\n\n```text\nsource_title\nsource_id\nchapter_number\nchapter_title\nsection_heading\nchunk_type\npage number\nfigure_id/caption if applicable\nrecord text\noptional previous/next section heading\noptional preceding/following page excerpt when needed for context\n```\n\n2. Select candidate tag families using the rules above.\n\n3. Score candidate tags deterministically only for candidate narrowing:\n\n```text\n+10 exact tag display phrase in section_heading\n+8 exact tag display phrase in chunk text\n+7 exact tag display phrase in chapter_title\n+5 all important disease/entity tokens present in chunk text\n+4 all important disease/entity tokens present in chapter/section context\n+3 site/category tokens match chapter/section\n+2 abbreviation/synonym match if alias table exists\n-5 candidate tag family conflicts with source/chapter context\n```\n\n4. Keep a practical top candidate list:\n\n```text\npage: up to 20\u201360 candidate tags\nchunk: up to 20\u201340 candidate tags\nfigure caption: up to 10\u201325 candidate tags\n```\n\n5. If no plausible candidate tag exists, mark:\n\n```json\n\"tagging_status\": \"no_candidate_tags\"\n```\n\nCandidate scoring is not semantic tag assignment. It only creates `candidate_tags`.\n\n## 13.6 Candidate tag output schema\n\nAdd these fields to page records, chunk records, and figure-caption records when tag files are supplied:\n\n```json\n{\n  \"candidate_tags\": [\n    \"HN::Sinonasal::Papilloma::Schneiderian_Papilloma\"\n  ],\n  \"candidate_tag_assignments\": [\n    {\n      \"tag\": \"HN::Sinonasal::Papilloma::Schneiderian_Papilloma\",\n      \"candidate_score\": 18,\n      \"candidate_basis\": [\"section_heading\", \"chunk_text\"],\n      \"support\": \"short matched phrase or context cue\",\n      \"scope\": \"chunk\",\n      \"tag_source_file\": \"HN_Tags.txt\",\n      \"tag_registry_version\": \"tags_YYYYMMDD\",\n      \"tagger\": \"deterministic_candidate_narrowing\",\n      \"assigned_at_utc\": \"...\"\n    }\n  ],\n  \"ai_tags\": [],\n  \"context_tags\": [],\n  \"reviewed_tags\": [],\n  \"ai_tag_assignments\": [],\n  \"tagging_status\": \"candidate_tags_only_no_llm\"\n}\n```\n\nIf no tag files are supplied:\n\n```json\n\"tagging_status\": \"not_attempted\"\n```\n\nIf tag files are supplied but no candidates are plausible:\n\n```json\n\"tagging_status\": \"no_candidate_tags\"\n```\n\nDo not populate `ai_tags` during candidate staging.\n\n## 13.7 Candidate tagging audit\n\nCreate:\n\n```text\n<SOURCE_ID>_candidate_tagging_audit.json\n```\n\nInclude:\n\n```json\n{\n  \"schema_version\": \"textbook_candidate_tagging_audit.v1\",\n  \"source_id\": \"hn_gnepp\",\n  \"tag_registry_version\": \"tags_YYYYMMDD\",\n  \"tag_files_used\": [\n    \"HN_Tags.txt\",\n    \"Endo_Tags.txt\",\n    \"Heme_Tags.txt\"\n  ],\n  \"candidate_tagging_performed\": true,\n  \"semantic_llm_tagging_performed\": false,\n  \"records_processed\": {\n    \"pages\": 0,\n    \"chunks\": 0,\n    \"figures\": 0\n  },\n  \"records_with_candidate_tags\": 0,\n  \"records_without_candidate_tags\": 0,\n  \"candidate_tag_count_total\": 0,\n  \"unique_candidate_tag_count\": 0,\n  \"top_candidate_tags\": [],\n  \"candidate_status_counts\": {\n    \"candidate_tags_only_no_llm\": 0,\n    \"no_candidate_tags\": 0,\n    \"not_attempted\": 0\n  },\n  \"manual_review_recommended\": true,\n  \"known_limitations\": [\n    \"Candidate tags are deterministic narrowing metadata, not final semantic tags.\",\n    \"No LLM/API semantic assignment was performed in this stage unless separately stated.\",\n    \"Candidate tags should not be used as strong retrieval filters.\",\n    \"Primary retrieval remains source text, chapter title, section heading, and figure caption text.\"\n  ]\n}\n```\n\n## 13.8 Optional selective semantic LLM/API tagging\n\nSemantic LLM tagging is optional and should be a second pass, not a prerequisite for returning the lean package.\n\nRun semantic LLM tagging only if:\n\n```text\nan actual LLM/API call is available\ncandidate tags exist\nthe user explicitly asks for semantic tagging or the workflow has API access configured\nthe record is worth tagging\n```\n\nRecommended records to semantically tag first:\n\n```text\nfigure captions\nsection-heading/title chunks\nentity-dense diagnostic chunks\ntables/lists of differential diagnoses\nhigh-confidence candidate records\npages with multiple figures or named entities\n```\n\nRecommended records to skip or defer:\n\n```text\ngeneric introduction text\nnormal anatomy unless the tag set contains useful anatomy tags\nhistory/background paragraphs\nlow-signal boilerplate-like prose\nrecords with no candidate tags\nrecords where candidate tags are only broad site terms\n```\n\nDo not send every chunk for semantic tagging by default. That is expensive and usually not needed.\n\n## 13.9 Required semantic LLM reasoning when optional semantic tagging is performed\n\nFor each record selected for semantic LLM tagging, the model must first interpret the record semantically as pathology content.\n\nThe LLM must consider:\n\n```text\nanatomic site\nchapter context\nsection heading\nlesion/process category\narchitecture\ncytology\ngrowth pattern\ndysplasia grade if applicable\nbenign/reactive/neoplastic distinction\nmimics and differential diagnoses\nmolecular/pathogenesis meaning if actually discussed\nwhether a term is the main entity or only a near-miss/differential\n```\n\nExamples of desired semantic behavior:\n\n```text\n\"twisting ducts lined by nonkeratinizing squamous epithelium, mucinous cells, rounded nests, lateral nasal wall\"\n\u2192 semantically maps to inverted Schneiderian papilloma if that exact controlled tag is available.\n\n\"oncocytic epithelium with intraepithelial mucinous cysts in sinonasal papilloma\"\n\u2192 semantically maps to oncocytic Schneiderian papilloma if available.\n\n\"necrotic seromucinous glands with squamous metaplasia retaining lobular architecture\"\n\u2192 semantically maps to necrotizing sialometaplasia if available.\n\n\"drop-shaped rete ridges, dyskeratosis, superficial mitoses, severe atypia\"\n\u2192 semantically maps to squamous dysplasia / carcinoma in situ category tags if available.\n\n\"p16 mentioned in normal tonsillar crypt epithelium\"\n\u2192 do not assign an HPV-associated carcinoma tag unless neoplasia is actually discussed.\n```\n\n## 13.10 Required LLM/API call rules\n\nFor every record marked semantically tagged, call the LLM/API.\n\nAcceptable batching:\n\n```text\none record per call\nor small batches of records if the prompt forces independent JSON output per record_id\n```\n\nEach LLM output must be tied back to:\n\n```text\nrecord_id or chunk_id\nsource_id\npage\nchunk_type\ncandidate_tags_version\ntag_registry_version\n```\n\nThe pipeline must validate that every returned tag is an exact string from the candidate tag list and the full tag catalog.\n\nIf JSON parsing fails, retry once with a strict JSON repair prompt. If still invalid, mark the record:\n\n```json\n\"tagging_status\": \"llm_parse_failed\"\n```\n\nDo not silently substitute candidate tags into `ai_tags`.\n\n## 13.11 Semantic LLM tag assignment prompt\n\nUse this prompt for each selected page/chunk/figure-caption record or an equivalent batch prompt with the same constraints.\n\n```text\nYou are assigning controlled Pathology Hub tags to a textbook record.\n\nYou are a subspecialty surgical pathologist. You must reason semantically about the pathology content before selecting tags.\n\nYou must choose only from the provided candidate tags.\nDo not invent tags.\nDo not rewrite tag strings.\nDo not assign a tag unless the record text or provided context supports it.\n\nImportant:\nThis is not keyword matching.\nA tag should reflect the diagnostic/pathologic concept of the record.\nUse morphology, architecture, cytology, site, chapter title, section heading, and figure caption context to infer the correct concept.\nPrefer tags directly supported by the record text.\nIf the record text is generic but the chapter/section clearly identifies the entity, you may assign a context-supported tag with lower confidence and basis=\"context\".\nIf no candidate tag is semantically supported, return an empty assigned_tags array.\n\nSemantic reasoning checklist:\n- Identify the main pathologic process or entity.\n- Decide whether the text is primarily about diagnosis, differential diagnosis, normal anatomy, mimic, ancillary testing, molecular pathogenesis, or prognosis.\n- Distinguish main entity from near-miss/differential entities.\n- Avoid assigning tags from incidental mentions.\n- Avoid assigning site-only tags when a specific entity tag is available.\n- Avoid assigning molecular tags unless molecular testing, mutation class, fusion, biomarker interpretation, or molecular pathogenesis is central to the text.\n- Avoid assigning cytology tags unless cytology/FNA/smear/category material is central to the text.\n\nTagging rules:\n- Prefer the most specific disease/entity tag.\n- Assign at most 5 tags per chunk.\n- Assign at most 8 tags per page.\n- Assign at most 3 tags per figure caption.\n- Use broad/site tags only if a specific entity tag is unavailable and the tag itself exists in the candidate list.\n- If a heading says one entity but the chunk discusses differential diagnosis, assign the main entity tag and only assign differential tags if they are substantially discussed.\n- If a tag is only inferred from chapter context and not the chunk text, mark basis=\"context\" and confidence <=0.80.\n- If support is weak, abstain.\n\nInput:\nsource_id: <SOURCE_ID>\nsource_title: <SOURCE_TITLE>\npage: <PAGE>\nrecord_id/chunk_id: <RECORD_ID_OR_CHUNK_ID>\nchunk_type: <page_text|figure_caption|page>\nchapter_number: <CHAPTER_NUMBER>\nchapter_title: <CHAPTER_TITLE>\nsection_heading: <SECTION_HEADING>\nfigure_id: <FIGURE_ID or null>\nrecord_text:\n<CHUNK_OR_PAGE_OR_CAPTION_TEXT>\n\nCandidate tags:\n<ONE TAG PER LINE>\n\nReturn strict JSON only:\n{\n  \"semantic_interpretation\": {\n    \"main_process_or_entity\": \"short label or null\",\n    \"record_role\": \"diagnosis|differential|normal_anatomy|mimic|ancillary|molecular|prognosis|mixed|unclear\",\n    \"support_summary\": \"one concise sentence\"\n  },\n  \"assigned_tags\": [\n    {\n      \"tag\": \"exact candidate tag\",\n      \"confidence\": 0.0,\n      \"basis\": \"text|heading|chapter|caption|context|mixed\",\n      \"support\": \"short exact phrase from text/context\",\n      \"scope\": \"page|chunk|figure_caption\",\n      \"reasoning_short\": \"one sentence explaining why this tag matches semantically\"\n    }\n  ],\n  \"rejected_near_misses\": [\n    {\n      \"tag\": \"candidate tag not assigned\",\n      \"reason\": \"why not\"\n    }\n  ],\n  \"missing_tag_suggestions\": [\n    {\n      \"suggested_label\": \"possible missing concept label\",\n      \"reason\": \"only if no allowed tag fits\"\n    }\n  ]\n}\n```\n\nStore only concise `support_summary` and `reasoning_short`; do not store long hidden reasoning.\n\n## 13.12 Semantic confidence rules\n\nUse these confidence bands only for actual LLM/API semantic assignments:\n\n```text\n0.95\u20131.00\nExact disease/entity name appears in chunk text or section heading and is the main topic.\n\n0.85\u20130.94\nStrong synonym, abbreviation, or morphology-based semantic match in chunk text, supported by chapter/section context.\n\n0.75\u20130.84\nEntity clearly identified by chapter/section heading; chunk text is about that entity but does not repeat exact name.\n\n0.65\u20130.74\nContext-only tag; useful as weak retrieval boost but not strong filter.\n\n<0.65\nDo not assign.\n```\n\nStore semantic tags with confidence. Do not collapse confidence into a single flat list without provenance.\n\n## 13.13 Page vs chunk vs figure tagging\n\nCandidate or semantic page tags:\n\n```text\nPage tags are broad context.\nAssign using chapter title, section heading, full cleaned page content, and figure captions on that page.\nPage tags may include multiple entities if the page is a table, list, differential diagnosis, or multi-entity figure page.\nRecommended max page tags: 8 semantic tags; candidate list may be larger but capped.\n```\n\nCandidate or semantic chunk tags:\n\n```text\nChunk tags should be more precise.\nUse chunk text first, section heading second, chapter title third.\nRecommended max semantic chunk tags: 5.\nIf a chunk is generic introduction text and only chapter context identifies the entity, either assign a context-supported semantic tag with confidence <=0.80 or leave ai_tags empty and keep only candidate/context tags.\n```\n\nCandidate or semantic figure-caption tags:\n\n```text\nFigure-caption tags should be assigned from the caption and page/chapter context.\nRecommended max semantic figure-caption tags: 3.\nDo not describe the image visually.\nOnly tag from caption/context text.\n```\n\n## 13.14 Semantic output schema additions\n\nWhen actual LLM/API semantic tagging is performed, populate:\n\n```json\n{\n  \"ai_tags\": [\n    \"HN::Oral_Cavity::Precursor::Oral_Epithelial_Dysplasia\"\n  ],\n  \"context_tags\": [\n    \"HN::Oral_Cavity::Precursor::Oral_Epithelial_Dysplasia\"\n  ],\n  \"ai_tag_assignments\": [\n    {\n      \"tag\": \"HN::Oral_Cavity::Precursor::Oral_Epithelial_Dysplasia\",\n      \"confidence\": 0.92,\n      \"basis\": \"section\",\n      \"support\": \"section_heading: Oral epithelial dysplasia\",\n      \"scope\": \"chunk\",\n      \"reasoning_short\": \"The section and text discuss epithelial dysplasia as the primary diagnostic concept.\",\n      \"tag_source_file\": \"HN_Tags.txt\",\n      \"tag_registry_version\": \"tags_YYYYMMDD\",\n      \"tagger\": \"llm_semantic_controlled_tag_assignment\",\n      \"llm_model\": \"<MODEL_NAME>\",\n      \"assigned_at_utc\": \"...\"\n    }\n  ],\n  \"tagging_status\": \"semantic_llm_assigned_unreviewed\"\n}\n```\n\nAllowed tagging statuses:\n\n```text\nnot_attempted\nno_candidate_tags\ncandidate_tags_only_no_llm\nllm_parse_failed\nsemantic_llm_assigned_unreviewed\nsemantic_llm_low_confidence\nreviewed_passed\nreviewed_corrected\n```\n\nKeep `candidate_tags`, `ai_tags`, `context_tags`, and `reviewed_tags` separate.\n\nDo not overwrite candidate tags with AI tags.\n\nDo not overwrite source tags with AI tags.\n\n## 13.15 Semantic tagging audit if optional LLM tagging is performed\n\nIf actual semantic LLM/API tagging is performed, create:\n\n```text\n<SOURCE_ID>_semantic_tagging_audit.json\n```\n\nInclude:\n\n```json\n{\n  \"schema_version\": \"textbook_semantic_tagging_audit.v1\",\n  \"source_id\": \"hn_gnepp\",\n  \"tag_registry_version\": \"tags_YYYYMMDD\",\n  \"llm_tagging_performed\": true,\n  \"llm_model\": \"<MODEL_NAME>\",\n  \"selection_policy\": \"selective_high_yield_records_only\",\n  \"records_selected_for_llm\": {\n    \"pages\": 0,\n    \"chunks\": 0,\n    \"figures\": 0\n  },\n  \"records_semantically_tagged\": {\n    \"pages\": 0,\n    \"chunks\": 0,\n    \"figures\": 0\n  },\n  \"records_with_candidate_tags_but_no_llm_call\": 0,\n  \"llm_parse_failures\": 0,\n  \"assigned_ai_tag_count\": 0,\n  \"unique_ai_tag_count\": 0,\n  \"tag_confidence_counts\": {\n    \"high_ge_0_90\": 0,\n    \"medium_0_75_0_89\": 0,\n    \"context_0_65_0_74\": 0\n  },\n  \"top_ai_tags\": [],\n  \"missing_tag_suggestions_top\": [],\n  \"manual_review_recommended\": true,\n  \"known_limitations\": [\n    \"AI tags are metadata for retrieval boost/filtering, not diagnostic truth.\",\n    \"Tags were selected from supplied controlled tag files only.\",\n    \"Semantic LLM tagging was selective, not exhaustive.\",\n    \"Spot audit is required before using AI tags as strong filters.\",\n    \"No visual interpretation was performed; figure tags are caption/context based only.\"\n  ]\n}\n```\n\n## 13.16 Retrieval use\n\nUse tags as metadata only:\n\n```text\nreviewed_tags: strong filter or strong boost\nhigh-confidence ai_tags: moderate boost\ncandidate_tags: candidate narrowing / weak boost only\ncontext_tags: weak boost only\nlow-confidence tags: display/audit only\n```\n\nNever use candidate tags or AI tags as the only retrieval path.\n\nThe primary searchable content remains:\n\n```text\npage/chunk text\nsection heading\nchapter title\nfigure caption text\nsource title\n```\n\n## 13.17 Quality checks for controlled tag staging and optional semantic tagging\n\nBefore returning output:\n\n```text\nAll candidate_tags are exact strings from the tag catalog.\nAll ai_tags are exact strings from the tag catalog.\nNo freeform tags exist in candidate_tags or ai_tags.\nEvery candidate_tag_assignments item includes tag, candidate_score, candidate_basis, support, scope, and tag_source_file.\nEvery ai_tag_assignments item includes tag, confidence, basis, support, scope, reasoning_short, tag_source_file, tagger, and llm_model.\nNo ai_tag has confidence <0.65.\nTag files used are recorded.\nLLM semantic tagging status is explicit.\nRecords with candidate tags but no LLM call are not marked semantic_llm_assigned.\nAt least 10 random candidate-tagged records are spot-checked.\nIf semantic LLM tagging was performed, at least 10 high-confidence semantic tags are spot-checked.\nIf semantic LLM tagging was performed, at least 10 context-only semantic tags are spot-checked.\nFor source families with expected tags, confirm that assigned tags use the expected prefix unless chapter context justifies an override.\n```\n\nIf tag quality is poor, do not force tags into the final answer. Mark tag assignment as experimental and keep tags as weak metadata only.\n\n---\n\n# 14. Build SQLite FTS index\n\nBuild a simple SQLite FTS5 index over chunks after cleaning and controlled tag candidate staging. If optional semantic LLM tagging was performed, include semantic tag metadata too.\n\nMinimum table:\n\n```sql\nCREATE TABLE textbook_chunks (\n  chunk_id TEXT PRIMARY KEY,\n  source_id TEXT,\n  chunk_type TEXT,\n  page INTEGER,\n  chapter_number TEXT,\n  chapter_title TEXT,\n  section_heading TEXT,\n  figure_id TEXT,\n  image_path TEXT,\n  text TEXT,\n  candidate_tags TEXT,\n  candidate_tag_assignments TEXT,\n  ai_tags TEXT,\n  context_tags TEXT,\n  reviewed_tags TEXT,\n  ai_tag_assignments TEXT,\n  tagging_status TEXT,\n  json TEXT\n);\n```\n\nFTS table:\n\n```sql\nCREATE VIRTUAL TABLE textbook_chunks_fts\nUSING fts5(text, chapter_title, section_heading, content='textbook_chunks', content_rowid='rowid');\n```\n\nInsert all page_text and figure_caption chunks.\n\nThe FTS index is keyword/search text only. It is not vector search and not semantic search.\n\nStore tag fields in the base table for filtering/boosting metadata, not as a substitute for source text retrieval.\n\n---\n\n# 15. Audit\n\nCreate:\n\n```text\n<SOURCE_ID>_LEAN_cleaning_audit.json\n```\n\nInclude only useful audit fields:\n\n```json\n{\n  \"source_id\": \"hn_gnepp\",\n  \"input_file\": \"HN_Gnepp_UNIFIED.json\",\n  \"raw_pages\": 1220,\n  \"output_pages\": 975,\n  \"raw_figures\": 2321,\n  \"output_figures\": 1108,\n  \"chunks\": 3582,\n  \"page_text_chunks\": 2500,\n  \"figure_caption_chunks\": 1082,\n  \"boilerplate_pages_removed\": 50,\n  \"reference_blocks_removed\": 193,\n  \"figure_captions_extracted\": 1000,\n  \"figure_captions_removed_from_page_text\": 1000,\n  \"fig1_bug_applied\": true,\n  \"dropped_fig1_records\": 973,\n  \"json_parse_pass\": true,\n  \"sqlite_fts_created\": true,\n  \"controlled_candidate_tagging_performed\": true,\n  \"semantic_llm_tagging_performed\": false,\n  \"candidate_tagging_audit_file\": \"<SOURCE_ID>_candidate_tagging_audit.json\",\n  \"semantic_tagging_audit_file\": null,\n  \"known_limitations\": [\n    \"Caption matching is text-based and may need spot audit for multi-panel figures.\",\n    \"No visual interpretation was performed.\",\n    \"No vector index was built.\",\n    \"Candidate tags are deterministic narrowing metadata, not final semantic tags.\",\n    \"AI tags are only present if actual LLM/API semantic assignment was performed.\"\n  ]\n}\n```\n\nDo not include long methodology unless asked.\n\n---\n\n# 16. Quality checks before final response\n\nBefore returning the ZIP, run these checks.\n\nJSON checks:\n\n```text\nUNIFIED_LEAN parses as JSON\npages JSONL parses line-by-line\nchunks JSONL parses line-by-line\nfigures JSONL parses line-by-line\ntag_catalog JSONL parses line-by-line if created\ncandidate_tagging_audit parses as JSON if created\nsemantic_tagging_audit parses as JSON if created\n```\n\nDuplication checks:\n\n```text\nNo `legend` field in final figure records.\nNo duplicate `caption` + `legend` fields.\nNo `image_gcs_uri_canonical` field.\nNo verbose raw_figure/provenance fields.\n```\n\nCaption checks:\n\n```text\nSearch final page content for caption starts:\nFig.\nFigure\nFIGURE\n```\n\nIt is okay if ordinary prose mentions a figure in parentheses, like `(Fig. 1.1)`, but full caption blocks should be gone.\n\nReference checks:\n\n```text\nSearch final chunks for headings:\nReferences\nBibliography\nSuggested Reading\nFurther Reading\n```\n\nThere should be no chapter-end reference chunks.\n\nFigure checks:\n\n```text\nEvery figure has image_path.\nEvery figure has page.\nEvery figure has figure_index.\nCaption may be null, but should not be duplicated in page text.\n```\n\nIf `fig1_bug_applied=true`:\n\n```text\nNo retained figure image_path should end with _fig1.jpg.\n```\n\nTag checks if tags were supplied:\n\n```text\nAll candidate_tags are exact controlled tag strings.\nNo candidate_tags appear outside tag_catalog.\nAll ai_tags are exact controlled tag strings if semantic LLM tagging was performed.\nNo ai_tags appear outside tag_catalog.\nNo ai_tag confidence <0.65.\nEvery candidate_tag_assignments item includes tag, candidate_score, candidate_basis, support, scope, tag_source_file, tagger.\nEvery ai_tag_assignments item includes tag, confidence, basis, support, scope, tag_source_file, tagger, llm_model.\nLLM semantic tagging was actually performed if tagging_status begins with semantic_llm_assigned.\nRecords with candidate tags but no LLM call use tagging_status=candidate_tags_only_no_llm.\n```\n\nFTS smoke test:\n\nRun at least 2\u20133 search tests using SQLite FTS.\n\nExample:\n\n```sql\nSELECT chunk_id, page, chunk_type, substr(text,1,250)\nFROM textbook_chunks_fts\nJOIN textbook_chunks ON textbook_chunks.rowid = textbook_chunks_fts.rowid\nWHERE textbook_chunks_fts MATCH 'squamous carcinoma'\nLIMIT 5;\n```\n\nReport whether FTS was created and smoke-tested.\n\n---\n\n# 17. Final response format to user\n\nKeep final answer concise.\n\nDo not dump giant methodology.\n\nIf only candidate tags were created, say:\n\n```text\nI cleaned it into a lean whole-book UNIFIED JSON.\nI kept chapter metadata inside page/chunk records, but did not create chapter JSON folders.\nI removed references from RAG content.\nI removed figure captions from page text and stored them only as figure records/caption chunks.\nI parsed the controlled tag registry and added candidate_tags as staging metadata.\nI did not mark tags as semantic ai_tags because no actual LLM/API semantic tagging pass was performed.\nI built pages/chunks/figures JSONL and SQLite FTS.\nHere is the ZIP.\n```\n\nIf semantic LLM/API tagging was actually performed, say:\n\n```text\nI cleaned it into a lean whole-book UNIFIED JSON.\nI kept chapter metadata inside page/chunk records, but did not create chapter JSON folders.\nI removed references from RAG content.\nI removed figure captions from page text and stored them only as figure records/caption chunks.\nI parsed the controlled tag registry, added candidate_tags, and assigned ai_tags only for records processed by actual semantic LLM/API calls.\nI built pages/chunks/figures JSONL and SQLite FTS.\nHere is the ZIP.\n```\n\nInclude the download link.\n\nInclude a short count summary.\n\nMention known limitations in 2\u20133 bullets only.\n\nDo not claim semantic tagging was performed unless actual LLM/API calls occurred.\n\n---\n\n# 18. Do not do these things\n\nDo not create separate chapter JSON folders by default.\n\nDo not make the output overly complicated.\n\nDo not include full reference lists in the searchable output.\n\nDo not keep figure captions duplicated in page content.\n\nDo not keep both `legend` and `caption`.\n\nDo not create separate `legacy` and `canonical` image path fields unless explicitly asked.\n\nDo not produce a long figshift/migration manifesto.\n\nDo not claim vectorization.\n\nDo not claim API exposure.\n\nDo not claim image interpretation.\n\nDo not rewrite medical content.\n\nDo not invent figure captions.\n\nDo not invent chapter titles if unsure.\n\nDo not invent tags.\n\nDo not assign final tags from keyword matching alone.\n\nDo not mark deterministic/fuzzy candidate tags as AI semantic tags.\n\nDo not use candidate tags or AI tags as diagnostic truth.\n\nDo not make tagging the bottleneck for lean source creation.\n\n---\n\n# 19. Pathology Hub handoff packet\n\nAt the end, include a small handoff packet:\n\n```text\nWorkstream name:\nTextbook + Tag RAG / Source Normalization\n\nPurpose:\nClean one page-based textbook UNIFIED JSON into a lean whole-book normalized source for future RAG, then optionally add controlled candidate tags for future semantic tagging, routing, filtering, boosting, and QA.\n\nInputs assumed:\nUploaded <SOURCE_FILE>.\nOptional uploaded controlled tag ZIP/folder.\n\nOutputs produced:\n<SOURCE_ID>_UNIFIED_LEAN.json\n<SOURCE_ID>_pages_LEAN.jsonl\n<SOURCE_ID>_chunks_LEAN.jsonl\n<SOURCE_ID>_figures_LEAN.jsonl\n<SOURCE_ID>_fts_LEAN.sqlite\n<SOURCE_ID>_LEAN_cleaning_audit.json\n<SOURCE_ID>_tag_catalog.jsonl, if tag files provided\n<SOURCE_ID>_candidate_tagging_audit.json, if tag files provided\n<SOURCE_ID>_semantic_tagging_audit.json, only if actual semantic LLM/API tagging performed\n\nSchemas used:\ntextbook_page.lean.v1\ntextbook_chunk.lean.v1\ntextbook_figure.lean.v1\ntextbook_cleaning_audit.lean.v1\ntag_catalog.controlled.v1\ntextbook_candidate_tag_assignment.v1\ntextbook_candidate_tagging_audit.v1\ntextbook_ai_tag_assignment.semantic_llm.v1, only if actual LLM/API tagging performed\ntextbook_semantic_tagging_audit.v1, only if actual LLM/API tagging performed\n\nGCS paths assumed:\nNone unless user asks to upload.\nFuture canonical target would be under gs://pathology_hub/02_normalized/textbooks/ and gs://pathology_hub/03_indexes/textbooks/.\n\nAPI endpoints needed/provided:\nNone provided.\nFuture endpoint would be POST /evidence/search with sources including textbooks, but do not claim this is live.\n\nIntegration points:\nFuture unified evidence API.\nFuture textbook FTS backend.\nFuture tag-aware retrieval/reranking.\nFuture selective semantic LLM tagging pass.\nFuture HTML teaching pages.\n\nTests/audits:\nJSON parse pass.\nJSONL parse pass.\nSQLite FTS created.\nCaption/reference duplication checks run.\nControlled candidate tag validation run if tags supplied.\nSemantic LLM tagging audit run only if actual LLM/API calls completed.\nCount summary included.\n\nKnown limitations:\nNo vector search.\nNo visual interpretation.\nCaption matching may need spot audit.\nCandidate tags are not final semantic tags.\nAI tags are unreviewed metadata and are present only if actual LLM/API semantic assignment occurred.\n\nNext steps:\nReview audit counts, spot-check 10 pages with figures, spot-check candidate-tagged records, then optionally run selective semantic LLM tagging for high-yield figure captions, section-heading chunks, and entity-dense chunks before uploading normalized artifacts to canonical GCS.\n```\n"

prompt_path = CHATGPT_JOB_DIR / "PROMPT_TEXTBOOK_UNIFIED_TO_LEAN_WITH_CONTROLLED_TAGGING_v4.txt"
tag_urls_path = CHATGPT_JOB_DIR / "CONTROLLED_TAG_URLS.json"
readme_path = CHATGPT_JOB_DIR / "README_CHATGPT_INTERMISSION.md"

prompt_path.write_text(CLEANER_PROMPT_TEXT, encoding="utf-8")
save_json({"tag_urls": TAG_URLS, "tag_registry_source": "https://storage.googleapis.com/pathology-hub-0/Tags/"}, tag_urls_path)

readme_lines = [
    "# ChatGPT Intermission — Pathology Hub Textbook Cleaner",
    "",
    "Upload this job ZIP to ChatGPT.",
    "",
    "Use/paste `PROMPT_TEXTBOOK_UNIFIED_TO_LEAN_WITH_CONTROLLED_TAGGING_v4.txt`.",
    "",
    "Input file in this ZIP:",
    "- one `*_UNIFIED.json`",
    "",
    "Required ChatGPT return:",
    "- one LEAN package ZIP containing:",
    "  - `<SOURCE_ID>_UNIFIED_LEAN.json`",
    "  - `<SOURCE_ID>_pages_LEAN.jsonl`",
    "  - `<SOURCE_ID>_chunks_LEAN.jsonl`",
    "  - `<SOURCE_ID>_figures_LEAN.jsonl`",
    "  - `<SOURCE_ID>_fts_LEAN.sqlite`",
    "  - `<SOURCE_ID>_LEAN_cleaning_audit.json`",
    "  - `<SOURCE_ID>_tag_catalog.jsonl` if tags used",
    "  - `<SOURCE_ID>_candidate_tagging_audit.json` if tags used",
    "  - `<SOURCE_ID>_semantic_tagging_audit.json` only if actual semantic LLM/API tagging was performed",
    "",
    "Important:",
    "- Candidate tags are not final diagnostic truth.",
    "- Do not claim vectorization/API exposure.",
    "- No visual interpretation.",
]
readme_path.write_text("\n".join(readme_lines), encoding="utf-8")

job_zips = []
raw_unified_files = sorted(STAGED_UNIFIED_DIR.glob("*_UNIFIED.json"))
if not raw_unified_files:
    raise RuntimeError("No raw *_UNIFIED.json files found. Run Stage 1 first.")

for unified_path in raw_unified_files:
    source_id = infer_source_id(unified_path.name)
    zip_path = CHATGPT_JOB_DIR / f"{source_id}_CHATGPT_LEAN_TAGGING_JOB.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(unified_path, arcname=unified_path.name)
        z.write(prompt_path, arcname=prompt_path.name)
        z.write(tag_urls_path, arcname=tag_urls_path.name)
        z.write(readme_path, arcname=readme_path.name)
    job_zips.append(zip_path)

print("Created ChatGPT job ZIPs:", len(job_zips))
for p in job_zips:
    print(" -", p, "|", p.stat().st_size, "bytes")

if SYNC_CHATGPT_JOBS_TO_GCS:
    sync_dir_to_gcs(CHATGPT_JOB_DIR, f"gs://{DEST_BUCKET}/01_staged/textbooks/chatgpt_jobs/")
    print("\nGCS ChatGPT jobs:", f"gs://{DEST_BUCKET}/01_staged/textbooks/chatgpt_jobs/")

print("\nSTOP HERE FOR CHATGPT.")
print("Upload one *_CHATGPT_LEAN_TAGGING_JOB.zip into ChatGPT, use the included prompt, then return here with the LEAN ZIP.")


# STAGE 3 — Upload ChatGPT-returned LEAN ZIPs

Run this **after** ChatGPT returns LEAN package ZIPs. The upload picker will place the packages in `/content/lean_returns`.

In [ ]:
# ============================================================
# STAGE 3A — UPLOAD CHATGPT-RETURNED LEAN ZIP PACKAGES
# ============================================================

LEAN_RETURN_DIR.mkdir(parents=True, exist_ok=True)

if RUN_UPLOAD_PICKER_FOR_LEAN_ZIPS:
    try:
        from google.colab import files
        uploaded = files.upload()
        for name in uploaded.keys():
            src = Path("/content") / name
            if src.exists() and src.suffix.lower() == ".zip":
                dest = LEAN_RETURN_DIR / src.name
                shutil.move(str(src), dest)
                print("Moved uploaded LEAN ZIP:", dest)
    except Exception as e:
        print("Upload picker skipped/failed:", e)

lean_zips = sorted(set(list(LEAN_RETURN_DIR.glob("*.zip")) + list(Path("/content").glob("*LEAN*.zip")) + list(Path("/content").glob("*lean*.zip"))))
print("LEAN ZIPs available:", len(lean_zips))
for p in lean_zips:
    print(" -", p)


In [ ]:
# ============================================================
# STAGE 3A FIX — LOAD COMPLETED LEAN ZIPs FROM GOOGLE DRIVE
#
# Use this instead of the notebook cell that says:
# "LEAN packages selected: 0"
#
# It copies completed LEAN ZIPs from Drive into a local runtime folder
# and defines INPUT_PACKAGES for the downstream 3B/3C integration cells.
# ============================================================

from pathlib import Path
from google.colab import drive
import shutil, datetime, json, os

# ----------------------------
# Find/mount Google Drive
# ----------------------------

if Path("/content/gdrive/MyDrive").exists():
    MOUNT = Path("/content/gdrive")
    print("Using existing Drive mount:", MOUNT)

elif Path("/content/drive/MyDrive").exists():
    MOUNT = Path("/content/drive")
    print("Using existing Drive mount:", MOUNT)

else:
    # Prefer /content/gdrive because /content/drive was previously dirty in this runtime.
    MOUNT = Path("/content/gdrive")
    if MOUNT.exists() and any(MOUNT.iterdir()):
        backup = Path(f"/content/gdrive_mountpoint_junk_{datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')}")
        print("Moving non-mounted /content/gdrive contents to:", backup)
        shutil.move(str(MOUNT), str(backup))
    MOUNT.mkdir(parents=True, exist_ok=True)
    drive.mount(str(MOUNT), force_remount=True)

assert (MOUNT / "MyDrive").exists(), "Google Drive mount failed."

# ----------------------------
# Drive completed LEAN ZIP source
# ----------------------------

DRIVE_ROOT = MOUNT / "MyDrive/4-Archives/Pathology_Hub_Intermission"
DRIVE_COMPLETED = DRIVE_ROOT / "completed_lean_zips"
DRIVE_MANIFESTS = DRIVE_ROOT / "manifests"
DRIVE_MANIFESTS.mkdir(parents=True, exist_ok=True)

assert DRIVE_COMPLETED.exists(), f"Completed LEAN ZIP folder not found: {DRIVE_COMPLETED}"

completed_zips = sorted(DRIVE_COMPLETED.glob("*_LEAN_PACKAGE.zip"))

print("Completed LEAN ZIPs in Drive:", len(completed_zips))
for p in completed_zips[:80]:
    print(" -", p.name)

assert completed_zips, "No *_LEAN_PACKAGE.zip files found in completed_lean_zips."

# ----------------------------
# Local runtime copy destination
# ----------------------------

BASE = Path(globals().get("BASE", "pathology_hub"))
BASE.mkdir(parents=True, exist_ok=True)

LEAN_RETURN_DIR = BASE / "00_runtime" / "completed_lean_zips_for_integration"
LEAN_RETURN_DIR.mkdir(parents=True, exist_ok=True)

# Optional: clear old local copies so this reflects Drive exactly.
for old in LEAN_RETURN_DIR.glob("*.zip"):
    old.unlink()

copied = []

for src in completed_zips:
    dst = LEAN_RETURN_DIR / src.name
    shutil.copy2(src, dst)
    copied.append(dst)

# ----------------------------
# Define variables expected by downstream notebook cells
# ----------------------------

INPUT_PACKAGES = sorted(copied)

# Also define common aliases in case later cells use another name.
LEAN_ZIPS = INPUT_PACKAGES
LEAN_PACKAGES = INPUT_PACKAGES
RETURNED_LEAN_ZIPS = INPUT_PACKAGES

globals()["BASE"] = BASE
globals()["LEAN_RETURN_DIR"] = LEAN_RETURN_DIR
globals()["INPUT_PACKAGES"] = INPUT_PACKAGES
globals()["LEAN_ZIPS"] = LEAN_ZIPS
globals()["LEAN_PACKAGES"] = LEAN_PACKAGES
globals()["RETURNED_LEAN_ZIPS"] = RETURNED_LEAN_ZIPS

# ----------------------------
# Audit manifest
# ----------------------------

manifest = {
    "schema_version": "pathology_hub_lean_integration_input_manifest.v1",
    "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "drive_completed_folder": str(DRIVE_COMPLETED),
    "local_lean_return_dir": str(LEAN_RETURN_DIR),
    "input_package_count": len(INPUT_PACKAGES),
    "input_packages": [str(p) for p in INPUT_PACKAGES],
}

manifest_path = DRIVE_MANIFESTS / "lean_integration_input_packages_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("\n✅ LEAN packages staged locally for integration.")
print("Local LEAN return dir:", LEAN_RETURN_DIR)
print("INPUT_PACKAGES:", len(INPUT_PACKAGES))
print("Manifest:", manifest_path)

assert len(INPUT_PACKAGES) == len(completed_zips)

## 3B LEAN integration helpers

In [ ]:
# ============================================================
# STAGE 3B — LEAN PACKAGE INTEGRATION HELPERS
# ============================================================

TAG_FIELD_NAMES = [
    "candidate_tags", "candidate_tag_assignments", "ai_tags", "context_tags", "reviewed_tags",
    "ai_tag_assignments", "tagging_status"
]


def unique_id(base, seen):
    base = str(base)
    candidate = base
    i = 2
    while candidate in seen:
        candidate = f"{base}:{i}"
        i += 1
    seen.add(candidate)
    return candidate


def find_one(root, patterns):
    root = Path(root)
    for pat in patterns:
        hits = sorted(root.rglob(pat))
        if hits:
            return hits[0]
    return None


def discover_lean_packages():
    paths = sorted(set(
        list(LEAN_RETURN_DIR.glob("*.zip")) +
        list(Path("/content").glob("*LEAN*.zip")) +
        list(Path("/content").glob("*lean*.zip"))
    ))
    # Do not accidentally ingest ChatGPT job ZIPs as returned LEAN packages.
    paths = [p for p in paths if "CHATGPT_LEAN_TAGGING_JOB" not in p.name]
    return paths


def infer_source_id_from_package(zip_path, audit, files):
    for key in ["source_id", "book_id"]:
        if isinstance(audit, dict) and audit.get(key):
            return slugify(audit[key])
    for k in ["unified", "pages", "chunks", "figures"]:
        p = files.get(k)
        if p:
            name = Path(p).name
            name = re.sub(r"_(UNIFIED_LEAN|pages_LEAN|chunks_LEAN|figures_LEAN|LEAN_cleaning_audit).*", "", name, flags=re.I)
            return infer_source_id(name)
    return infer_source_id(Path(zip_path).stem)


def infer_source_title(source_id, audit):
    if isinstance(audit, dict):
        for key in ["source_title", "book_title", "title"]:
            if audit.get(key):
                return str(audit[key])
    return source_id.replace("_", " ").title()


def carry_tag_fields(src, dest):
    for k in TAG_FIELD_NAMES:
        if k in src:
            dest[k] = src.get(k)
        elif k.endswith("tags") or k.endswith("assignments"):
            dest[k] = []
        elif k == "tagging_status":
            dest[k] = src.get("tagging_status", "not_attempted")
    return dest


def validate_exact_tags_against_catalog(records, tag_catalog):
    allowed = set(tag_catalog)
    bad = []
    for rec in records:
        for field in ["candidate_tags", "ai_tags", "context_tags", "reviewed_tags"]:
            for tag in rec.get(field) or []:
                if tag not in allowed:
                    bad.append({"record_id": rec.get("chunk_id") or rec.get("page_id") or rec.get("figure_record_id"), "field": field, "tag": tag})
    return bad


## 3C Extract, validate, combine, and preserve LEAN packages

In [ ]:
# ============================================================
# STAGE 3C — EXTRACT / VALIDATE / COMBINE LEAN PACKAGES
# ============================================================

authenticate_gcp_colab()

# Clear only LEAN integration outputs, not Stage 1 raw UNIFIED files.
for d in [STAGED_LEAN_PACKAGE_DIR, LEAN_NORM_DIR, LEAN_INDEX_DIR, LEAN_AUDIT_DIR]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

INPUT_PACKAGES = discover_lean_packages()
print("LEAN packages selected:", len(INPUT_PACKAGES))
for p in INPUT_PACKAGES:
    print(" -", p)
if not INPUT_PACKAGES:
    raise RuntimeError("No returned LEAN ZIP packages found. Upload ChatGPT-returned LEAN ZIPs first.")

EXTRACT_DIR = BASE / "_work" / "textbook_lean_packages"
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

package_infos = []
combined_tag_catalog_records = []

for zip_path in INPUT_PACKAGES:
    zip_path = Path(zip_path)
    staged_pkg = STAGED_LEAN_PACKAGE_DIR / zip_path.name
    shutil.copy2(zip_path, staged_pkg)

    pkg_dir = EXTRACT_DIR / slugify(zip_path.stem)
    pkg_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(pkg_dir)

    files = {
        "unified": find_one(pkg_dir, ["*UNIFIED*_LEAN.json", "*_UNIFIED_LEAN.json"]),
        "pages": find_one(pkg_dir, ["*pages*_LEAN.jsonl", "*_pages_LEAN.jsonl"]),
        "chunks": find_one(pkg_dir, ["*chunks*_LEAN.jsonl", "*_chunks_LEAN.jsonl"]),
        "figures": find_one(pkg_dir, ["*figures*_LEAN.jsonl", "*_figures_LEAN.jsonl"]),
        "sqlite": find_one(pkg_dir, ["*fts*_LEAN.sqlite", "*_fts_LEAN.sqlite", "*.sqlite"]),
        "cleaning_audit": find_one(pkg_dir, ["*LEAN_cleaning_audit.json", "*cleaning_audit*.json", "*audit*.json"]),
        "tag_catalog": find_one(pkg_dir, ["*tag_catalog*.jsonl"]),
        "candidate_tagging_audit": find_one(pkg_dir, ["*candidate_tagging_audit*.json"]),
        "semantic_tagging_audit": find_one(pkg_dir, ["*semantic_tagging_audit*.json"]),
    }

    missing = [k for k in ["unified", "pages", "chunks", "figures", "cleaning_audit"] if not files[k]]
    if missing:
        raise RuntimeError(f"Package missing required LEAN files {missing}: {zip_path}\nFound: {files}")

    audit = read_json(files["cleaning_audit"])
    unified = read_json(files["unified"])
    pages = read_jsonl(files["pages"])
    chunks = read_jsonl(files["chunks"])
    figures = read_jsonl(files["figures"])
    tag_catalog = read_jsonl(files["tag_catalog"]) if files.get("tag_catalog") else []
    combined_tag_catalog_records.extend(tag_catalog)

    source_id = infer_source_id_from_package(zip_path, audit, files)
    source_title = infer_source_title(source_id, audit)

    info = {
        "package_name": zip_path.name,
        "package_path": str(zip_path),
        "staged_package_path": str(staged_pkg),
        "package_sha256": sha256_file(zip_path),
        "extract_dir": str(pkg_dir),
        "source_id": source_id,
        "source_title": source_title,
        "files": {k: str(v) if v else None for k, v in files.items()},
        "audit": audit,
        "counts": {
            "unified_pages": len(unified) if isinstance(unified, list) else None,
            "pages": len(pages),
            "chunks": len(chunks),
            "figures": len(figures),
            "tag_catalog_records": len(tag_catalog),
            "sqlite_present": bool(files.get("sqlite")),
            "candidate_tagging_audit_present": bool(files.get("candidate_tagging_audit")),
            "semantic_tagging_audit_present": bool(files.get("semantic_tagging_audit")),
        },
        "_loaded": {"pages": pages, "chunks": chunks, "figures": figures, "tag_catalog": tag_catalog},
    }
    package_infos.append(info)

print("\nPackage summary:")
print(json.dumps([{k:v for k,v in i.items() if k not in ["audit", "_loaded"]} for i in package_infos], indent=2)[:12000])

# Preserve exact per-source outputs.
per_source_norm_dir = LEAN_NORM_DIR / "per_source"
per_source_index_dir = LEAN_INDEX_DIR / "per_source"
per_source_norm_dir.mkdir(parents=True, exist_ok=True)
per_source_index_dir.mkdir(parents=True, exist_ok=True)

for info in package_infos:
    source_id = info["source_id"]
    files = {k: Path(v) if v else None for k, v in info["files"].items()}
    src_norm = per_source_norm_dir / source_id
    src_index = per_source_index_dir / source_id
    src_norm.mkdir(parents=True, exist_ok=True)
    src_index.mkdir(parents=True, exist_ok=True)
    for k in ["unified", "pages", "chunks", "figures", "cleaning_audit", "tag_catalog", "candidate_tagging_audit", "semantic_tagging_audit"]:
        if files.get(k):
            shutil.copy2(files[k], src_norm / Path(files[k]).name)
    if files.get("sqlite"):
        shutil.copy2(files["sqlite"], src_index / Path(files["sqlite"]).name)

# Build combined records.
textbook_sources, textbook_pages, textbook_chunks, textbook_figures = [], [], [], []
seen_source_ids, seen_page_ids, seen_chunk_ids, seen_figure_record_ids = set(), set(), set(), set()

for info in package_infos:
    source_id = unique_id(info["source_id"], seen_source_ids)
    source_title = info["source_title"]
    pages = info["_loaded"]["pages"]
    chunks = info["_loaded"]["chunks"]
    figures = info["_loaded"]["figures"]

    textbook_sources.append({
        "schema_version": "textbook_source.lean_integrated.v1",
        "source_id": source_id,
        "source_title": source_title,
        "source_family": "textbooks",
        "source_type": "textbook",
        "package_name": info["package_name"],
        "package_sha256": info["package_sha256"],
        "input_files": info["files"],
        "input_audit_summary": info["audit"],
        "counts_input": info["counts"],
        "normalized_at_utc": now_utc(),
    })

    page_fig_counter = defaultdict(int)
    figure_lookup = {}

    for rf in figures:
        page = rf.get("page") or rf.get("source_page")
        try:
            page_i = int(page) if page is not None and str(page).isdigit() else page
        except Exception:
            page_i = page
        page_key = f"p{int(page_i):04d}" if isinstance(page_i, int) else f"p{slugify(page_i)}"
        page_fig_counter[page_key] += 1
        local_idx = rf.get("figure_index") or page_fig_counter[page_key]
        figure_id = rf.get("figure_id") or rf.get("figure_label")
        image_path = rf.get("image_path") or rf.get("path") or rf.get("figure_url")
        fig_rec = {
            "schema_version": "textbook_figure.lean_integrated.v1",
            "figure_record_id": unique_id(f"tbfig:{source_id}:{page_key}:{int(local_idx):03d}", seen_figure_record_ids),
            "source_id": source_id,
            "source_title": source_title,
            "source_family": "textbooks",
            "source_type": "textbook_figure",
            "page": page_i,
            "chapter_number": rf.get("chapter_number"),
            "chapter_title": rf.get("chapter_title"),
            "section_heading": rf.get("section_heading"),
            "figure_id": figure_id,
            "caption": clean_text(rf.get("caption")) or None,
            "image_path": image_path,
            "image_url": gs_to_https(image_path),
            "figure_index": local_idx,
            "package_name": info["package_name"],
        }
        carry_tag_fields(rf, fig_rec)
        textbook_figures.append(fig_rec)
        for key in [(str(page_i), str(figure_id)), (str(page_i), str(image_path)), (str(page_i), str(local_idx))]:
            figure_lookup[key] = fig_rec["figure_record_id"]

    for rp in pages:
        page = rp.get("page") or rp.get("page_start")
        try:
            page_i = int(page) if page is not None and str(page).isdigit() else page
        except Exception:
            page_i = page
        page_key = f"p{int(page_i):04d}" if isinstance(page_i, int) else f"p{slugify(page_i)}"
        raw_fig_ids = rp.get("figure_ids") or []
        figure_record_ids = []
        for fig in raw_fig_ids:
            fid = figure_lookup.get((str(page_i), str(fig)))
            if fid:
                figure_record_ids.append(fid)
        page_rec = {
            "schema_version": "textbook_page.lean_integrated.v1",
            "page_id": unique_id(f"tbpage:{source_id}:{page_key}", seen_page_ids),
            "source_id": source_id,
            "source_title": source_title,
            "source_family": "textbooks",
            "source_type": "textbook_page",
            "page": page_i,
            "chapter_number": rp.get("chapter_number"),
            "chapter_title": rp.get("chapter_title"),
            "section_heading": rp.get("section_heading"),
            "content": clean_text(rp.get("content") or rp.get("text")),
            "figure_ids": raw_fig_ids,
            "figure_record_ids": sorted(set(figure_record_ids)),
            "package_name": info["package_name"],
        }
        carry_tag_fields(rp, page_rec)
        textbook_pages.append(page_rec)

    for rc in chunks:
        page = rc.get("page") or rc.get("page_start")
        try:
            page_i = int(page) if page is not None and str(page).isdigit() else page
        except Exception:
            page_i = page
        legacy_chunk_id = rc.get("chunk_id")
        base = f"tbchunk:{source_id}:{slugify(legacy_chunk_id, max_len=160)}" if legacy_chunk_id else f"tbchunk:{source_id}:{len(textbook_chunks)+1:06d}"
        figure_id = rc.get("figure_id")
        image_path = rc.get("image_path") or rc.get("path")
        figure_record_id = None
        for key in [(str(page_i), str(figure_id)), (str(page_i), str(image_path))]:
            if key in figure_lookup:
                figure_record_id = figure_lookup[key]
                break
        chunk_rec = {
            "schema_version": "textbook_chunk.lean_integrated.v1",
            "chunk_id": unique_id(base, seen_chunk_ids),
            "source_id": source_id,
            "source_title": source_title,
            "source_family": "textbooks",
            "source_type": "textbook_chunk",
            "chunk_type": rc.get("chunk_type"),
            "page": page_i,
            "chapter_number": rc.get("chapter_number"),
            "chapter_title": rc.get("chapter_title"),
            "section_heading": rc.get("section_heading"),
            "figure_id": figure_id,
            "figure_record_id": figure_record_id,
            "figure_record_ids": [figure_record_id] if figure_record_id else [],
            "image_path": image_path,
            "image_url": gs_to_https(image_path),
            "text": clean_text(rc.get("text") or rc.get("chunk_text") or rc.get("content")),
            "package_name": info["package_name"],
        }
        carry_tag_fields(rc, chunk_rec)
        textbook_chunks.append(chunk_rec)

# Write combined artifacts.
sources_path = LEAN_NORM_DIR / "textbook_lean_sources.jsonl"
pages_path = LEAN_NORM_DIR / "textbook_lean_pages.jsonl"
chunks_path = LEAN_NORM_DIR / "textbook_lean_chunks.jsonl"
figures_path = LEAN_NORM_DIR / "textbook_lean_figures.jsonl"
tag_catalog_path = LEAN_NORM_DIR / "textbook_lean_tag_catalog.jsonl"

save_jsonl(textbook_sources, sources_path)
save_jsonl(textbook_pages, pages_path)
save_jsonl(textbook_chunks, chunks_path)
save_jsonl(textbook_figures, figures_path)
if combined_tag_catalog_records:
    save_jsonl(combined_tag_catalog_records, tag_catalog_path)

print("Combined records:")
print(" sources", len(textbook_sources), sources_path)
print(" pages", len(textbook_pages), pages_path)
print(" chunks", len(textbook_chunks), chunks_path)
print(" figures", len(textbook_figures), figures_path)
print(" tag_catalog", len(combined_tag_catalog_records), tag_catalog_path if combined_tag_catalog_records else "none")


## 3D Build consolidated SQLite FTS index from LEAN chunks

In [ ]:
# ============================================================
# STAGE 3D — BUILD SQLITE FTS OVER LEAN CHUNKS
# ============================================================

textbook_fts_path = LEAN_INDEX_DIR / "textbook_lean_fts.sqlite"
if textbook_fts_path.exists():
    textbook_fts_path.unlink()

con = sqlite3.connect(textbook_fts_path)
cur = con.cursor()
cur.execute("""
CREATE TABLE textbook_chunks (
    chunk_id TEXT PRIMARY KEY,
    source_id TEXT,
    source_title TEXT,
    chunk_type TEXT,
    page INTEGER,
    chapter_number TEXT,
    chapter_title TEXT,
    section_heading TEXT,
    figure_id TEXT,
    figure_record_id TEXT,
    image_path TEXT,
    image_url TEXT,
    text TEXT,
    candidate_tags TEXT,
    candidate_tag_assignments TEXT,
    ai_tags TEXT,
    context_tags TEXT,
    reviewed_tags TEXT,
    ai_tag_assignments TEXT,
    tagging_status TEXT,
    json TEXT
)
""")
cur.execute("""
CREATE VIRTUAL TABLE textbook_chunks_fts
USING fts5(text, source_title, chapter_title, section_heading, content='textbook_chunks', content_rowid='rowid')
""")

for rec in textbook_chunks:
    cur.execute("""
    INSERT INTO textbook_chunks
    (chunk_id, source_id, source_title, chunk_type, page, chapter_number, chapter_title,
     section_heading, figure_id, figure_record_id, image_path, image_url, text,
     candidate_tags, candidate_tag_assignments, ai_tags, context_tags, reviewed_tags,
     ai_tag_assignments, tagging_status, json)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        rec.get("chunk_id"), rec.get("source_id"), rec.get("source_title"), rec.get("chunk_type"), rec.get("page"),
        rec.get("chapter_number"), rec.get("chapter_title"), rec.get("section_heading"), rec.get("figure_id"),
        rec.get("figure_record_id"), rec.get("image_path"), rec.get("image_url"), rec.get("text"),
        json.dumps(rec.get("candidate_tags") or [], ensure_ascii=False),
        json.dumps(rec.get("candidate_tag_assignments") or [], ensure_ascii=False),
        json.dumps(rec.get("ai_tags") or [], ensure_ascii=False),
        json.dumps(rec.get("context_tags") or [], ensure_ascii=False),
        json.dumps(rec.get("reviewed_tags") or [], ensure_ascii=False),
        json.dumps(rec.get("ai_tag_assignments") or [], ensure_ascii=False),
        rec.get("tagging_status"),
        json.dumps(rec, ensure_ascii=False),
    ))

cur.execute("""
INSERT INTO textbook_chunks_fts(rowid, text, source_title, chapter_title, section_heading)
SELECT rowid, text, source_title, chapter_title, section_heading FROM textbook_chunks
""")
con.commit()
con.close()

print("Textbook LEAN FTS:", textbook_fts_path, textbook_fts_path.stat().st_size, "bytes")


def search_textbook_lean_fts(query, limit=5, source_id=None):
    con = sqlite3.connect(textbook_fts_path)
    con.row_factory = sqlite3.Row
    try:
        tokens = re.findall(r"[A-Za-z0-9]+", query)
        tokens = [t for t in tokens if len(t) > 1][:12]
        fts_query = " AND ".join(tokens) if tokens else query
        params = [fts_query]
        where = ""
        if source_id:
            where = "AND c.source_id = ?"
            params.append(source_id)
        params.append(limit)
        sql = f"""
        SELECT c.*, bm25(textbook_chunks_fts) AS score
        FROM textbook_chunks_fts
        JOIN textbook_chunks c ON c.rowid = textbook_chunks_fts.rowid
        WHERE textbook_chunks_fts MATCH ?
        {where}
        ORDER BY score
        LIMIT ?
        """
        rows = con.execute(sql, params).fetchall()
        if not rows and tokens:
            fts_query = " OR ".join(tokens)
            params = [fts_query]
            if source_id:
                params.append(source_id)
            params.append(limit)
            rows = con.execute(sql, params).fetchall()
        return [dict(r) for r in rows]
    finally:
        con.close()

SMOKE_TEST_QUERIES = ["squamous carcinoma", "immunohistochemistry", "molecular mutation"]
smoke_results = {}
for q in SMOKE_TEST_QUERIES:
    hits = search_textbook_lean_fts(q, limit=3)
    smoke_results[q] = len(hits)
    print("\nQUERY:", q, "hits:", len(hits))
    for h in hits[:3]:
        print(" -", h.get("source_id"), "p.", h.get("page"), h.get("chunk_type"), "::", (h.get("text") or "")[:250].replace("\n", " "))


## 3E Audit, manifest, and upload LEAN outputs

In [ ]:
# ============================================================
# STAGE 3E — AUDIT + MANIFEST + GCS SYNC
# ============================================================

source_counts = defaultdict(lambda: {"pages": 0, "chunks": 0, "figures": 0})
chunk_type_counts = Counter()
tagging_status_counts = Counter()

for r in textbook_pages:
    source_counts[r["source_id"]]["pages"] += 1
    tagging_status_counts[r.get("tagging_status") or "missing"] += 1
for r in textbook_chunks:
    source_counts[r["source_id"]]["chunks"] += 1
    chunk_type_counts[r.get("chunk_type") or "unknown"] += 1
    tagging_status_counts[r.get("tagging_status") or "missing"] += 1
for r in textbook_figures:
    source_counts[r["source_id"]]["figures"] += 1
    tagging_status_counts[r.get("tagging_status") or "missing"] += 1

legend_field_count = sum(1 for r in textbook_figures if "legend" in r)
reference_heading_hits = [r.get("chunk_id") for r in textbook_chunks if re.search(r"(^|\n)\s*(References|Bibliography|Suggested Reading|Further Reading|Selected References)\s*($|\n)", r.get("text") or "", flags=re.I)]
caption_start_hits_in_pages = [r.get("page_id") for r in textbook_pages if re.search(r"(^|\n)\s*(Fig\.|Figure|FIGURE)\s+\d", r.get("content") or "")]

# Validate tags if a tag catalog exists.
tag_catalog_strings = set()
for rec in combined_tag_catalog_records:
    if isinstance(rec, dict) and rec.get("tag"):
        tag_catalog_strings.add(rec["tag"])
    elif isinstance(rec, str):
        tag_catalog_strings.add(rec)
invalid_tag_records = []
if tag_catalog_strings:
    invalid_tag_records = validate_exact_tags_against_catalog(textbook_pages + textbook_chunks + textbook_figures, tag_catalog_strings)

audit = {
    "schema_version": "textbook_lean_integration_audit.v1",
    "created_at_utc": now_utc(),
    "workstream": "Textbook + Tag RAG / Source Normalization",
    "purpose": "Integrate ChatGPT-cleaner LEAN textbook packages and build consolidated SQLite FTS index.",
    "stage1_raw_unified_staged_count": len(list(STAGED_UNIFIED_DIR.glob("*_UNIFIED.json"))),
    "input_packages": [
        {k: v for k, v in info.items() if k not in ["audit", "_loaded"]}
        for info in package_infos
    ],
    "record_counts": {
        "sources": len(textbook_sources),
        "pages": len(textbook_pages),
        "chunks": len(textbook_chunks),
        "figures": len(textbook_figures),
        "tag_catalog_records": len(combined_tag_catalog_records),
    },
    "chunk_type_counts": dict(chunk_type_counts),
    "tagging_status_counts": dict(tagging_status_counts),
    "source_counts": dict(source_counts),
    "validations": {
        "lean_package_json_parse_pass": True,
        "lean_package_jsonl_parse_pass": True,
        "chunk_id_unique": len(textbook_chunks) == len({r["chunk_id"] for r in textbook_chunks}),
        "page_id_unique": len(textbook_pages) == len({r["page_id"] for r in textbook_pages}),
        "figure_record_id_unique": len(textbook_figures) == len({r["figure_record_id"] for r in textbook_figures}),
        "sqlite_fts_created": textbook_fts_path.exists(),
        "fts_smoke_tests": smoke_results,
        "no_legend_field_in_combined_figures": legend_field_count == 0,
        "reference_heading_hits_in_combined_chunks": len(reference_heading_hits),
        "caption_start_hits_in_combined_pages": len(caption_start_hits_in_pages),
        "controlled_tag_catalog_present": bool(tag_catalog_strings),
        "invalid_tags_outside_catalog_count": len(invalid_tag_records),
        "invalid_tags_examples": invalid_tag_records[:25],
    },
    "search_mode": "sqlite_fts5",
    "vectorized": False,
    "api_exposed": False,
    "known_limitations": [
        "Raw UNIFIED files are staged only and are not canonical normalized/indexed records.",
        "LEAN cleaning and tag assignment are performed in ChatGPT, not in Colab.",
        "SQLite FTS is keyword search only; no vector index was built.",
        "No image pixel interpretation was performed.",
        "AI tags, if present, are unreviewed metadata unless separately human-reviewed."
    ],
}

audit_path = LEAN_AUDIT_DIR / "textbook_lean_integration_audit.json"
save_json(audit, audit_path)

manifest = {
    "schema_version": "textbook_lean_index_manifest.v1",
    "created_at_utc": now_utc(),
    "corpus": "textbooks_lean",
    "index_type": "sqlite_fts5",
    "searchable": True,
    "vectorized": False,
    "api_exposed": False,
    "input_artifacts": {
        "lean_packages": [str(p) for p in INPUT_PACKAGES],
        "staged_raw_unified_dir": str(STAGED_UNIFIED_DIR),
    },
    "output_artifacts": {
        "sources_jsonl": str(sources_path),
        "pages_jsonl": str(pages_path),
        "chunks_jsonl": str(chunks_path),
        "figures_jsonl": str(figures_path),
        "tag_catalog_jsonl": str(tag_catalog_path) if combined_tag_catalog_records else None,
        "sqlite_fts": str(textbook_fts_path),
        "audit": str(audit_path),
    },
    "gcs_paths": {
        "normalized": f"gs://{DEST_BUCKET}/02_normalized/textbooks/lean/",
        "index": f"gs://{DEST_BUCKET}/03_indexes/textbooks/lean/",
        "audit": f"gs://{DEST_BUCKET}/06_audits/textbooks/lean/",
    },
    "record_counts": audit["record_counts"],
    "validations": audit["validations"],
}
manifest_path = LEAN_INDEX_DIR / "textbook_lean_index_manifest.json"
save_json(manifest, manifest_path)

print("Audit:", audit_path)
print("Manifest:", manifest_path)
print(json.dumps(audit["record_counts"], indent=2))

if SYNC_LEAN_OUTPUTS_TO_GCS:
    print("\nUploading final LEAN artifacts to canonical GCS...")
    sync_jobs = []
    sync_jobs.append(sync_dir_to_gcs(STAGED_LEAN_PACKAGE_DIR, f"gs://{DEST_BUCKET}/01_staged/textbooks/lean_packages/"))
    sync_jobs.append(sync_dir_to_gcs(LEAN_NORM_DIR, f"gs://{DEST_BUCKET}/02_normalized/textbooks/lean/"))
    sync_jobs.append(sync_dir_to_gcs(LEAN_INDEX_DIR, f"gs://{DEST_BUCKET}/03_indexes/textbooks/lean/"))
    sync_jobs.append(sync_dir_to_gcs(LEAN_AUDIT_DIR, f"gs://{DEST_BUCKET}/06_audits/textbooks/lean/"))
    save_json(sync_jobs, LEAN_AUDIT_DIR / "stage3_gcs_sync_status.json")
    print("\nFinal paths:")
    print("LEAN packages staged:", f"gs://{DEST_BUCKET}/01_staged/textbooks/lean_packages/")
    print("LEAN normalized:", f"gs://{DEST_BUCKET}/02_normalized/textbooks/lean/")
    print("LEAN index:", f"gs://{DEST_BUCKET}/03_indexes/textbooks/lean/")
    print("LEAN audits:", f"gs://{DEST_BUCKET}/06_audits/textbooks/lean/")


## 3F Verify GCS outputs

In [ ]:
# ============================================================
# STAGE 3F — VERIFY GCS OUTPUTS
# ============================================================

print("=== Raw UNIFIED staged only ===")
!gcloud storage ls gs://{DEST_BUCKET}/01_staged/textbooks/unified_raw/ | head -50

print("\n=== ChatGPT job packages ===")
!gcloud storage ls gs://{DEST_BUCKET}/01_staged/textbooks/chatgpt_jobs/ | head -50

print("\n=== LEAN packages staged ===")
!gcloud storage ls gs://{DEST_BUCKET}/01_staged/textbooks/lean_packages/ | head -50

print("\n=== LEAN normalized artifacts ===")
!gcloud storage ls gs://{DEST_BUCKET}/02_normalized/textbooks/lean/ | head -50

print("\n=== LEAN index artifacts ===")
!gcloud storage ls gs://{DEST_BUCKET}/03_indexes/textbooks/lean/ | head -50

print("\n=== LEAN audit ===")
!gcloud storage cat gs://{DEST_BUCKET}/06_audits/textbooks/lean/textbook_lean_integration_audit.json | head -100


In [ ]:
# ============================================================
# FIX DEPLOY: USE ARTIFACT REGISTRY INSTEAD OF gcr.io
#
# Previous failure was only image push to old gcr.io.
# This cell:
#   1. Creates/uses Artifact Registry Docker repo
#   2. Builds image to us-central1-docker.pkg.dev
#   3. Deploys pathology-hub-v04
#   4. Smoke tests textbooks + combined search
#   5. Writes updated OpenAPI YAML
# ============================================================

from pathlib import Path
import subprocess, json, requests, shutil, datetime

PROJECT_ID = "pathology-annotation-project"
REGION = "us-central1"
SERVICE_NAME = "pathology-hub-v04"
AR_REPO = "pathology-hub"
IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/{SERVICE_NAME}:latest"

APP_DIR = Path("/content/pathology_hub_v04_textbook_api")
assert APP_DIR.exists(), f"Missing app dir: {APP_DIR}. Rerun the app-generation cell first."
assert (APP_DIR / "app.py").exists(), "Missing app.py. Rerun the app-generation cell first."
assert (APP_DIR / "Dockerfile").exists(), "Missing Dockerfile. Rerun the app-generation cell first."

TEXTBOOK_SQLITE_GCS = "gs://pathology_hub/03_indexes/textbooks/lean/textbook_lean_fts.sqlite"
TEXTBOOK_MANIFEST_GCS = "gs://pathology_hub/03_indexes/textbooks/lean/textbook_lean_index_manifest.json"
UPSTREAM_EVIDENCE_URL = "https://pathology-hub-830130787988.us-central1.run.app/evidence/search"

def run(cmd, cwd=None, check=True):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout:
        print(p.stdout[-10000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

# ------------------------------------------------------------
# Project + APIs
# ------------------------------------------------------------

run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

for api in [
    "artifactregistry.googleapis.com",
    "cloudbuild.googleapis.com",
    "run.googleapis.com",
    "secretmanager.googleapis.com",
    "storage.googleapis.com",
]:
    run(["gcloud", "services", "enable", api], check=True)

# ------------------------------------------------------------
# Create Artifact Registry Docker repo if missing
# ------------------------------------------------------------

print("\nChecking Artifact Registry repo...")
check = run([
    "gcloud", "artifacts", "repositories", "describe", AR_REPO,
    "--location", REGION,
], check=False)

if check.returncode != 0:
    print(f"\nCreating Artifact Registry repo: {AR_REPO}")
    run([
        "gcloud", "artifacts", "repositories", "create", AR_REPO,
        "--repository-format", "docker",
        "--location", REGION,
        "--description", "Pathology Hub Docker images",
    ], check=True)
else:
    print("Artifact Registry repo already exists.")

# Docker auth for Artifact Registry
run(["gcloud", "auth", "configure-docker", f"{REGION}-docker.pkg.dev", "--quiet"], check=True)

# ------------------------------------------------------------
# Confirm textbook artifacts
# ------------------------------------------------------------

print("\nChecking textbook artifacts in GCS...")
run(["gcloud", "storage", "ls", TEXTBOOK_SQLITE_GCS], check=True)
run(["gcloud", "storage", "ls", TEXTBOOK_MANIFEST_GCS], check=True)

# ------------------------------------------------------------
# Build to Artifact Registry
# ------------------------------------------------------------

print("\nBuilding container to Artifact Registry...")
run(["gcloud", "builds", "submit", "--tag", IMAGE, "."], cwd=APP_DIR, check=True)

# ------------------------------------------------------------
# Deploy Cloud Run
# ------------------------------------------------------------

print("\nDeploying Cloud Run service...")
deploy_cmd = [
    "gcloud", "run", "deploy", SERVICE_NAME,
    "--image", IMAGE,
    "--region", REGION,
    "--platform", "managed",
    "--allow-unauthenticated",
    "--memory", "4Gi",
    "--cpu", "2",
    "--timeout", "300",
    "--min-instances", "1",
    "--set-env-vars",
    f"TEXTBOOK_SQLITE_GCS={TEXTBOOK_SQLITE_GCS},TEXTBOOK_MANIFEST_GCS={TEXTBOOK_MANIFEST_GCS},UPSTREAM_EVIDENCE_URL={UPSTREAM_EVIDENCE_URL}",
    "--set-secrets", "PATHOLOGY_HUB_API_KEY=pathology-hub-api-key:latest",
    "--quiet",
]
run(deploy_cmd, check=True)

service_url = subprocess.check_output([
    "gcloud", "run", "services", "describe", SERVICE_NAME,
    "--region", REGION,
    "--format", "value(status.url)"
], text=True).strip()

print("\nSERVICE URL:", service_url)

# ------------------------------------------------------------
# Smoke tests
# ------------------------------------------------------------

print("\nGetting API key from Secret Manager for smoke test...")
api_key = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", "pathology-hub-api-key"
], text=True).strip()

print("\nHealth check...")
r = requests.get(f"{service_url}/health", timeout=300)
print("health status:", r.status_code)
print(json.dumps(r.json(), indent=2)[:5000])
assert r.status_code == 200

print("\nTextbook-only smoke test...")
payload = {
    "query": "STIC p53 Ki67 diagnostic criteria",
    "sources": ["textbooks"],
    "max_results": 1,
    "include_figures": False,
    "max_figures": 0,
    "compact": True,
    "excerpt_char_limit": 900,
}
r = requests.post(
    f"{service_url}/evidence/search",
    headers={"X-API-Key": api_key, "Content-Type": "application/json"},
    json=payload,
    timeout=300,
)
print("search status:", r.status_code)
resp = r.json()
print(json.dumps(resp, indent=2)[:7000])
assert r.status_code == 200
assert "textbook_results" in resp

print("\nCombined smoke test: textbooks + pathout")
payload2 = {
    "query": "bladder CIS p53 CK20",
    "sources": ["textbooks", "pathout"],
    "max_results": 1,
    "include_figures": False,
    "max_figures": 0,
    "compact": True,
    "excerpt_char_limit": 700,
}
r2 = requests.post(
    f"{service_url}/evidence/search",
    headers={"X-API-Key": api_key, "Content-Type": "application/json"},
    json=payload2,
    timeout=300,
)
print("combined status:", r2.status_code)
print(json.dumps(r2.json(), indent=2)[:7000])
assert r2.status_code == 200

# ------------------------------------------------------------
# Write updated GPT Action OpenAPI YAML
# ------------------------------------------------------------

openapi_yaml = f"""openapi: 3.1.0
info:
  title: Pathology Hub Unified Evidence API
  version: 1.5.0
  description: Unified evidence search for Pathology Hub with Journal, PathOut, and Textbook SQLite FTS support.

servers:
  - url: {service_url}

components:
  securitySchemes:
    ApiKeyAuth:
      type: apiKey
      in: header
      name: X-API-Key

  schemas:
    EvidenceSearchRequest:
      type: object
      required:
        - query
      properties:
        query:
          type: string
          description: Short keyword-style pathology evidence query.
        sources:
          type: array
          description: Sources to search.
          items:
            type: string
            enum:
              - journals
              - pathout
              - textbooks
          default:
            - journals
        max_results:
          type: integer
          minimum: 1
          maximum: 10
          default: 1
        include_figures:
          type: boolean
          default: false
        max_figures:
          type: integer
          minimum: 0
          maximum: 10
          default: 0
        compact:
          type: boolean
          default: true
        excerpt_char_limit:
          type: integer
          minimum: 200
          maximum: 4000
          default: 900

    EvidenceItem:
      type: object
      properties:
        title:
          type: string
        journal:
          type: string
        doi:
          type: string
        source_url:
          type: string
        url:
          type: string
        excerpt:
          type: string
        text:
          type: string
        chunk_text:
          type: string
        source_name:
          type: string
        source_type:
          type: string
        source_id:
          type: string
        chunk_id:
          type: string
        chunk_type:
          type: string
        page:
          type: integer
        chapter_title:
          type: string
        section:
          type: string
        section_heading:
          type: string
        figure_id:
          type: string
        image_path:
          type: string
        candidate_tags:
          type: array
          items:
            type: string
        ai_tags:
          type: array
          items:
            type: string
        tagging_status:
          type: string
      additionalProperties: true

    FigureItem:
      type: object
      properties:
        title:
          type: string
        caption:
          type: string
        figure_id:
          type: string
        figure_url:
          type: string
        source_url:
          type: string
        url:
          type: string
        source_name:
          type: string
        page:
          type: integer
      additionalProperties: true

    SourceStatus:
      type: object
      properties:
        journals:
          type: string
        pathout:
          type: string
        textbooks:
          type: string
      additionalProperties: true

    EvidenceSearchResponse:
      type: object
      properties:
        schema_version:
          type: string
        query:
          type: string
        source_status:
          $ref: "#/components/schemas/SourceStatus"
        journal_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        pathout_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        textbook_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        figures:
          type: array
          items:
            $ref: "#/components/schemas/FigureItem"
        warnings:
          type: array
          items:
            type: string
      additionalProperties: true

    ErrorResponse:
      type: object
      properties:
        error:
          type: string
        detail:
          type: string
      additionalProperties: true

paths:
  /evidence/search:
    post:
      operationId: searchEvidence
      summary: Search Pathology Hub evidence.
      description: Search Journal RAG, Pathology Outlines, and/or Textbook SQLite FTS evidence.
      security:
        - ApiKeyAuth: []
      x-openai-isConsequential: false
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: "#/components/schemas/EvidenceSearchRequest"
      responses:
        "200":
          description: Evidence search results.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/EvidenceSearchResponse"
        "400":
          description: Bad request.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/ErrorResponse"
        "401":
          description: Unauthorized.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/ErrorResponse"
        "500":
          description: Server error.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/ErrorResponse"
"""

yaml_path = APP_DIR / "openapi_pathology_hub_unified_searchEvidence_textbooks_v1_5_0.yaml"
yaml_path.write_text(openapi_yaml, encoding="utf-8")

handoff = {
    "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "workstream": "Backend API / Textbook RAG integration",
    "service": SERVICE_NAME,
    "service_url": service_url,
    "image": IMAGE,
    "textbook_sqlite_gcs": TEXTBOOK_SQLITE_GCS,
    "textbook_manifest_gcs": TEXTBOOK_MANIFEST_GCS,
    "search_mode": "SQLite FTS textbook search plus upstream proxy",
    "vectorized": False,
    "api_exposed": True,
    "openapi_yaml": str(yaml_path),
}
handoff_path = APP_DIR / "HANDOFF_PATHOLOGY_HUB_V04_TEXTBOOK_API.json"
handoff_path.write_text(json.dumps(handoff, indent=2), encoding="utf-8")

print("\n✅ DEPLOYED v04 TEXTBOOK API")
print("Service:", service_url)
print("Image:", IMAGE)
print("OpenAPI YAML:", yaml_path)
print("Handoff:", handoff_path)
print("\nNext: in GPT Builder, replace the Action schema with the generated YAML and keep auth header X-API-Key using pathology-hub-api-key.")

In [ ]:
# ============================================================
# FIX BROKEN WHO PATCH + REDEPLOY v04.1
#
# Restores app.py from app_pre_who_patch_* backup, patches cleanly,
# deploys v04.1 with WHO passthrough + textbooks, smoke-tests both,
# and writes OpenAPI YAML with sources: who, journals, pathout, textbooks.
# ============================================================

from pathlib import Path
import subprocess, json, requests, shutil, datetime, re

PROJECT_ID = "pathology-annotation-project"
REGION = "us-central1"
SERVICE_NAME = "pathology-hub-v04"
AR_REPO = "pathology-hub"
IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/{SERVICE_NAME}:latest"

APP_DIR = Path("/content/pathology_hub_v04_textbook_api")
APP_PATH = APP_DIR / "app.py"
assert APP_PATH.exists(), f"Missing app.py: {APP_PATH}"

UPSTREAM_EVIDENCE_URL = "https://pathology-hub-830130787988.us-central1.run.app/evidence/search"
TEXTBOOK_SQLITE_GCS = "gs://pathology_hub/03_indexes/textbooks/lean/textbook_lean_fts.sqlite"
TEXTBOOK_MANIFEST_GCS = "gs://pathology_hub/03_indexes/textbooks/lean/textbook_lean_index_manifest.json"

def run(cmd, cwd=None, check=True):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout:
        print(p.stdout[-10000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

def replace_once(text, old, new, label):
    if old not in text:
        raise RuntimeError(f"Patch target not found for: {label}")
    return text.replace(old, new, 1)

run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

# ------------------------------------------------------------
# 1. Restore from latest pre-WHO backup
# ------------------------------------------------------------

backups = sorted(APP_DIR.glob("app_pre_who_patch_*.py"))
assert backups, "No app_pre_who_patch_*.py backup found. Need to regenerate app.py from the v04 app cell."
backup = backups[-1]
print("Restoring app.py from:", backup)
shutil.copy2(backup, APP_PATH)

app = APP_PATH.read_text(encoding="utf-8")

# ------------------------------------------------------------
# 2. Clean WHO passthrough patch
# ------------------------------------------------------------

app = app.replace(
    'APP_VERSION = "1.5.0-textbooks-v04"',
    'APP_VERSION = "1.5.1-textbooks-who-v04"'
)

app = replace_once(
    app,
    'allowed = {"journals", "pathout", "textbooks"}',
    'allowed = {"journals", "pathout", "textbooks", "who"}',
    "allowed sources"
)

app = replace_once(
    app,
'''        "source_status": {
            "journals": "not_requested",
            "pathout": "not_requested",
            "textbooks": "not_requested",
        },''',
'''        "source_status": {
            "journals": "not_requested",
            "pathout": "not_requested",
            "textbooks": "not_requested",
            "who": "not_requested",
        },''',
    "source_status who"
)

app = replace_once(
    app,
'''        "journal_results": [],
        "pathout_results": [],
        "textbook_results": [],''',
'''        "journal_results": [],
        "pathout_results": [],
        "who_results": [],
        "textbook_results": [],''',
    "who_results default"
)

app = replace_once(
    app,
    'upstream_sources = [s for s in sources if s in {"journals", "pathout"}]',
    'upstream_sources = [s for s in sources if s in {"journals", "pathout", "who"}]',
    "upstream sources who"
)

app = replace_once(
    app,
'''                    response["journal_results"] = u.get("journal_results", [])
                    response["pathout_results"] = u.get("pathout_results", [])
                    response["figures"].extend(u.get("figures", []) or [])''',
'''                    response["journal_results"] = u.get("journal_results", [])
                    response["pathout_results"] = u.get("pathout_results", [])
                    response["who_results"] = u.get("who_results", [])
                    response["figures"].extend(u.get("figures", []) or [])''',
    "copy upstream who_results"
)

# Leave warning block alone; avoids syntax issues.
APP_PATH.write_text(app, encoding="utf-8")

print("Patched app.py cleanly.")

# Syntax check
run(["python", "-m", "py_compile", str(APP_PATH)], check=True)

# ------------------------------------------------------------
# 3. Build + redeploy
# ------------------------------------------------------------

for api in [
    "artifactregistry.googleapis.com",
    "cloudbuild.googleapis.com",
    "run.googleapis.com",
    "secretmanager.googleapis.com",
    "storage.googleapis.com",
]:
    run(["gcloud", "services", "enable", api], check=True)

run(["gcloud", "auth", "configure-docker", f"{REGION}-docker.pkg.dev", "--quiet"], check=True)

print("\nChecking GCS textbook artifacts...")
run(["gcloud", "storage", "ls", TEXTBOOK_SQLITE_GCS], check=True)
run(["gcloud", "storage", "ls", TEXTBOOK_MANIFEST_GCS], check=True)

print("\nBuilding v04.1 image...")
run(["gcloud", "builds", "submit", "--tag", IMAGE, "."], cwd=APP_DIR, check=True)

print("\nDeploying v04.1...")
run([
    "gcloud", "run", "deploy", SERVICE_NAME,
    "--image", IMAGE,
    "--region", REGION,
    "--platform", "managed",
    "--allow-unauthenticated",
    "--memory", "4Gi",
    "--cpu", "2",
    "--timeout", "300",
    "--min-instances", "1",
    "--set-env-vars",
    f"TEXTBOOK_SQLITE_GCS={TEXTBOOK_SQLITE_GCS},TEXTBOOK_MANIFEST_GCS={TEXTBOOK_MANIFEST_GCS},UPSTREAM_EVIDENCE_URL={UPSTREAM_EVIDENCE_URL}",
    "--set-secrets", "PATHOLOGY_HUB_API_KEY=pathology-hub-api-key:latest",
    "--quiet",
], check=True)

service_url = subprocess.check_output([
    "gcloud", "run", "services", "describe", SERVICE_NAME,
    "--region", REGION,
    "--format", "value(status.url)"
], text=True).strip()

print("\nSERVICE URL:", service_url)

# ------------------------------------------------------------
# 4. Smoke tests
# ------------------------------------------------------------

api_key = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", "pathology-hub-api-key"
], text=True).strip()

print("\nHealth check...")
rh = requests.get(f"{service_url}/health", timeout=300)
print("health:", rh.status_code)
print(json.dumps(rh.json(), indent=2)[:3000])
assert rh.status_code == 200

who_payload = {
    "query": "cribriform morular thyroid carcinoma",
    "sources": ["who"],
    "max_results": 1,
    "include_figures": False,
    "max_figures": 0,
    "compact": True,
    "excerpt_char_limit": 900,
}

print("\nWHO smoke test via v04.1...")
rw = requests.post(
    f"{service_url}/evidence/search",
    headers={"X-API-Key": api_key, "Content-Type": "application/json"},
    json=who_payload,
    timeout=300,
)
print("who status:", rw.status_code)
who_resp = rw.json()
print(json.dumps(who_resp, indent=2)[:7000])
assert rw.status_code == 200
assert who_resp.get("source_status", {}).get("who") == "ok"
assert len(who_resp.get("who_results", [])) >= 1

print("\nTextbook smoke test via v04.1...")
rt = requests.post(
    f"{service_url}/evidence/search",
    headers={"X-API-Key": api_key, "Content-Type": "application/json"},
    json={
        "query": "STIC p53 Ki67 diagnostic criteria",
        "sources": ["textbooks"],
        "max_results": 1,
        "include_figures": False,
        "max_figures": 0,
        "compact": True,
        "excerpt_char_limit": 900,
    },
    timeout=300,
)
print("textbook status:", rt.status_code)
print(json.dumps(rt.json(), indent=2)[:5000])
assert rt.status_code == 200
assert rt.json().get("source_status", {}).get("textbooks") == "ok"

print("\nCombined WHO + textbooks smoke test...")
rc = requests.post(
    f"{service_url}/evidence/search",
    headers={"X-API-Key": api_key, "Content-Type": "application/json"},
    json={
        "query": "STIC p53 Ki67 diagnostic criteria",
        "sources": ["who", "textbooks"],
        "max_results": 1,
        "include_figures": False,
        "max_figures": 0,
        "compact": True,
        "excerpt_char_limit": 700,
    },
    timeout=300,
)
print("combined status:", rc.status_code)
print(json.dumps(rc.json(), indent=2)[:7000])
assert rc.status_code == 200

# ------------------------------------------------------------
# 5. Write OpenAPI YAML with WHO included
# ------------------------------------------------------------

openapi_yaml = f"""openapi: 3.1.0
info:
  title: Pathology Hub Unified Evidence API
  version: 1.5.1
  description: Unified evidence search for Pathology Hub with WHO passthrough, Journal, PathOut, and Textbook SQLite FTS support.

servers:
  - url: {service_url}

components:
  securitySchemes:
    ApiKeyAuth:
      type: apiKey
      in: header
      name: X-API-Key

  schemas:
    EvidenceSearchRequest:
      type: object
      required:
        - query
      properties:
        query:
          type: string
          description: Short keyword-style pathology evidence query.
        sources:
          type: array
          description: Sources to search.
          items:
            type: string
            enum:
              - who
              - journals
              - pathout
              - textbooks
          default:
            - textbooks
        max_results:
          type: integer
          minimum: 1
          maximum: 10
          default: 1
        include_figures:
          type: boolean
          default: false
        max_figures:
          type: integer
          minimum: 0
          maximum: 10
          default: 0
        compact:
          type: boolean
          default: true
        excerpt_char_limit:
          type: integer
          minimum: 200
          maximum: 4000
          default: 900

    EvidenceItem:
      type: object
      properties:
        title:
          type: string
        source:
          type: string
        source_name:
          type: string
        source_family:
          type: string
        volume_code:
          type: string
        entity_name:
          type: string
        journal:
          type: string
        doi:
          type: string
        source_url:
          type: string
        url:
          type: string
        excerpt:
          type: string
        text:
          type: string
        chunk_text:
          type: string
        source_type:
          type: string
        source_id:
          type: string
        record_id:
          type: string
        chunk_id:
          type: string
        chunk_type:
          type: string
        page:
          type: integer
        chapter_title:
          type: string
        section:
          type: string
        section_heading:
          type: string
        figure_id:
          type: string
        image_path:
          type: string
        candidate_tags:
          type: array
          items:
            type: string
        ai_tags:
          type: array
          items:
            type: string
        tagging_status:
          type: string
      additionalProperties: true

    FigureItem:
      type: object
      properties:
        title:
          type: string
        caption:
          type: string
        figure_id:
          type: string
        figure_url:
          type: string
        source_url:
          type: string
        url:
          type: string
        source_name:
          type: string
        page:
          type: integer
      additionalProperties: true

    SourceStatus:
      type: object
      properties:
        who:
          type: string
        journals:
          type: string
        pathout:
          type: string
        textbooks:
          type: string
      additionalProperties: true

    EvidenceSearchResponse:
      type: object
      properties:
        schema_version:
          type: string
        query:
          type: string
        source_status:
          $ref: "#/components/schemas/SourceStatus"
        who_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        journal_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        pathout_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        textbook_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        figures:
          type: array
          items:
            $ref: "#/components/schemas/FigureItem"
        warnings:
          type: array
          items:
            type: string
      additionalProperties: true

    ErrorResponse:
      type: object
      properties:
        error:
          type: string
        detail:
          type: string
      additionalProperties: true

paths:
  /evidence/search:
    post:
      operationId: searchEvidence
      summary: Search Pathology Hub evidence.
      description: Search WHO, Journal RAG, Pathology Outlines, and/or Textbook SQLite FTS evidence.
      security:
        - ApiKeyAuth: []
      x-openai-isConsequential: false
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: "#/components/schemas/EvidenceSearchRequest"
      responses:
        "200":
          description: Evidence search results.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/EvidenceSearchResponse"
        "400":
          description: Bad request.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/ErrorResponse"
        "401":
          description: Unauthorized.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/ErrorResponse"
        "500":
          description: Server error.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/ErrorResponse"
"""

yaml_path = APP_DIR / "openapi_pathology_hub_unified_searchEvidence_who_textbooks_v1_5_1.yaml"
yaml_path.write_text(openapi_yaml, encoding="utf-8")

handoff = {
    "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "workstream": "Backend API / WHO + Textbook RAG integration",
    "service": SERVICE_NAME,
    "service_url": service_url,
    "image": IMAGE,
    "who_support": "proxied_to_upstream",
    "textbook_support": "local_sqlite_fts",
    "textbook_sqlite_gcs": TEXTBOOK_SQLITE_GCS,
    "textbook_manifest_gcs": TEXTBOOK_MANIFEST_GCS,
    "search_mode": "SQLite/keyword FTS; not vectorized",
    "vectorized": False,
    "api_exposed": True,
    "openapi_yaml": str(yaml_path),
}
handoff_path = APP_DIR / "HANDOFF_PATHOLOGY_HUB_V04_1_WHO_TEXTBOOK_API.json"
handoff_path.write_text(json.dumps(handoff, indent=2), encoding="utf-8")

print("\n✅ DEPLOYED v04.1 WITH WHO + TEXTBOOKS")
print("Service:", service_url)
print("OpenAPI YAML:", yaml_path)
print("Handoff:", handoff_path)
print("\nUse this YAML in GPT Builder:")
print(yaml_path)

In [ ]:
import subprocess, requests, json

SERVICE_URL = "https://pathology-hub-v04-vorn5q2kga-uc.a.run.app"

api_key = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", "pathology-hub-api-key"
], text=True).strip()

tests = [
    {
        "name": "WHO",
        "payload": {
            "query": "cribriform morular thyroid carcinoma",
            "sources": ["who"],
            "max_results": 1,
            "include_figures": False,
            "max_figures": 0,
            "compact": True,
            "excerpt_char_limit": 900
        }
    },
    {
        "name": "Textbooks",
        "payload": {
            "query": "STIC p53 Ki67 diagnostic criteria",
            "sources": ["textbooks"],
            "max_results": 1,
            "include_figures": False,
            "max_figures": 0,
            "compact": True,
            "excerpt_char_limit": 900
        }
    },
    {
        "name": "PathOut",
        "payload": {
            "query": "bladder carcinoma in situ CK20 p53",
            "sources": ["pathout"],
            "max_results": 1,
            "include_figures": False,
            "max_figures": 0,
            "compact": True,
            "excerpt_char_limit": 900
        }
    },
    {
        "name": "Journals",
        "payload": {
            "query": "STK11 KEAP1 lung adenocarcinoma immune checkpoint inhibitor",
            "sources": ["journals"],
            "max_results": 1,
            "include_figures": False,
            "max_figures": 0,
            "compact": True,
            "excerpt_char_limit": 900
        }
    },
    {
        "name": "Combined",
        "payload": {
            "query": "bladder CIS p53 CK20",
            "sources": ["who", "textbooks", "pathout"],
            "max_results": 1,
            "include_figures": False,
            "max_figures": 0,
            "compact": True,
            "excerpt_char_limit": 900
        }
    }
]

for t in tests:
    print("\n==============================")
    print(t["name"])
    r = requests.post(
        f"{SERVICE_URL}/evidence/search",
        headers={"X-API-Key": api_key, "Content-Type": "application/json"},
        json=t["payload"],
        timeout=300
    )
    print("status:", r.status_code)
    data = r.json()
    print("source_status:", data.get("source_status"))
    print("who_results:", len(data.get("who_results", [])))
    print("textbook_results:", len(data.get("textbook_results", [])))
    print("pathout_results:", len(data.get("pathout_results", [])))
    print("journal_results:", len(data.get("journal_results", [])))
    print("first 1500 chars:")
    print(json.dumps(data, indent=2)[:1500])

In [ ]:
from pathlib import Path
import subprocess, shutil, datetime, json

APP_DIR = Path("/content/pathology_hub_v04_textbook_api")

DRIVE_OUT = Path("/content/gdrive/MyDrive/4-Archives/Pathology_Hub_Intermission/api_handoffs")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

files = [
    APP_DIR / "openapi_pathology_hub_unified_searchEvidence_who_textbooks_v1_5_1.yaml",
    APP_DIR / "HANDOFF_PATHOLOGY_HUB_V04_1_WHO_TEXTBOOK_API.json",
]

for f in files:
    assert f.exists(), f"Missing: {f}"
    shutil.copy2(f, DRIVE_OUT / f.name)
    print("Copied to Drive:", DRIVE_OUT / f.name)

subprocess.run([
    "gcloud", "storage", "cp",
    str(APP_DIR / "openapi_pathology_hub_unified_searchEvidence_who_textbooks_v1_5_1.yaml"),
    "gs://pathology_hub/04_api_artifacts/openapi_pathology_hub_unified_searchEvidence_who_textbooks_v1_5_1.yaml"
], check=True)

subprocess.run([
    "gcloud", "storage", "cp",
    str(APP_DIR / "HANDOFF_PATHOLOGY_HUB_V04_1_WHO_TEXTBOOK_API.json"),
    "gs://pathology_hub/06_audits/handoff_packets/HANDOFF_PATHOLOGY_HUB_V04_1_WHO_TEXTBOOK_API.json"
], check=True)

print("✅ Saved v04.1 API schema + handoff.")

In [ ]:
# ============================================================
# PATHOLOGY HUB — TEXTBOOK VECTOR BUILD v1
#
# Workstream: Textbook Vector / Hybrid Search
#
# Input:
#   gs://pathology_hub/02_normalized/textbooks/lean/textbook_lean_chunks.jsonl
#
# Outputs:
#   gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_embeddings.npy
#   gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_faiss.index
#   gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_docstore.jsonl
#   gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_manifest.json
#   gs://pathology_hub/06_audits/textbooks/vector/textbook_lean_vector_build_audit.json
#
# This DOES NOT patch the live API.
# v04.1 remains FTS-only for textbooks until v04.2 hybrid patch.
# ============================================================

import os, sys, json, time, math, random, hashlib, getpass, subprocess, shutil, datetime
from pathlib import Path

# ----------------------------
# Config
# ----------------------------

PROJECT_ID = "pathology-annotation-project"

GCS_INPUT_CHUNKS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_lean_chunks.jsonl"

GCS_VECTOR_DIR = "gs://pathology_hub/03_indexes/textbooks/vector/"
GCS_AUDIT_DIR = "gs://pathology_hub/06_audits/textbooks/vector/"

WORK_DIR = Path("/content/pathology_hub_textbook_vector_build")
INPUT_DIR = WORK_DIR / "input"
OUT_DIR = WORK_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_CHUNKS = INPUT_DIR / "textbook_lean_chunks.jsonl"

EMBEDDING_MODEL = "text-embedding-3-small"
BATCH_SIZE = 96
MAX_EMBED_CHARS = 6000
MAX_DOCSTORE_TEXT_CHARS = 10000
MAX_RETRIES = 8

# Set to e.g. 500 for a quick test. Leave None for full 81k-ish corpus.
DRY_RUN_LIMIT = None

MODEL_SAFE = EMBEDDING_MODEL.replace("/", "_").replace(":", "_")
BATCH_DIR = OUT_DIR / f"embedding_batches_{MODEL_SAFE}_chars{MAX_EMBED_CHARS}_batch{BATCH_SIZE}"
BATCH_DIR.mkdir(parents=True, exist_ok=True)

EMB_PATH = OUT_DIR / "textbook_lean_embeddings.npy"
FAISS_PATH = OUT_DIR / "textbook_lean_faiss.index"
DOCSTORE_PATH = OUT_DIR / "textbook_lean_vector_docstore.jsonl"
MANIFEST_PATH = OUT_DIR / "textbook_lean_vector_manifest.json"
AUDIT_PATH = OUT_DIR / "textbook_lean_vector_build_audit.json"

# ----------------------------
# Helpers
# ----------------------------

def run(cmd, check=True, cwd=None):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=cwd,
    )
    if p.stdout:
        print(p.stdout[-8000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

def sha256_file(path: Path, block_size=1024 * 1024 * 8):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            h.update(block)
    return h.hexdigest()

def utc_now():
    return datetime.datetime.now(datetime.UTC).isoformat()

def coalesce(*vals):
    for v in vals:
        if v is not None and str(v).strip():
            return v
    return ""

def parse_maybe_json_list(x):
    if isinstance(x, list):
        return [str(v) for v in x]
    if isinstance(x, str) and x.strip():
        try:
            y = json.loads(x)
            if isinstance(y, list):
                return [str(v) for v in y]
        except Exception:
            return []
    return []

def clean_ws(s):
    return " ".join(str(s or "").replace("\x00", " ").split())

def truncate(s, n):
    s = str(s or "")
    return s if len(s) <= n else s[:n]

def build_embedding_text(obj):
    text = coalesce(
        obj.get("text"),
        obj.get("chunk_text"),
        obj.get("content"),
        obj.get("clean_text"),
        obj.get("page_text"),
        obj.get("caption"),
        obj.get("figure_caption"),
    )
    text = clean_ws(text)

    source_title = clean_ws(coalesce(obj.get("source_title"), obj.get("title"), obj.get("source_id")))
    source_id = clean_ws(obj.get("source_id"))
    chunk_type = clean_ws(obj.get("chunk_type"))
    chapter_title = clean_ws(obj.get("chapter_title"))
    section_heading = clean_ws(coalesce(obj.get("section_heading"), obj.get("section")))
    page = coalesce(obj.get("page"), obj.get("source_page"))

    tags = []
    for key in ["candidate_tags", "ai_tags", "reviewed_tags", "context_tags"]:
        tags.extend(parse_maybe_json_list(obj.get(key)))
    tags = tags[:10]

    parts = []
    if source_title:
        parts.append(f"Source: {source_title}")
    if source_id:
        parts.append(f"Source ID: {source_id}")
    if page:
        parts.append(f"Page: {page}")
    if chapter_title:
        parts.append(f"Chapter: {chapter_title}")
    if section_heading:
        parts.append(f"Section: {section_heading}")
    if chunk_type:
        parts.append(f"Chunk type: {chunk_type}")
    if tags:
        parts.append("Tags: " + "; ".join(tags))
    parts.append("Text: " + text)

    return truncate("\n".join(parts), MAX_EMBED_CHARS), text

# ----------------------------
# Install dependencies
# ----------------------------

run([sys.executable, "-m", "pip", "install", "-q", "openai>=1.0.0", "faiss-cpu", "tqdm"], check=True)

import numpy as np
import faiss
from tqdm.auto import tqdm
from openai import OpenAI

# ----------------------------
# Auth
# ----------------------------

run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

api_key = os.environ.get("OPEN_AI_KEY_01", "").strip()

if not api_key:
    try:
        from google.colab import userdata
        api_key = (userdata.get("OPEN_AI_KEY_01") or "").strip()
    except Exception:
        api_key = ""

if not api_key:
    api_key = getpass.getpass("Paste OPEN_AI_KEY_01, input hidden: ").strip()

assert api_key, "OPEN_AI_KEY_01 missing."
os.environ["OPEN_AI_KEY_01"] = api_key
client = OpenAI(api_key=api_key)

# ----------------------------
# Download input chunks
# ----------------------------

if not LOCAL_CHUNKS.exists() or LOCAL_CHUNKS.stat().st_size < 1024:
    run(["gcloud", "storage", "cp", GCS_INPUT_CHUNKS, str(LOCAL_CHUNKS)], check=True)
else:
    print("Using existing local chunks:", LOCAL_CHUNKS)

input_sha256 = sha256_file(LOCAL_CHUNKS)
print("Input:", LOCAL_CHUNKS)
print("Input size:", LOCAL_CHUNKS.stat().st_size)
print("Input sha256:", input_sha256)

# ----------------------------
# Load records
# ----------------------------

records = []
skipped_empty = 0

with LOCAL_CHUNKS.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue
        try:
            obj = json.loads(line)
        except Exception:
            continue

        embed_text, raw_text = build_embedding_text(obj)
        if len(clean_ws(raw_text)) < 20 or len(clean_ws(embed_text)) < 20:
            skipped_empty += 1
            continue

        source_id = coalesce(obj.get("source_id"), "unknown_source")
        chunk_id = coalesce(obj.get("chunk_id"), obj.get("id"), f"line:{line_no}")

        doc = {
            "vector_row": None,
            "chunk_id": chunk_id,
            "source_id": source_id,
            "source_title": coalesce(obj.get("source_title"), obj.get("title"), source_id),
            "chunk_type": coalesce(obj.get("chunk_type"), ""),
            "page": obj.get("page", obj.get("source_page")),
            "chapter_number": obj.get("chapter_number"),
            "chapter_title": obj.get("chapter_title"),
            "section": coalesce(obj.get("section_heading"), obj.get("section")),
            "figure_id": obj.get("figure_id"),
            "image_path": obj.get("image_path"),
            "candidate_tags": parse_maybe_json_list(obj.get("candidate_tags")),
            "ai_tags": parse_maybe_json_list(obj.get("ai_tags")),
            "tagging_status": obj.get("tagging_status"),
            "text": truncate(raw_text, MAX_DOCSTORE_TEXT_CHARS),
            "embedding_text_sha256": hashlib.sha256(embed_text.encode("utf-8")).hexdigest(),
        }

        records.append({
            "doc": doc,
            "embedding_text": embed_text,
        })

        if DRY_RUN_LIMIT is not None and len(records) >= DRY_RUN_LIMIT:
            break

print("Records loaded for embedding:", len(records))
print("Skipped empty/short:", skipped_empty)
assert records, "No records loaded."

# ----------------------------
# Embedding API with retry
# ----------------------------

def embed_batch(texts):
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.embeddings.create(
                model=EMBEDDING_MODEL,
                input=texts,
                encoding_format="float",
            )
            # Ensure original order by index
            data = sorted(resp.data, key=lambda x: x.index)
            arr = np.array([d.embedding for d in data], dtype=np.float32)
            if arr.shape[0] != len(texts):
                raise RuntimeError(f"Embedding count mismatch: got {arr.shape[0]}, expected {len(texts)}")
            return arr
        except Exception as e:
            wait = min(90, (2 ** attempt) + random.random() * 3)
            print(f"Embedding batch failed attempt {attempt+1}/{MAX_RETRIES}: {repr(e)}")
            print(f"Sleeping {wait:.1f}s")
            time.sleep(wait)
    raise RuntimeError("Embedding batch failed after retries.")

# ----------------------------
# Build/resume embedding batches
# ----------------------------

n = len(records)
num_batches = math.ceil(n / BATCH_SIZE)
print("Embedding batches:", num_batches, "batch_size:", BATCH_SIZE)

for batch_id, start in enumerate(tqdm(range(0, n, BATCH_SIZE), total=num_batches)):
    end = min(start + BATCH_SIZE, n)
    batch_file = BATCH_DIR / f"emb_batch_{batch_id:06d}_{start:07d}_{end:07d}.npy"
    meta_file = BATCH_DIR / f"emb_batch_{batch_id:06d}_{start:07d}_{end:07d}.json"

    if batch_file.exists() and meta_file.exists():
        try:
            meta = json.loads(meta_file.read_text(encoding="utf-8"))
            arr = np.load(batch_file)
            if meta.get("start") == start and meta.get("end") == end and arr.shape[0] == (end - start):
                continue
        except Exception:
            pass

    texts = [records[i]["embedding_text"] for i in range(start, end)]
    arr = embed_batch(texts)

    np.save(batch_file, arr)
    meta_file.write_text(json.dumps({
        "batch_id": batch_id,
        "start": start,
        "end": end,
        "count": end - start,
        "shape": list(arr.shape),
        "model": EMBEDDING_MODEL,
        "created_at_utc": utc_now(),
    }, indent=2), encoding="utf-8")

print("Embedding batches complete:", BATCH_DIR)

# ----------------------------
# Combine embeddings
# ----------------------------

arrays = []
expected_start = 0
dim = None

batch_files = sorted(BATCH_DIR.glob("emb_batch_*.npy"))
assert batch_files, "No embedding batch files found."

for bf in batch_files:
    mf = bf.with_suffix(".json")
    meta = json.loads(mf.read_text(encoding="utf-8"))
    arr = np.load(bf).astype(np.float32)

    if meta["start"] != expected_start:
        raise RuntimeError(f"Batch continuity error at {bf.name}: expected start {expected_start}, got {meta['start']}")
    if arr.shape[0] != meta["count"]:
        raise RuntimeError(f"Batch count mismatch at {bf.name}")
    if dim is None:
        dim = arr.shape[1]
    if arr.shape[1] != dim:
        raise RuntimeError(f"Embedding dim mismatch at {bf.name}")

    arrays.append(arr)
    expected_start = meta["end"]

assert expected_start == n, f"Only embedded through row {expected_start}, expected {n}"

embeddings = np.vstack(arrays).astype(np.float32)
assert embeddings.shape[0] == n
print("Combined embedding matrix:", embeddings.shape)

# Normalize for cosine / inner product search
faiss.normalize_L2(embeddings)

# Save embeddings
np.save(EMB_PATH, embeddings)
print("Saved embeddings:", EMB_PATH, EMB_PATH.stat().st_size)

# ----------------------------
# Build FAISS index
# ----------------------------

index = faiss.IndexFlatIP(dim)
index.add(embeddings)
assert index.ntotal == n

faiss.write_index(index, str(FAISS_PATH))
print("Saved FAISS index:", FAISS_PATH, FAISS_PATH.stat().st_size)

# ----------------------------
# Write docstore
# ----------------------------

with DOCSTORE_PATH.open("w", encoding="utf-8") as f:
    for i, rec in enumerate(records):
        d = dict(rec["doc"])
        d["vector_row"] = i
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

print("Saved docstore:", DOCSTORE_PATH, DOCSTORE_PATH.stat().st_size)

# ----------------------------
# Smoke-test vector search
# ----------------------------

docstore = []
with DOCSTORE_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        docstore.append(json.loads(line))

def vector_search(query, k=5):
    q = embed_batch([query])
    faiss.normalize_L2(q)
    D, I = index.search(q.astype(np.float32), k)
    hits = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0:
            continue
        d = docstore[int(idx)]
        hits.append({
            "score": float(score),
            "vector_row": int(idx),
            "source_id": d.get("source_id"),
            "source_title": d.get("source_title"),
            "page": d.get("page"),
            "chapter_title": d.get("chapter_title"),
            "section": d.get("section"),
            "chunk_id": d.get("chunk_id"),
            "text_preview": truncate(clean_ws(d.get("text")), 350),
        })
    return hits

smoke_queries = [
    "fallopian tube precursor lesion abnormal p53 increased proliferation",
    "flat bladder high grade lesion CK20 p53",
    "tumor with morules nuclear beta catenin",
    "distinguish reactive atypia from carcinoma in situ bladder",
    "soft tissue tumor loss of SMARCB1 INI1",
]

smoke_results = []
for q in smoke_queries:
    hits = vector_search(q, k=5)
    smoke_results.append({"query": q, "hits": hits})
    print("\nQUERY:", q)
    for h in hits[:3]:
        print(
            f" - {h['score']:.4f} | {h['source_id']} p.{h.get('page')} | "
            f"{h.get('chapter_title') or ''} / {h.get('section') or ''}"
        )
        print("   ", h["text_preview"][:260])

# ----------------------------
# Manifest + audit
# ----------------------------

manifest = {
    "schema_version": "textbook_lean_vector_manifest.v1",
    "created_at_utc": utc_now(),
    "workstream": "Textbook Vector / Hybrid Search",
    "purpose": "Vectorize LEAN cleaned textbook chunks for future hybrid FTS+vector retrieval.",
    "corpus": "textbooks_lean",
    "input_chunks_gcs": GCS_INPUT_CHUNKS,
    "input_chunks_local": str(LOCAL_CHUNKS),
    "input_chunks_sha256": input_sha256,
    "embedding_model": EMBEDDING_MODEL,
    "embedding_count": int(n),
    "embedding_dim": int(dim),
    "max_embed_chars": MAX_EMBED_CHARS,
    "docstore_text_max_chars": MAX_DOCSTORE_TEXT_CHARS,
    "faiss_index_type": "IndexFlatIP",
    "similarity": "cosine_via_l2_normalized_inner_product",
    "embeddings_l2_normalized": True,
    "vectorized": True,
    "searchable": True,
    "api_exposed": False,
    "live_api_modified": False,
    "outputs": {
        "embeddings_npy": str(EMB_PATH),
        "faiss_index": str(FAISS_PATH),
        "docstore_jsonl": str(DOCSTORE_PATH),
        "manifest_json": str(MANIFEST_PATH),
        "audit_json": str(AUDIT_PATH),
        "gcs_vector_dir": GCS_VECTOR_DIR,
        "gcs_audit_dir": GCS_AUDIT_DIR,
    },
    "notes": [
        "This build creates vector artifacts only.",
        "The live GPT/API remains v04.1 until a separate v04.2 hybrid API patch is deployed.",
        "Do not claim textbook API uses vector search until v04.2 is deployed and audited."
    ],
}

MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

audit = {
    "schema_version": "textbook_lean_vector_build_audit.v1",
    "created_at_utc": utc_now(),
    "workstream": "Textbook Vector / Hybrid Search",
    "input_record_count": int(n),
    "skipped_empty_or_short": int(skipped_empty),
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dim": int(dim),
    "embedding_matrix_shape": list(embeddings.shape),
    "faiss_index_ntotal": int(index.ntotal),
    "artifact_sizes_bytes": {
        "embeddings_npy": EMB_PATH.stat().st_size,
        "faiss_index": FAISS_PATH.stat().st_size,
        "docstore_jsonl": DOCSTORE_PATH.stat().st_size,
        "manifest_json": MANIFEST_PATH.stat().st_size,
    },
    "smoke_queries": smoke_results,
    "vectorized": True,
    "api_exposed": False,
    "known_limitations": [
        "Vector artifacts are not yet used by the live API.",
        "Vector search can retrieve semantically related but off-target chunks; final deployment should use hybrid FTS+vector, not vector-only.",
        "OCR/chunking errors in source text remain present in embeddings.",
        "Figure pixels are not interpreted."
    ],
}

AUDIT_PATH.write_text(json.dumps(audit, indent=2), encoding="utf-8")

print("\nSaved manifest:", MANIFEST_PATH)
print("Saved audit:", AUDIT_PATH)

# ----------------------------
# Upload final artifacts to GCS
# ----------------------------

print("\nUploading vector artifacts to GCS...")
for f in [EMB_PATH, FAISS_PATH, DOCSTORE_PATH, MANIFEST_PATH]:
    run(["gcloud", "storage", "cp", str(f), GCS_VECTOR_DIR], check=True)

print("\nUploading audit artifact to GCS...")
run(["gcloud", "storage", "cp", str(AUDIT_PATH), GCS_AUDIT_DIR], check=True)

print("\n✅ TEXTBOOK VECTOR BUILD COMPLETE")
print("Vector artifacts:", GCS_VECTOR_DIR)
print("Audit:", GCS_AUDIT_DIR)
print("Embedding count:", n)
print("Embedding dim:", dim)
print("FAISS ntotal:", index.ntotal)
print("\nNext step after this passes: deploy v04.2 hybrid API using FTS + FAISS reciprocal-rank fusion.")

In [ ]:
# ============================================================
# TEXTBOOK FIGURE AUDIT v2
# Converts storage.googleapis.com URLs back to gs:// paths,
# checks object existence, downloads samples, verifies PIL images.
# ============================================================

import json, re, subprocess, urllib.parse
from pathlib import Path
from collections import Counter
import pandas as pd

try:
    from PIL import Image
except Exception:
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pillow"], check=True)
    from PIL import Image

LOCAL = Path("/content/textbook_lean_figures.jsonl")
SAMPLE_DIR = Path("/content/textbook_figure_asset_audit_v2_samples")
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

assert LOCAL.exists(), f"Missing {LOCAL}; rerun prior download cell."

SEARCH_TERMS = ["merkel", "stic", "bladder", "ck20", "squamous"]
MAX_CHECK = 50

def run(cmd):
    p = subprocess.run(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    return p.returncode, p.stdout

def storage_https_to_gs(url):
    if not isinstance(url, str):
        return None
    url = url.strip()
    if url.startswith("gs://"):
        return url
    if url.startswith("https://storage.googleapis.com/"):
        rest = url.replace("https://storage.googleapis.com/", "", 1)
        parts = rest.split("/", 1)
        if len(parts) == 2:
            bucket, key = parts
            return f"gs://{bucket}/{urllib.parse.unquote(key)}"
    return None

def clean(s):
    return " ".join(str(s or "").split())

def pick_url(obj):
    for k in ["image_url", "figure_url", "image_path", "path", "gcs_uri", "url", "source_url"]:
        v = obj.get(k)
        if isinstance(v, str) and v.strip():
            return k, v.strip()
    return None, None

records = []
matches = []

with LOCAL.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue
        obj = json.loads(line)
        key, url = pick_url(obj)
        if not url:
            continue

        gs = storage_https_to_gs(url)

        blob = " ".join([
            clean(obj.get("source_id")),
            clean(obj.get("source_title")),
            clean(obj.get("figure_id")),
            clean(obj.get("caption")),
            clean(obj.get("legend")),
            clean(obj.get("text")),
            clean(url),
        ]).lower()

        rec = {
            "line_no": line_no,
            "source_id": obj.get("source_id"),
            "source_title": obj.get("source_title"),
            "page": obj.get("page") or obj.get("source_page"),
            "figure_id": obj.get("figure_id"),
            "caption": clean(obj.get("caption") or obj.get("legend") or obj.get("text"))[:500],
            "path_field": key,
            "url": url,
            "gs_uri": gs,
        }

        records.append(rec)

        if any(t in blob for t in SEARCH_TERMS):
            matches.append(rec)

print("Records with paths:", len(records))
print("Matches:", len(matches))

sample = matches[:MAX_CHECK] if matches else records[:MAX_CHECK]

rows = []
for i, r in enumerate(sample):
    row = dict(r)
    row.update({
        "gcs_exists": False,
        "gcs_size": None,
        "gcs_content_type": None,
        "download_ok": False,
        "pil_ok": False,
        "pil_format": None,
        "pil_size": None,
        "local_file": None,
    })

    gs = r["gs_uri"]
    if gs:
        rc, out = run(["gcloud", "storage", "ls", "-L", gs])
        row["gcs_exists"] = rc == 0

        if rc == 0:
            m_size = re.search(r"Content-Length:\s*(\d+)", out)
            m_type = re.search(r"Content-Type:\s*([^\n]+)", out)
            if m_size:
                row["gcs_size"] = int(m_size.group(1))
            if m_type:
                row["gcs_content_type"] = m_type.group(1).strip()

            suffix = Path(gs).suffix or ".img"
            local_file = SAMPLE_DIR / f"sample_{i:03d}{suffix}"
            rc2, out2 = run(["gcloud", "storage", "cp", gs, str(local_file)])
            row["download_ok"] = rc2 == 0 and local_file.exists() and local_file.stat().st_size > 0
            row["local_file"] = str(local_file) if row["download_ok"] else None

            if row["download_ok"]:
                try:
                    im = Image.open(local_file)
                    im.verify()
                    im = Image.open(local_file)
                    row["pil_ok"] = True
                    row["pil_format"] = im.format
                    row["pil_size"] = list(im.size)
                except Exception as e:
                    row["pil_error"] = repr(e)

    rows.append(row)

df = pd.DataFrame(rows)

print("\nCounts:")
for c in ["gcs_exists", "download_ok", "pil_ok"]:
    print(c, df[c].value_counts(dropna=False).to_dict())

pd.set_option("display.max_colwidth", 120)
display(df[[
    "source_id", "page", "figure_id", "gs_uri",
    "gcs_exists", "gcs_size", "gcs_content_type",
    "download_ok", "pil_ok", "pil_format", "pil_size",
    "local_file", "caption"
]])

out = Path("/content/textbook_figure_asset_audit_v2.csv")
df.to_csv(out, index=False)
print("\nSaved:", out)
print("Downloaded samples:", SAMPLE_DIR)

In [ ]:
# ============================================================
# FIX v04.2 DEPLOY — create/use Secret Manager secret from local key
# Supports local env/userdata key named OPEN_AI_KEY_01.
# Cloud Run app env var remains OPENAI_API_KEY.
# No rebuild.
# ============================================================

from pathlib import Path
import os, subprocess, json, requests, datetime, time, getpass

PROJECT_ID = "pathology-annotation-project"
REGION = "us-central1"
SERVICE_NAME = "pathology-hub-v04"
AR_REPO = "pathology-hub"
IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/{SERVICE_NAME}:latest"

# Secret Manager secret name we will use.
OPENAI_SECRET_NAME = "OPEN_AI_KEY_01"

TEXTBOOK_SQLITE_GCS = "gs://pathology_hub/03_indexes/textbooks/lean/textbook_lean_fts.sqlite"
TEXTBOOK_MANIFEST_GCS = "gs://pathology_hub/03_indexes/textbooks/lean/textbook_lean_index_manifest.json"
TEXTBOOK_FAISS_GCS = "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_faiss.index"
TEXTBOOK_DOCSTORE_GCS = "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_docstore.jsonl"
TEXTBOOK_VECTOR_MANIFEST_GCS = "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_manifest.json"
UPSTREAM_EVIDENCE_URL = "https://pathology-hub-830130787988.us-central1.run.app/evidence/search"

def run(cmd, check=True, input_text=None):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        input=input_text,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout:
        print(p.stdout[-12000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

run(["gcloud", "config", "set", "project", PROJECT_ID])

# ------------------------------------------------------------
# Find the actual OpenAI key locally
# ------------------------------------------------------------

openai_key = os.environ.get("OPEN_AI_KEY_01", "").strip()

if not openai_key:
    openai_key = os.environ.get("OPENAI_API_KEY", "").strip()

if not openai_key:
    try:
        from google.colab import userdata
        openai_key = (userdata.get("OPEN_AI_KEY_01") or "").strip()
    except Exception:
        openai_key = ""

if not openai_key:
    try:
        from google.colab import userdata
        openai_key = (userdata.get("OPENAI_API_KEY") or "").strip()
    except Exception:
        openai_key = ""

if not openai_key:
    openai_key = getpass.getpass("Paste OpenAI API key, hidden: ").strip()

assert openai_key, "No OpenAI key found."

# Optional quick validation against OpenAI before saving
from openai import OpenAI
client = OpenAI(api_key=openai_key, base_url="https://api.openai.com/v1")
test = client.embeddings.create(
    model="text-embedding-3-small",
    input=["pathology hub cloud run key validation"],
    encoding_format="float",
)
print("✅ Local OpenAI key validated. Dim:", len(test.data[0].embedding))

# ------------------------------------------------------------
# Create or update Secret Manager secret OPEN_AI_KEY_01
# ------------------------------------------------------------

exists = run(["gcloud", "secrets", "describe", OPENAI_SECRET_NAME], check=False)

if exists.returncode != 0:
    print(f"Creating Secret Manager secret: {OPENAI_SECRET_NAME}")
    run(
        ["gcloud", "secrets", "create", OPENAI_SECRET_NAME, "--data-file=-"],
        input_text=openai_key,
        check=True,
    )
else:
    print(f"Adding new version to existing secret: {OPENAI_SECRET_NAME}")
    run(
        ["gcloud", "secrets", "versions", "add", OPENAI_SECRET_NAME, "--data-file=-"],
        input_text=openai_key,
        check=True,
    )

# ------------------------------------------------------------
# Grant Cloud Run runtime service account access to secrets
# ------------------------------------------------------------

PROJECT_NUMBER = subprocess.check_output([
    "gcloud", "projects", "describe", PROJECT_ID,
    "--format", "value(projectNumber)"
], text=True).strip()

RUNTIME_SA = f"{PROJECT_NUMBER}-compute@developer.gserviceaccount.com"
print("Cloud Run runtime service account:", RUNTIME_SA)

for secret in [OPENAI_SECRET_NAME, "pathology-hub-api-key"]:
    run([
        "gcloud", "secrets", "add-iam-policy-binding", secret,
        "--member", f"serviceAccount:{RUNTIME_SA}",
        "--role", "roles/secretmanager.secretAccessor"
    ], check=True)

time.sleep(10)

# ------------------------------------------------------------
# Redeploy already-built v04.2 image
# ------------------------------------------------------------

print("\nRedeploying existing v04.2 image — no rebuild.")
run([
    "gcloud", "run", "deploy", SERVICE_NAME,
    "--image", IMAGE,
    "--region", REGION,
    "--platform", "managed",
    "--allow-unauthenticated",
    "--memory", "8Gi",
    "--cpu", "4",
    "--timeout", "300",
    "--min-instances", "1",
    "--set-env-vars",
    f"TEXTBOOK_SQLITE_GCS={TEXTBOOK_SQLITE_GCS},TEXTBOOK_MANIFEST_GCS={TEXTBOOK_MANIFEST_GCS},TEXTBOOK_FAISS_GCS={TEXTBOOK_FAISS_GCS},TEXTBOOK_DOCSTORE_GCS={TEXTBOOK_DOCSTORE_GCS},TEXTBOOK_VECTOR_MANIFEST_GCS={TEXTBOOK_VECTOR_MANIFEST_GCS},UPSTREAM_EVIDENCE_URL={UPSTREAM_EVIDENCE_URL},EMBEDDING_MODEL=text-embedding-3-small",
    "--set-secrets",
    f"PATHOLOGY_HUB_API_KEY=pathology-hub-api-key:latest,OPENAI_API_KEY={OPENAI_SECRET_NAME}:latest",
    "--quiet",
], check=True)

service_url = subprocess.check_output([
    "gcloud", "run", "services", "describe", SERVICE_NAME,
    "--region", REGION,
    "--format", "value(status.url)"
], text=True).strip()

print("\nSERVICE URL:", service_url)

api_key = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", "pathology-hub-api-key"
], text=True).strip()

# ------------------------------------------------------------
# Smoke tests
# ------------------------------------------------------------

print("\nHealth check...")
r = requests.get(f"{service_url}/health", timeout=300)
print("health:", r.status_code)
health = r.json()
print(json.dumps(health, indent=2)[:5000])
assert r.status_code == 200
assert health.get("vectorized") is True
assert health.get("textbook_search_mode") == "hybrid_fts_faiss_vector_rrf"

print("\nHybrid textbook smoke test...")
payload = {
    "query": "fallopian tube precursor lesion abnormal p53 increased proliferation",
    "sources": ["textbooks"],
    "max_results": 3,
    "include_figures": False,
    "max_figures": 0,
    "compact": True,
    "excerpt_char_limit": 900
}

rr = requests.post(
    f"{service_url}/evidence/search",
    headers={"X-API-Key": api_key, "Content-Type": "application/json"},
    json=payload,
    timeout=300,
)
print("status:", rr.status_code)
data = rr.json()
print("source_status:", data.get("source_status"))
print("search_mode:", data.get("search_mode"))
print(json.dumps(data, indent=2)[:7000])
assert rr.status_code == 200
assert data.get("source_status", {}).get("textbooks") == "ok"

print("\n✅ v04.2 deployed using Secret Manager secret:", OPENAI_SECRET_NAME)
print("Service:", service_url)

In [ ]:
from pathlib import Path
import subprocess, json, datetime

APP_DIR = Path("/content/pathology_hub_v04_textbook_api")
APP_DIR.mkdir(parents=True, exist_ok=True)

SERVICE_URL = "https://pathology-hub-v04-vorn5q2kga-uc.a.run.app"

yaml_path = APP_DIR / "openapi_pathology_hub_unified_searchEvidence_hybrid_textbooks_v1_5_2.yaml"
handoff_path = APP_DIR / "HANDOFF_PATHOLOGY_HUB_V04_2_HYBRID_TEXTBOOK_API.json"

openapi_yaml = f"""openapi: 3.1.0
info:
  title: Pathology Hub Unified Evidence API
  version: 1.5.2
  description: Unified evidence search with WHO passthrough, Journal, PathOut, and hybrid Textbook SQLite FTS + FAISS vector support.

servers:
  - url: {SERVICE_URL}

components:
  securitySchemes:
    ApiKeyAuth:
      type: apiKey
      in: header
      name: X-API-Key

  schemas:
    EvidenceSearchRequest:
      type: object
      required: [query]
      properties:
        query:
          type: string
        sources:
          type: array
          items:
            type: string
            enum: [who, journals, pathout, textbooks]
          default: [textbooks]
        max_results:
          type: integer
          minimum: 1
          maximum: 10
          default: 1
        include_figures:
          type: boolean
          default: false
        max_figures:
          type: integer
          minimum: 0
          maximum: 10
          default: 0
        compact:
          type: boolean
          default: true
        excerpt_char_limit:
          type: integer
          minimum: 200
          maximum: 4000
          default: 900

    EvidenceItem:
      type: object
      additionalProperties: true

    FigureItem:
      type: object
      additionalProperties: true

    EvidenceSearchResponse:
      type: object
      additionalProperties: true
      properties:
        schema_version:
          type: string
        query:
          type: string
        source_status:
          type: object
          additionalProperties: true
        who_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        journal_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        pathout_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        textbook_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        figures:
          type: array
          items:
            $ref: "#/components/schemas/FigureItem"
        warnings:
          type: array
          items:
            type: string

paths:
  /evidence/search:
    post:
      operationId: searchEvidence
      summary: Search Pathology Hub evidence.
      security:
        - ApiKeyAuth: []
      x-openai-isConsequential: false
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: "#/components/schemas/EvidenceSearchRequest"
      responses:
        "200":
          description: Evidence search results.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/EvidenceSearchResponse"
"""

yaml_path.write_text(openapi_yaml, encoding="utf-8")

handoff = {
    "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "workstream": "Backend API / Textbook Hybrid Search",
    "purpose": "Expose textbook hybrid SQLite FTS + FAISS vector retrieval through existing searchEvidence Action.",
    "service": "pathology-hub-v04",
    "service_url": SERVICE_URL,
    "sources_supported": ["who", "textbooks", "pathout", "journals"],
    "textbook_search_mode": "hybrid_fts_faiss_vector_rrf",
    "vectorized": True,
    "api_exposed": True,
    "textbook_sqlite_gcs": "gs://pathology_hub/03_indexes/textbooks/lean/textbook_lean_fts.sqlite",
    "textbook_faiss_gcs": "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_faiss.index",
    "textbook_docstore_gcs": "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_docstore.jsonl",
    "textbook_vector_manifest_gcs": "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_manifest.json",
    "openapi_yaml": str(yaml_path),
    "known_limitations": [
        "Textbook figure image serving is not fixed in v04.2.",
        "JPX/TIFF/non-web textbook assets require v04.3 web derivatives or proxy/signed URLs.",
        "Vector search may retrieve semantically related but off-target chunks; hybrid ranking should be judged."
    ],
    "next_steps": [
        "Update GPT Action schema to v1.5.2 YAML.",
        "Patch GPT instructions: textbooks now hybrid FTS+vector.",
        "Build v04.3 figure serving layer."
    ]
}

handoff_path.write_text(json.dumps(handoff, indent=2), encoding="utf-8")

subprocess.run(["gcloud", "storage", "cp", str(yaml_path), "gs://pathology_hub/04_api_artifacts/"], check=True)
subprocess.run(["gcloud", "storage", "cp", str(handoff_path), "gs://pathology_hub/06_audits/handoff_packets/"], check=True)

print("✅ Saved v04.2 YAML + handoff")
print("YAML:", yaml_path)
print("Handoff:", handoff_path)

In [ ]:

# ============================================================
# DEPLOY PATHOLOGY HUB v04.3
# Textbook hybrid retrieval + controlled textbook figure proxy
#
# Keeps:
#   who/pathout/journals -> upstream proxy
#   textbooks -> hybrid FTS + FAISS vector
#
# Adds:
#   /figures/textbook?u=...&exp=...&sig=...
#   - expiring HMAC URLs
#   - streams private GCS textbook figures
#   - converts JPX/TIFF/non-web images to JPEG on the fly
#   - returns verified figure_url fields when include_figures=true
#
# Does NOT make pathology_hub bucket public.
# ============================================================

from pathlib import Path
import os, subprocess, json, requests, datetime, time

PROJECT_ID = "pathology-annotation-project"
REGION = "us-central1"
SERVICE_NAME = "pathology-hub-v04"
AR_REPO = "pathology-hub"
IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/{SERVICE_NAME}:latest"

APP_DIR = Path("/content/pathology_hub_v04_textbook_api")
APP_DIR.mkdir(parents=True, exist_ok=True)

TEXTBOOK_SQLITE_GCS = "gs://pathology_hub/03_indexes/textbooks/lean/textbook_lean_fts.sqlite"
TEXTBOOK_MANIFEST_GCS = "gs://pathology_hub/03_indexes/textbooks/lean/textbook_lean_index_manifest.json"
TEXTBOOK_FAISS_GCS = "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_faiss.index"
TEXTBOOK_DOCSTORE_GCS = "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_docstore.jsonl"
TEXTBOOK_VECTOR_MANIFEST_GCS = "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_manifest.json"
TEXTBOOK_FIGURES_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_lean_figures.jsonl"

UPSTREAM_EVIDENCE_URL = "https://pathology-hub-830130787988.us-central1.run.app/evidence/search"
OPENAI_SECRET_NAME = "OPEN_AI_KEY_01"

def run(cmd, cwd=None, check=True):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout:
        print(p.stdout[-12000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

# Confirm required secrets/artifacts.
for secret in ["pathology-hub-api-key", OPENAI_SECRET_NAME]:
    run(["gcloud", "secrets", "describe", secret], check=True)

for uri in [
    TEXTBOOK_SQLITE_GCS,
    TEXTBOOK_MANIFEST_GCS,
    TEXTBOOK_FAISS_GCS,
    TEXTBOOK_DOCSTORE_GCS,
    TEXTBOOK_VECTOR_MANIFEST_GCS,
    TEXTBOOK_FIGURES_GCS,
]:
    run(["gcloud", "storage", "ls", uri], check=True)

# Ensure Cloud Run default runtime service account can read secrets.
PROJECT_NUMBER = subprocess.check_output([
    "gcloud", "projects", "describe", PROJECT_ID,
    "--format", "value(projectNumber)"
], text=True).strip()
RUNTIME_SA = f"{PROJECT_NUMBER}-compute@developer.gserviceaccount.com"
print("Cloud Run runtime service account:", RUNTIME_SA)

for secret in ["pathology-hub-api-key", OPENAI_SECRET_NAME]:
    run([
        "gcloud", "secrets", "add-iam-policy-binding", secret,
        "--member", f"serviceAccount:{RUNTIME_SA}",
        "--role", "roles/secretmanager.secretAccessor"
    ], check=True)

time.sleep(5)

# ------------------------------------------------------------
# Write v04.3 app
# ------------------------------------------------------------

(APP_DIR / "app.py").write_text(r'''
import os, re, json, sqlite3, time, hmac, hashlib, base64, io, urllib.parse
from pathlib import Path
from typing import Any, Dict, List, Optional
import requests
import numpy as np
import faiss
from fastapi import FastAPI, Header, HTTPException, Request, Query
from fastapi.responses import Response
from pydantic import BaseModel, Field
from google.cloud import storage
from openai import OpenAI
from PIL import Image

APP_VERSION = "1.5.3-textbooks-hybrid-figproxy-v04"

TEXTBOOK_SQLITE_GCS = os.environ.get("TEXTBOOK_SQLITE_GCS")
TEXTBOOK_MANIFEST_GCS = os.environ.get("TEXTBOOK_MANIFEST_GCS")
TEXTBOOK_FAISS_GCS = os.environ.get("TEXTBOOK_FAISS_GCS")
TEXTBOOK_DOCSTORE_GCS = os.environ.get("TEXTBOOK_DOCSTORE_GCS")
TEXTBOOK_VECTOR_MANIFEST_GCS = os.environ.get("TEXTBOOK_VECTOR_MANIFEST_GCS")
TEXTBOOK_FIGURES_GCS = os.environ.get("TEXTBOOK_FIGURES_GCS")
UPSTREAM_EVIDENCE_URL = os.environ.get("UPSTREAM_EVIDENCE_URL", "")

EXPECTED_API_KEY = os.environ.get("PATHOLOGY_HUB_API_KEY", "")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
FIGURE_PROXY_SECRET = os.environ.get("FIGURE_PROXY_SECRET", "") or EXPECTED_API_KEY

EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")
RRF_K = int(os.environ.get("RRF_K", "60"))
FTS_POOL = int(os.environ.get("FTS_POOL", "25"))
VECTOR_POOL = int(os.environ.get("VECTOR_POOL", "25"))
FIGURE_URL_TTL_SECONDS = int(os.environ.get("FIGURE_URL_TTL_SECONDS", "21600"))

DATA_DIR = Path("/tmp/pathology_hub_textbooks")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "textbook_lean_fts.sqlite"
MANIFEST_PATH = DATA_DIR / "textbook_lean_index_manifest.json"
FAISS_PATH = DATA_DIR / "textbook_lean_faiss.index"
DOCSTORE_PATH = DATA_DIR / "textbook_lean_vector_docstore.jsonl"
VECTOR_MANIFEST_PATH = DATA_DIR / "textbook_lean_vector_manifest.json"
FIGURES_PATH = DATA_DIR / "textbook_lean_figures.jsonl"

app = FastAPI(
    title="Pathology Hub Unified Evidence API v04.3",
    version=APP_VERSION,
    description="Unified evidence API with WHO/Journals/PathOut proxy, hybrid textbook search, and controlled textbook figure proxy."
)

class EvidenceSearchRequest(BaseModel):
    query: str = Field(..., description="Short keyword-style pathology evidence query.")
    sources: List[str] = Field(default_factory=lambda: ["textbooks"])
    max_results: int = Field(1, ge=1, le=10)
    include_figures: bool = False
    max_figures: int = Field(0, ge=0, le=10)
    compact: bool = True
    excerpt_char_limit: int = Field(900, ge=200, le=4000)

_INDEX = None
_DOCSTORE = None
_OPENAI_CLIENT = None
_FIGURES = None
_FIGURES_BY_SOURCE_PAGE = None

def _parse_gs_uri(uri: str):
    assert uri.startswith("gs://")
    rest = uri[5:]
    bucket, key = rest.split("/", 1)
    return bucket, key

def _download_gcs(uri: str, dest: Path):
    bucket_name, blob_name = _parse_gs_uri(uri)
    client = storage.Client()
    blob = client.bucket(bucket_name).blob(blob_name)
    dest.parent.mkdir(parents=True, exist_ok=True)
    blob.download_to_filename(str(dest))
    return dest

def ensure_artifacts():
    required = [
        (TEXTBOOK_SQLITE_GCS, DB_PATH),
        (TEXTBOOK_MANIFEST_GCS, MANIFEST_PATH),
        (TEXTBOOK_FAISS_GCS, FAISS_PATH),
        (TEXTBOOK_DOCSTORE_GCS, DOCSTORE_PATH),
        (TEXTBOOK_VECTOR_MANIFEST_GCS, VECTOR_MANIFEST_PATH),
        (TEXTBOOK_FIGURES_GCS, FIGURES_PATH),
    ]
    for uri, dest in required:
        if not uri:
            continue
        if not dest.exists() or dest.stat().st_size < 1024:
            _download_gcs(uri, dest)

def require_key(x_api_key: Optional[str]):
    if EXPECTED_API_KEY:
        if not x_api_key or x_api_key != EXPECTED_API_KEY:
            raise HTTPException(status_code=401, detail="Unauthorized")
    return True

def storage_https_to_gs(url: Optional[str]):
    if not isinstance(url, str):
        return None
    url = url.strip()
    if url.startswith("gs://"):
        return url
    if url.startswith("https://storage.googleapis.com/"):
        rest = url.replace("https://storage.googleapis.com/", "", 1)
        parts = rest.split("/", 1)
        if len(parts) == 2:
            bucket, key = parts
            return f"gs://{bucket}/{urllib.parse.unquote(key)}"
    return None

def is_allowed_figure_gs(gs_uri: str):
    return isinstance(gs_uri, str) and gs_uri.startswith("gs://pathology_hub/01_staged/textbooks/assets/figure_images/")

def _b64url(s: str):
    return base64.urlsafe_b64encode(s.encode("utf-8")).decode("ascii").rstrip("=")

def _b64url_decode(s: str):
    pad = "=" * (-len(s) % 4)
    return base64.urlsafe_b64decode((s + pad).encode("ascii")).decode("utf-8")

def _sign_payload(payload: str):
    secret = FIGURE_PROXY_SECRET or EXPECTED_API_KEY or "dev-secret"
    return hmac.new(secret.encode("utf-8"), payload.encode("utf-8"), hashlib.sha256).hexdigest()

def make_figure_proxy_url(base_url: str, gs_uri: str):
    if not gs_uri or not is_allowed_figure_gs(gs_uri):
        return None
    exp = str(int(time.time()) + FIGURE_URL_TTL_SECONDS)
    u = _b64url(gs_uri)
    sig = _sign_payload(f"{u}.{exp}")
    return f"{base_url.rstrip('/')}/figures/textbook?u={u}&exp={exp}&sig={sig}"

def verify_figure_sig(u: str, exp: str, sig: str):
    try:
        if int(exp) < int(time.time()):
            return False
    except Exception:
        return False
    expected = _sign_payload(f"{u}.{exp}")
    return hmac.compare_digest(expected, sig or "")

@app.get("/figures/textbook")
def get_textbook_figure(
    u: str = Query(...),
    exp: str = Query(...),
    sig: str = Query(...)
):
    if not verify_figure_sig(u, exp, sig):
        raise HTTPException(status_code=403, detail="Invalid or expired figure URL")

    try:
        gs_uri = _b64url_decode(u)
    except Exception:
        raise HTTPException(status_code=400, detail="Bad figure token")

    if not is_allowed_figure_gs(gs_uri):
        raise HTTPException(status_code=403, detail="Figure path not allowed")

    try:
        bucket_name, blob_name = _parse_gs_uri(gs_uri)
        blob = storage.Client().bucket(bucket_name).blob(blob_name)
        raw = blob.download_as_bytes()
    except Exception as e:
        raise HTTPException(status_code=404, detail=f"Figure object not found: {repr(e)}")

    suffix = Path(blob_name).suffix.lower()
    content_type = blob.content_type or ""

    if suffix in {".jpg", ".jpeg"} or content_type in {"image/jpeg", "image/jpg"}:
        return Response(content=raw, media_type="image/jpeg")
    if suffix == ".png" or content_type == "image/png":
        return Response(content=raw, media_type="image/png")
    if suffix == ".webp" or content_type == "image/webp":
        return Response(content=raw, media_type="image/webp")
    if suffix == ".gif" or content_type == "image/gif":
        return Response(content=raw, media_type="image/gif")

    try:
        im = Image.open(io.BytesIO(raw))
        im.load()
        if im.mode != "RGB":
            im = im.convert("RGB")
        out = io.BytesIO()
        im.save(out, format="JPEG", quality=88, optimize=True)
        return Response(content=out.getvalue(), media_type="image/jpeg")
    except Exception as e:
        raise HTTPException(status_code=415, detail=f"Unsupported image format or conversion failed: {repr(e)}")

def tokenize_for_fts(query: str):
    terms = re.findall(r"[A-Za-z0-9_]+", query.lower())
    stop = {"the","and","or","of","in","to","for","with","a","an","on","by","from","is","are","as"}
    return [t for t in terms if len(t) > 1 and t not in stop][:12]

def build_fts_queries(query: str):
    terms = tokenize_for_fts(query)
    if not terms:
        return [query]
    quoted = [f'"{t}"' for t in terms]
    if len(quoted) > 1:
        return [" AND ".join(quoted), " OR ".join(quoted)]
    return [quoted[0]]

def parse_jsonish(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str) and x.strip():
        try:
            y = json.loads(x)
            return y if isinstance(y, list) else []
        except Exception:
            return []
    return []

def make_excerpt(text: str, query: str, limit: int):
    text = text or ""
    if len(text) <= limit:
        return text
    terms = tokenize_for_fts(query)
    low = text.lower()
    pos = -1
    for t in terms:
        pos = low.find(t.lower())
        if pos >= 0:
            break
    if pos < 0:
        return text[:limit]
    start = max(0, pos - limit // 3)
    end = min(len(text), start + limit)
    return ("..." if start > 0 else "") + text[start:end] + ("..." if end < len(text) else "")

def table_columns(conn, table_name):
    try:
        return {r[1] for r in conn.execute(f"PRAGMA table_info({table_name})").fetchall()}
    except Exception:
        return set()

def detect_tables(conn):
    names = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type IN ('table','view')").fetchall()]
    base = "textbook_chunks" if "textbook_chunks" in names else None
    fts = "textbook_chunks_fts" if "textbook_chunks_fts" in names else None
    if not base:
        for n in names:
            if n.endswith("_chunks") or n == "chunks":
                base = n
                break
    if not fts:
        for n in names:
            if "fts" in n.lower():
                fts = n
                break
    if not base or not fts:
        raise RuntimeError(f"Could not detect chunk/FTS tables. Tables={names}")
    return base, fts

def load_figures():
    global _FIGURES, _FIGURES_BY_SOURCE_PAGE
    ensure_artifacts()
    if _FIGURES is not None and _FIGURES_BY_SOURCE_PAGE is not None:
        return _FIGURES, _FIGURES_BY_SOURCE_PAGE

    figures = []
    by_sp = {}

    with FIGURES_PATH.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue

            img = obj.get("image_path") or obj.get("image_url") or obj.get("figure_url") or obj.get("path") or obj.get("gcs_uri") or obj.get("url")
            gs = storage_https_to_gs(img)
            if not gs:
                continue

            source_id = obj.get("source_id")
            page = obj.get("page") or obj.get("source_page")
            cap = obj.get("caption") or obj.get("legend") or obj.get("text") or ""

            rec = {
                "line_no": line_no,
                "source_id": source_id,
                "source_title": obj.get("source_title") or obj.get("title") or source_id,
                "page": page,
                "figure_id": obj.get("figure_id"),
                "caption": cap,
                "image_path": gs,
                "original_image_url": img,
                "chunk_id": obj.get("chunk_id"),
            }
            figures.append(rec)
            key = (str(source_id), str(page))
            by_sp.setdefault(key, []).append(rec)

    _FIGURES = figures
    _FIGURES_BY_SOURCE_PAGE = by_sp
    return figures, by_sp

def figure_to_response(rec: dict, base_url: str, rank: int = None):
    gs = rec.get("image_path")
    proxy = make_figure_proxy_url(base_url, gs)
    return {
        "rank": rank,
        "title": rec.get("source_title") or rec.get("source_id"),
        "caption": rec.get("caption"),
        "figure_id": rec.get("figure_id"),
        "figure_url": proxy,
        "image_url": proxy,
        "image_path": gs,
        "original_image_path": gs,
        "original_image_url": rec.get("original_image_url"),
        "source": "textbooks",
        "source_name": "textbooks",
        "source_id": rec.get("source_id"),
        "page": rec.get("page"),
    }

def collect_textbook_figures(textbook_results: list, base_url: str, max_figures: int):
    if max_figures <= 0:
        return []

    figures, by_sp = load_figures()
    out = []
    seen = set()

    def add_rec(rec):
        if not rec:
            return
        gs = rec.get("image_path")
        if not gs or gs in seen:
            return
        url = make_figure_proxy_url(base_url, gs)
        if not url:
            return
        seen.add(gs)
        out.append(figure_to_response(rec, base_url, rank=len(out)+1))

    for r in textbook_results:
        img = r.get("image_path")
        gs = storage_https_to_gs(img)
        if gs:
            add_rec({
                "source_id": r.get("source_id"),
                "source_title": r.get("title") or r.get("source_title"),
                "page": r.get("page"),
                "figure_id": r.get("figure_id"),
                "caption": r.get("text") or r.get("excerpt"),
                "image_path": gs,
                "original_image_url": img,
            })
            if len(out) >= max_figures:
                return out

    for r in textbook_results:
        key = (str(r.get("source_id")), str(r.get("page")))
        for rec in by_sp.get(key, []):
            add_rec(rec)
            if len(out) >= max_figures:
                return out

    return out

def row_to_textbook_result(d, query, limit, rank=None, retrieval_mode="fts", extra=None):
    text = d.get("text") or d.get("chunk_text") or ""
    result = {
        "rank": rank,
        "title": d.get("source_title") or d.get("source_id"),
        "source_name": "textbooks",
        "source_type": "textbook_chunk",
        "source_id": d.get("source_id"),
        "chunk_id": d.get("chunk_id"),
        "chunk_type": d.get("chunk_type"),
        "page": d.get("page"),
        "chapter_number": d.get("chapter_number"),
        "chapter_title": d.get("chapter_title"),
        "section": d.get("section") or d.get("section_heading"),
        "section_heading": d.get("section_heading") or d.get("section"),
        "figure_id": d.get("figure_id"),
        "image_path": d.get("image_path"),
        "excerpt": make_excerpt(text, query, limit),
        "text": make_excerpt(text, query, limit),
        "candidate_tags": parse_jsonish(d.get("candidate_tags")),
        "ai_tags": parse_jsonish(d.get("ai_tags")),
        "context_tags": parse_jsonish(d.get("context_tags")),
        "reviewed_tags": parse_jsonish(d.get("reviewed_tags")),
        "tagging_status": d.get("tagging_status"),
        "retrieval_mode": retrieval_mode,
    }
    if extra:
        result.update(extra)
    return result

def fts_search_pool(query: str, pool_size: int):
    conn = sqlite3.connect(str(DB_PATH))
    conn.row_factory = sqlite3.Row
    base, fts = detect_tables(conn)
    cols = table_columns(conn, base)

    wanted = [
        "chunk_id", "source_id", "source_title", "chunk_type", "page",
        "chapter_number", "chapter_title", "section_heading",
        "figure_id", "image_path", "text",
        "candidate_tags", "ai_tags", "context_tags", "reviewed_tags",
        "tagging_status"
    ]
    select_parts = []
    for c in wanted:
        select_parts.append(f"c.{c} AS {c}" if c in cols else f"NULL AS {c}")
    select_sql = ", ".join(select_parts)

    rows = []
    for fts_q in build_fts_queries(query):
        try:
            sql = f"""
                SELECT {select_sql}, bm25({fts}) AS bm25_score
                FROM {fts}
                JOIN {base} c ON c.rowid = {fts}.rowid
                WHERE {fts} MATCH ?
                ORDER BY bm25_score
                LIMIT ?
            """
            rows = conn.execute(sql, (fts_q, pool_size)).fetchall()
            if rows:
                break
        except Exception:
            rows = []

    if not rows:
        terms = tokenize_for_fts(query)
        term = terms[0] if terms else query
        try:
            sql = f"""
                SELECT {select_sql}, 9999.0 AS bm25_score
                FROM {base} c
                WHERE c.text LIKE ?
                LIMIT ?
            """
            rows = conn.execute(sql, (f"%{term}%", pool_size)).fetchall()
        except Exception:
            rows = []

    conn.close()
    out = []
    for i, r in enumerate(rows, start=1):
        d = dict(r)
        d["_fts_rank"] = i
        d["_bm25_score"] = d.get("bm25_score")
        out.append(d)
    return out

def get_openai_client():
    global _OPENAI_CLIENT
    if _OPENAI_CLIENT is None:
        if not OPENAI_API_KEY:
            raise RuntimeError("OPENAI_API_KEY not configured")
        _OPENAI_CLIENT = OpenAI(api_key=OPENAI_API_KEY, base_url="https://api.openai.com/v1")
    return _OPENAI_CLIENT

def load_vector_assets():
    global _INDEX, _DOCSTORE
    ensure_artifacts()
    if _INDEX is None:
        _INDEX = faiss.read_index(str(FAISS_PATH))
    if _DOCSTORE is None:
        docs = []
        with DOCSTORE_PATH.open("r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    docs.append(json.loads(line))
        _DOCSTORE = docs
    return _INDEX, _DOCSTORE

def embed_query(query: str):
    client = get_openai_client()
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=[query], encoding_format="float")
    q = np.array([resp.data[0].embedding], dtype=np.float32)
    faiss.normalize_L2(q)
    return q

def vector_search_pool(query: str, pool_size: int):
    index, docs = load_vector_assets()
    q = embed_query(query)
    D, I = index.search(q, pool_size)
    out = []
    for rank, (score, idx) in enumerate(zip(D[0], I[0]), start=1):
        if idx < 0:
            continue
        d = dict(docs[int(idx)])
        d["_vector_rank"] = rank
        d["_vector_score"] = float(score)
        out.append(d)
    return out

def hybrid_textbook_search(query: str, max_results: int, excerpt_char_limit: int):
    ensure_artifacts()
    fts_hits = fts_search_pool(query, max(FTS_POOL, max_results))
    vector_hits = vector_search_pool(query, max(VECTOR_POOL, max_results))
    merged = {}

    for h in fts_hits:
        key = h.get("chunk_id") or f"fts:{len(merged)}"
        if key not in merged:
            merged[key] = {"doc": h, "fts_rank": None, "vector_rank": None, "bm25_score": None, "vector_score": None}
        merged[key]["doc"].update({k: v for k, v in h.items() if v is not None})
        merged[key]["fts_rank"] = h.get("_fts_rank")
        merged[key]["bm25_score"] = h.get("_bm25_score")

    for h in vector_hits:
        key = h.get("chunk_id") or f"vec:{h.get('vector_row')}"
        if key not in merged:
            merged[key] = {"doc": h, "fts_rank": None, "vector_rank": None, "bm25_score": None, "vector_score": None}
        merged[key]["doc"].update({k: v for k, v in h.items() if v is not None})
        merged[key]["vector_rank"] = h.get("_vector_rank")
        merged[key]["vector_score"] = h.get("_vector_score")

    ranked = []
    for key, item in merged.items():
        score = 0.0
        if item["fts_rank"]:
            score += 1.0 / (RRF_K + item["fts_rank"])
        if item["vector_rank"]:
            score += 1.0 / (RRF_K + item["vector_rank"])
        if item["fts_rank"] and item["vector_rank"]:
            score += 0.005
        item["fusion_score"] = score
        ranked.append(item)
    ranked.sort(key=lambda x: x["fusion_score"], reverse=True)

    results = []
    for rank, item in enumerate(ranked[:max_results], start=1):
        extra = {
            "fusion_score": item["fusion_score"],
            "fts_rank": item["fts_rank"],
            "vector_rank": item["vector_rank"],
            "bm25_score": item["bm25_score"],
            "vector_score": item["vector_score"],
        }
        if item["fts_rank"] and item["vector_rank"]:
            mode = "hybrid_fts_vector"
        elif item["vector_rank"]:
            mode = "vector_only"
        else:
            mode = "fts_only"
        results.append(row_to_textbook_result(item["doc"], query, excerpt_char_limit, rank, mode, extra))

    warnings = [
        "Textbook retrieval uses hybrid SQLite FTS + FAISS vector search with reciprocal-rank fusion.",
        "Vector search can retrieve semantically related but off-target chunks; judge relevance.",
        "Textbook figure URLs, when returned, are expiring proxy URLs for private GCS assets."
    ]
    return results, warnings

def manifest_summary(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}

@app.on_event("startup")
def startup_event():
    try:
        ensure_artifacts()
    except Exception as e:
        print(f"Startup artifact download warning: {e}")

@app.get("/health")
def health():
    ensure_artifacts()
    fig_count = 0
    try:
        figs, _ = load_figures()
        fig_count = len(figs)
    except Exception:
        fig_count = -1
    return {
        "schema_version": "pathology_hub_health.v1.5.3",
        "service": "pathology-hub-v04",
        "version": APP_VERSION,
        "loaded": True,
        "textbook_search_mode": "hybrid_fts_faiss_vector_rrf",
        "textbook_figure_mode": "expiring_hmac_proxy_with_on_the_fly_web_conversion",
        "textbook_sqlite_size_bytes": DB_PATH.stat().st_size if DB_PATH.exists() else 0,
        "textbook_faiss_size_bytes": FAISS_PATH.stat().st_size if FAISS_PATH.exists() else 0,
        "textbook_docstore_size_bytes": DOCSTORE_PATH.stat().st_size if DOCSTORE_PATH.exists() else 0,
        "textbook_figures_size_bytes": FIGURES_PATH.stat().st_size if FIGURES_PATH.exists() else 0,
        "textbook_figure_records_loaded": fig_count,
        "embedding_model": EMBEDDING_MODEL,
        "vectorized": True,
        "api_exposed": True,
        "figure_proxy_enabled": True,
        "manifest_summary": manifest_summary(MANIFEST_PATH),
        "vector_manifest_summary": manifest_summary(VECTOR_MANIFEST_PATH),
        "upstream_evidence_url": UPSTREAM_EVIDENCE_URL,
    }

@app.post("/evidence/search")
def search_evidence(req: EvidenceSearchRequest, request: Request, x_api_key: Optional[str] = Header(None, alias="X-API-Key")):
    require_key(x_api_key)
    proto = request.headers.get("x-forwarded-proto") or request.url.scheme
    host = request.headers.get("x-forwarded-host") or request.headers.get("host") or request.url.netloc
    base_url = f"{proto}://{host}".rstrip("/")

    sources = [s.lower() for s in (req.sources or ["textbooks"])]
    allowed = {"who", "journals", "pathout", "textbooks"}
    bad = [s for s in sources if s not in allowed]
    if bad:
        raise HTTPException(status_code=400, detail=f"Unsupported source(s): {bad}")

    response = {
        "schema_version": "evidence_search_response.v1.5.3",
        "query": req.query,
        "source_status": {"who": "not_requested", "journals": "not_requested", "pathout": "not_requested", "textbooks": "not_requested"},
        "who_results": [],
        "journal_results": [],
        "pathout_results": [],
        "textbook_results": [],
        "figures": [],
        "warnings": [],
        "search_mode": {"textbooks": "hybrid_fts_faiss_vector_rrf", "who": "upstream", "journals": "upstream", "pathout": "upstream"}
    }

    if "textbooks" in sources:
        try:
            results, warnings = hybrid_textbook_search(req.query, req.max_results, req.excerpt_char_limit)
            response["textbook_results"] = results
            response["source_status"]["textbooks"] = "ok"
            response["warnings"].extend(warnings)
            if req.include_figures and req.max_figures > 0:
                response["figures"].extend(collect_textbook_figures(results, base_url, req.max_figures))
        except Exception as e:
            response["source_status"]["textbooks"] = "error"
            response["warnings"].append(f"textbook_hybrid_error: {repr(e)}")

    upstream_sources = [s for s in sources if s in {"who", "journals", "pathout"}]
    if upstream_sources:
        if not UPSTREAM_EVIDENCE_URL:
            for s in upstream_sources:
                response["source_status"][s] = "error_no_upstream"
        else:
            try:
                payload = req.dict()
                payload["sources"] = upstream_sources
                headers = {"Content-Type": "application/json"}
                if x_api_key:
                    headers["X-API-Key"] = x_api_key
                r = requests.post(UPSTREAM_EVIDENCE_URL, headers=headers, json=payload, timeout=90)
                if r.status_code >= 400:
                    for s in upstream_sources:
                        response["source_status"][s] = f"upstream_http_{r.status_code}"
                    response["warnings"].append(f"Upstream error {r.status_code}: {r.text[:500]}")
                else:
                    u = r.json()
                    response["who_results"] = u.get("who_results", [])
                    response["journal_results"] = u.get("journal_results", [])
                    response["pathout_results"] = u.get("pathout_results", [])
                    response["figures"].extend(u.get("figures", []) or [])
                    uss = u.get("source_status", {}) or {}
                    for s in upstream_sources:
                        response["source_status"][s] = uss.get(s, "ok")
                    response["warnings"].extend(u.get("warnings", []) or [])
            except Exception as e:
                for s in upstream_sources:
                    response["source_status"][s] = "upstream_error"
                response["warnings"].append(f"upstream_proxy_error: {repr(e)}")

    if not req.include_figures:
        response["figures"] = []
    else:
        response["figures"] = response["figures"][:req.max_figures]

    return response
''', encoding="utf-8")

# Requirements + Dockerfile
(APP_DIR / "requirements.txt").write_text("""\
fastapi==0.115.6
uvicorn[standard]==0.34.0
google-cloud-storage==2.19.0
requests==2.32.3
pydantic==2.10.4
numpy==1.26.4
faiss-cpu==1.8.0.post1
openai>=1.0.0
pillow>=10.0.0
""", encoding="utf-8")

(APP_DIR / "Dockerfile").write_text("""\
FROM python:3.11-slim

ENV PYTHONUNBUFFERED=1
WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \\
    ca-certificates \\
    libopenjp2-7 \\
    libtiff6 \\
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8080"]
""", encoding="utf-8")

# ------------------------------------------------------------
# Build and deploy
# ------------------------------------------------------------

for api in [
    "artifactregistry.googleapis.com",
    "cloudbuild.googleapis.com",
    "run.googleapis.com",
    "secretmanager.googleapis.com",
    "storage.googleapis.com",
]:
    run(["gcloud", "services", "enable", api], check=True)

run(["gcloud", "auth", "configure-docker", f"{REGION}-docker.pkg.dev", "--quiet"], check=True)

print("\nBuilding v04.3 image...")
run(["gcloud", "builds", "submit", "--tag", IMAGE, "."], cwd=APP_DIR, check=True)

print("\nDeploying v04.3...")
run([
    "gcloud", "run", "deploy", SERVICE_NAME,
    "--image", IMAGE,
    "--region", REGION,
    "--platform", "managed",
    "--allow-unauthenticated",
    "--memory", "8Gi",
    "--cpu", "4",
    "--timeout", "300",
    "--min-instances", "1",
    "--set-env-vars",
    f"TEXTBOOK_SQLITE_GCS={TEXTBOOK_SQLITE_GCS},TEXTBOOK_MANIFEST_GCS={TEXTBOOK_MANIFEST_GCS},TEXTBOOK_FAISS_GCS={TEXTBOOK_FAISS_GCS},TEXTBOOK_DOCSTORE_GCS={TEXTBOOK_DOCSTORE_GCS},TEXTBOOK_VECTOR_MANIFEST_GCS={TEXTBOOK_VECTOR_MANIFEST_GCS},TEXTBOOK_FIGURES_GCS={TEXTBOOK_FIGURES_GCS},UPSTREAM_EVIDENCE_URL={UPSTREAM_EVIDENCE_URL},EMBEDDING_MODEL=text-embedding-3-small",
    "--set-secrets",
    f"PATHOLOGY_HUB_API_KEY=pathology-hub-api-key:latest,OPENAI_API_KEY={OPENAI_SECRET_NAME}:latest,FIGURE_PROXY_SECRET=pathology-hub-api-key:latest",
    "--quiet",
], check=True)

service_url = subprocess.check_output([
    "gcloud", "run", "services", "describe", SERVICE_NAME,
    "--region", REGION,
    "--format", "value(status.url)"
], text=True).strip()

print("\nSERVICE URL:", service_url)

api_key = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", "pathology-hub-api-key"
], text=True).strip()

# ------------------------------------------------------------
# Smoke tests
# ------------------------------------------------------------

print("\nHealth check...")
r = requests.get(f"{service_url}/health", timeout=300)
print("health:", r.status_code)
health = r.json()
print(json.dumps(health, indent=2)[:5000])
assert r.status_code == 200
assert health.get("vectorized") is True
assert health.get("figure_proxy_enabled") is True

print("\nTextbook figure smoke test...")
payload = {
    "query": "fallopian tube precursor lesion abnormal p53 increased proliferation",
    "sources": ["textbooks"],
    "max_results": 3,
    "include_figures": True,
    "max_figures": 3,
    "compact": True,
    "excerpt_char_limit": 900
}

rr = requests.post(
    f"{service_url}/evidence/search",
    headers={"X-API-Key": api_key, "Content-Type": "application/json"},
    json=payload,
    timeout=300,
)
print("status:", rr.status_code)
data = rr.json()
print("source_status:", data.get("source_status"))
print("figures:", len(data.get("figures", [])))
print(json.dumps(data, indent=2)[:7000])
assert rr.status_code == 200
assert data.get("source_status", {}).get("textbooks") == "ok"
assert len(data.get("figures", [])) >= 1

fig_url = data["figures"][0]["figure_url"]
print("\nTesting first figure URL:", fig_url[:300], "...")
img_resp = requests.get(fig_url, timeout=120)
print("figure status:", img_resp.status_code)
print("figure content-type:", img_resp.headers.get("content-type"))
print("figure bytes:", len(img_resp.content))
assert img_resp.status_code == 200
assert img_resp.headers.get("content-type", "").startswith("image/")
assert len(img_resp.content) > 1000

# ------------------------------------------------------------
# Save OpenAPI + handoff
# ------------------------------------------------------------

openapi_yaml = f"""openapi: 3.1.0
info:
  title: Pathology Hub Unified Evidence API
  version: 1.5.3
  description: Unified evidence search with WHO passthrough, Journal, PathOut, hybrid Textbook FTS+vector support, and controlled textbook figure proxy URLs.

servers:
  - url: {service_url}

components:
  securitySchemes:
    ApiKeyAuth:
      type: apiKey
      in: header
      name: X-API-Key

  schemas:
    EvidenceSearchRequest:
      type: object
      required:
        - query
      properties:
        query:
          type: string
        sources:
          type: array
          items:
            type: string
            enum:
              - who
              - journals
              - pathout
              - textbooks
          default:
            - textbooks
        max_results:
          type: integer
          minimum: 1
          maximum: 10
          default: 1
        include_figures:
          type: boolean
          default: false
        max_figures:
          type: integer
          minimum: 0
          maximum: 10
          default: 0
        compact:
          type: boolean
          default: true
        excerpt_char_limit:
          type: integer
          minimum: 200
          maximum: 4000
          default: 900

    SourceStatus:
      type: object
      properties:
        who:
          type: string
        journals:
          type: string
        pathout:
          type: string
        textbooks:
          type: string
      additionalProperties: true

    EvidenceItem:
      type: object
      properties:
        rank:
          type: integer
        title:
          type: string
        source:
          type: string
        source_name:
          type: string
        source_family:
          type: string
        volume_code:
          type: string
        entity_name:
          type: string
        journal:
          type: string
        doi:
          type: string
        source_url:
          type: string
        url:
          type: string
        excerpt:
          type: string
        text:
          type: string
        chunk_text:
          type: string
        source_type:
          type: string
        source_id:
          type: string
        record_id:
          type: string
        chunk_id:
          type: string
        chunk_type:
          type: string
        page:
          type: integer
        chapter_number:
          type: string
        chapter_title:
          type: string
        section:
          type: string
        section_heading:
          type: string
        figure_id:
          type: string
        image_path:
          type: string
        retrieval_mode:
          type: string
        fusion_score:
          type: number
        fts_rank:
          type: integer
        vector_rank:
          type: integer
        bm25_score:
          type: number
        vector_score:
          type: number
        score:
          type: number
      additionalProperties: true

    FigureItem:
      type: object
      properties:
        rank:
          type: integer
        title:
          type: string
        caption:
          type: string
        figure_id:
          type: string
        figure_url:
          type: string
        image_url:
          type: string
        image_path:
          type: string
        original_image_path:
          type: string
        original_image_url:
          type: string
        source:
          type: string
        source_name:
          type: string
        source_id:
          type: string
        page:
          type: integer
        score:
          type: number
      additionalProperties: true

    EvidenceSearchResponse:
      type: object
      properties:
        schema_version:
          type: string
        query:
          type: string
        source_status:
          $ref: "#/components/schemas/SourceStatus"
        who_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        journal_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        pathout_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        textbook_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        figures:
          type: array
          items:
            $ref: "#/components/schemas/FigureItem"
        warnings:
          type: array
          items:
            type: string
        search_mode:
          type: object
          additionalProperties: true
      additionalProperties: true

    ErrorResponse:
      type: object
      properties:
        error:
          type: string
        detail:
          type: string
        message:
          type: string
      additionalProperties: true

paths:
  /evidence/search:
    post:
      operationId: searchEvidence
      summary: Search Pathology Hub evidence.
      description: Search WHO, Journal, PathOut, and hybrid Textbook evidence. May return expiring figure proxy URLs when include_figures=true.
      security:
        - ApiKeyAuth: []
      x-openai-isConsequential: false
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: "#/components/schemas/EvidenceSearchRequest"
      responses:
        "200":
          description: Evidence search results.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/EvidenceSearchResponse"
        "400":
          description: Bad request.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/ErrorResponse"
        "401":
          description: Unauthorized.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/ErrorResponse"
        "500":
          description: Server error.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/ErrorResponse"
"""

yaml_path = APP_DIR / "openapi_pathology_hub_unified_searchEvidence_hybrid_textbooks_figures_v1_5_3.yaml"
yaml_path.write_text(openapi_yaml, encoding="utf-8")

handoff = {
    "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "workstream": "Backend API / Textbook Hybrid Search + Figure Proxy",
    "purpose": "Expose hybrid textbook retrieval and controlled renderable textbook figure URLs through searchEvidence.",
    "service": SERVICE_NAME,
    "service_url": service_url,
    "image": IMAGE,
    "sources_supported": ["who", "textbooks", "pathout", "journals"],
    "textbook_search_mode": "hybrid_fts_faiss_vector_rrf",
    "textbook_figure_mode": "expiring_hmac_proxy_with_on_the_fly_web_conversion",
    "vectorized": True,
    "api_exposed": True,
    "figure_proxy_enabled": True,
    "textbook_sqlite_gcs": TEXTBOOK_SQLITE_GCS,
    "textbook_faiss_gcs": TEXTBOOK_FAISS_GCS,
    "textbook_docstore_gcs": TEXTBOOK_DOCSTORE_GCS,
    "textbook_vector_manifest_gcs": TEXTBOOK_VECTOR_MANIFEST_GCS,
    "textbook_figures_gcs": TEXTBOOK_FIGURES_GCS,
    "openapi_yaml": str(yaml_path),
    "known_limitations": [
        "Figure proxy URLs expire.",
        "JPX/TIFF/non-web images are converted to JPEG on request.",
        "Figure pixels are served but not interpreted by searchEvidence.",
        "Vector search may retrieve semantically related but off-target chunks; relevance must be judged."
    ],
    "next_steps": [
        "Update GPT Action schema to v1.5.3 YAML.",
        "Patch GPT instructions: textbook figures use verified expiring proxy URLs.",
        "Regression test image atlas behavior."
    ]
}

handoff_path = APP_DIR / "HANDOFF_PATHOLOGY_HUB_V04_3_HYBRID_TEXTBOOK_FIGURE_API.json"
handoff_path.write_text(json.dumps(handoff, indent=2), encoding="utf-8")

run(["gcloud", "storage", "cp", str(yaml_path), "gs://pathology_hub/04_api_artifacts/"], check=True)
run(["gcloud", "storage", "cp", str(handoff_path), "gs://pathology_hub/06_audits/handoff_packets/"], check=True)

print("\n✅ DEPLOYED v04.3 HYBRID TEXTBOOK + FIGURE PROXY API")
print("Service:", service_url)
print("OpenAPI YAML:", yaml_path)
print("Handoff:", handoff_path)


In [ ]:

# ============================================================
# PATHOLOGY HUB — PUBLIC TEXTBOOK FIGURE WEB LIBRARY v1
#
# Creates a dedicated public bucket containing only web-safe figure images.
# Copies JPEG/PNG/WebP/GIF as-is and converts JPX/JP2/TIFF/non-web formats to JPEG.
# Does NOT make the main pathology_hub bucket public.
# ============================================================

from pathlib import Path
import os, re, io, json, time, urllib.parse, mimetypes, subprocess, concurrent.futures, threading, datetime
from collections import Counter

try:
    from PIL import Image
except Exception:
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pillow"], check=True)
    from PIL import Image

try:
    from google.cloud import storage
except Exception:
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-cloud-storage"], check=True)
    from google.cloud import storage

try:
    from tqdm.auto import tqdm
except Exception:
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tqdm"], check=True)
    from tqdm.auto import tqdm

PROJECT_ID = "pathology-annotation-project"
FIGURES_JSONL_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_lean_figures.jsonl"

PUBLIC_BUCKET = "pathology-hub-public-figures-830130787988"
PUBLIC_PREFIX = "textbook_figures_web_v1"

MAP_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_figure_web_map_v1.jsonl"
AUDIT_GCS = "gs://pathology_hub/06_audits/textbooks/figures/textbook_figure_web_derivatives_audit_v1.json"

WORK_DIR = Path("/content/pathology_hub_public_textbook_figures")
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_FIGURES = WORK_DIR / "textbook_lean_figures.jsonl"
LOCAL_MAP = WORK_DIR / "textbook_figure_web_map_v1.jsonl"
LOCAL_AUDIT = WORK_DIR / "textbook_figure_web_derivatives_audit_v1.json"

# Set to None for all ~68k. Set to 200 for a quick dry run.
MAX_RECORDS = None

SKIP_IF_DEST_EXISTS = True
MAX_WORKERS = 12

WEB_SAFE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".gif"}

def run(cmd, check=True):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run([str(x) for x in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if p.stdout:
        print(p.stdout[-5000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

def utc_now():
    return datetime.datetime.now(datetime.UTC).isoformat()

def parse_gs(uri):
    assert uri.startswith("gs://"), uri
    rest = uri[5:]
    bucket, key = rest.split("/", 1)
    return bucket, key

def gs_to_https(gs_uri):
    bucket, key = parse_gs(gs_uri)
    return f"https://storage.googleapis.com/{bucket}/{urllib.parse.quote(key, safe='/')}"

def storage_https_to_gs(url):
    if not isinstance(url, str):
        return None
    url = url.strip()
    if url.startswith("gs://"):
        return url
    if url.startswith("https://storage.googleapis.com/"):
        rest = url.replace("https://storage.googleapis.com/", "", 1)
        parts = rest.split("/", 1)
        if len(parts) == 2:
            bucket, key = parts
            return f"gs://{bucket}/{urllib.parse.unquote(key)}"
    return None

def pick_path(obj):
    for k in ["image_url", "figure_url", "image_path", "path", "gcs_uri", "gcs_path", "asset_path", "uri", "url", "source_url"]:
        v = obj.get(k)
        if isinstance(v, str) and v.strip():
            return k, v.strip()
    return None, None

def rel_from_figure_gs(gs_uri):
    marker = "/01_staged/textbooks/assets/figure_images/"
    if marker not in gs_uri:
        return None
    return gs_uri.split(marker, 1)[1]

def safe_caption(obj):
    return " ".join(str(obj.get("caption") or obj.get("legend") or obj.get("text") or "").split())

def content_type_for_ext(ext):
    ext = ext.lower()
    if ext in [".jpg", ".jpeg"]:
        return "image/jpeg"
    if ext == ".png":
        return "image/png"
    if ext == ".webp":
        return "image/webp"
    if ext == ".gif":
        return "image/gif"
    return mimetypes.guess_type("x" + ext)[0] or "application/octet-stream"

run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

exists = run(["gcloud", "storage", "buckets", "describe", f"gs://{PUBLIC_BUCKET}"], check=False)
if exists.returncode != 0:
    run([
        "gcloud", "storage", "buckets", "create", f"gs://{PUBLIC_BUCKET}",
        "--location=US",
        "--uniform-bucket-level-access"
    ], check=True)

# gcloud versions differ here. Bucket describe already shows public_access_prevention: inherited.
# Use --clear-pap if supported, and do not fail if there is nothing to clear.
run([
    "gcloud", "storage", "buckets", "update", f"gs://{PUBLIC_BUCKET}",
    "--clear-pap"
], check=False)

run([
    "gcloud", "storage", "buckets", "add-iam-policy-binding", f"gs://{PUBLIC_BUCKET}",
    "--member=allUsers",
    "--role=roles/storage.objectViewer"
], check=False)

if not LOCAL_FIGURES.exists():
    run(["gcloud", "storage", "cp", FIGURES_JSONL_GCS, str(LOCAL_FIGURES)], check=True)

client = storage.Client(project=PROJECT_ID)
pub_bucket = client.bucket(PUBLIC_BUCKET)

jobs = []
skipped_no_path = 0
skipped_bad_prefix = 0

with LOCAL_FIGURES.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue
        obj = json.loads(line)
        field, path = pick_path(obj)
        if not path:
            skipped_no_path += 1
            continue
        original_gs = storage_https_to_gs(path)
        if not original_gs:
            skipped_no_path += 1
            continue

        rel = rel_from_figure_gs(original_gs)
        if not rel:
            skipped_bad_prefix += 1
            continue

        ext = Path(rel).suffix.lower()
        if ext in WEB_SAFE_EXTS:
            dest_rel = rel
            mode = "copy_websafe"
        else:
            dest_rel = str(Path(rel).with_suffix(".jpg"))
            mode = "convert_to_jpeg"

        public_gs = f"gs://{PUBLIC_BUCKET}/{PUBLIC_PREFIX}/{dest_rel}"
        public_url = gs_to_https(public_gs)

        jobs.append({
            "line_no": line_no,
            "source_id": obj.get("source_id"),
            "source_title": obj.get("source_title") or obj.get("title") or obj.get("source_id"),
            "page": obj.get("page") or obj.get("source_page"),
            "figure_id": obj.get("figure_id"),
            "caption": safe_caption(obj),
            "path_field": field,
            "original_path_value": path,
            "original_gs_uri": original_gs,
            "original_ext": ext,
            "mode": mode,
            "public_gs_uri": public_gs,
            "public_url": public_url,
            "public_rel_path": f"{PUBLIC_PREFIX}/{dest_rel}",
        })

        if MAX_RECORDS is not None and len(jobs) >= MAX_RECORDS:
            break

print("Jobs:", len(jobs))
print("Skipped no path:", skipped_no_path)
print("Skipped bad prefix:", skipped_bad_prefix)
print("Modes:", Counter(j["mode"] for j in jobs))
print("Original exts:", Counter(j["original_ext"] for j in jobs).most_common(20))

done_originals = set()
if LOCAL_MAP.exists():
    with LOCAL_MAP.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    done_originals.add(json.loads(line).get("original_gs_uri"))
                except Exception:
                    pass

print("Already mapped locally:", len(done_originals))

def process_job(job):
    try:
        if job["original_gs_uri"] in done_originals:
            return {"status": "already_mapped", **job}

        src_bucket_name, src_key = parse_gs(job["original_gs_uri"])
        dst_bucket_name, dst_key = parse_gs(job["public_gs_uri"])

        src_bucket = client.bucket(src_bucket_name)
        src_blob = src_bucket.blob(src_key)
        dst_blob = pub_bucket.blob(dst_key)

        if SKIP_IF_DEST_EXISTS and dst_blob.exists(client):
            dst_blob.reload(client)
            rec = dict(job)
            rec.update({
                "status": "exists",
                "converted": job["mode"] == "convert_to_jpeg",
                "public_content_type": dst_blob.content_type,
                "public_size": dst_blob.size,
                "width": None,
                "height": None,
                "error": None,
            })
            return rec

        if job["mode"] == "copy_websafe":
            src_blob.reload(client)
            copied = src_bucket.copy_blob(src_blob, pub_bucket, dst_key)
            copied.content_type = content_type_for_ext(job["original_ext"])
            copied.cache_control = "public, max-age=31536000"
            copied.patch()
            rec = dict(job)
            rec.update({
                "status": "copied",
                "converted": False,
                "public_content_type": copied.content_type,
                "public_size": copied.size,
                "width": None,
                "height": None,
                "error": None,
            })
            return rec

        raw = src_blob.download_as_bytes()
        im = Image.open(io.BytesIO(raw))
        im.load()
        width, height = im.size
        if im.mode not in ("RGB", "L"):
            im = im.convert("RGB")
        elif im.mode == "L":
            im = im.convert("RGB")

        out = io.BytesIO()
        im.save(out, format="JPEG", quality=88, optimize=True)
        data = out.getvalue()

        dst_blob.cache_control = "public, max-age=31536000"
        dst_blob.upload_from_string(data, content_type="image/jpeg")

        rec = dict(job)
        rec.update({
            "status": "converted",
            "converted": True,
            "public_content_type": "image/jpeg",
            "public_size": len(data),
            "width": width,
            "height": height,
            "error": None,
        })
        return rec

    except Exception as e:
        rec = dict(job)
        rec.update({
            "status": "error",
            "converted": job.get("mode") == "convert_to_jpeg",
            "public_content_type": None,
            "public_size": None,
            "width": None,
            "height": None,
            "error": repr(e)[:1000],
        })
        return rec

status_counter = Counter()
ext_counter = Counter()
errors = []
start = time.time()

with LOCAL_MAP.open("a", encoding="utf-8") as out_f:
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = [ex.submit(process_job, j) for j in jobs]
        for fut in tqdm(concurrent.futures.as_completed(futs), total=len(futs)):
            rec = fut.result()
            status_counter[rec["status"]] += 1
            ext_counter[rec.get("original_ext")] += 1
            if rec["status"] == "error":
                errors.append(rec)
            out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            if sum(status_counter.values()) % 500 == 0:
                out_f.flush()

elapsed = time.time() - start

print("Elapsed seconds:", round(elapsed, 1))
print("Status:", status_counter)
print("Errors:", len(errors))
if errors[:5]:
    print("First errors:")
    for e in errors[:5]:
        print(json.dumps({k:e.get(k) for k in ["source_id","page","figure_id","original_gs_uri","error"]}, indent=2)[:1200])

# Test public URLs, including converted JPX if possible.
import urllib.request

test_rows = []
converted_rows = []
with LOCAL_MAP.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rec = json.loads(line)
            if rec.get("status") in {"copied", "converted", "exists"}:
                if len(test_rows) < 10:
                    test_rows.append(rec)
                if rec.get("converted") and len(converted_rows) < 10:
                    converted_rows.append(rec)
        if len(test_rows) >= 10 and len(converted_rows) >= 10:
            break

test_rows = converted_rows + test_rows

print("\nTesting public URLs...")
public_test = []
for rec in test_rows[:20]:
    url = rec["public_url"]
    row = {"url": url, "status": None, "content_type": None, "bytes": None, "source_id": rec.get("source_id"), "figure_id": rec.get("figure_id"), "original_ext": rec.get("original_ext"), "converted": rec.get("converted")}
    try:
        req = urllib.request.Request(url, method="GET")
        with urllib.request.urlopen(req, timeout=20) as resp:
            data = resp.read(2048)
            row["status"] = resp.status
            row["content_type"] = resp.headers.get("content-type")
            row["bytes"] = len(data)
    except Exception as e:
        row["status"] = "ERR"
        row["error"] = repr(e)[:300]
    public_test.append(row)
    print(row)

audit = {
    "schema_version": "textbook_figure_web_derivatives_audit.v1",
    "created_at_utc": utc_now(),
    "workstream": "Textbook Figure Serving / Web Derivatives",
    "purpose": "Create public browser-safe textbook figure URLs without exposing the main pathology_hub bucket.",
    "input_figures_jsonl_gcs": FIGURES_JSONL_GCS,
    "public_bucket": PUBLIC_BUCKET,
    "public_prefix": PUBLIC_PREFIX,
    "map_gcs": MAP_GCS,
    "audit_gcs": AUDIT_GCS,
    "jobs_total": len(jobs),
    "status_counts": dict(status_counter),
    "original_ext_counts": dict(ext_counter),
    "errors_count": len(errors),
    "errors_sample": errors[:20],
    "public_url_tests": public_test,
    "notes": [
        "Main pathology_hub bucket was not made public.",
        "Dedicated public bucket contains only figure web derivatives/copies.",
        "JPX/JP2/TIFF/non-web images are converted to JPEG.",
        "Web-safe images are copied as-is.",
        "This map can be used by v04.4 to prefer direct public figure URLs over proxy URLs."
    ],
}

LOCAL_AUDIT.write_text(json.dumps(audit, indent=2), encoding="utf-8")

run(["gcloud", "storage", "cp", str(LOCAL_MAP), MAP_GCS], check=True)
run(["gcloud", "storage", "cp", str(LOCAL_AUDIT), AUDIT_GCS], check=True)

print("\n✅ PUBLIC TEXTBOOK FIGURE WEB LIBRARY COMPLETE")
print("Public bucket:", f"gs://{PUBLIC_BUCKET}/{PUBLIC_PREFIX}/")
print("Map:", MAP_GCS)
print("Audit:", AUDIT_GCS)
print("Example public URL:", public_test[0]["url"] if public_test else "none")


In [ ]:
# ============================================================
# PATHOLOGY HUB COLAB CONFIG / RESET CELL
# For testing v04.4 API after runtime disconnect
# ============================================================

import os, json, subprocess, requests, textwrap, sys
from pathlib import Path

# ----------------------------
# Project / service config
# ----------------------------

PROJECT_ID = "pathology-annotation-project"
REGION = "us-central1"

SERVICE_URL = "https://pathology-hub-v04-vorn5q2kga-uc.a.run.app"

API_SECRET_NAME = "pathology-hub-api-key"
OPENAI_SECRET_NAME = "OPEN_AI_KEY_01"

# Canonical artifacts
GCS_OPENAPI_V044 = "gs://pathology_hub/04_api_artifacts/openapi_pathology_hub_unified_searchEvidence_hybrid_textbooks_publicfigs_v1_5_4.yaml"
GCS_HANDOFF_V044 = "gs://pathology_hub/06_audits/handoff_packets/HANDOFF_PATHOLOGY_HUB_V04_4_PUBLIC_TEXTBOOK_FIGURES_API.json"

GCS_TEXTBOOK_FIG_MAP_FILTERED = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_figure_web_map_v1_FILTERED_NO_MCKEE_DORFMAN.jsonl"
PUBLIC_FIGURE_BUCKET = "pathology-hub-public-figures-830130787988"

LOCAL_ARTIFACT_DIR = Path("/content/pathology_hub_runtime")
LOCAL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_OPENAPI_V044 = LOCAL_ARTIFACT_DIR / "openapi_pathology_hub_unified_searchEvidence_hybrid_textbooks_publicfigs_v1_5_4.yaml"
LOCAL_HANDOFF_V044 = LOCAL_ARTIFACT_DIR / "HANDOFF_PATHOLOGY_HUB_V04_4_PUBLIC_TEXTBOOK_FIGURES_API.json"

# ----------------------------
# Helpers
# ----------------------------

def run(cmd, check=True, quiet=False):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout and not quiet:
        print(p.stdout[-8000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

# ----------------------------
# Authenticate Colab to GCP
# ----------------------------

try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Colab authenticated to Google.")
except Exception as e:
    print("Auth note:", repr(e))

run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

# ----------------------------
# Confirm secrets exist
# ----------------------------

run(["gcloud", "secrets", "describe", API_SECRET_NAME], check=True)
run(["gcloud", "secrets", "describe", OPENAI_SECRET_NAME], check=False)

# Retrieve Pathology Hub API key for direct tests
PATHOLOGY_HUB_API_KEY = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", API_SECRET_NAME
], text=True).strip()

assert PATHOLOGY_HUB_API_KEY, "Failed to retrieve pathology-hub-api-key"
print("✅ Retrieved Pathology Hub API key from Secret Manager.")

# ----------------------------
# Download current v04.4 Action YAML + handoff
# ----------------------------

run(["gcloud", "storage", "cp", GCS_OPENAPI_V044, str(LOCAL_OPENAPI_V044)], check=False)
run(["gcloud", "storage", "cp", GCS_HANDOFF_V044, str(LOCAL_HANDOFF_V044)], check=False)

print("\nLocal v04.4 OpenAPI YAML:")
print(LOCAL_OPENAPI_V044)

print("\nLocal v04.4 handoff:")
print(LOCAL_HANDOFF_V044)

# ----------------------------
# API helper functions
# ----------------------------

def searchEvidence(
    query,
    sources=("textbooks",),
    max_results=1,
    include_figures=False,
    max_figures=0,
    compact=True,
    excerpt_char_limit=900,
    timeout=300,
):
    payload = {
        "query": query,
        "sources": list(sources),
        "max_results": max_results,
        "include_figures": include_figures,
        "max_figures": max_figures,
        "compact": compact,
        "excerpt_char_limit": excerpt_char_limit,
    }

    r = requests.post(
        f"{SERVICE_URL}/evidence/search",
        headers={
            "X-API-Key": PATHOLOGY_HUB_API_KEY,
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=timeout,
    )

    print("\nQUERY:", query)
    print("SOURCES:", sources)
    print("HTTP:", r.status_code)

    try:
        data = r.json()
    except Exception:
        print(r.text[:3000])
        raise

    print("source_status:", data.get("source_status"))
    print("search_mode:", data.get("search_mode"))
    print("counts:", {
        "who": len(data.get("who_results", [])),
        "textbooks": len(data.get("textbook_results", [])),
        "pathout": len(data.get("pathout_results", [])),
        "journals": len(data.get("journal_results", [])),
        "figures": len(data.get("figures", [])),
    })

    return data

def show_first_hits(data, max_chars=900):
    for group in ["who_results", "textbook_results", "pathout_results", "journal_results"]:
        hits = data.get(group, [])
        if not hits:
            continue
        print("\n" + "="*80)
        print(group)
        for i, h in enumerate(hits[:3], 1):
            print(f"\nHIT {i}")
            print("title:", h.get("title") or h.get("entity_name"))
            print("source:", h.get("source_name") or h.get("source"))
            print("page:", h.get("page"))
            print("section:", h.get("section") or h.get("section_heading"))
            print("retrieval_mode:", h.get("retrieval_mode"))
            print("url:", h.get("source_url") or h.get("url"))
            print("excerpt:", (h.get("excerpt") or h.get("text") or h.get("chunk_text") or "")[:max_chars])

    figs = data.get("figures", [])
    if figs:
        print("\n" + "="*80)
        print("figures")
        for i, f in enumerate(figs[:5], 1):
            print(f"\nFIG {i}")
            print("title:", f.get("title"))
            print("source:", f.get("source_id") or f.get("source"))
            print("page:", f.get("page"))
            print("figure_id:", f.get("figure_id"))
            print("figure_url:", f.get("figure_url") or f.get("image_url"))
            print("caption:", (f.get("caption") or "")[:500])

# ----------------------------
# Health check
# ----------------------------

print("\nChecking v04.4 /health ...")
health = requests.get(f"{SERVICE_URL}/health", timeout=300)
print("health HTTP:", health.status_code)
health_json = health.json()
print(json.dumps({
    "version": health_json.get("version"),
    "textbook_search_mode": health_json.get("textbook_search_mode"),
    "textbook_figure_mode": health_json.get("textbook_figure_mode"),
    "vectorized": health_json.get("vectorized"),
    "api_exposed": health_json.get("api_exposed"),
    "public_figure_map_enabled": health_json.get("public_figure_map_enabled"),
    "public_figure_map_records_loaded": health_json.get("public_figure_map_records_loaded"),
}, indent=2))

assert health.status_code == 200
assert health_json.get("vectorized") is True
assert health_json.get("public_figure_map_enabled") is True

# ----------------------------
# Quick smoke tests
# ----------------------------

print("\nRunning quick source smoke tests...")

textbook_test = searchEvidence(
    "fallopian tube precursor lesion abnormal p53 increased proliferation",
    sources=("textbooks",),
    max_results=3,
    include_figures=True,
    max_figures=3,
)

show_first_hits(textbook_test, max_chars=700)

journal_test = searchEvidence(
    "sinonasal adenocarcinoma wood dust",
    sources=("journals",),
    max_results=3,
    include_figures=False,
    max_figures=0,
)

show_first_hits(journal_test, max_chars=700)

print("\n✅ CONFIG READY")
print("Use searchEvidence(...) for further API tests.")
print("Use LOCAL_OPENAPI_V044 for GPT Builder Action schema if needed:")
print(LOCAL_OPENAPI_V044)

In [ ]:
# ============================================================
# PATHOLOGY HUB — JOURNAL VECTOR BUILD
#
# Workstream: Journal RAG / Vectorization
#
# Builds:
# gs://pathology_hub/03_indexes/journals/vector/journal_embeddings.npy
# gs://pathology_hub/03_indexes/journals/vector/journal_faiss.index
# gs://pathology_hub/03_indexes/journals/vector/journal_vector_docstore.jsonl
# gs://pathology_hub/03_indexes/journals/vector/journal_vector_manifest.json
# gs://pathology_hub/06_audits/journals/vector/journal_vector_build_audit.json
#
# NOTE:
# - This makes journals vectorized/searchable as artifacts.
# - API is NOT updated until a later v04.5 deploy patches journals to hybrid FTS+vector.
# ============================================================

import os, sys, json, time, math, gzip, random, hashlib, subprocess, gc
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

# ----------------------------
# CONFIG
# ----------------------------

PROJECT_ID = "pathology-annotation-project"

# Leave None to autodetect. Override if needed:
JOURNAL_CHUNKS_GCS = None
# Example:
# JOURNAL_CHUNKS_GCS = "gs://pathology_hub/02_normalized/journals/journal_chunks.jsonl"

OPENAI_SECRET_NAME = "OPEN_AI_KEY_01"
EMBEDDING_MODEL = "text-embedding-3-small"

MIN_TEXT_CHARS = 25
MAX_EMBED_CHARS = 7000
MAX_DOCSTORE_TEXT_CHARS = 6000

EMBED_BATCH_SIZE = 96
SHARD_SIZE = 2048

OUT_PREFIX_GCS = "gs://pathology_hub/03_indexes/journals/vector"
AUDIT_PREFIX_GCS = "gs://pathology_hub/06_audits/journals/vector"
CHECKPOINT_PREFIX_GCS = f"{OUT_PREFIX_GCS}/_build_shards_v1"

LOCAL = Path("/content/pathology_hub_journal_vector_build")
LOCAL.mkdir(parents=True, exist_ok=True)

LOCAL_CHUNKS = LOCAL / "journal_chunks_source.jsonl"
LOCAL_DOCSTORE = LOCAL / "journal_vector_docstore.jsonl"
LOCAL_EMBEDDINGS = LOCAL / "journal_embeddings.npy"
LOCAL_FAISS = LOCAL / "journal_faiss.index"
LOCAL_MANIFEST = LOCAL / "journal_vector_manifest.json"
LOCAL_AUDIT = LOCAL / "journal_vector_build_audit.json"

# ----------------------------
# BASIC HELPERS
# ----------------------------

def run(cmd, check=True, quiet=False):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout and not quiet:
        print(p.stdout[-8000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

def gcs_exists(uri):
    p = run(["gcloud", "storage", "ls", uri], check=False, quiet=True)
    return p.returncode == 0

def gcs_size(uri):
    p = run(["gcloud", "storage", "du", uri], check=False, quiet=True)
    if p.returncode != 0:
        return -1
    for line in p.stdout.splitlines():
        toks = line.strip().split()
        if toks:
            try:
                return int(toks[0])
            except Exception:
                pass
    return -1

def open_text(path):
    path = Path(path)
    if str(path).endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8", errors="replace")
    return open(path, "r", encoding="utf-8", errors="replace")

def sha256_text(s):
    return hashlib.sha256((s or "").encode("utf-8", errors="ignore")).hexdigest()

def first_nonempty(*vals):
    for v in vals:
        if v is None:
            continue
        if isinstance(v, str) and v.strip():
            return v.strip()
        if not isinstance(v, str) and v:
            return v
    return None

def nested(obj, *keys):
    cur = obj
    for k in keys:
        if not isinstance(cur, dict):
            return None
        cur = cur.get(k)
    return cur

# ----------------------------
# AUTH + INSTALL
# ----------------------------

try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Colab authenticated.")
except Exception as e:
    print("Auth note:", repr(e))

run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

print("\nInstalling packages...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "openai>=1.0.0", "faiss-cpu==1.8.0.post1", "tqdm", "numpy"],
    check=True,
)

import numpy as np
import faiss
from tqdm.auto import tqdm
from openai import OpenAI

OPENAI_API_KEY = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", OPENAI_SECRET_NAME,
], text=True).strip()

assert OPENAI_API_KEY, "Could not retrieve OPEN_AI_KEY_01"
client = OpenAI(api_key=OPENAI_API_KEY)
print("✅ OpenAI key loaded from Secret Manager.")

# ----------------------------
# DISCOVER JOURNAL CHUNKS
# ----------------------------

def discover_journal_chunks():
    candidates = []

    defaults = [
        "gs://pathology_hub/02_normalized/journals/journal_chunks.jsonl",
        "gs://pathology_hub/02_normalized/journals/journal_chunks_all.jsonl",
        "gs://pathology_hub/02_normalized/journals/all/journal_chunks.jsonl",
        "gs://pathology_hub/02_normalized/journals/combined/journal_chunks.jsonl",
    ]

    for uri in defaults:
        if gcs_exists(uri):
            candidates.append(uri)

    p = run(
        ["gcloud", "storage", "ls", "--recursive", "gs://pathology_hub/02_normalized/journals/"],
        check=False,
        quiet=True,
    )
    if p.returncode == 0:
        for line in p.stdout.splitlines():
            uri = line.strip()
            low = uri.lower()
            if uri.startswith("gs://") and ("chunk" in low) and (low.endswith(".jsonl") or low.endswith(".jsonl.gz")):
                candidates.append(uri)

    # dedupe
    candidates = sorted(set(candidates))
    if not candidates:
        raise RuntimeError("Could not find journal chunks JSONL under gs://pathology_hub/02_normalized/journals/")

    scored = []
    for uri in candidates:
        scored.append((gcs_size(uri), uri))

    scored.sort(reverse=True)
    print("\nCandidate journal chunk files:")
    for size, uri in scored[:20]:
        print(size, uri)

    return scored[0][1]

if JOURNAL_CHUNKS_GCS is None:
    JOURNAL_CHUNKS_GCS = discover_journal_chunks()

print("\nUSING JOURNAL CHUNKS:")
print(JOURNAL_CHUNKS_GCS)

run(["gcloud", "storage", "cp", JOURNAL_CHUNKS_GCS, str(LOCAL_CHUNKS)], check=True)

# ----------------------------
# NORMALIZE RECORDS FOR EMBEDDING
# ----------------------------

def extract_text(obj):
    prov = obj.get("provenance") if isinstance(obj.get("provenance"), dict) else {}
    vals = [
        obj.get("chunk_text"),
        obj.get("text"),
        obj.get("excerpt"),
        obj.get("caption"),
        obj.get("figure_caption"),
        obj.get("abstract"),
        obj.get("body"),
        obj.get("content"),
        prov.get("chunk_text") if isinstance(prov, dict) else None,
    ]
    txt = first_nonempty(*vals)
    if isinstance(txt, list):
        txt = " ".join(str(x) for x in txt)
    return " ".join(str(txt or "").replace("\x00", " ").split())

def get_meta(obj):
    prov = obj.get("provenance") if isinstance(obj.get("provenance"), dict) else {}

    title = first_nonempty(
        obj.get("title"),
        obj.get("article_title"),
        obj.get("paper_title"),
        prov.get("title") if isinstance(prov, dict) else None,
        prov.get("article_title") if isinstance(prov, dict) else None,
    )

    journal = first_nonempty(
        obj.get("journal"),
        obj.get("source_name"),
        obj.get("source"),
        prov.get("journal") if isinstance(prov, dict) else None,
        prov.get("source_name") if isinstance(prov, dict) else None,
    )

    doi = first_nonempty(obj.get("doi"), prov.get("doi") if isinstance(prov, dict) else None)
    url = first_nonempty(
        obj.get("source_url"),
        obj.get("url"),
        obj.get("article_url"),
        prov.get("source_url") if isinstance(prov, dict) else None,
        prov.get("url") if isinstance(prov, dict) else None,
    )

    article_id = first_nonempty(obj.get("article_id"), obj.get("source_id"), prov.get("article_id") if isinstance(prov, dict) else None)
    source_id = first_nonempty(obj.get("source_id"), obj.get("journal_id"), journal)
    section = first_nonempty(obj.get("section"), obj.get("section_heading"), obj.get("heading"), prov.get("section") if isinstance(prov, dict) else None)
    chunk_type = first_nonempty(obj.get("chunk_type"), obj.get("type"), prov.get("chunk_type") if isinstance(prov, dict) else None)
    year = first_nonempty(obj.get("year"), obj.get("publication_year"), prov.get("year") if isinstance(prov, dict) else None)

    return {
        "title": str(title or ""),
        "journal": str(journal or ""),
        "doi": str(doi or ""),
        "url": str(url or ""),
        "article_id": str(article_id or ""),
        "source_id": str(source_id or ""),
        "section": str(section or ""),
        "chunk_type": str(chunk_type or ""),
        "year": str(year or ""),
    }

def make_embedding_text(meta, text):
    parts = []
    for label, key in [
        ("Title", "title"),
        ("Journal", "journal"),
        ("Year", "year"),
        ("DOI", "doi"),
        ("Section", "section"),
        ("Chunk type", "chunk_type"),
    ]:
        v = meta.get(key)
        if v:
            parts.append(f"{label}: {v}")
    parts.append("Text: " + text)
    return "\n".join(parts)[:MAX_EMBED_CHARS]

docs = []
embed_texts = []
seen_chunk_ids = set()
stats = Counter()

print("\nLoading and normalizing journal chunks...")

with open_text(LOCAL_CHUNKS) as f:
    for line_no, line in enumerate(tqdm(f), start=1):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except Exception:
            stats["bad_json"] += 1
            continue

        text = extract_text(obj)
        if len(text) < MIN_TEXT_CHARS:
            stats["skipped_short"] += 1
            continue

        meta = get_meta(obj)

        chunk_id = first_nonempty(
            obj.get("chunk_id"),
            obj.get("id"),
            f"{meta.get('article_id')}:{obj.get('chunk_index', line_no)}",
        )
        chunk_id = str(chunk_id)

        if chunk_id in seen_chunk_ids:
            stats["duplicate_chunk_id"] += 1
            continue
        seen_chunk_ids.add(chunk_id)

        emb_text = make_embedding_text(meta, text)
        if len(emb_text) < MIN_TEXT_CHARS:
            stats["skipped_empty_embedding_text"] += 1
            continue

        vector_row = len(docs)

        docs.append({
            "schema_version": "journal_vector_docstore.v1",
            "vector_row": vector_row,
            "chunk_id": chunk_id,
            "article_id": meta.get("article_id", ""),
            "source_id": meta.get("source_id", ""),
            "title": meta.get("title", ""),
            "journal": meta.get("journal", ""),
            "year": meta.get("year", ""),
            "doi": meta.get("doi", ""),
            "url": meta.get("url", ""),
            "section": meta.get("section", ""),
            "chunk_type": meta.get("chunk_type", ""),
            "text": text[:MAX_DOCSTORE_TEXT_CHARS],
            "embedding_text_sha256": sha256_text(emb_text),
        })
        embed_texts.append(emb_text)
        stats["kept"] += 1

N = len(docs)
print("\nNormalization stats:")
print(json.dumps(dict(stats), indent=2))
print("Kept docs:", N)

assert N > 0, "No journal chunks loaded."

# ----------------------------
# EMBEDDING WITH SHARDED GCS CHECKPOINTS
# ----------------------------

def embed_batch(texts):
    cleaned = [str(t or "").replace("\x00", " ")[:MAX_EMBED_CHARS] for t in texts]

    for attempt in range(10):
        try:
            resp = client.embeddings.create(
                model=EMBEDDING_MODEL,
                input=cleaned,
                encoding_format="float",
            )
            return [d.embedding for d in resp.data]
        except Exception as e:
            wait = min(60, (2 ** attempt) + random.random() * 3)
            print(f"Embedding error attempt {attempt+1}/10: {repr(e)[:300]}")
            print(f"Sleeping {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError("Embedding batch failed after retries.")

num_shards = math.ceil(N / SHARD_SIZE)
print(f"\nEmbedding {N} docs in {num_shards} shards of {SHARD_SIZE}...")

local_shard_npys = []
local_shard_docs = []

for shard_id in range(num_shards):
    start = shard_id * SHARD_SIZE
    end = min(N, start + SHARD_SIZE)

    shard_npy = LOCAL / f"journal_embeddings_shard_{shard_id:05d}.npy"
    shard_doc = LOCAL / f"journal_docstore_shard_{shard_id:05d}.jsonl"

    gcs_npy = f"{CHECKPOINT_PREFIX_GCS}/journal_embeddings_shard_{shard_id:05d}.npy"
    gcs_doc = f"{CHECKPOINT_PREFIX_GCS}/journal_docstore_shard_{shard_id:05d}.jsonl"

    if gcs_exists(gcs_npy) and gcs_exists(gcs_doc):
        print(f"Shard {shard_id:05d}: checkpoint exists in GCS; downloading.")
        if not shard_npy.exists():
            run(["gcloud", "storage", "cp", gcs_npy, str(shard_npy)], check=True, quiet=True)
        if not shard_doc.exists():
            run(["gcloud", "storage", "cp", gcs_doc, str(shard_doc)], check=True, quiet=True)

        local_shard_npys.append(shard_npy)
        local_shard_docs.append(shard_doc)
        continue

    print(f"\nShard {shard_id+1}/{num_shards}: rows {start}:{end}")

    shard_embs = []
    shard_texts = embed_texts[start:end]

    for b0 in tqdm(range(0, len(shard_texts), EMBED_BATCH_SIZE), desc=f"embed shard {shard_id:05d}"):
        batch = shard_texts[b0:b0 + EMBED_BATCH_SIZE]
        batch_embs = embed_batch(batch)
        shard_embs.extend(batch_embs)

    arr = np.asarray(shard_embs, dtype="float32")
    assert arr.shape[0] == end - start, f"Shard row mismatch: {arr.shape}"

    np.save(shard_npy, arr)

    with shard_doc.open("w", encoding="utf-8") as f:
        for d in docs[start:end]:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

    run(["gcloud", "storage", "cp", str(shard_npy), gcs_npy], check=True, quiet=True)
    run(["gcloud", "storage", "cp", str(shard_doc), gcs_doc], check=True, quiet=True)

    local_shard_npys.append(shard_npy)
    local_shard_docs.append(shard_doc)

print("\n✅ All embedding shards complete or recovered from checkpoint.")

# ----------------------------
# CONCATENATE, NORMALIZE, BUILD FAISS
# ----------------------------

print("\nConcatenating shards...")
arrays = []
for p in local_shard_npys:
    arrays.append(np.load(p))

embeddings = np.vstack(arrays).astype("float32")
assert embeddings.shape[0] == N, f"Expected {N}, got {embeddings.shape[0]}"

dim = embeddings.shape[1]
print("Embedding matrix:", embeddings.shape)

print("L2-normalizing embeddings for cosine/IP search...")
faiss.normalize_L2(embeddings)

print("Building FAISS IndexFlatIP...")
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print("FAISS rows:", index.ntotal)

np.save(LOCAL_EMBEDDINGS, embeddings)
faiss.write_index(index, str(LOCAL_FAISS))

with LOCAL_DOCSTORE.open("w", encoding="utf-8") as f:
    for d in docs:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

# ----------------------------
# VECTOR SMOKE TEST
# ----------------------------

def vector_search(query, k=5):
    q_emb = np.asarray(embed_batch([query]), dtype="float32")
    faiss.normalize_L2(q_emb)
    scores, ids = index.search(q_emb, k)
    out = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        d = docs[int(idx)]
        out.append({
            "score": float(score),
            "vector_row": int(idx),
            "title": d.get("title"),
            "journal": d.get("journal"),
            "doi": d.get("doi"),
            "url": d.get("url"),
            "section": d.get("section"),
            "chunk_type": d.get("chunk_type"),
            "excerpt": d.get("text", "")[:500],
        })
    return out

smoke_queries = [
    "intestinal type adenocarcinoma sinonasal occupational wood dust",
    "sinonasal adenocarcinoma CK20 CDX2 SATB2",
    "SMARCB1 deficient sinonasal carcinoma",
    "MTAP loss mesothelioma",
    "EGFR exon 20 insertion lung adenocarcinoma",
    "MSI endometrial carcinoma MLH1 methylation",
]

smoke_results = {}
print("\nVector smoke tests:")
for q in smoke_queries:
    hits = vector_search(q, k=5)
    smoke_results[q] = hits
    print("\n" + "="*80)
    print("QUERY:", q)
    for i, h in enumerate(hits[:3], 1):
        print(f"\nHIT {i} | score={h['score']:.4f}")
        print("title:", h["title"])
        print("journal:", h["journal"])
        print("doi:", h["doi"])
        print("url:", h["url"])
        print("excerpt:", h["excerpt"])

# ----------------------------
# MANIFEST + AUDIT
# ----------------------------

created_at = datetime.now(timezone.utc).isoformat()

manifest = {
    "schema_version": "journal_vector_manifest.v1",
    "created_at_utc": created_at,
    "corpus": "journals",
    "workstream": "Journal RAG / Vectorization",
    "index_type": "faiss_IndexFlatIP_cosine_normalized",
    "searchable": True,
    "vectorized": True,
    "api_exposed": False,
    "api_exposed_note": "Vector artifacts built only. API requires v04.5 patch before journal vector retrieval is exposed.",
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dim": int(dim),
    "record_count": int(N),
    "source_chunks_gcs": JOURNAL_CHUNKS_GCS,
    "artifact_paths": {
        "embeddings_npy": f"{OUT_PREFIX_GCS}/journal_embeddings.npy",
        "faiss_index": f"{OUT_PREFIX_GCS}/journal_faiss.index",
        "docstore_jsonl": f"{OUT_PREFIX_GCS}/journal_vector_docstore.jsonl",
        "manifest_json": f"{OUT_PREFIX_GCS}/journal_vector_manifest.json",
        "audit_json": f"{AUDIT_PREFIX_GCS}/journal_vector_build_audit.json",
        "checkpoint_prefix": CHECKPOINT_PREFIX_GCS,
    },
    "normalization_stats": dict(stats),
    "smoke_queries": smoke_queries,
}

audit = {
    "schema_version": "journal_vector_build_audit.v1",
    "created_at_utc": created_at,
    "pass": True,
    "checks": {
        "records_loaded": int(N),
        "embedding_rows_match_docstore": bool(embeddings.shape[0] == N),
        "faiss_rows_match_docstore": bool(index.ntotal == N),
        "embedding_dim": int(dim),
        "smoke_query_count": len(smoke_queries),
    },
    "smoke_results_top1": {
        q: (hits[0] if hits else None)
        for q, hits in smoke_results.items()
    },
    "known_limitations": [
        "Journal vector artifacts are not API-exposed until v04.5 or later.",
        "Hybrid retrieval should combine existing journal FTS with this FAISS vector index; do not use vector-only for final API.",
        "Source metadata quality depends on normalized journal_chunks.jsonl fields.",
    ],
}

LOCAL_MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
LOCAL_AUDIT.write_text(json.dumps(audit, indent=2), encoding="utf-8")

# ----------------------------
# UPLOAD FINAL ARTIFACTS
# ----------------------------

print("\nUploading final journal vector artifacts...")

run(["gcloud", "storage", "cp", str(LOCAL_EMBEDDINGS), f"{OUT_PREFIX_GCS}/journal_embeddings.npy"], check=True)
run(["gcloud", "storage", "cp", str(LOCAL_FAISS), f"{OUT_PREFIX_GCS}/journal_faiss.index"], check=True)
run(["gcloud", "storage", "cp", str(LOCAL_DOCSTORE), f"{OUT_PREFIX_GCS}/journal_vector_docstore.jsonl"], check=True)
run(["gcloud", "storage", "cp", str(LOCAL_MANIFEST), f"{OUT_PREFIX_GCS}/journal_vector_manifest.json"], check=True)
run(["gcloud", "storage", "cp", str(LOCAL_AUDIT), f"{AUDIT_PREFIX_GCS}/journal_vector_build_audit.json"], check=True)

# ----------------------------
# FINAL SUMMARY
# ----------------------------

print("\n✅ JOURNAL VECTOR BUILD COMPLETE")
print("Records vectorized:", N)
print("Embedding dim:", dim)
print("Model:", EMBEDDING_MODEL)
print("\nGCS artifacts:")
print(f"{OUT_PREFIX_GCS}/journal_embeddings.npy")
print(f"{OUT_PREFIX_GCS}/journal_faiss.index")
print(f"{OUT_PREFIX_GCS}/journal_vector_docstore.jsonl")
print(f"{OUT_PREFIX_GCS}/journal_vector_manifest.json")
print(f"{AUDIT_PREFIX_GCS}/journal_vector_build_audit.json")
print("\nNext step: deploy v04.5 to use journals hybrid FTS + FAISS vector RRF.")

In [ ]:
# ============================================================
# JOURNAL VECTOR BUILD RECOVERY CELL
#
# Use after the original build completed all embedding shards but failed at:
# faiss.normalize_L2(embeddings) -> ValueError: input not a numpy array
#
# This DOES NOT re-embed the journal corpus.
# It resumes from GCS shard checkpoints and builds final FAISS/docstore/manifest/audit.
# ============================================================

import os, sys, json, time, subprocess, hashlib, gc
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

PROJECT_ID = "pathology-annotation-project"
OPENAI_SECRET_NAME = "OPEN_AI_KEY_01"
EMBEDDING_MODEL = "text-embedding-3-small"

CHECKPOINT_PREFIX_GCS = "gs://pathology_hub/03_indexes/journals/vector/_build_shards_v1"
OUT_PREFIX_GCS = "gs://pathology_hub/03_indexes/journals/vector"
AUDIT_PREFIX_GCS = "gs://pathology_hub/06_audits/journals/vector"

LOCAL = Path("/content/pathology_hub_journal_vector_recovery")
LOCAL.mkdir(parents=True, exist_ok=True)

LOCAL_DOCSTORE = LOCAL / "journal_vector_docstore.jsonl"
LOCAL_EMBEDDINGS = LOCAL / "journal_embeddings.npy"
LOCAL_FAISS = LOCAL / "journal_faiss.index"
LOCAL_MANIFEST = LOCAL / "journal_vector_manifest.json"
LOCAL_AUDIT = LOCAL / "journal_vector_build_audit.json"

def run(cmd, check=True, quiet=False):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout and not quiet:
        print(p.stdout[-8000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

# ----------------------------
# Auth + packages
# ----------------------------

try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Colab authenticated.")
except Exception as e:
    print("Auth note:", repr(e))

run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

# Keep this minimal; avoid re-running original embedding build.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "numpy==1.26.4", "faiss-cpu==1.8.0.post1", "openai>=1.0.0", "tqdm"],
    check=True,
)

import numpy as np
import faiss
from tqdm.auto import tqdm
from openai import OpenAI

print("numpy:", np.__version__)
print("faiss:", faiss.__version__ if hasattr(faiss, "__version__") else "unknown")

# ----------------------------
# Discover saved shard checkpoints
# ----------------------------

p = run(["gcloud", "storage", "ls", "--recursive", CHECKPOINT_PREFIX_GCS + "/"], check=True, quiet=True)
uris = [x.strip() for x in p.stdout.splitlines() if x.strip().startswith("gs://")]

npy_uris = sorted([u for u in uris if u.endswith(".npy") and "journal_embeddings_shard_" in u])
doc_uris = sorted([u for u in uris if u.endswith(".jsonl") and "journal_docstore_shard_" in u])

print("Embedding shard count:", len(npy_uris))
print("Docstore shard count:", len(doc_uris))

assert len(npy_uris) > 0, "No embedding shards found in GCS."
assert len(npy_uris) == len(doc_uris), "Embedding/docstore shard count mismatch."

# ----------------------------
# Download shards
# ----------------------------

local_npys = []
local_docs = []

for uri in tqdm(npy_uris, desc="download npy shards"):
    local_path = LOCAL / Path(uri).name
    if not local_path.exists():
        run(["gcloud", "storage", "cp", uri, str(local_path)], check=True, quiet=True)
    local_npys.append(local_path)

for uri in tqdm(doc_uris, desc="download docstore shards"):
    local_path = LOCAL / Path(uri).name
    if not local_path.exists():
        run(["gcloud", "storage", "cp", uri, str(local_path)], check=True, quiet=True)
    local_docs.append(local_path)

# ----------------------------
# Concatenate docstore
# ----------------------------

print("\nConcatenating docstore shards...")
doc_count = 0
first_docs = []

with LOCAL_DOCSTORE.open("w", encoding="utf-8") as fout:
    for pth in local_docs:
        with pth.open("r", encoding="utf-8") as fin:
            for line in fin:
                if not line.strip():
                    continue
                fout.write(line)
                doc_count += 1
                if len(first_docs) < 5:
                    try:
                        first_docs.append(json.loads(line))
                    except Exception:
                        pass

print("Docstore rows:", doc_count)

# ----------------------------
# Concatenate embeddings
# ----------------------------

print("\nLoading embedding shards...")
arrays = []
row_counts = []

for pth in tqdm(local_npys, desc="load npy shards"):
    arr = np.load(pth)
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim != 2:
        raise ValueError(f"Bad shard shape {pth}: {arr.shape}")
    arrays.append(arr)
    row_counts.append(arr.shape[0])

N = int(sum(row_counts))
dim = int(arrays[0].shape[1])
print("Expected rows:", N)
print("Dim:", dim)

assert N == doc_count, f"Row mismatch: embeddings={N}, docstore={doc_count}"

print("\nConcatenating embeddings...")
embeddings = np.concatenate(arrays, axis=0)
del arrays
gc.collect()

# Critical fix: force a real C-contiguous float32 ndarray.
embeddings = np.asarray(embeddings, dtype=np.float32, order="C")
embeddings = np.ascontiguousarray(embeddings)

print("Embedding matrix:", embeddings.shape)
print("type:", type(embeddings))
print("dtype:", embeddings.dtype)
print("C_CONTIGUOUS:", embeddings.flags["C_CONTIGUOUS"])
print("ALIGNED:", embeddings.flags["ALIGNED"])

assert isinstance(embeddings, np.ndarray)
assert embeddings.dtype == np.float32
assert embeddings.flags["C_CONTIGUOUS"]

# ----------------------------
# Manual L2 normalization
# Avoid faiss.normalize_L2 because that was the failure point.
# ----------------------------

print("\nManual L2 normalization...")
block_size = 10000
zero_norm_rows = 0

for start in tqdm(range(0, embeddings.shape[0], block_size), desc="normalize blocks"):
    end = min(embeddings.shape[0], start + block_size)
    block = embeddings[start:end]
    norms = np.linalg.norm(block, axis=1, keepdims=True).astype(np.float32)
    zero_norm_rows += int((norms[:, 0] == 0).sum())
    norms = np.maximum(norms, np.float32(1e-12))
    embeddings[start:end] = block / norms

embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
print("Zero-norm rows:", zero_norm_rows)

# Save normalized embeddings
print("\nSaving normalized embeddings...")
np.save(LOCAL_EMBEDDINGS, embeddings)

# ----------------------------
# Build FAISS
# ----------------------------

print("\nBuilding FAISS IndexFlatIP...")
index = faiss.IndexFlatIP(dim)

try:
    index.add(embeddings)
except Exception as e:
    print("FAISS index.add failed:", repr(e))
    print("\nIf this happens, restart Colab runtime and run this recovery cell again as the first non-auth cell.")
    raise

print("FAISS ntotal:", index.ntotal)
assert index.ntotal == embeddings.shape[0]

faiss.write_index(index, str(LOCAL_FAISS))

# ----------------------------
# Smoke-test vector search
# ----------------------------

print("\nLoading docs for smoke test...")
docs = []
with LOCAL_DOCSTORE.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            docs.append(json.loads(line))

assert len(docs) == embeddings.shape[0]

OPENAI_API_KEY = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", OPENAI_SECRET_NAME,
], text=True).strip()

client = OpenAI(api_key=OPENAI_API_KEY)

def embed_query(q):
    resp = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=[q],
        encoding_format="float",
    )
    qv = np.asarray([resp.data[0].embedding], dtype=np.float32, order="C")
    # manual normalize query
    qv /= max(float(np.linalg.norm(qv)), 1e-12)
    return np.ascontiguousarray(qv, dtype=np.float32)

def vector_search(q, k=5):
    qv = embed_query(q)
    scores, ids = index.search(qv, k)
    hits = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        d = docs[int(idx)]
        hits.append({
            "score": float(score),
            "vector_row": int(idx),
            "title": d.get("title"),
            "journal": d.get("journal"),
            "doi": d.get("doi"),
            "url": d.get("url"),
            "section": d.get("section"),
            "chunk_type": d.get("chunk_type"),
            "excerpt": (d.get("text") or "")[:600],
        })
    return hits

smoke_queries = [
    "intestinal type adenocarcinoma sinonasal occupational wood dust",
    "sinonasal adenocarcinoma CK20 CDX2 SATB2",
    "SMARCB1 deficient sinonasal carcinoma",
    "MTAP loss mesothelioma",
    "EGFR exon 20 insertion lung adenocarcinoma",
    "MSI endometrial carcinoma MLH1 methylation",
]

smoke_results = {}

print("\nVector smoke tests:")
for q in smoke_queries:
    hits = vector_search(q, k=5)
    smoke_results[q] = hits
    print("\n" + "=" * 80)
    print("QUERY:", q)
    for i, h in enumerate(hits[:3], start=1):
        print(f"\nHIT {i} | score={h['score']:.4f}")
        print("title:", h.get("title"))
        print("journal:", h.get("journal"))
        print("doi:", h.get("doi"))
        print("url:", h.get("url"))
        print("excerpt:", h.get("excerpt"))

# ----------------------------
# Manifest + audit
# ----------------------------

created_at = datetime.now(timezone.utc).isoformat()

manifest = {
    "schema_version": "journal_vector_manifest.v1",
    "created_at_utc": created_at,
    "corpus": "journals",
    "workstream": "Journal RAG / Vectorization",
    "index_type": "faiss_IndexFlatIP_cosine_normalized",
    "searchable": True,
    "vectorized": True,
    "api_exposed": False,
    "api_exposed_note": "Vector artifacts built. API requires v04.5 patch before journal vector retrieval is exposed.",
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dim": int(dim),
    "record_count": int(N),
    "source_checkpoint_prefix_gcs": CHECKPOINT_PREFIX_GCS,
    "artifact_paths": {
        "embeddings_npy": f"{OUT_PREFIX_GCS}/journal_embeddings.npy",
        "faiss_index": f"{OUT_PREFIX_GCS}/journal_faiss.index",
        "docstore_jsonl": f"{OUT_PREFIX_GCS}/journal_vector_docstore.jsonl",
        "manifest_json": f"{OUT_PREFIX_GCS}/journal_vector_manifest.json",
        "audit_json": f"{AUDIT_PREFIX_GCS}/journal_vector_build_audit.json",
        "checkpoint_prefix": CHECKPOINT_PREFIX_GCS,
    },
    "checks": {
        "embedding_rows": int(embeddings.shape[0]),
        "docstore_rows": int(doc_count),
        "faiss_ntotal": int(index.ntotal),
        "zero_norm_rows": int(zero_norm_rows),
        "shard_count": int(len(local_npys)),
    },
    "smoke_queries": smoke_queries,
}

audit = {
    "schema_version": "journal_vector_build_audit.v1",
    "created_at_utc": created_at,
    "pass": True,
    "checks": {
        "shards_recovered_from_gcs": int(len(local_npys)),
        "embedding_rows_match_docstore": bool(embeddings.shape[0] == doc_count),
        "faiss_rows_match_docstore": bool(index.ntotal == doc_count),
        "embedding_dim": int(dim),
        "smoke_query_count": len(smoke_queries),
    },
    "smoke_results_top1": {
        q: (hits[0] if hits else None)
        for q, hits in smoke_results.items()
    },
    "known_limitations": [
        "Journal vector artifacts are not API-exposed until v04.5 or later.",
        "Final API should use hybrid retrieval: existing journal FTS plus this FAISS vector index.",
        "Vector-only retrieval should not replace exact keyword matching for genes, markers, and entity names.",
    ],
}

LOCAL_MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
LOCAL_AUDIT.write_text(json.dumps(audit, indent=2), encoding="utf-8")

# ----------------------------
# Upload final artifacts
# ----------------------------

print("\nUploading final journal vector artifacts...")

run(["gcloud", "storage", "cp", str(LOCAL_EMBEDDINGS), f"{OUT_PREFIX_GCS}/journal_embeddings.npy"], check=True)
run(["gcloud", "storage", "cp", str(LOCAL_FAISS), f"{OUT_PREFIX_GCS}/journal_faiss.index"], check=True)
run(["gcloud", "storage", "cp", str(LOCAL_DOCSTORE), f"{OUT_PREFIX_GCS}/journal_vector_docstore.jsonl"], check=True)
run(["gcloud", "storage", "cp", str(LOCAL_MANIFEST), f"{OUT_PREFIX_GCS}/journal_vector_manifest.json"], check=True)
run(["gcloud", "storage", "cp", str(LOCAL_AUDIT), f"{AUDIT_PREFIX_GCS}/journal_vector_build_audit.json"], check=True)

print("\n✅ JOURNAL VECTOR RECOVERY COMPLETE")
print("Records vectorized:", N)
print("Embedding dim:", dim)
print("FAISS ntotal:", index.ntotal)
print("\nGCS artifacts:")
print(f"{OUT_PREFIX_GCS}/journal_embeddings.npy")
print(f"{OUT_PREFIX_GCS}/journal_faiss.index")
print(f"{OUT_PREFIX_GCS}/journal_vector_docstore.jsonl")
print(f"{OUT_PREFIX_GCS}/journal_vector_manifest.json")
print(f"{AUDIT_PREFIX_GCS}/journal_vector_build_audit.json")
print("\nNext step: deploy v04.5 so sources=['journals'] uses hybrid FTS + FAISS vector RRF.")

In [ ]:
# ============================================================
# JOURNAL VECTOR RECOVERY AFTER NUMPY/FAISS RESTART
# No re-embedding. Builds final FAISS/docstore/manifest/audit from GCS shards.
# ============================================================

import os, sys, json, time, subprocess, gc
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ID = "pathology-annotation-project"
OPENAI_SECRET_NAME = "OPEN_AI_KEY_01"
EMBEDDING_MODEL = "text-embedding-3-small"

CHECKPOINT_PREFIX_GCS = "gs://pathology_hub/03_indexes/journals/vector/_build_shards_v1"
OUT_PREFIX_GCS = "gs://pathology_hub/03_indexes/journals/vector"
AUDIT_PREFIX_GCS = "gs://pathology_hub/06_audits/journals/vector"

LOCAL = Path("/content/pathology_hub_journal_vector_recovery")
LOCAL.mkdir(parents=True, exist_ok=True)

LOCAL_DOCSTORE = LOCAL / "journal_vector_docstore.jsonl"
LOCAL_EMBEDDINGS = LOCAL / "journal_embeddings.npy"
LOCAL_FAISS = LOCAL / "journal_faiss.index"
LOCAL_MANIFEST = LOCAL / "journal_vector_manifest.json"
LOCAL_AUDIT = LOCAL / "journal_vector_build_audit.json"

def run(cmd, check=True, quiet=False):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run([str(x) for x in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if p.stdout and not quiet:
        print(p.stdout[-5000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Colab authenticated.")
except Exception as e:
    print("Auth note:", repr(e))

run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

import numpy as np
import faiss
from tqdm.auto import tqdm

print("numpy:", np.__version__)
print("faiss:", faiss.__version__ if hasattr(faiss, "__version__") else "unknown")

assert np.__version__.startswith("1.26"), "NumPy is not 1.26.x. Run the restart-fix cell first."

# Discover shards
p = run(["gcloud", "storage", "ls", "--recursive", CHECKPOINT_PREFIX_GCS + "/"], check=True, quiet=True)
uris = [x.strip() for x in p.stdout.splitlines() if x.strip().startswith("gs://")]

npy_uris = sorted([u for u in uris if u.endswith(".npy") and "journal_embeddings_shard_" in u])
doc_uris = sorted([u for u in uris if u.endswith(".jsonl") and "journal_docstore_shard_" in u])

print("Embedding shard count:", len(npy_uris))
print("Docstore shard count:", len(doc_uris))
assert len(npy_uris) == 51
assert len(doc_uris) == 51

# Download shards
local_npys = []
local_docs = []

for uri in tqdm(npy_uris, desc="download npy shards"):
    local_path = LOCAL / Path(uri).name
    if not local_path.exists():
        run(["gcloud", "storage", "cp", uri, str(local_path)], check=True, quiet=True)
    local_npys.append(local_path)

for uri in tqdm(doc_uris, desc="download docstore shards"):
    local_path = LOCAL / Path(uri).name
    if not local_path.exists():
        run(["gcloud", "storage", "cp", uri, str(local_path)], check=True, quiet=True)
    local_docs.append(local_path)

# Concatenate docstore
doc_count = 0
with LOCAL_DOCSTORE.open("w", encoding="utf-8") as fout:
    for pth in local_docs:
        with pth.open("r", encoding="utf-8") as fin:
            for line in fin:
                if line.strip():
                    fout.write(line)
                    doc_count += 1

print("Docstore rows:", doc_count)

# Concatenate embeddings
arrays = []
for pth in tqdm(local_npys, desc="load npy shards"):
    arr = np.load(pth)
    arr = np.asarray(arr, dtype=np.float32, order="C")
    arrays.append(arr)

embeddings = np.concatenate(arrays, axis=0)
del arrays
gc.collect()

embeddings = np.asarray(embeddings, dtype=np.float32, order="C")
embeddings = np.ascontiguousarray(embeddings)

print("Embedding matrix:", embeddings.shape)
print("dtype:", embeddings.dtype)
print("C_CONTIGUOUS:", embeddings.flags["C_CONTIGUOUS"])
assert embeddings.shape[0] == doc_count
assert embeddings.shape[1] == 1536

# Manual normalize
block_size = 10000
zero_norm_rows = 0
for start in tqdm(range(0, embeddings.shape[0], block_size), desc="normalize"):
    end = min(embeddings.shape[0], start + block_size)
    block = embeddings[start:end]
    norms = np.linalg.norm(block, axis=1, keepdims=True).astype(np.float32)
    zero_norm_rows += int((norms[:, 0] == 0).sum())
    norms = np.maximum(norms, np.float32(1e-12))
    embeddings[start:end] = block / norms

embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
np.save(LOCAL_EMBEDDINGS, embeddings)

# Build FAISS
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("FAISS ntotal:", index.ntotal)
assert index.ntotal == doc_count

faiss.write_index(index, str(LOCAL_FAISS))

# Basic local doc sample
docs_sample = []
with LOCAL_DOCSTORE.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            docs_sample.append(json.loads(line))
        if len(docs_sample) >= 3:
            break

created_at = datetime.now(timezone.utc).isoformat()

manifest = {
    "schema_version": "journal_vector_manifest.v1",
    "created_at_utc": created_at,
    "corpus": "journals",
    "workstream": "Journal RAG / Vectorization",
    "index_type": "faiss_IndexFlatIP_cosine_normalized",
    "searchable": True,
    "vectorized": True,
    "api_exposed": False,
    "api_exposed_note": "Vector artifacts built. API requires v04.5 patch before journal vector retrieval is exposed.",
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dim": int(dim),
    "record_count": int(doc_count),
    "source_checkpoint_prefix_gcs": CHECKPOINT_PREFIX_GCS,
    "artifact_paths": {
        "embeddings_npy": f"{OUT_PREFIX_GCS}/journal_embeddings.npy",
        "faiss_index": f"{OUT_PREFIX_GCS}/journal_faiss.index",
        "docstore_jsonl": f"{OUT_PREFIX_GCS}/journal_vector_docstore.jsonl",
        "manifest_json": f"{OUT_PREFIX_GCS}/journal_vector_manifest.json",
        "audit_json": f"{AUDIT_PREFIX_GCS}/journal_vector_build_audit.json",
        "checkpoint_prefix": CHECKPOINT_PREFIX_GCS,
    },
    "checks": {
        "embedding_rows": int(embeddings.shape[0]),
        "docstore_rows": int(doc_count),
        "faiss_ntotal": int(index.ntotal),
        "zero_norm_rows": int(zero_norm_rows),
        "shard_count": int(len(local_npys)),
    },
}

audit = {
    "schema_version": "journal_vector_build_audit.v1",
    "created_at_utc": created_at,
    "pass": True,
    "checks": {
        "shards_recovered_from_gcs": int(len(local_npys)),
        "embedding_rows_match_docstore": bool(embeddings.shape[0] == doc_count),
        "faiss_rows_match_docstore": bool(index.ntotal == doc_count),
        "embedding_dim": int(dim),
        "numpy_version": np.__version__,
    },
    "sample_docs": docs_sample,
    "known_limitations": [
        "Journal vector artifacts are not API-exposed until v04.5 or later.",
        "Final API should use hybrid retrieval: existing journal FTS plus this FAISS vector index.",
        "Vector-only retrieval should not replace exact keyword matching for genes, markers, and entity names."
    ],
}

LOCAL_MANIFEST.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
LOCAL_AUDIT.write_text(json.dumps(audit, indent=2), encoding="utf-8")

# Upload
run(["gcloud", "storage", "cp", str(LOCAL_EMBEDDINGS), f"{OUT_PREFIX_GCS}/journal_embeddings.npy"], check=True)
run(["gcloud", "storage", "cp", str(LOCAL_FAISS), f"{OUT_PREFIX_GCS}/journal_faiss.index"], check=True)
run(["gcloud", "storage", "cp", str(LOCAL_DOCSTORE), f"{OUT_PREFIX_GCS}/journal_vector_docstore.jsonl"], check=True)
run(["gcloud", "storage", "cp", str(LOCAL_MANIFEST), f"{OUT_PREFIX_GCS}/journal_vector_manifest.json"], check=True)
run(["gcloud", "storage", "cp", str(LOCAL_AUDIT), f"{AUDIT_PREFIX_GCS}/journal_vector_build_audit.json"], check=True)

print("\n✅ JOURNAL VECTOR ARTIFACTS COMPLETE")
print("Records vectorized:", doc_count)
print("Embedding dim:", dim)
print("FAISS ntotal:", index.ntotal)
print(f"{OUT_PREFIX_GCS}/journal_embeddings.npy")
print(f"{OUT_PREFIX_GCS}/journal_faiss.index")
print(f"{OUT_PREFIX_GCS}/journal_vector_docstore.jsonl")
print(f"{OUT_PREFIX_GCS}/journal_vector_manifest.json")
print(f"{AUDIT_PREFIX_GCS}/journal_vector_build_audit.json")

In [ ]:
# ============================================================
# CHECK CURRENT DEPLOY STATE AFTER INTERRUPTED v04.5 CELL
# ============================================================

import subprocess, requests, json

PROJECT_ID = "pathology-annotation-project"
REGION = "us-central1"
SERVICE_NAME = "pathology-hub-v04"
SERVICE_URL = "https://pathology-hub-v04-vorn5q2kga-uc.a.run.app"

def run(cmd, check=True):
    print("\nRUN:", " ".join(cmd))
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout[-6000:])
    if check and p.returncode != 0:
        raise RuntimeError("Command failed")
    return p

run(["gcloud", "config", "set", "project", PROJECT_ID])

print("\nCloud Run service:")
run([
    "gcloud", "run", "services", "describe", SERVICE_NAME,
    "--region", REGION,
    "--format",
    "json(status.url,status.latestReadyRevisionName,status.traffic)"
])

api_key = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", "pathology-hub-api-key"
], text=True).strip()

print("\nHealth:")
r = requests.get(f"{SERVICE_URL}/health", timeout=300)
print(r.status_code)
health = r.json()
print(json.dumps({
    "version": health.get("version"),
    "journal_search_mode": health.get("journal_search_mode"),
    "journal_vectorized": health.get("journal_vectorized"),
    "journal_vector_records": health.get("journal_vector_records"),
    "textbook_search_mode": health.get("textbook_search_mode"),
    "public_figure_map_records_loaded": health.get("public_figure_map_records_loaded"),
}, indent=2))

print("\nJournal smoke:")
payload = {
    "query": "intestinal type adenocarcinoma sinonasal occupational wood dust",
    "sources": ["journals"],
    "max_results": 3,
    "include_figures": False,
    "max_figures": 0,
    "compact": True,
    "excerpt_char_limit": 900,
}
rr = requests.post(
    f"{SERVICE_URL}/evidence/search",
    headers={"X-API-Key": api_key, "Content-Type": "application/json"},
    json=payload,
    timeout=300,
)
print(rr.status_code)
data = rr.json()
print("source_status:", data.get("source_status"))
print("search_mode:", data.get("search_mode"))
print("journal count:", len(data.get("journal_results", [])))
print("warnings:", data.get("warnings"))
print(json.dumps(data, indent=2)[:5000])

In [ ]:
import subprocess, requests, json

SERVICE_URL = "https://pathology-hub-v04-vorn5q2kga-uc.a.run.app"

api_key = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", "pathology-hub-api-key"
], text=True).strip()

queries = [
    "intestinal type sinonasal adenocarcinoma",
    "sinonasal intestinal type adenocarcinoma",
    "sinonasal adenocarcinoma wood dust",
    "ITAC sinonasal KRAS TP53",
    "intestinal type adenocarcinoma sinonasal occupational wood dust",
    "sinonasal adenocarcinoma CDX2 CK20 SATB2",
    "sinonasal adenocarcinoma hardwood dust",
]

for q in queries:
    payload = {
        "query": q,
        "sources": ["journals"],
        "max_results": 3,
        "include_figures": False,
        "max_figures": 0,
        "compact": True,
        "excerpt_char_limit": 900,
    }

    r = requests.post(
        f"{SERVICE_URL}/evidence/search",
        headers={"X-API-Key": api_key, "Content-Type": "application/json"},
        json=payload,
        timeout=300,
    )

    print("\n" + "="*80)
    print("QUERY:", q)
    print("HTTP:", r.status_code)
    data = r.json()
    print("source_status:", data.get("source_status"))
    print("journal_results:", len(data.get("journal_results", [])))
    print("warnings:", data.get("warnings", [])[:5])

    for i, hit in enumerate(data.get("journal_results", [])[:3], start=1):
        print(f"\nHIT {i}")
        print("title:", hit.get("title"))
        print("journal:", hit.get("journal") or hit.get("source_name"))
        print("doi:", hit.get("doi"))
        print("url:", hit.get("source_url") or hit.get("url"))
        print("excerpt:", (hit.get("excerpt") or hit.get("text") or "")[:700])

In [ ]:
payload = {
    "query": "sinonasal adenocarcinoma",
    "sources": ["journals"],
    "max_results": 5,
    "include_figures": False,
    "max_figures": 0,
    "compact": True,
    "excerpt_char_limit": 1200,
}
r = requests.post(
    f"{SERVICE_URL}/evidence/search",
    headers={"X-API-Key": api_key, "Content-Type": "application/json"},
    json=payload,
    timeout=300,
)
print(r.status_code)
print(json.dumps(r.json(), indent=2)[:8000])

In [ ]:
# ============================================================
# AUDIT PUBLIC TEXTBOOK FIGURE WEB LIBRARY BY SOURCE/BOOK
#
# Checks each source_id/book:
# - how many figure records exist
# - how many have web_map records
# - how many public URLs return 200 image/*
# - which sources have failures
#
# Outputs:
# /content/public_figure_source_audit.csv
# /content/public_figure_failed_urls.csv
# /content/public_figure_missing_sources.csv
# ============================================================

import json, subprocess, sys, time, urllib.request, urllib.error
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

PROJECT_ID = "pathology-annotation-project"

FIGURES_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_lean_figures.jsonl"
WEB_MAP_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_figure_web_map_v1.jsonl"

LOCAL_DIR = Path("/content/public_figure_audit")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_LOCAL = LOCAL_DIR / "textbook_lean_figures.jsonl"
WEB_MAP_LOCAL = LOCAL_DIR / "textbook_figure_web_map_v1.jsonl"

MAX_WORKERS = 32
TIMEOUT = 10

def run(cmd, check=True):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout:
        print(p.stdout[-4000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(map(str, cmd))}")
    return p

def clean(s):
    return " ".join(str(s or "").split())

def get_source_id(obj):
    return obj.get("source_id") or obj.get("source") or "unknown"

def get_fig_key(obj):
    # Stable-ish figure identity across original and map.
    source_id = get_source_id(obj)
    page = obj.get("page") or obj.get("source_page") or ""
    figure_id = obj.get("figure_id") or ""
    original = obj.get("original_gs_uri") or obj.get("image_path") or obj.get("image_url") or obj.get("path") or ""
    return f"{source_id}|{page}|{figure_id}|{original}"

def find_url(obj):
    for k in ["public_url", "web_url", "figure_url", "image_url", "url"]:
        v = obj.get(k)
        if isinstance(v, str) and v.startswith("http"):
            return v
    return None

def head_url(row):
    url = row["url"]
    try:
        req = urllib.request.Request(url, method="HEAD")
        with urllib.request.urlopen(req, timeout=TIMEOUT) as resp:
            status = resp.status
            ctype = resp.headers.get("content-type", "")
            return {
                **row,
                "status": status,
                "content_type": ctype,
                "ok": status == 200 and ctype.startswith("image/")
            }
    except Exception as e:
        return {
            **row,
            "status": "ERR",
            "content_type": "",
            "ok": False,
            "error": repr(e)[:300]
        }

run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

if not FIGURES_LOCAL.exists():
    run(["gcloud", "storage", "cp", FIGURES_GCS, str(FIGURES_LOCAL)], check=True)

run(["gcloud", "storage", "cp", WEB_MAP_GCS, str(WEB_MAP_LOCAL)], check=True)

# ----------------------------
# Load original figure counts
# ----------------------------

orig_counts = Counter()
orig_keys_by_source = defaultdict(set)

with FIGURES_LOCAL.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        sid = get_source_id(obj)
        orig_counts[sid] += 1
        orig_keys_by_source[sid].add(get_fig_key(obj))

# ----------------------------
# Load web map
# ----------------------------

map_counts = Counter()
map_rows = []
map_keys_by_source = defaultdict(set)

with WEB_MAP_LOCAL.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        sid = get_source_id(obj)
        url = find_url(obj)
        if not url:
            continue

        key = get_fig_key(obj)
        map_counts[sid] += 1
        map_keys_by_source[sid].add(key)

        map_rows.append({
            "source_id": sid,
            "page": obj.get("page") or obj.get("source_page"),
            "figure_id": obj.get("figure_id"),
            "url": url,
            "original_gs_uri": obj.get("original_gs_uri") or obj.get("image_path") or obj.get("image_url"),
            "mode": obj.get("mode") or obj.get("status"),
            "converted": obj.get("converted"),
        })

print("Original sources:", len(orig_counts))
print("Original figure records:", sum(orig_counts.values()))
print("Mapped URL rows:", len(map_rows))

# ----------------------------
# HEAD-check every mapped public URL
# ----------------------------

checked = []
t0 = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = [ex.submit(head_url, r) for r in map_rows]
    for i, fut in enumerate(as_completed(futs), start=1):
        checked.append(fut.result())
        if i % 5000 == 0:
            print(f"Checked {i}/{len(map_rows)} in {time.time()-t0:.1f}s")

df = pd.DataFrame(checked)

# ----------------------------
# Summarize by source/book
# ----------------------------

summary_rows = []
for sid in sorted(orig_counts):
    src_df = df[df["source_id"] == sid] if not df.empty else pd.DataFrame()
    orig_n = orig_counts[sid]
    mapped_n = map_counts.get(sid, 0)
    ok_n = int(src_df["ok"].sum()) if not src_df.empty and "ok" in src_df else 0
    fail_n = mapped_n - ok_n
    missing_n = max(0, orig_n - mapped_n)

    summary_rows.append({
        "source_id": sid,
        "original_figures": orig_n,
        "mapped_urls": mapped_n,
        "public_ok": ok_n,
        "public_failed": fail_n,
        "missing_from_web_map_est": missing_n,
        "public_ok_rate": round(ok_n / mapped_n, 4) if mapped_n else 0,
        "coverage_rate_vs_original": round(ok_n / orig_n, 4) if orig_n else 0,
    })

summary = pd.DataFrame(summary_rows).sort_values(
    ["coverage_rate_vs_original", "public_failed", "missing_from_web_map_est"],
    ascending=[True, False, False]
)

failed = df[df["ok"] != True].copy() if not df.empty else pd.DataFrame()

summary_path = Path("/content/public_figure_source_audit.csv")
failed_path = Path("/content/public_figure_failed_urls.csv")
missing_path = Path("/content/public_figure_missing_sources.csv")

summary.to_csv(summary_path, index=False)
failed.to_csv(failed_path, index=False)

missing_summary = summary[(summary["public_failed"] > 0) | (summary["missing_from_web_map_est"] > 0)]
missing_summary.to_csv(missing_path, index=False)

print("\n=== WORST SOURCES ===")
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 120)
display(summary.head(80))

print("\n=== FAIL COUNTS ===")
print("Mapped URLs:", len(df))
print("Public OK:", int(df["ok"].sum()) if not df.empty else 0)
print("Public failed:", len(failed))
print("Sources with any issue:", len(missing_summary))

print("\nSaved:")
print(summary_path)
print(failed_path)
print(missing_path)

print("\nTop failures:")
if not failed.empty:
    display(failed.head(50))

In [ ]:
# ============================================================
# REPAIR PUBLIC TEXTBOOK FIGURES
#
# Reprocesses any figure that:
# - is missing from web map, OR
# - has public URL failure
#
# Uses original textbook_lean_figures.jsonl and overwrites/creates
# public web derivatives.
#
# Output:
# /content/textbook_figure_web_map_v1_REPAIRED.jsonl
# gs://pathology_hub/02_normalized/textbooks/lean/textbook_figure_web_map_v1_REPAIRED.jsonl
# ============================================================

import json, re, io, urllib.parse, subprocess, sys, mimetypes, time
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    from PIL import Image
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pillow"], check=True)
    from PIL import Image

from google.cloud import storage

PROJECT_ID = "pathology-annotation-project"
PUBLIC_BUCKET = "pathology-hub-public-figures-830130787988"
PUBLIC_PREFIX = "textbook_figures_web_v1"
PRIVATE_FIGURES_LOCAL = Path("/content/public_figure_audit/textbook_lean_figures.jsonl")
FAILED_CSV = Path("/content/public_figure_failed_urls.csv")
SUMMARY_CSV = Path("/content/public_figure_source_audit.csv")

OUT_MAP = Path("/content/textbook_figure_web_map_v1_REPAIRED.jsonl")
OUT_AUDIT = Path("/content/textbook_figure_web_repair_audit_v1.json")

MAX_WORKERS = 24

def run(cmd, check=True):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run([str(x) for x in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if p.stdout:
        print(p.stdout[-4000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(map(str, cmd))}")
    return p

def storage_https_to_gs(url):
    if not isinstance(url, str):
        return None
    if url.startswith("gs://"):
        return url
    if url.startswith("https://storage.googleapis.com/"):
        rest = url.replace("https://storage.googleapis.com/", "", 1)
        bucket, key = rest.split("/", 1)
        return f"gs://{bucket}/{urllib.parse.unquote(key)}"
    return None

def pick_path(obj):
    for k in ["image_url", "figure_url", "image_path", "path", "gcs_uri", "url"]:
        v = obj.get(k)
        if isinstance(v, str) and v.strip():
            gs = storage_https_to_gs(v.strip())
            if gs:
                return gs
    return None

def parse_gs(gs):
    rest = gs[5:]
    bucket, key = rest.split("/", 1)
    return bucket, key

def safe_name(s):
    s = str(s or "unknown").lower()
    s = re.sub(r"[^a-z0-9_.-]+", "_", s).strip("_")
    return s or "unknown"

def public_key_for(obj, original_gs):
    source_id = safe_name(obj.get("source_id"))
    original_name = Path(original_gs).name
    stem = Path(original_name).stem
    ext = Path(original_name).suffix.lower()

    if ext in [".jpg", ".jpeg", ".png", ".webp", ".gif"]:
        out_ext = ext
    else:
        out_ext = ".jpg"

    return f"{PUBLIC_PREFIX}/{source_id}/{stem}{out_ext}"

def public_url(public_key):
    return f"https://storage.googleapis.com/{PUBLIC_BUCKET}/{public_key}"

def convert_to_jpeg_bytes(raw):
    im = Image.open(io.BytesIO(raw))
    im.load()
    if im.mode != "RGB":
        im = im.convert("RGB")
    out = io.BytesIO()
    im.save(out, format="JPEG", quality=88, optimize=True)
    return out.getvalue(), "image/jpeg"

def process_one(obj):
    storage_client = storage.Client()
    original_gs = pick_path(obj)
    if not original_gs:
        return {"status": "skipped_no_path", "source_id": obj.get("source_id"), "figure_id": obj.get("figure_id")}

    src_bucket, src_key = parse_gs(original_gs)
    src_blob = storage_client.bucket(src_bucket).blob(src_key)
    public_key = public_key_for(obj, original_gs)
    dst_blob = storage_client.bucket(PUBLIC_BUCKET).blob(public_key)

    ext = Path(original_gs).suffix.lower()
    try:
        if ext in [".jpg", ".jpeg", ".png", ".webp", ".gif"]:
            # download/upload instead of rewrite+patch; avoids prior PATCH NotFound weirdness
            raw = src_blob.download_as_bytes()
            content_type = mimetypes.guess_type(public_key)[0] or "application/octet-stream"
            dst_blob.upload_from_string(raw, content_type=content_type)
            converted = False
            mode = "copy_websafe_reupload"
        else:
            raw = src_blob.download_as_bytes()
            out_bytes, content_type = convert_to_jpeg_bytes(raw)
            dst_blob.upload_from_string(out_bytes, content_type=content_type)
            converted = True
            mode = "convert_to_jpeg"

        # Make cache public-friendly metadata, ignore if metadata patch fails
        try:
            dst_blob.cache_control = "public, max-age=31536000, immutable"
            dst_blob.patch()
        except Exception:
            pass

        return {
            "status": "ok",
            "mode": mode,
            "source_id": obj.get("source_id"),
            "source_title": obj.get("source_title") or obj.get("title") or obj.get("source_id"),
            "page": obj.get("page") or obj.get("source_page"),
            "figure_id": obj.get("figure_id"),
            "caption": obj.get("caption") or obj.get("legend") or obj.get("text"),
            "original_gs_uri": original_gs,
            "public_gs_uri": f"gs://{PUBLIC_BUCKET}/{public_key}",
            "public_url": public_url(public_key),
            "converted": converted,
            "original_ext": ext,
        }
    except Exception as e:
        return {
            "status": "error",
            "source_id": obj.get("source_id"),
            "page": obj.get("page") or obj.get("source_page"),
            "figure_id": obj.get("figure_id"),
            "original_gs_uri": original_gs,
            "error": repr(e)[:500],
        }

run(["gcloud", "config", "set", "project", PROJECT_ID])

assert PRIVATE_FIGURES_LOCAL.exists(), f"Missing {PRIVATE_FIGURES_LOCAL}; run audit cell first."

# Reprocess all records from sources with issues. This is safer than trying to match exact keys.
import pandas as pd
summary = pd.read_csv(SUMMARY_CSV)
bad_sources = set(summary[(summary.public_failed > 0) | (summary.missing_from_web_map_est > 0)]["source_id"].astype(str))

print("Sources to repair:", len(bad_sources))
print(sorted(list(bad_sources))[:100])

jobs = []
with PRIVATE_FIGURES_LOCAL.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        if str(obj.get("source_id")) in bad_sources:
            jobs.append(obj)

print("Repair jobs:", len(jobs))

results = []
t0 = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = [ex.submit(process_one, obj) for obj in jobs]
    for i, fut in enumerate(as_completed(futs), start=1):
        results.append(fut.result())
        if i % 1000 == 0:
            print(f"Repaired {i}/{len(jobs)} in {time.time()-t0:.1f}s")

counts = Counter(r["status"] for r in results)
modes = Counter(r.get("mode") for r in results if r.get("mode"))

print("Counts:", counts)
print("Modes:", modes)
print("First errors:")
for r in results:
    if r["status"] == "error":
        print(json.dumps(r, indent=2)[:1000])
        break

with OUT_MAP.open("w", encoding="utf-8") as f:
    for r in results:
        if r.get("status") == "ok":
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

OUT_AUDIT.write_text(json.dumps({
    "created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "sources_repaired": sorted(bad_sources),
    "jobs": len(jobs),
    "counts": dict(counts),
    "modes": dict(modes),
    "output_map": str(OUT_MAP),
}, indent=2), encoding="utf-8")

run(["gcloud", "storage", "cp", str(OUT_MAP), "gs://pathology_hub/02_normalized/textbooks/lean/textbook_figure_web_map_v1_REPAIRED.jsonl"], check=True)
run(["gcloud", "storage", "cp", str(OUT_AUDIT), "gs://pathology_hub/06_audits/textbooks/figures/textbook_figure_web_repair_audit_v1.json"], check=True)

print("\n✅ Repair complete.")
print("Repair map:", OUT_MAP)
print("Repair audit:", OUT_AUDIT)

In [ ]:
# ============================================================
# AUDIT PUBLIC TEXTBOOK FIGURE WEB LIBRARY BY SOURCE/BOOK
#
# Checks each source_id/book:
# - how many figure records exist
# - how many have web_map records
# - how many public URLs return 200 image/*
# - which sources have failures
#
# Outputs:
# /content/public_figure_source_audit.csv
# /content/public_figure_failed_urls.csv
# /content/public_figure_missing_sources.csv
# ============================================================

import json, subprocess, sys, time, urllib.request, urllib.error
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

PROJECT_ID = "pathology-annotation-project"

FIGURES_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_lean_figures.jsonl"
WEB_MAP_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_figure_web_map_v1.jsonl"

LOCAL_DIR = Path("/content/public_figure_audit")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_LOCAL = LOCAL_DIR / "textbook_lean_figures.jsonl"
WEB_MAP_LOCAL = LOCAL_DIR / "textbook_figure_web_map_v1.jsonl"

MAX_WORKERS = 32
TIMEOUT = 10

def run(cmd, check=True):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout:
        print(p.stdout[-4000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(map(str, cmd))}")
    return p

def clean(s):
    return " ".join(str(s or "").split())

def get_source_id(obj):
    return obj.get("source_id") or obj.get("source") or "unknown"

def get_fig_key(obj):
    # Stable-ish figure identity across original and map.
    source_id = get_source_id(obj)
    page = obj.get("page") or obj.get("source_page") or ""
    figure_id = obj.get("figure_id") or ""
    original = obj.get("original_gs_uri") or obj.get("image_path") or obj.get("image_url") or obj.get("path") or ""
    return f"{source_id}|{page}|{figure_id}|{original}"

def find_url(obj):
    for k in ["public_url", "web_url", "figure_url", "image_url", "url"]:
        v = obj.get(k)
        if isinstance(v, str) and v.startswith("http"):
            return v
    return None

def head_url(row):
    url = row["url"]
    try:
        req = urllib.request.Request(url, method="HEAD")
        with urllib.request.urlopen(req, timeout=TIMEOUT) as resp:
            status = resp.status
            ctype = resp.headers.get("content-type", "")
            return {
                **row,
                "status": status,
                "content_type": ctype,
                "ok": status == 200 and ctype.startswith("image/")
            }
    except Exception as e:
        return {
            **row,
            "status": "ERR",
            "content_type": "",
            "ok": False,
            "error": repr(e)[:300]
        }

run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

if not FIGURES_LOCAL.exists():
    run(["gcloud", "storage", "cp", FIGURES_GCS, str(FIGURES_LOCAL)], check=True)

run(["gcloud", "storage", "cp", WEB_MAP_GCS, str(WEB_MAP_LOCAL)], check=True)

# ----------------------------
# Load original figure counts
# ----------------------------

orig_counts = Counter()
orig_keys_by_source = defaultdict(set)

with FIGURES_LOCAL.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        sid = get_source_id(obj)
        orig_counts[sid] += 1
        orig_keys_by_source[sid].add(get_fig_key(obj))

# ----------------------------
# Load web map
# ----------------------------

map_counts = Counter()
map_rows = []
map_keys_by_source = defaultdict(set)

with WEB_MAP_LOCAL.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        sid = get_source_id(obj)
        url = find_url(obj)
        if not url:
            continue

        key = get_fig_key(obj)
        map_counts[sid] += 1
        map_keys_by_source[sid].add(key)

        map_rows.append({
            "source_id": sid,
            "page": obj.get("page") or obj.get("source_page"),
            "figure_id": obj.get("figure_id"),
            "url": url,
            "original_gs_uri": obj.get("original_gs_uri") or obj.get("image_path") or obj.get("image_url"),
            "mode": obj.get("mode") or obj.get("status"),
            "converted": obj.get("converted"),
        })

print("Original sources:", len(orig_counts))
print("Original figure records:", sum(orig_counts.values()))
print("Mapped URL rows:", len(map_rows))

# ----------------------------
# HEAD-check every mapped public URL
# ----------------------------

checked = []
t0 = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = [ex.submit(head_url, r) for r in map_rows]
    for i, fut in enumerate(as_completed(futs), start=1):
        checked.append(fut.result())
        if i % 5000 == 0:
            print(f"Checked {i}/{len(map_rows)} in {time.time()-t0:.1f}s")

df = pd.DataFrame(checked)

# ----------------------------
# Summarize by source/book
# ----------------------------

summary_rows = []
for sid in sorted(orig_counts):
    src_df = df[df["source_id"] == sid] if not df.empty else pd.DataFrame()
    orig_n = orig_counts[sid]
    mapped_n = map_counts.get(sid, 0)
    ok_n = int(src_df["ok"].sum()) if not src_df.empty and "ok" in src_df else 0
    fail_n = mapped_n - ok_n
    missing_n = max(0, orig_n - mapped_n)

    summary_rows.append({
        "source_id": sid,
        "original_figures": orig_n,
        "mapped_urls": mapped_n,
        "public_ok": ok_n,
        "public_failed": fail_n,
        "missing_from_web_map_est": missing_n,
        "public_ok_rate": round(ok_n / mapped_n, 4) if mapped_n else 0,
        "coverage_rate_vs_original": round(ok_n / orig_n, 4) if orig_n else 0,
    })

summary = pd.DataFrame(summary_rows).sort_values(
    ["coverage_rate_vs_original", "public_failed", "missing_from_web_map_est"],
    ascending=[True, False, False]
)

failed = df[df["ok"] != True].copy() if not df.empty else pd.DataFrame()

summary_path = Path("/content/public_figure_source_audit.csv")
failed_path = Path("/content/public_figure_failed_urls.csv")
missing_path = Path("/content/public_figure_missing_sources.csv")

summary.to_csv(summary_path, index=False)
failed.to_csv(failed_path, index=False)

missing_summary = summary[(summary["public_failed"] > 0) | (summary["missing_from_web_map_est"] > 0)]
missing_summary.to_csv(missing_path, index=False)

print("\n=== WORST SOURCES ===")
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 120)
display(summary.head(80))

print("\n=== FAIL COUNTS ===")
print("Mapped URLs:", len(df))
print("Public OK:", int(df["ok"].sum()) if not df.empty else 0)
print("Public failed:", len(failed))
print("Sources with any issue:", len(missing_summary))

print("\nSaved:")
print(summary_path)
print(failed_path)
print(missing_path)

print("\nTop failures:")
if not failed.empty:
    display(failed.head(50))

In [ ]:
# ============================================================
# QUARANTINE BAD TEXTBOOK FIGURE SOURCES
#
# Excludes derm_mckee and bone_dorfman from public figure serving.
# Keeps their text/chunks searchable.
#
# Outputs:
# - filtered public web map
# - exclusion audit JSON
# - optionally removes public derivative folders for those sources
# ============================================================

import json, subprocess, datetime
from pathlib import Path
from collections import Counter

PROJECT_ID = "pathology-annotation-project"

EXCLUDE_SOURCES = {
    "derm_mckee": "Poor public derivative audit: only 225/5888 public URLs worked; 5663 failures.",
    "bone_dorfman": "Incomplete public derivative coverage: 1065/1206 worked; 141 missing. Excluded for clean figure-serving behavior."
}

WEB_MAP_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_figure_web_map_v1.jsonl"
FILTERED_MAP_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_figure_web_map_v1_FILTERED_NO_MCKEE_DORFMAN.jsonl"
EXCLUSION_AUDIT_GCS = "gs://pathology_hub/06_audits/textbooks/figures/textbook_figure_source_exclusions_v1.json"

PUBLIC_BUCKET = "gs://pathology-hub-public-figures-830130787988"
PUBLIC_PREFIX = "textbook_figures_web_v1"

WORK = Path("/content/public_figure_filter")
WORK.mkdir(parents=True, exist_ok=True)

WEB_MAP_LOCAL = WORK / "textbook_figure_web_map_v1.jsonl"
FILTERED_LOCAL = WORK / "textbook_figure_web_map_v1_FILTERED_NO_MCKEE_DORFMAN.jsonl"
AUDIT_LOCAL = WORK / "textbook_figure_source_exclusions_v1.json"

def run(cmd, check=True):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout:
        print(p.stdout[-5000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(map(str, cmd))}")
    return p

run(["gcloud", "config", "set", "project", PROJECT_ID])
run(["gcloud", "storage", "cp", WEB_MAP_GCS, str(WEB_MAP_LOCAL)])

kept = 0
excluded = 0
by_source_total = Counter()
by_source_excluded = Counter()

with WEB_MAP_LOCAL.open("r", encoding="utf-8") as fin, FILTERED_LOCAL.open("w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip():
            continue
        obj = json.loads(line)
        sid = obj.get("source_id") or obj.get("source") or "unknown"
        by_source_total[sid] += 1

        if sid in EXCLUDE_SOURCES:
            excluded += 1
            by_source_excluded[sid] += 1
            continue

        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
        kept += 1

audit = {
    "schema_version": "textbook_figure_source_exclusions.v1",
    "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "purpose": "Exclude known-problem textbook sources from public figure serving while preserving their text retrieval.",
    "excluded_sources": EXCLUDE_SOURCES,
    "input_map_gcs": WEB_MAP_GCS,
    "filtered_map_gcs": FILTERED_MAP_GCS,
    "counts": {
        "input_rows": kept + excluded,
        "kept_rows": kept,
        "excluded_rows": excluded,
        "excluded_by_source": dict(by_source_excluded),
    },
    "notes": [
        "This does not remove textbook text/chunks/searchability.",
        "This only removes these sources from public figure-map based serving.",
        "Future v04.4 API should use the filtered map and should not return figures from excluded sources."
    ]
}

AUDIT_LOCAL.write_text(json.dumps(audit, indent=2), encoding="utf-8")

run(["gcloud", "storage", "cp", str(FILTERED_LOCAL), FILTERED_MAP_GCS])
run(["gcloud", "storage", "cp", str(AUDIT_LOCAL), EXCLUSION_AUDIT_GCS])

print("\nFiltered map written:")
print(FILTERED_MAP_GCS)
print("\nExclusion audit written:")
print(EXCLUSION_AUDIT_GCS)
print("\nCounts:")
print(json.dumps(audit["counts"], indent=2))

# Optional but recommended: delete the public derivative folders for excluded sources
# so they cannot be accidentally linked from old maps.
for sid in EXCLUDE_SOURCES:
    public_prefix = f"{PUBLIC_BUCKET}/{PUBLIC_PREFIX}/{sid}/"
    print("\nRemoving public derivative folder:", public_prefix)
    run(["gcloud", "storage", "rm", "--recursive", public_prefix], check=False)

print("\n✅ Done. McKee and Dorfman figures are quarantined from public derivative serving.")

In [ ]:
# ============================================================
# DEPLOY PATHOLOGY HUB v04.4
# Hybrid textbook search + public derivative figure URLs
#
# Uses filtered public figure map:
#   gs://pathology_hub/02_normalized/textbooks/lean/textbook_figure_web_map_v1_FILTERED_NO_MCKEE_DORFMAN.jsonl
#
# Behavior:
# - Textbook retrieval remains hybrid SQLite FTS + FAISS vector.
# - include_figures=true prefers direct public derivative URLs.
# - Falls back to v04.3 proxy only when a public derivative is missing.
# - Never returns figures from excluded sources: derm_mckee, bone_dorfman.
# ============================================================

from pathlib import Path
import os, subprocess, json, requests, datetime, re, time

PROJECT_ID = "pathology-annotation-project"
REGION = "us-central1"
SERVICE_NAME = "pathology-hub-v04"
AR_REPO = "pathology-hub"
IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/{SERVICE_NAME}:latest"

APP_DIR = Path("/content/pathology_hub_v04_textbook_api")
APP_PATH = APP_DIR / "app.py"
assert APP_PATH.exists(), f"Missing v04.3 app.py at {APP_PATH}. Run v04.3 deploy first."

TEXTBOOK_SQLITE_GCS = "gs://pathology_hub/03_indexes/textbooks/lean/textbook_lean_fts.sqlite"
TEXTBOOK_MANIFEST_GCS = "gs://pathology_hub/03_indexes/textbooks/lean/textbook_lean_index_manifest.json"
TEXTBOOK_FAISS_GCS = "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_faiss.index"
TEXTBOOK_DOCSTORE_GCS = "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_docstore.jsonl"
TEXTBOOK_VECTOR_MANIFEST_GCS = "gs://pathology_hub/03_indexes/textbooks/vector/textbook_lean_vector_manifest.json"
TEXTBOOK_FIGURES_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_lean_figures.jsonl"
TEXTBOOK_WEB_MAP_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_figure_web_map_v1_FILTERED_NO_MCKEE_DORFMAN.jsonl"

UPSTREAM_EVIDENCE_URL = "https://pathology-hub-830130787988.us-central1.run.app/evidence/search"
OPENAI_SECRET_NAME = "OPEN_AI_KEY_01"

def run(cmd, cwd=None, check=True):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout:
        print(p.stdout[-12000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

for secret in ["pathology-hub-api-key", OPENAI_SECRET_NAME]:
    run(["gcloud", "secrets", "describe", secret], check=True)

for uri in [
    TEXTBOOK_SQLITE_GCS,
    TEXTBOOK_MANIFEST_GCS,
    TEXTBOOK_FAISS_GCS,
    TEXTBOOK_DOCSTORE_GCS,
    TEXTBOOK_VECTOR_MANIFEST_GCS,
    TEXTBOOK_FIGURES_GCS,
    TEXTBOOK_WEB_MAP_GCS,
]:
    run(["gcloud", "storage", "ls", uri], check=True)

# Backup current app
backup_path = APP_DIR / f"app_pre_v044_{datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')}.py"
backup_path.write_text(APP_PATH.read_text(encoding="utf-8"), encoding="utf-8")
print("Backup:", backup_path)

app = APP_PATH.read_text(encoding="utf-8")

# Ensure we are patching the v04.3 app.
if "APP_VERSION = \"1.5.3-textbooks-hybrid-figproxy-v04\"" not in app:
    print("WARNING: Did not find exact v04.3 APP_VERSION string; patching best-effort.")

app = app.replace(
    'APP_VERSION = "1.5.3-textbooks-hybrid-figproxy-v04"',
    'APP_VERSION = "1.5.4-textbooks-hybrid-publicfigs-v04"'
)

# Add web map env/path/global if missing.
if 'TEXTBOOK_WEB_MAP_GCS = os.environ.get("TEXTBOOK_WEB_MAP_GCS")' not in app:
    app = app.replace(
        'TEXTBOOK_FIGURES_GCS = os.environ.get("TEXTBOOK_FIGURES_GCS")\nUPSTREAM_EVIDENCE_URL',
        'TEXTBOOK_FIGURES_GCS = os.environ.get("TEXTBOOK_FIGURES_GCS")\nTEXTBOOK_WEB_MAP_GCS = os.environ.get("TEXTBOOK_WEB_MAP_GCS")\nUPSTREAM_EVIDENCE_URL'
    )

if 'WEB_MAP_PATH = DATA_DIR / "textbook_figure_web_map_v1_FILTERED_NO_MCKEE_DORFMAN.jsonl"' not in app:
    app = app.replace(
        'FIGURES_PATH = DATA_DIR / "textbook_lean_figures.jsonl"',
        'FIGURES_PATH = DATA_DIR / "textbook_lean_figures.jsonl"\nWEB_MAP_PATH = DATA_DIR / "textbook_figure_web_map_v1_FILTERED_NO_MCKEE_DORFMAN.jsonl"'
    )

if '(TEXTBOOK_WEB_MAP_GCS, WEB_MAP_PATH),' not in app:
    app = app.replace(
        '(TEXTBOOK_FIGURES_GCS, FIGURES_PATH),',
        '(TEXTBOOK_FIGURES_GCS, FIGURES_PATH),\n        (TEXTBOOK_WEB_MAP_GCS, WEB_MAP_PATH),'
    )

if '_WEB_FIGURE_MAP = None' not in app:
    app = app.replace(
        '_FIGURES_BY_SOURCE_PAGE = None',
        '_FIGURES_BY_SOURCE_PAGE = None\n_WEB_FIGURE_MAP = None'
    )

# Replace the figure metadata block from figure_to_response through collect_textbook_figures.
new_figure_block = """EXCLUDED_FIGURE_SOURCE_IDS = {"derm_mckee", "bone_dorfman"}

def load_web_figure_map():
    # Map original private GCS figure paths to public web-safe derivative URLs.
    # Uses the filtered map that excludes derm_mckee and bone_dorfman.
    global _WEB_FIGURE_MAP
    ensure_artifacts()
    if _WEB_FIGURE_MAP is not None:
        return _WEB_FIGURE_MAP

    m = {}
    if WEB_MAP_PATH.exists():
        with WEB_MAP_PATH.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                except Exception:
                    continue

                sid = str(obj.get("source_id") or obj.get("source") or "")
                if sid in EXCLUDED_FIGURE_SOURCE_IDS:
                    continue

                orig = (
                    obj.get("original_gs_uri")
                    or obj.get("original_image_path")
                    or obj.get("image_path")
                    or obj.get("image_url")
                    or obj.get("path")
                    or obj.get("url")
                )
                orig_gs = storage_https_to_gs(orig)

                pub = (
                    obj.get("public_url")
                    or obj.get("web_url")
                    or obj.get("figure_url")
                    or obj.get("image_url")
                    or obj.get("url")
                )

                if orig_gs and isinstance(pub, str) and pub.startswith("https://"):
                    m[orig_gs] = pub

    _WEB_FIGURE_MAP = m
    return m

def figure_to_response(rec: dict, base_url: str, rank: int = None):
    sid = str(rec.get("source_id") or "")
    if sid in EXCLUDED_FIGURE_SOURCE_IDS:
        return None

    gs = storage_https_to_gs(rec.get("image_path") or rec.get("original_image_url"))
    if not gs:
        return None

    web_map = load_web_figure_map()
    public_url = web_map.get(gs)

    if public_url:
        final_url = public_url
        mode = "public_web_derivative"
        proxy_fallback_used = False
    else:
        final_url = make_figure_proxy_url(base_url, gs)
        mode = "expiring_proxy_fallback"
        proxy_fallback_used = True

    if not final_url:
        return None

    return {
        "rank": rank,
        "title": rec.get("source_title") or rec.get("source_id"),
        "caption": rec.get("caption"),
        "figure_id": rec.get("figure_id"),
        "figure_url": final_url,
        "image_url": final_url,
        "image_path": gs,
        "original_image_path": gs,
        "original_image_url": rec.get("original_image_url"),
        "public_derivative_url": public_url,
        "proxy_fallback_used": proxy_fallback_used,
        "figure_serving_mode": mode,
        "source": "textbooks",
        "source_name": "textbooks",
        "source_id": rec.get("source_id"),
        "page": rec.get("page"),
    }

def collect_textbook_figures(textbook_results: list, base_url: str, max_figures: int):
    if max_figures <= 0:
        return []

    figures, by_sp = load_figures()
    out = []
    seen = set()

    def add_rec(rec):
        if not rec:
            return
        sid = str(rec.get("source_id") or "")
        if sid in EXCLUDED_FIGURE_SOURCE_IDS:
            return

        gs = storage_https_to_gs(rec.get("image_path") or rec.get("original_image_url"))
        if not gs or gs in seen:
            return

        response_rec = figure_to_response(rec, base_url, rank=len(out) + 1)
        if not response_rec:
            return

        seen.add(gs)
        out.append(response_rec)

    # First: exact image_path on figure-caption hits.
    for r in textbook_results:
        if str(r.get("source_id") or "") in EXCLUDED_FIGURE_SOURCE_IDS:
            continue
        img = r.get("image_path")
        gs = storage_https_to_gs(img)
        if gs:
            add_rec({
                "source_id": r.get("source_id"),
                "source_title": r.get("title") or r.get("source_title"),
                "page": r.get("page"),
                "figure_id": r.get("figure_id"),
                "caption": r.get("text") or r.get("excerpt"),
                "image_path": gs,
                "original_image_url": img,
            })
            if len(out) >= max_figures:
                return out

    # Second: figures from the same source/page as top hits.
    for r in textbook_results:
        if str(r.get("source_id") or "") in EXCLUDED_FIGURE_SOURCE_IDS:
            continue
        key = (str(r.get("source_id")), str(r.get("page")))
        for rec in by_sp.get(key, []):
            add_rec(rec)
            if len(out) >= max_figures:
                return out

    return out
"""

# Replace the v04.3 figure_to_response / collect_textbook_figures block.
# Do this with boundary-based string search instead of a brittle regex.
start = app.find("def figure_to_response(")
if start < 0:
    raise RuntimeError("Could not find def figure_to_response(...) in app.py")

end_candidates = []
for marker in [
    "\n# ----------------------------\n# Hybrid textbook retrieval",
    "\n# ----------------------------\n# Text helpers",
    "\ndef row_to_textbook_result(",
    "\ndef fts_search_pool(",
]:
    pos = app.find(marker, start)
    if pos >= 0:
        end_candidates.append((pos, marker))

if not end_candidates:
    raise RuntimeError("Could not find end of figure block after def figure_to_response(...).")

end, marker = min(end_candidates, key=lambda x: x[0])
print("Replacing figure block from char", start, "to", end, "ending at marker:", repr(marker))

# Preserve the downstream marker/block. If marker is a function, add a spacer only.
app = app[:start] + new_figure_block + "\n\n" + app[end:]

# Patch warning text.
app = app.replace(
    "Textbook figure URLs, when returned, are expiring proxy URLs for private GCS assets.",
    "Textbook figure URLs prefer direct public web-safe derivative URLs and fall back to expiring proxy URLs if needed."
)

# Patch health return mode and add web map stats.
app = app.replace(
    '"textbook_figure_mode": "expiring_hmac_proxy_with_on_the_fly_web_conversion",',
    '"textbook_figure_mode": "public_web_derivative_urls_with_proxy_fallback",'
)
app = app.replace(
    '"figure_proxy_enabled": True,',
    '"figure_proxy_enabled": True,\n        "public_figure_map_enabled": True,\n        "public_figure_map_size_bytes": WEB_MAP_PATH.stat().st_size if WEB_MAP_PATH.exists() else 0,\n        "public_figure_map_records_loaded": len(load_web_figure_map()) if WEB_MAP_PATH.exists() else 0,'
)

APP_PATH.write_text(app, encoding="utf-8")

print("Patched app.py to v04.4")
run(["python", "-m", "py_compile", str(APP_PATH)], check=True)

# Build/deploy
for api in [
    "artifactregistry.googleapis.com",
    "cloudbuild.googleapis.com",
    "run.googleapis.com",
    "secretmanager.googleapis.com",
    "storage.googleapis.com",
]:
    run(["gcloud", "services", "enable", api], check=True)

run(["gcloud", "auth", "configure-docker", f"{REGION}-docker.pkg.dev", "--quiet"], check=True)

print("\nBuilding v04.4 image...")
run(["gcloud", "builds", "submit", "--tag", IMAGE, "."], cwd=APP_DIR, check=True)

print("\nDeploying v04.4...")
run([
    "gcloud", "run", "deploy", SERVICE_NAME,
    "--image", IMAGE,
    "--region", REGION,
    "--platform", "managed",
    "--allow-unauthenticated",
    "--memory", "8Gi",
    "--cpu", "4",
    "--timeout", "300",
    "--min-instances", "1",
    "--set-env-vars",
    f"TEXTBOOK_SQLITE_GCS={TEXTBOOK_SQLITE_GCS},TEXTBOOK_MANIFEST_GCS={TEXTBOOK_MANIFEST_GCS},TEXTBOOK_FAISS_GCS={TEXTBOOK_FAISS_GCS},TEXTBOOK_DOCSTORE_GCS={TEXTBOOK_DOCSTORE_GCS},TEXTBOOK_VECTOR_MANIFEST_GCS={TEXTBOOK_VECTOR_MANIFEST_GCS},TEXTBOOK_FIGURES_GCS={TEXTBOOK_FIGURES_GCS},TEXTBOOK_WEB_MAP_GCS={TEXTBOOK_WEB_MAP_GCS},UPSTREAM_EVIDENCE_URL={UPSTREAM_EVIDENCE_URL},EMBEDDING_MODEL=text-embedding-3-small",
    "--set-secrets",
    f"PATHOLOGY_HUB_API_KEY=pathology-hub-api-key:latest,OPENAI_API_KEY={OPENAI_SECRET_NAME}:latest,FIGURE_PROXY_SECRET=pathology-hub-api-key:latest",
    "--quiet",
], check=True)

service_url = subprocess.check_output([
    "gcloud", "run", "services", "describe", SERVICE_NAME,
    "--region", REGION,
    "--format", "value(status.url)"
], text=True).strip()

print("\nSERVICE URL:", service_url)

api_key = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", "pathology-hub-api-key"
], text=True).strip()

print("\nHealth check...")
r = requests.get(f"{service_url}/health", timeout=300)
print("health:", r.status_code)
health = r.json()
print(json.dumps(health, indent=2)[:6000])
assert r.status_code == 200
assert health.get("figure_proxy_enabled") is True
assert health.get("public_figure_map_enabled") is True

# Smoke test: Gyn Essentials should return direct public derivative URL.
print("\nPublic figure smoke test: Gyn Essentials")
payload = {
    "query": "fallopian tube precursor lesion abnormal p53 increased proliferation",
    "sources": ["textbooks"],
    "max_results": 3,
    "include_figures": True,
    "max_figures": 3,
    "compact": True,
    "excerpt_char_limit": 900
}

rr = requests.post(
    f"{service_url}/evidence/search",
    headers={"X-API-Key": api_key, "Content-Type": "application/json"},
    json=payload,
    timeout=300,
)
print("status:", rr.status_code)
data = rr.json()
print("source_status:", data.get("source_status"))
print("figures:", len(data.get("figures", [])))
print(json.dumps(data, indent=2)[:9000])
assert rr.status_code == 200
assert data.get("source_status", {}).get("textbooks") == "ok"
assert len(data.get("figures", [])) >= 1

first_url = data["figures"][0]["figure_url"]
print("First figure URL:", first_url)
assert "storage.googleapis.com/pathology-hub-public-figures-830130787988" in first_url

img = requests.get(first_url, timeout=120)
print("public figure status:", img.status_code)
print("public figure content-type:", img.headers.get("content-type"))
print("public figure bytes:", len(img.content))
assert img.status_code == 200
assert img.headers.get("content-type", "").startswith("image/")
assert len(img.content) > 1000

# Smoke test: excluded source should not return derm_mckee figures.
print("\nExcluded-source smoke test: derm_mckee figures should be suppressed")
payload2 = {
    "query": "Merkel cell carcinoma CK20 derm mckee",
    "sources": ["textbooks"],
    "max_results": 5,
    "include_figures": True,
    "max_figures": 5,
    "compact": True,
    "excerpt_char_limit": 700
}
rr2 = requests.post(
    f"{service_url}/evidence/search",
    headers={"X-API-Key": api_key, "Content-Type": "application/json"},
    json=payload2,
    timeout=300,
)
print("status:", rr2.status_code)
data2 = rr2.json()
print(json.dumps(data2, indent=2)[:9000])
assert rr2.status_code == 200
for fig in data2.get("figures", []):
    assert fig.get("source_id") not in {"derm_mckee", "bone_dorfman"}

# Write OpenAPI and handoff.
openapi_yaml = f"""openapi: 3.1.0
info:
  title: Pathology Hub Unified Evidence API
  version: 1.5.4
  description: Unified evidence search with WHO passthrough, Journal, PathOut, hybrid Textbook FTS+vector support, and public textbook figure derivative URLs with proxy fallback.

servers:
  - url: {service_url}

components:
  securitySchemes:
    ApiKeyAuth:
      type: apiKey
      in: header
      name: X-API-Key

  schemas:
    EvidenceSearchRequest:
      type: object
      required:
        - query
      properties:
        query:
          type: string
        sources:
          type: array
          items:
            type: string
            enum:
              - who
              - journals
              - pathout
              - textbooks
          default:
            - textbooks
        max_results:
          type: integer
          minimum: 1
          maximum: 10
          default: 1
        include_figures:
          type: boolean
          default: false
        max_figures:
          type: integer
          minimum: 0
          maximum: 10
          default: 0
        compact:
          type: boolean
          default: true
        excerpt_char_limit:
          type: integer
          minimum: 200
          maximum: 4000
          default: 900

    SourceStatus:
      type: object
      properties:
        who:
          type: string
        journals:
          type: string
        pathout:
          type: string
        textbooks:
          type: string
      additionalProperties: true

    EvidenceItem:
      type: object
      properties:
        rank:
          type: integer
        title:
          type: string
        source:
          type: string
        source_name:
          type: string
        source_family:
          type: string
        volume_code:
          type: string
        entity_name:
          type: string
        journal:
          type: string
        doi:
          type: string
        source_url:
          type: string
        url:
          type: string
        excerpt:
          type: string
        text:
          type: string
        chunk_text:
          type: string
        source_type:
          type: string
        source_id:
          type: string
        record_id:
          type: string
        chunk_id:
          type: string
        chunk_type:
          type: string
        page:
          type: integer
        chapter_number:
          type: string
        chapter_title:
          type: string
        section:
          type: string
        section_heading:
          type: string
        figure_id:
          type: string
        image_path:
          type: string
        retrieval_mode:
          type: string
        fusion_score:
          type: number
        fts_rank:
          type: integer
        vector_rank:
          type: integer
        bm25_score:
          type: number
        vector_score:
          type: number
        score:
          type: number
      additionalProperties: true

    FigureItem:
      type: object
      properties:
        rank:
          type: integer
        title:
          type: string
        caption:
          type: string
        figure_id:
          type: string
        figure_url:
          type: string
        image_url:
          type: string
        image_path:
          type: string
        original_image_path:
          type: string
        original_image_url:
          type: string
        public_derivative_url:
          type: string
        proxy_fallback_used:
          type: boolean
        figure_serving_mode:
          type: string
        source:
          type: string
        source_name:
          type: string
        source_id:
          type: string
        page:
          type: integer
        score:
          type: number
      additionalProperties: true

    EvidenceSearchResponse:
      type: object
      properties:
        schema_version:
          type: string
        query:
          type: string
        source_status:
          $ref: "#/components/schemas/SourceStatus"
        who_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        journal_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        pathout_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        textbook_results:
          type: array
          items:
            $ref: "#/components/schemas/EvidenceItem"
        figures:
          type: array
          items:
            $ref: "#/components/schemas/FigureItem"
        warnings:
          type: array
          items:
            type: string
        search_mode:
          type: object
          additionalProperties: true
      additionalProperties: true

    ErrorResponse:
      type: object
      properties:
        error:
          type: string
        detail:
          type: string
        message:
          type: string
      additionalProperties: true

paths:
  /evidence/search:
    post:
      operationId: searchEvidence
      summary: Search Pathology Hub evidence.
      description: Search WHO, Journal, PathOut, and hybrid Textbook evidence. May return public textbook figure derivative URLs when include_figures=true.
      security:
        - ApiKeyAuth: []
      x-openai-isConsequential: false
      requestBody:
        required: true
        content:
          application/json:
            schema:
              $ref: "#/components/schemas/EvidenceSearchRequest"
      responses:
        "200":
          description: Evidence search results.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/EvidenceSearchResponse"
        "400":
          description: Bad request.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/ErrorResponse"
        "401":
          description: Unauthorized.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/ErrorResponse"
        "500":
          description: Server error.
          content:
            application/json:
              schema:
                $ref: "#/components/schemas/ErrorResponse"
"""

yaml_path = APP_DIR / "openapi_pathology_hub_unified_searchEvidence_hybrid_textbooks_publicfigs_v1_5_4.yaml"
yaml_path.write_text(openapi_yaml, encoding="utf-8")

handoff = {
    "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "workstream": "Backend API / Textbook Hybrid Search + Public Figure Derivatives",
    "purpose": "Return public web-safe textbook figure derivative URLs when include_figures=true, with proxy fallback.",
    "service": SERVICE_NAME,
    "service_url": service_url,
    "image": IMAGE,
    "sources_supported": ["who", "textbooks", "pathout", "journals"],
    "textbook_search_mode": "hybrid_fts_faiss_vector_rrf",
    "textbook_figure_mode": "public_web_derivative_urls_with_proxy_fallback",
    "excluded_figure_sources": ["derm_mckee", "bone_dorfman"],
    "vectorized": True,
    "api_exposed": True,
    "figure_proxy_enabled": True,
    "public_figure_map_gcs": TEXTBOOK_WEB_MAP_GCS,
    "openapi_yaml": str(yaml_path),
    "known_limitations": [
        "derm_mckee and bone_dorfman figures are intentionally excluded from public figure serving due poor derivative audit.",
        "Their text retrieval remains active.",
        "Proxy fallback may be used for non-excluded figures missing from the public map.",
        "Figure pixels are served, not interpreted by searchEvidence."
    ],
    "next_steps": [
        "Update GPT Action schema to v1.5.4 YAML.",
        "Patch GPT instructions: textbook figures use public derivative URLs with proxy fallback; McKee/Dorfman figures excluded.",
        "Regression test HTML/image atlas behavior."
    ]
}

handoff_path = APP_DIR / "HANDOFF_PATHOLOGY_HUB_V04_4_PUBLIC_TEXTBOOK_FIGURES_API.json"
handoff_path.write_text(json.dumps(handoff, indent=2), encoding="utf-8")

run(["gcloud", "storage", "cp", str(yaml_path), "gs://pathology_hub/04_api_artifacts/"], check=True)
run(["gcloud", "storage", "cp", str(handoff_path), "gs://pathology_hub/06_audits/handoff_packets/"], check=True)

print("\n✅ DEPLOYED v04.4 PUBLIC TEXTBOOK FIGURE API")
print("Service:", service_url)
print("OpenAPI YAML:", yaml_path)
print("Handoff:", handoff_path)


In [ ]:
# ============================================================
# CONFIRM v04.5 IS SAFE FOR GPT BUILDER
# Run AFTER the unhashable-fix redeploy cell.
#
# PASS condition:
#   prints ✅ GPT_BUILDER_IMPORT_OK
# ============================================================

import subprocess, requests, json, sys

PROJECT_ID = "pathology-annotation-project"
SERVICE_URL = "https://pathology-hub-v04-vorn5q2kga-uc.a.run.app"

def no_go(msg, data=None):
    print("\n❌ NO-GO:", msg)
    if data is not None:
        print(json.dumps(data, indent=2)[:9000])
    raise SystemExit(1)

def yes(msg):
    print("✅", msg)

def post_search(api_key, payload):
    r = requests.post(
        f"{SERVICE_URL}/evidence/search",
        headers={"X-API-Key": api_key, "Content-Type": "application/json"},
        json=payload,
        timeout=300,
    )
    try:
        data = r.json()
    except Exception:
        no_go(f"Non-JSON response HTTP {r.status_code}: {r.text[:1000]}")
    if r.status_code != 200:
        no_go(f"HTTP {r.status_code}", data)
    return data

subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID], check=True)

api_key = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", "pathology-hub-api-key"
], text=True).strip()

# ----------------------------
# 1. Health
# ----------------------------

h = requests.get(f"{SERVICE_URL}/health", timeout=300)
if h.status_code != 200:
    no_go(f"Health HTTP {h.status_code}", {"text": h.text[:1000]})

health = h.json()

print("\n=== HEALTH ===")
print(json.dumps({
    "version": health.get("version"),
    "journal_search_mode": health.get("journal_search_mode"),
    "journal_vectorized": health.get("journal_vectorized"),
    "journal_vector_records": health.get("journal_vector_records"),
    "textbook_search_mode": health.get("textbook_search_mode"),
    "public_figure_map_records_loaded": health.get("public_figure_map_records_loaded"),
}, indent=2))

if health.get("version") != "1.5.5-textbooks-journals-hybrid-publicfigs-v04":
    no_go("Wrong version; v04.5 is not active", health)

if health.get("journal_search_mode") != "hybrid_upstream_fts_faiss_vector_rrf":
    no_go("Journal search mode is not hybrid", health)

if health.get("journal_vectorized") is not True:
    no_go("journal_vectorized is not true", health)

if int(health.get("journal_vector_records") or 0) != 103830:
    no_go("journal_vector_records is not 103830", health)

yes("Health confirms v04.5 journal hybrid/vector exposure.")

# ----------------------------
# 2. Journal-only smoke
# ----------------------------

journal_data = post_search(api_key, {
    "query": "intestinal type adenocarcinoma sinonasal occupational wood dust",
    "sources": ["journals"],
    "max_results": 5,
    "include_figures": False,
    "max_figures": 0,
    "compact": True,
    "excerpt_char_limit": 1000,
})

print("\n=== JOURNAL SMOKE ===")
print("source_status:", journal_data.get("source_status"))
print("search_mode:", journal_data.get("search_mode"))
print("journal_results:", len(journal_data.get("journal_results", [])))
print("warnings:", journal_data.get("warnings"))

warning_text = " ".join(journal_data.get("warnings") or "")

if "unhashable" in warning_text or "journal_hybrid_error" in warning_text:
    no_go("Journal hybrid bug still present", journal_data)

if journal_data.get("source_status", {}).get("journals") != "ok":
    no_go("source_status.journals is not ok", journal_data)

if len(journal_data.get("journal_results", [])) < 1:
    no_go("No journal results returned", journal_data)

if journal_data.get("search_mode", {}).get("journals") != "hybrid_upstream_fts_faiss_vector_rrf":
    no_go("search_mode.journals is not hybrid", journal_data)

retrieval_modes = {h.get("retrieval_mode") for h in journal_data.get("journal_results", [])}
print("retrieval_modes:", retrieval_modes)

if not retrieval_modes.intersection({"hybrid_fts_vector", "vector_only", "fts_only"}):
    no_go("No valid journal retrieval_mode found", journal_data)

top = journal_data["journal_results"][0]
print("\nTop journal hit:")
print("title:", top.get("title"))
print("journal:", top.get("journal") or top.get("source_name"))
print("doi:", top.get("doi"))
print("retrieval_mode:", top.get("retrieval_mode"))

yes("Journal-only hybrid smoke passed.")

# ----------------------------
# 3. Combined smoke
# ----------------------------

combined_data = post_search(api_key, {
    "query": "bladder CIS p53 CK20",
    "sources": ["textbooks", "pathout", "journals"],
    "max_results": 2,
    "include_figures": False,
    "max_figures": 0,
    "compact": True,
    "excerpt_char_limit": 800,
})

print("\n=== COMBINED SMOKE ===")
print("source_status:", combined_data.get("source_status"))
print("counts:", {
    "textbooks": len(combined_data.get("textbook_results", [])),
    "pathout": len(combined_data.get("pathout_results", [])),
    "journals": len(combined_data.get("journal_results", [])),
})

if combined_data.get("source_status", {}).get("textbooks") != "ok":
    no_go("Combined smoke: textbooks not ok", combined_data)

if combined_data.get("source_status", {}).get("journals") != "ok":
    no_go("Combined smoke: journals not ok", combined_data)

if combined_data.get("source_status", {}).get("pathout") != "ok":
    no_go("Combined smoke: pathout not ok", combined_data)

yes("Combined source smoke passed.")

# ----------------------------
# 4. Textbook figure smoke
# ----------------------------

fig_data = post_search(api_key, {
    "query": "fallopian tube precursor lesion abnormal p53 increased proliferation",
    "sources": ["textbooks"],
    "max_results": 3,
    "include_figures": True,
    "max_figures": 3,
    "compact": True,
    "excerpt_char_limit": 900,
})

print("\n=== TEXTBOOK FIGURE SMOKE ===")
print("source_status:", fig_data.get("source_status"))
print("figures:", len(fig_data.get("figures", [])))

if fig_data.get("source_status", {}).get("textbooks") != "ok":
    no_go("Figure smoke: textbooks not ok", fig_data)

if len(fig_data.get("figures", [])) < 1:
    no_go("Figure smoke: no figures returned", fig_data)

fig_url = fig_data["figures"][0].get("figure_url") or fig_data["figures"][0].get("image_url")
print("first figure_url:", fig_url)

if not fig_url:
    no_go("Figure smoke: no figure_url/image_url", fig_data)

if "storage.googleapis.com/pathology-hub-public-figures-830130787988" not in fig_url:
    no_go("Figure smoke: first figure is not public derivative URL", fig_data)

img = requests.get(fig_url, timeout=120)
print("first figure HTTP:", img.status_code)
print("first figure content-type:", img.headers.get("content-type"))
print("first figure bytes:", len(img.content))

if img.status_code != 200:
    no_go("Figure URL did not return HTTP 200", {
        "status": img.status_code,
        "url": fig_url,
    })

if not img.headers.get("content-type", "").startswith("image/"):
    no_go("Figure URL did not return image content-type", {
        "content_type": img.headers.get("content-type"),
        "url": fig_url,
    })

if len(img.content) < 1000:
    no_go("Figure URL returned too few bytes", {
        "bytes": len(img.content),
        "url": fig_url,
    })

yes("Textbook figure smoke passed.")

print("\n============================================================")
print("✅ GPT_BUILDER_IMPORT_OK")
print("Now import the v1.5.5 YAML into GPT Builder.")
print("YAML local path:")
print("/content/pathology_hub_v04_textbook_api/openapi_pathology_hub_unified_searchEvidence_journal_hybrid_v1_5_5.yaml")
print("============================================================")

# Handoff packet

| Field | Value |
|---|---|
| Workstream name | Textbook + Tag RAG / Source Normalization |
| Purpose | Run a complete staged textbook workflow: GCS PDFs → raw UNIFIED extraction → ChatGPT LEAN/tagging intermission → LEAN integration → SQLite FTS → GCS upload. |
| Inputs assumed | PDFs under `gs://pathology-hub-0/source_pdfs`; ChatGPT-returned LEAN package ZIPs after intermission. |
| Outputs produced | Raw staged `*_UNIFIED.json`; ChatGPT job ZIPs; LEAN normalized JSONL; consolidated SQLite FTS; integration audit; index manifest. |
| Schemas used | `textbook_unified_page.raw.v1`, `textbook_page.lean_integrated.v1`, `textbook_chunk.lean_integrated.v1`, `textbook_figure.lean_integrated.v1`, `textbook_lean_index_manifest.v1`. |
| GCS paths assumed | Source PDFs: `gs://pathology-hub-0/source_pdfs`; staged raw/job/package outputs under `gs://pathology_hub/01_staged/textbooks/`; LEAN normalized under `gs://pathology_hub/02_normalized/textbooks/lean/`; index under `gs://pathology_hub/03_indexes/textbooks/lean/`; audits under `gs://pathology_hub/06_audits/textbooks/`. |
| API endpoints needed/provided | None provided. Textbook API exposure remains future work. |
| Integration points | Future `/evidence/search` textbook backend can consume `textbook_lean_fts.sqlite` after audit. |
| Tests/audits | PDF-to-UNIFIED batch audit; JSON/JSONL parse checks for LEAN packages; SQLite FTS smoke tests; tag catalog exact-string validation when catalog present. |
| Known limitations | No vector search; no image pixel interpretation; ChatGPT-cleaned outputs require spot audit; candidate tags are weak metadata unless reviewed; API exposure not live. |
| Next steps | Run Stage 1 for selected PDFs; process job ZIPs in ChatGPT; upload returned LEAN ZIPs; run Stage 3; inspect audit and spot-check figure/tag records before backend integration. |


In [ ]:
# Lecture corpus inventory — Pathology Hub
# Read-only exploration of gs://pathology-hub-0/_content_library/lectures/

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
import json, hashlib, os, re, collections, datetime
from pathlib import Path

PROJECT_ID = "pathology-annotation-project"
SOURCE_BUCKET = "pathology-hub-0"
LECTURE_PREFIX = "_content_library/lectures/"
ASSET_PREFIX_DEFAULT = "_asset_library/lectures/"
OUT_BUCKET = "pathology_hub"
OUT_PREFIX = "00_manifests/lectures/"

client = storage.Client(project=PROJECT_ID)
src_bucket = client.bucket(SOURCE_BUCKET)
out_bucket = client.bucket(OUT_BUCKET)

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def detect_format(obj):
    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
        keys = set(obj[0].keys())
        if {"segment_id", "start_time", "end_time", "image_path"} <= keys:
            return "lecture_segment_json"
        if {"entity_name", "definition", "related_figures"} <= keys:
            return "lecture_entity_master_json"
        return "list_of_dict_unknown"
    if isinstance(obj, dict):
        return "dict_unknown"
    return type(obj).__name__

def normalize_image_path(path):
    if not path:
        return None
    path = str(path).strip()
    if path.startswith("gs://"):
        return path
    if path.startswith("pathology-hub-0/"):
        return "gs://" + path
    if path.startswith("_asset_library/"):
        return f"gs://{SOURCE_BUCKET}/{path}"
    if path.startswith("/_asset_library/"):
        return f"gs://{SOURCE_BUCKET}{path}"
    return path

def guess_subspecialty(name):
    n = name.lower()
    if n.startswith("gu_") or "/gu" in n or "prostate" in n or "kidney" in n:
        return "GU"
    if n.startswith("derm_") or "/derm" in n or "skin" in n:
        return "Derm"
    if n.startswith("hn_") or "head" in n or "neck" in n or "thyroid" in n:
        return "HN_Endo"
    if n.startswith("cyto_") or "cyto" in n:
        return "Cyto"
    if "breast" in n:
        return "Breast"
    if "gi_" in n or "colon" in n or "stomach" in n:
        return "GI"
    return "Unknown"

def collect_image_refs(obj):
    refs = []
    if isinstance(obj, list):
        for rec in obj:
            if not isinstance(rec, dict):
                continue
            if rec.get("image_path"):
                refs.append(rec.get("image_path"))
            for fig in rec.get("related_figures") or []:
                if isinstance(fig, dict):
                    refs.append(fig.get("gcs_path") or fig.get("src") or fig.get("image_path"))
    return [r for r in refs if r]

rows = []
errors = []

for blob in client.list_blobs(SOURCE_BUCKET, prefix=LECTURE_PREFIX):
    if not blob.name.lower().endswith(".json"):
        continue

    row = {
        "source_gcs_uri": f"gs://{SOURCE_BUCKET}/{blob.name}",
        "size_bytes": blob.size,
        "updated": blob.updated.isoformat() if blob.updated else None,
        "json_parse_ok": False,
    }

    try:
        data_bytes = blob.download_as_bytes()
        row["sha256"] = sha256_bytes(data_bytes)
        obj = json.loads(data_bytes.decode("utf-8"))
        row["json_parse_ok"] = True
        row["top_level_type"] = type(obj).__name__
        row["record_count"] = len(obj) if hasattr(obj, "__len__") else None
        row["detected_format"] = detect_format(obj)

        keys = collections.Counter()
        if isinstance(obj, list):
            for rec in obj:
                if isinstance(rec, dict):
                    keys.update(rec.keys())
        elif isinstance(obj, dict):
            keys.update(obj.keys())

        row["keys_observed"] = dict(keys.most_common())
        row["lecture_id_guess"] = Path(blob.name).stem
        row["subspecialty_guess"] = guess_subspecialty(blob.name)

        image_refs = collect_image_refs(obj)
        normalized_refs = [normalize_image_path(x) for x in image_refs]
        row["image_ref_count"] = len(image_refs)
        row["relative_image_ref_count"] = sum(str(x).startswith("_asset_library/") for x in image_refs)
        row["gcs_image_ref_count"] = sum(str(x).startswith("gs://") for x in image_refs)
        row["normalized_image_ref_examples"] = normalized_refs[:5]

    except Exception as e:
        row["error"] = repr(e)
        errors.append(row)

    rows.append(row)

# duplicate content groups
sha_groups = collections.defaultdict(list)
for r in rows:
    if r.get("sha256"):
        sha_groups[r["sha256"]].append(r["source_gcs_uri"])

for r in rows:
    group = sha_groups.get(r.get("sha256"), [])
    r["duplicate_exact_count"] = len(group)
    r["duplicate_exact_uris"] = group if len(group) > 1 else []

summary = {
    "schema_version": "lecture_inventory_audit.v0_proposed",
    "created_at_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "source_prefix": f"gs://{SOURCE_BUCKET}/{LECTURE_PREFIX}",
    "json_file_count": len(rows),
    "parse_ok_count": sum(r.get("json_parse_ok") for r in rows),
    "parse_error_count": sum(not r.get("json_parse_ok") for r in rows),
    "detected_format_counts": dict(collections.Counter(r.get("detected_format", "parse_error") for r in rows)),
    "subspecialty_guess_counts": dict(collections.Counter(r.get("subspecialty_guess", "Unknown") for r in rows)),
    "exact_duplicate_file_groups": sum(1 for v in sha_groups.values() if len(v) > 1),
}

local_jsonl = "/content/lecture_gcs_inventory_20260623.jsonl"
local_summary = "/content/lecture_gcs_inventory_summary_20260623.json"

with open(local_jsonl, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open(local_summary, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2))

# Upload inventory manifests
out_bucket.blob(OUT_PREFIX + "lecture_gcs_inventory_20260623.jsonl").upload_from_filename(local_jsonl)
out_bucket.blob(OUT_PREFIX + "lecture_gcs_inventory_summary_20260623.json").upload_from_filename(local_summary)

print("Uploaded:")
print(f"gs://{OUT_BUCKET}/{OUT_PREFIX}lecture_gcs_inventory_20260623.jsonl")
print(f"gs://{OUT_BUCKET}/{OUT_PREFIX}lecture_gcs_inventory_summary_20260623.json")

In [ ]:
# ============================================================
# Pathology Hub — Lecture RAG / Curriculum Content Mapping CONFIG
# ============================================================
# Workstream: Lecture RAG / Curriculum Content Mapping
# Goal: inventory + normalize lecture JSONs from legacy source bucket,
#       stage derived artifacts under canonical pathology_hub paths,
#       then later build FTS/vector/API artifacts after audit.
#
# IMPORTANT:
# - This cell should not write/delete anything by itself.
# - Do not claim lectures are indexed/vectorized/API-exposed until
#   manifests + smoke tests exist.
# ============================================================

from dataclasses import dataclass, asdict
from pathlib import Path
import os, re, json, datetime, hashlib, collections

# ----------------------------
# GCP / bucket configuration
# ----------------------------

PROJECT_ID = "pathology-annotation-project"

# Legacy source bucket where lecture JSON and lecture slide assets currently live
SOURCE_BUCKET = "pathology-hub-0"
LECTURE_JSON_PREFIX = "_content_library/lectures/"
LECTURE_ASSET_PREFIX = "_asset_library/lectures/"

# Canonical derived-artifact bucket
DEST_BUCKET = "pathology_hub"

# ----------------------------
# Canonical output prefixes
# ----------------------------

LECTURE_WORKSTREAM_DATE = "20260623"

OUT_BASE_PREFIX = "lectures"

STAGED_PREFIX = "01_staged/lectures/"
NORMALIZED_PREFIX = "02_normalized/lectures/"
INDEX_PREFIX = "03_indexes/lectures/"
API_ARTIFACT_PREFIX = "04_api_artifacts/lectures/"
HTML_PREFIX = "05_html/lectures/"
AUDIT_PREFIX = "06_audits/lectures/"
MANIFEST_PREFIX = "00_manifests/lectures/"

# Proposed normalized outputs
LECTURE_SOURCES_JSONL = NORMALIZED_PREFIX + "lecture_sources.jsonl"
LECTURE_SEGMENTS_JSONL = NORMALIZED_PREFIX + "lecture_segments.jsonl"
LECTURE_ENTITIES_JSONL = NORMALIZED_PREFIX + "lecture_entities.jsonl"
LECTURE_CHUNKS_JSONL = NORMALIZED_PREFIX + "lecture_chunks.jsonl"
LECTURE_FIGURES_JSONL = NORMALIZED_PREFIX + "lecture_figures.jsonl"
LECTURE_TAG_CATALOG_JSONL = NORMALIZED_PREFIX + "lecture_tag_catalog.jsonl"

# Proposed audits/manifests
LECTURE_INVENTORY_JSONL = MANIFEST_PREFIX + f"lecture_gcs_inventory_{LECTURE_WORKSTREAM_DATE}.jsonl"
LECTURE_INVENTORY_SUMMARY_JSON = MANIFEST_PREFIX + f"lecture_gcs_inventory_summary_{LECTURE_WORKSTREAM_DATE}.json"
LECTURE_NORMALIZATION_AUDIT_JSON = AUDIT_PREFIX + f"lecture_normalization_audit_{LECTURE_WORKSTREAM_DATE}.json"

# Proposed future indexes
LECTURE_FTS_SQLITE = INDEX_PREFIX + "fts/lecture_fts.sqlite"
LECTURE_FTS_MANIFEST_JSON = INDEX_PREFIX + "fts/lecture_fts_index_manifest.json"

LECTURE_VECTOR_PREFIX = INDEX_PREFIX + "vector/"
LECTURE_VECTOR_EMBEDDINGS_NPY = LECTURE_VECTOR_PREFIX + "lecture_embeddings.npy"
LECTURE_VECTOR_FAISS_INDEX = LECTURE_VECTOR_PREFIX + "lecture_faiss.index"
LECTURE_VECTOR_DOCSTORE_JSONL = LECTURE_VECTOR_PREFIX + "lecture_vector_docstore.jsonl"
LECTURE_VECTOR_MANIFEST_JSON = LECTURE_VECTOR_PREFIX + "lecture_vector_manifest.json"

# ----------------------------
# Runtime / mode flags
# ----------------------------

DRY_RUN = True              # keep True for inventory/debug
WRITE_OUTPUTS = False       # set True only after reviewing inventory
OVERWRITE_OUTPUTS = False   # set True only for deliberate rebuilds
MAX_FILES = None            # e.g. 20 for pilot; None = all JSON files
PILOT_ONLY = False

# Good first pilot filters
PILOT_SUBSPECIALTIES = {"GU"}   # set to {"GU", "Derm"} etc.
PILOT_NAME_REGEX = None         # e.g. r"GU_Lecture_0|GU_Lecture_3"

# ----------------------------
# Format detection / adapters
# ----------------------------

SUPPORTED_FORMATS = {
    "lecture_segment_json",        # segment_id/start_time/end_time/image_path/transcript
    "lecture_entity_master_json",  # entity_name/definition/related_figures/tags
}

SEGMENT_REQUIRED_KEYS = {
    "segment_id",
    "start_time",
    "end_time",
    "image_path",
}

SEGMENT_OPTIONAL_TEXT_KEYS = [
    "title",
    "summary",
    "cleaned_transcript",
    "raw_transcript",
]

ENTITY_REQUIRED_KEYS = {
    "entity_name",
    "definition",
}

ENTITY_OPTIONAL_KEYS = [
    "tags",
    "clinical",
    "pathogenesis",
    "macroscopic",
    "microscopic",
    "ancillary_studies",
    "differential_diagnosis",
    "staging",
    "prognosis_and_prediction",
    "related_figures",
]

# ----------------------------
# Chunking config
# ----------------------------

CHUNK_SCHEMA_VERSION = "lecture_chunk.v0_proposed"

# Segment lecture chunks
INCLUDE_RAW_TRANSCRIPT_IN_CHUNKS = True
INCLUDE_NEIGHBOR_CONTEXT = True
NEIGHBOR_SEGMENTS_EACH_SIDE = 1

# Entity-master chunks
ENTITY_CHUNK_ONE_RECORD_PER_ENTITY = True

# Character safety limits
MAX_CHUNK_TEXT_CHARS = 6000
MIN_CHUNK_TEXT_CHARS = 80

# ----------------------------
# Tag config
# ----------------------------

PRESERVE_EXISTING_TAGS = True
ALLOW_FILENAME_SUBSPECIALTY_GUESS = True
ALLOW_LLM_TAGGING_LATER = True

TAG_SOURCE_EXISTING = "existing"
TAG_SOURCE_FILENAME_RULE = "filename_rule"
TAG_SOURCE_LLM_CANDIDATE = "llm_candidate"
TAG_SOURCE_CURATED = "curated"

# Controlled tags are metadata only, not diagnostic truth
TAGS_ARE_DIAGNOSTIC_TRUTH = False

# Optional: point to mounted/extracted controlled tag registry later
CONTROLLED_TAG_ROOT_LOCAL = None  # e.g. "/content/Tags"

# ----------------------------
# Figure/image config
# ----------------------------

NORMALIZE_RELATIVE_IMAGE_PATHS = True

# Do not expose gs:// directly in HTML/GPT. This is only source normalization.
LECTURE_PUBLIC_FIGURES_ENABLED = False

# Future public derivative location if/when built
PUBLIC_FIGURE_BUCKET = "pathology-hub-public-figures-830130787988"
PUBLIC_LECTURE_FIGURE_PREFIX = "lecture_figures_web_v1/"

# Until public derivatives/proxy exist, output source_gcs_uri only.
FIGURE_OUTPUT_PUBLIC_URLS_ONLY_IF_VERIFIED = True

# ----------------------------
# Embedding/vector config — future use only
# ----------------------------

BUILD_VECTORS = False
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536
VECTOR_INDEX_TYPE = "faiss_IndexFlatIP_cosine_normalized"

# ----------------------------
# API integration config — future use only
# ----------------------------

API_SOURCE_NAME_PROPOSED = "lectures"
API_EXPOSE_LECTURES = False

# Current active evidence API does NOT yet include lectures.
# Do not set sources=["lectures"] in GPT until API contract/schema/health confirm it.
ACTIVE_API_SERVICE = "pathology-hub-v04"
ACTIVE_API_OPERATION = "POST /evidence/search"
ACTIVE_API_VERSION_CONFIRMED = "1.5.4-textbooks-hybrid-publicfigs-v04"

# ----------------------------
# Helpers
# ----------------------------

def utc_now_iso():
    return datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def slugify(s: str) -> str:
    s = str(s or "").strip()
    s = re.sub(r"\.[A-Za-z0-9]+$", "", s)
    s = re.sub(r"[^A-Za-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s.lower() or "unknown"

def guess_subspecialty_from_name(path_or_name: str) -> str:
    n = str(path_or_name).lower()
    if re.search(r"(^|[/_ -])gu([/_ -]|$)", n) or any(x in n for x in ["prostate", "kidney", "bladder", "testis", "renal"]):
        return "GU"
    if "derm" in n or "skin" in n or "follicul" in n:
        return "Derm"
    if "breast" in n:
        return "Breast"
    if re.search(r"(^|[/_ -])gi([/_ -]|$)", n) or any(x in n for x in ["colon", "stomach", "pancreas", "liver", "bowel"]):
        return "GI"
    if any(x in n for x in ["hn", "head", "neck", "thyroid", "salivary", "oral", "sinonasal"]):
        return "HN_Endo"
    if "cyto" in n:
        return "Cyto"
    if "heme" in n or "lymph" in n:
        return "Heme"
    if "gyn" in n or "uter" in n or "ovary" in n:
        return "Gyn"
    if "thor" in n or "lung" in n:
        return "Thoracic"
    if "molecular" in n or "mol" in n:
        return "Molecular"
    return "Unknown"

def normalize_gcs_image_path(path: str) -> str | None:
    """
    Normalize lecture image references to full gs:// source paths.
    Does NOT create public/browser URLs.
    """
    if not path:
        return None
    p = str(path).strip()

    if p.startswith("gs://"):
        return p

    if p.startswith(f"{SOURCE_BUCKET}/"):
        return "gs://" + p

    if p.startswith("_asset_library/"):
        return f"gs://{SOURCE_BUCKET}/{p}"

    if p.startswith("/_asset_library/"):
        return f"gs://{SOURCE_BUCKET}{p}"

    # If a file only gives a relative lecture asset path, preserve for audit
    # rather than guessing too aggressively.
    return p

def detect_lecture_json_format(obj) -> str:
    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
        keys = set(obj[0].keys())

        if SEGMENT_REQUIRED_KEYS <= keys:
            return "lecture_segment_json"

        if ENTITY_REQUIRED_KEYS <= keys and (
            "related_figures" in keys or "microscopic" in keys or "differential_diagnosis" in keys
        ):
            return "lecture_entity_master_json"

        return "list_of_dict_unknown"

    if isinstance(obj, list) and not obj:
        return "empty_list"

    if isinstance(obj, dict):
        return "dict_unknown"

    return type(obj).__name__

def lecture_source_id_from_gcs_uri(gcs_uri: str) -> str:
    name = Path(str(gcs_uri).replace("gs://", "").split("/", 1)[-1]).stem
    return "lecture::" + slugify(name)

def make_segment_id(lecture_source_id: str, segment_id) -> str:
    try:
        seg = int(segment_id)
        return f"{lecture_source_id}::segment::{seg:06d}"
    except Exception:
        return f"{lecture_source_id}::segment::{slugify(segment_id)}"

def make_entity_id(lecture_source_id: str, entity_name: str) -> str:
    return f"{lecture_source_id}::entity::{slugify(entity_name)}"

def make_chunk_id(parent_id: str, chunk_ix: int = 0) -> str:
    return f"{parent_id}::chunk::{chunk_ix:03d}"

def make_figure_id(lecture_source_id: str, image_path: str | None = None, fallback_id: str | None = None) -> str:
    raw = fallback_id or Path(str(image_path or "figure")).stem
    return f"{lecture_source_id}::figure::{slugify(raw)}"

@dataclass
class LectureRagConfig:
    project_id: str = PROJECT_ID
    source_bucket: str = SOURCE_BUCKET
    dest_bucket: str = DEST_BUCKET
    lecture_json_prefix: str = LECTURE_JSON_PREFIX
    lecture_asset_prefix: str = LECTURE_ASSET_PREFIX
    normalized_prefix: str = NORMALIZED_PREFIX
    audit_prefix: str = AUDIT_PREFIX
    manifest_prefix: str = MANIFEST_PREFIX
    dry_run: bool = DRY_RUN
    write_outputs: bool = WRITE_OUTPUTS
    overwrite_outputs: bool = OVERWRITE_OUTPUTS
    max_files: int | None = MAX_FILES
    pilot_only: bool = PILOT_ONLY
    pilot_subspecialties: set | None = None
    pilot_name_regex: str | None = PILOT_NAME_REGEX
    chunk_schema_version: str = CHUNK_SCHEMA_VERSION
    api_source_name_proposed: str = API_SOURCE_NAME_PROPOSED
    api_expose_lectures: bool = API_EXPOSE_LECTURES
    public_figures_enabled: bool = LECTURE_PUBLIC_FIGURES_ENABLED
    build_vectors: bool = BUILD_VECTORS
    embedding_model: str = EMBEDDING_MODEL
    embedding_dim: int = EMBEDDING_DIM

CONFIG = LectureRagConfig(
    pilot_subspecialties=PILOT_SUBSPECIALTIES
)

print("Lecture RAG config loaded:")
print(json.dumps({
    **asdict(CONFIG),
    "pilot_subspecialties": sorted(CONFIG.pilot_subspecialties) if CONFIG.pilot_subspecialties else None,
    "source_root": f"gs://{SOURCE_BUCKET}/{LECTURE_JSON_PREFIX}",
    "asset_root": f"gs://{SOURCE_BUCKET}/{LECTURE_ASSET_PREFIX}",
    "dest_root": f"gs://{DEST_BUCKET}/{NORMALIZED_PREFIX}",
    "inventory_jsonl": f"gs://{DEST_BUCKET}/{LECTURE_INVENTORY_JSONL}",
    "inventory_summary": f"gs://{DEST_BUCKET}/{LECTURE_INVENTORY_SUMMARY_JSON}",
    "notes": [
        "read-only by default",
        "lectures are not yet an API source",
        "do not expose gs:// image paths as HTML image URLs",
        "tags are metadata, not diagnostic truth"
    ]
}, indent=2))

In [ ]:
# ============================================================
# Pathology Hub — Lecture Normalizer v0 Proposed
# ============================================================
# Inputs:
#   gs://pathology_hub/00_manifests/lectures/lecture_gcs_inventory_20260623.jsonl
#
# Outputs:
#   gs://pathology_hub/02_normalized/lectures/lecture_sources.jsonl
#   gs://pathology_hub/02_normalized/lectures/lecture_segments.jsonl
#   gs://pathology_hub/02_normalized/lectures/lecture_entities.jsonl
#   gs://pathology_hub/02_normalized/lectures/lecture_chunks.jsonl
#   gs://pathology_hub/02_normalized/lectures/lecture_figures.jsonl
#   gs://pathology_hub/02_normalized/lectures/lecture_tag_catalog.jsonl
#   gs://pathology_hub/06_audits/lectures/lecture_normalization_audit_20260623.json
#
# Important:
# - Normalization only.
# - No indexing/vectorization/API exposure is claimed by this cell.
# - Lecture figures remain source_gcs_uri only unless public derivative/proxy audit exists.
# ============================================================

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, re, collections, os, math

# ----------------------------
# Config fallbacks
# ----------------------------

PROJECT_ID = globals().get("PROJECT_ID", "pathology-annotation-project")
SOURCE_BUCKET = globals().get("SOURCE_BUCKET", "pathology-hub-0")
DEST_BUCKET = globals().get("DEST_BUCKET", "pathology_hub")

INVENTORY_JSONL_GCS = "gs://pathology_hub/00_manifests/lectures/lecture_gcs_inventory_20260623.jsonl"

NORMALIZED_PREFIX = globals().get("NORMALIZED_PREFIX", "02_normalized/lectures/")
AUDIT_PREFIX = globals().get("AUDIT_PREFIX", "06_audits/lectures/")

LECTURE_WORKSTREAM_DATE = globals().get("LECTURE_WORKSTREAM_DATE", "20260623")

OUT_SOURCES = NORMALIZED_PREFIX + "lecture_sources.jsonl"
OUT_SEGMENTS = NORMALIZED_PREFIX + "lecture_segments.jsonl"
OUT_ENTITIES = NORMALIZED_PREFIX + "lecture_entities.jsonl"
OUT_CHUNKS = NORMALIZED_PREFIX + "lecture_chunks.jsonl"
OUT_FIGURES = NORMALIZED_PREFIX + "lecture_figures.jsonl"
OUT_TAGS = NORMALIZED_PREFIX + "lecture_tag_catalog.jsonl"
OUT_AUDIT = AUDIT_PREFIX + f"lecture_normalization_audit_{LECTURE_WORKSTREAM_DATE}.json"

LOCAL_OUT_DIR = Path("/content/lecture_normalized_v0")
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

NORMALIZER_WRITE_OUTPUTS = True
NORMALIZER_OVERWRITE_OUTPUTS = True
NORMALIZER_MAX_FILES = None          # e.g. 50 for pilot; None = all files
SKIP_EXACT_DUPLICATES = True

# For pilot, set e.g. {"GU"} or None for all.
NORMALIZER_SUBSPECIALTY_FILTER = None

# Do not create public URLs here.
LECTURE_PUBLIC_FIGURES_ENABLED = False

client = storage.Client(project=PROJECT_ID)

# ----------------------------
# Helpers
# ----------------------------

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def parse_gcs_uri(uri):
    assert uri.startswith("gs://"), uri
    rest = uri[5:]
    bucket, name = rest.split("/", 1)
    return bucket, name

def download_text_from_gcs(gcs_uri):
    bucket_name, blob_name = parse_gcs_uri(gcs_uri)
    return client.bucket(bucket_name).blob(blob_name).download_as_text(encoding="utf-8")

def download_bytes_from_gcs(gcs_uri):
    bucket_name, blob_name = parse_gcs_uri(gcs_uri)
    return client.bucket(bucket_name).blob(blob_name).download_as_bytes()

def download_json_from_gcs(gcs_uri):
    return json.loads(download_text_from_gcs(gcs_uri))

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def slugify(s):
    s = str(s or "").strip()
    s = re.sub(r"\.[A-Za-z0-9]+$", "", s)
    s = re.sub(r"[^A-Za-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_").lower()
    return s or "unknown"

def strip_known_suffixes(stem):
    s = str(stem)
    suffixes = [
        "_SLIDE_SESSION_MASTER",
        "_SESSION_MASTER",
        "_MASTER",
        "_RAW",
        "_CONTENT",
        "_FIGURES",
    ]
    changed = True
    while changed:
        changed = False
        for suf in suffixes:
            if s.endswith(suf):
                s = s[: -len(suf)]
                changed = True
    return s

def lecture_source_id_from_gcs_uri(gcs_uri):
    stem = Path(gcs_uri).stem
    return "lecture::" + slugify(stem)

def lecture_family_id_from_gcs_uri(gcs_uri):
    stem = Path(gcs_uri).stem
    return "lecture_family::" + slugify(strip_known_suffixes(stem))

def make_segment_id(lecture_source_id, segment_id, fallback_ix):
    raw = segment_id if segment_id is not None else fallback_ix
    try:
        seg = int(raw)
        return f"{lecture_source_id}::segment::{seg:06d}"
    except Exception:
        return f"{lecture_source_id}::segment::{slugify(raw)}"

def make_entity_id(lecture_source_id, entity_name, fallback_ix):
    raw = entity_name if entity_name else fallback_ix
    return f"{lecture_source_id}::entity::{slugify(raw)}"

def make_chunk_id(parent_id, chunk_ix=0):
    return f"{parent_id}::chunk::{chunk_ix:03d}"

def make_figure_id(lecture_source_id, image_path=None, fallback_id=None):
    raw = None
    if image_path:
        raw = Path(str(image_path)).stem
    raw = fallback_id or raw or "figure"
    return f"{lecture_source_id}::figure::{slugify(raw)}"

def guess_subspecialty_from_name(path_or_name):
    n = str(path_or_name).lower()
    if re.search(r"(^|[/_ -])gu([/_ -]|$)", n) or any(x in n for x in ["prostate", "kidney", "bladder", "testis", "renal"]):
        return "GU"
    if "derm" in n or "skin" in n or "follicul" in n:
        return "Derm"
    if "breast" in n:
        return "Breast"
    if re.search(r"(^|[/_ -])gi([/_ -]|$)", n) or any(x in n for x in ["colon", "stomach", "pancreas", "liver", "bowel"]):
        return "GI"
    if any(x in n for x in ["hn", "head", "neck", "thyroid", "salivary", "oral", "sinonasal"]):
        return "HN_Endo"
    if "cyto" in n:
        return "Cyto"
    if "heme" in n or "lymph" in n:
        return "Heme"
    if "gyn" in n or "uter" in n or "ovary" in n:
        return "Gyn"
    if "thor" in n or "lung" in n:
        return "Thoracic"
    if "molecular" in n or "mol" in n or "amp" in n:
        return "Molecular"
    return "Unknown"

def normalize_gcs_path(path):
    """
    Normalize lecture image/media references to full gs:// paths when safely possible.
    Does not create browser/public URLs.
    """
    if not path:
        return None

    p = str(path).strip()
    if not p:
        return None

    if p.startswith("gs://"):
        return p

    if p.startswith(f"{SOURCE_BUCKET}/"):
        return "gs://" + p

    if p.startswith("_asset_library/"):
        return f"gs://{SOURCE_BUCKET}/{p}"

    if p.startswith("/_asset_library/"):
        return f"gs://{SOURCE_BUCKET}{p}"

    if p.startswith("_content_library/"):
        return f"gs://{SOURCE_BUCKET}/{p}"

    if p.startswith("/_content_library/"):
        return f"gs://{SOURCE_BUCKET}{p}"

    return p

def is_gcs_uri(x):
    return isinstance(x, str) and x.startswith("gs://")

def safe_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [v for v in x if v is not None and str(v).strip() != ""]
    if isinstance(x, str):
        if not x.strip():
            return []
        return [x.strip()]
    return [x]

def as_clean_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return re.sub(r"\s+", " ", x).strip()
    if isinstance(x, list):
        vals = [as_clean_text(v) for v in x]
        return "; ".join([v for v in vals if v])
    if isinstance(x, dict):
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    return str(x).strip()

def parse_time_to_seconds(v):
    if v is None:
        return None
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        if math.isfinite(float(v)):
            return float(v)
        return None
    s = str(v).strip()
    if not s:
        return None
    try:
        return float(s)
    except Exception:
        pass

    # HH:MM:SS or MM:SS
    parts = s.split(":")
    try:
        nums = [float(p) for p in parts]
        if len(nums) == 3:
            return nums[0] * 3600 + nums[1] * 60 + nums[2]
        if len(nums) == 2:
            return nums[0] * 60 + nums[1]
    except Exception:
        return None
    return None

def get_first(rec, keys, default=None):
    for k in keys:
        if k in rec and rec.get(k) not in [None, ""]:
            return rec.get(k)
    return default

def flatten_nested_records(obj):
    """
    Flatten top-level nested lists if all leaves are dict records.
    Returns (records, flattened_ok, error).
    """
    records = []

    def walk(x):
        if isinstance(x, dict):
            records.append(x)
        elif isinstance(x, list):
            for y in x:
                walk(y)
        else:
            raise TypeError(f"Non-dict/list leaf: {type(x).__name__}")

    try:
        walk(obj)
        return records, True, None
    except Exception as e:
        return [], False, repr(e)

def detect_adapter(records):
    if not records:
        return "empty_or_unusable"

    key_union = set()
    first_keys = set(records[0].keys()) if isinstance(records[0], dict) else set()
    for rec in records[:25]:
        if isinstance(rec, dict):
            key_union.update(rec.keys())

    # Standard transcript/slide segment JSON
    if {"segment_id", "start_time", "end_time", "image_path"} <= key_union:
        return "segment_standard"

    # RAW slide segment variants
    raw_segment_keys = {"raw_transcript", "timestamp_start", "timestamp_end"}
    raw_imageish = {"image_path", "gcs_path", "path"}
    if raw_segment_keys <= key_union and ("slide_title" in key_union or "visual_desc" in key_union) and (raw_imageish & key_union):
        return "segment_raw_variant"

    # Segment without image field but still useful
    if raw_segment_keys <= key_union and ("slide_title" in key_union or "visual_desc" in key_union):
        return "segment_raw_variant_no_image"

    # Entity master variants, including Cyto-style masters
    if {"entity_name", "definition"} <= key_union:
        return "entity_master_variant"

    return "unsupported"

def add_section(parts, label, value):
    txt = as_clean_text(value)
    if txt:
        parts.append(f"{label}: {txt}")

def build_segment_chunk_text(seg):
    parts = []
    add_section(parts, "Lecture", seg.get("lecture_title"))
    add_section(parts, "Slide/segment title", seg.get("title"))
    if seg.get("start_time_sec") is not None or seg.get("end_time_sec") is not None:
        parts.append(f"Time: {seg.get('start_time_sec')}–{seg.get('end_time_sec')} sec")
    add_section(parts, "Summary", seg.get("summary"))
    add_section(parts, "Visual description", seg.get("visual_desc"))
    add_section(parts, "Key points", seg.get("key_points"))
    add_section(parts, "Labels", seg.get("labels"))
    add_section(parts, "Cleaned transcript", seg.get("cleaned_transcript"))
    add_section(parts, "Raw transcript", seg.get("raw_transcript"))
    return "\n".join(parts).strip()

def build_entity_chunk_text(ent):
    parts = []
    add_section(parts, "Entity", ent.get("entity_name"))
    add_section(parts, "Synonyms", ent.get("synonyms"))
    add_section(parts, "Definition", ent.get("definition"))
    add_section(parts, "Clinical", ent.get("clinical"))
    add_section(parts, "Pathogenesis", ent.get("pathogenesis"))
    add_section(parts, "Macroscopic", ent.get("macroscopic"))
    add_section(parts, "Microscopic", ent.get("microscopic"))
    add_section(parts, "Cytology", ent.get("cytology"))
    add_section(parts, "Ancillary studies", ent.get("ancillary_studies"))
    add_section(parts, "Differential diagnosis", ent.get("differential_diagnosis"))
    add_section(parts, "Reporting category", ent.get("reporting_category"))
    add_section(parts, "Staging", ent.get("staging"))
    add_section(parts, "Prognosis and prediction", ent.get("prognosis_and_prediction"))
    add_section(parts, "Tags", ent.get("tags"))
    return "\n".join(parts).strip()

def write_jsonl_local(rows, local_path):
    with open(local_path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def upload_file(local_path, dest_blob_name):
    blob = client.bucket(DEST_BUCKET).blob(dest_blob_name)
    if blob.exists() and not NORMALIZER_OVERWRITE_OUTPUTS:
        raise FileExistsError(f"Refusing to overwrite gs://{DEST_BUCKET}/{dest_blob_name}")
    blob.upload_from_filename(str(local_path))
    return f"gs://{DEST_BUCKET}/{dest_blob_name}"

def load_inventory():
    local_inv = Path("/content/lecture_gcs_inventory_20260623.jsonl")
    if local_inv.exists():
        with open(local_inv, "r", encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]
    text = download_text_from_gcs(INVENTORY_JSONL_GCS)
    return [json.loads(line) for line in text.splitlines() if line.strip()]

# ----------------------------
# Load inventory and prep duplicate groups
# ----------------------------

inventory = load_inventory()
inventory = [r for r in inventory if r.get("json_parse_ok")]

if NORMALIZER_SUBSPECIALTY_FILTER:
    inventory = [
        r for r in inventory
        if r.get("subspecialty_guess") in NORMALIZER_SUBSPECIALTY_FILTER
    ]

inventory = sorted(inventory, key=lambda r: r["source_gcs_uri"])

if NORMALIZER_MAX_FILES is not None:
    inventory = inventory[:NORMALIZER_MAX_FILES]

sha_groups = collections.defaultdict(list)
for r in inventory:
    if r.get("sha256"):
        sha_groups[r["sha256"]].append(r["source_gcs_uri"])

canonical_by_sha = {}
for sha, uris in sha_groups.items():
    canonical_by_sha[sha] = sorted(uris)[0]

# ----------------------------
# Normalize
# ----------------------------

created_at = utc_now_iso()

sources_out = []
segments_out = []
entities_out = []
chunks_out = []
figures_out = []
tags_out = []
unsupported = []
skipped_duplicates = []

adapter_counts = collections.Counter()
source_status_counts = collections.Counter()
image_stats = collections.Counter()

seen_figure_rows = set()

for file_ix, inv in enumerate(inventory):
    source_gcs_uri = inv["source_gcs_uri"]
    source_sha = inv.get("sha256")
    canonical_uri = canonical_by_sha.get(source_sha, source_gcs_uri)

    lecture_source_id = lecture_source_id_from_gcs_uri(source_gcs_uri)
    lecture_family_id = lecture_family_id_from_gcs_uri(source_gcs_uri)
    subspecialty = inv.get("subspecialty_guess") or guess_subspecialty_from_name(source_gcs_uri)

    source_blob_name = parse_gcs_uri(source_gcs_uri)[1]
    source_stem = Path(source_blob_name).stem

    source_base = {
        "schema_version": "lecture_source.v0_proposed",
        "lecture_source_id": lecture_source_id,
        "lecture_family_id": lecture_family_id,
        "source_gcs_uri": source_gcs_uri,
        "source_bucket": SOURCE_BUCKET,
        "source_blob_name": source_blob_name,
        "source_filename": Path(source_blob_name).name,
        "source_stem": source_stem,
        "source_subspecialty_guess": subspecialty,
        "inventory_detected_format": inv.get("detected_format"),
        "record_count_inventory": inv.get("record_count"),
        "sha256": source_sha,
        "size_bytes": inv.get("size_bytes"),
        "updated": inv.get("updated"),
        "created_at_utc": created_at,
    }

    if SKIP_EXACT_DUPLICATES and source_gcs_uri != canonical_uri:
        row = {
            **source_base,
            "normalization_status": "skipped_exact_duplicate",
            "duplicate_of_source_gcs_uri": canonical_uri,
        }
        sources_out.append(row)
        skipped_duplicates.append(row)
        source_status_counts["skipped_exact_duplicate"] += 1
        continue

    try:
        obj = download_json_from_gcs(source_gcs_uri)
        records, flattened_ok, flatten_error = flatten_nested_records(obj)
        adapter = detect_adapter(records)

        adapter_counts[adapter] += 1

        source_row = {
            **source_base,
            "normalization_status": "normalized" if adapter != "unsupported" else "unsupported",
            "adapter": adapter,
            "flattened_nested_records": flattened_ok,
            "flatten_error": flatten_error,
            "records_after_flatten": len(records),
        }
        sources_out.append(source_row)

        if adapter == "unsupported":
            source_status_counts["unsupported"] += 1
            key_union = sorted(set().union(*[set(x.keys()) for x in records[:25] if isinstance(x, dict)])) if records else []
            unsupported.append({
                "source_gcs_uri": source_gcs_uri,
                "lecture_source_id": lecture_source_id,
                "inventory_detected_format": inv.get("detected_format"),
                "records_after_flatten": len(records),
                "key_union_first_25": key_union,
                "first_record_preview": records[0] if records else None,
            })
            continue

        source_status_counts["normalized"] += 1

        # ----------------------------
        # Segment adapters
        # ----------------------------
        if adapter in {"segment_standard", "segment_raw_variant", "segment_raw_variant_no_image"}:
            for rec_ix, rec in enumerate(records):
                if not isinstance(rec, dict):
                    continue

                source_record_id = get_first(rec, ["segment_id", "id"], rec_ix)
                segment_id = make_segment_id(lecture_source_id, source_record_id, rec_ix)

                start_raw = get_first(rec, ["start_time", "timestamp_start"])
                end_raw = get_first(rec, ["end_time", "timestamp_end"])

                image_original = get_first(rec, ["image_path", "gcs_path", "path"])
                image_norm = normalize_gcs_path(image_original)

                if image_original:
                    image_stats["image_ref_total"] += 1
                    if str(image_original).startswith("gs://"):
                        image_stats["image_ref_original_gcs"] += 1
                    elif str(image_original).startswith("_asset_library/") or str(image_original).startswith("/_asset_library/"):
                        image_stats["image_ref_original_relative_asset"] += 1

                    if is_gcs_uri(image_norm):
                        image_stats["image_ref_normalized_gcs"] += 1
                    else:
                        image_stats["image_ref_unresolved_or_non_gcs"] += 1

                key_points = safe_list(rec.get("key_points"))
                labels = []
                labels.extend(safe_list(rec.get("labels")))
                labels.extend(safe_list(rec.get("extracted_labels")))

                seg = {
                    "schema_version": "lecture_segment.v0_proposed",
                    "lecture_source_id": lecture_source_id,
                    "lecture_family_id": lecture_family_id,
                    "lecture_segment_id": segment_id,
                    "source_gcs_uri": source_gcs_uri,
                    "source_record_id": source_record_id,
                    "segment_index": rec_ix,
                    "adapter": adapter,
                    "subspecialty_guess": subspecialty,
                    "lecture_title": strip_known_suffixes(source_stem),
                    "title": get_first(rec, ["title", "slide_title"]),
                    "start_time_raw": start_raw,
                    "end_time_raw": end_raw,
                    "start_time_sec": parse_time_to_seconds(start_raw),
                    "end_time_sec": parse_time_to_seconds(end_raw),
                    "raw_transcript": rec.get("raw_transcript"),
                    "cleaned_transcript": rec.get("cleaned_transcript"),
                    "summary": rec.get("summary"),
                    "visual_desc": rec.get("visual_desc"),
                    "key_points": key_points,
                    "labels": labels,
                    "image_path_original": image_original,
                    "image_gcs_uri": image_norm if is_gcs_uri(image_norm) else None,
                    "image_path_unresolved": image_norm if image_norm and not is_gcs_uri(image_norm) else None,
                    "created_at_utc": created_at,
                }
                segments_out.append(seg)

                chunk_text = build_segment_chunk_text(seg)
                if chunk_text:
                    chunks_out.append({
                        "schema_version": "lecture_chunk.v0_proposed",
                        "lecture_chunk_id": make_chunk_id(segment_id, 0),
                        "lecture_source_id": lecture_source_id,
                        "lecture_family_id": lecture_family_id,
                        "parent_id": segment_id,
                        "parent_type": "segment",
                        "source_gcs_uri": source_gcs_uri,
                        "adapter": adapter,
                        "subspecialty_guess": subspecialty,
                        "title": seg.get("title") or seg.get("lecture_title"),
                        "entity_name": None,
                        "start_time_sec": seg.get("start_time_sec"),
                        "end_time_sec": seg.get("end_time_sec"),
                        "image_gcs_uri": seg.get("image_gcs_uri"),
                        "tags": labels,
                        "text": chunk_text,
                        "created_at_utc": created_at,
                    })

                # Figure row from slide image
                if seg.get("image_gcs_uri") or seg.get("image_path_unresolved"):
                    fig_id = make_figure_id(
                        lecture_source_id,
                        seg.get("image_gcs_uri") or seg.get("image_path_unresolved"),
                        fallback_id=source_record_id,
                    )
                    fig_key = (fig_id, segment_id)
                    if fig_key not in seen_figure_rows:
                        seen_figure_rows.add(fig_key)
                        figures_out.append({
                            "schema_version": "lecture_figure.v0_proposed",
                            "lecture_figure_id": fig_id,
                            "lecture_source_id": lecture_source_id,
                            "lecture_family_id": lecture_family_id,
                            "parent_id": segment_id,
                            "parent_type": "segment",
                            "source_gcs_uri": source_gcs_uri,
                            "source_record_id": source_record_id,
                            "figure_label": str(source_record_id),
                            "caption": seg.get("title") or seg.get("summary") or seg.get("visual_desc"),
                            "legend": seg.get("visual_desc"),
                            "image_path_original": seg.get("image_path_original"),
                            "image_gcs_uri": seg.get("image_gcs_uri"),
                            "image_path_unresolved": seg.get("image_path_unresolved"),
                            "figure_url": None,
                            "image_url": None,
                            "public_url_verified": False,
                            "created_at_utc": created_at,
                        })

                # Preserve labels as tag-like metadata, not diagnostic truth
                for tag_ix, tag in enumerate(labels):
                    tags_out.append({
                        "schema_version": "lecture_tag_catalog.v0_proposed",
                        "lecture_source_id": lecture_source_id,
                        "lecture_family_id": lecture_family_id,
                        "parent_id": segment_id,
                        "parent_type": "segment",
                        "tag": str(tag),
                        "tag_source": "labels_or_extracted_labels",
                        "tag_confidence": None,
                        "diagnostic_truth": False,
                        "created_at_utc": created_at,
                    })

        # ----------------------------
        # Entity-master adapter
        # ----------------------------
        elif adapter == "entity_master_variant":
            for rec_ix, rec in enumerate(records):
                if not isinstance(rec, dict):
                    continue

                entity_name = rec.get("entity_name") or rec.get("diagnosis") or rec.get("title")
                entity_id = make_entity_id(lecture_source_id, entity_name, rec_ix)

                tags = safe_list(rec.get("tags"))
                synonyms = safe_list(rec.get("synonyms"))

                ent = {
                    "schema_version": "lecture_entity.v0_proposed",
                    "lecture_source_id": lecture_source_id,
                    "lecture_family_id": lecture_family_id,
                    "lecture_entity_id": entity_id,
                    "source_gcs_uri": source_gcs_uri,
                    "source_record_index": rec_ix,
                    "adapter": adapter,
                    "subspecialty_guess": subspecialty,
                    "lecture_title": strip_known_suffixes(source_stem),
                    "entity_name": entity_name,
                    "synonyms": synonyms,
                    "definition": rec.get("definition"),
                    "clinical": rec.get("clinical"),
                    "pathogenesis": rec.get("pathogenesis"),
                    "macroscopic": rec.get("macroscopic"),
                    "microscopic": rec.get("microscopic"),
                    "cytology": rec.get("cytology"),
                    "ancillary_studies": rec.get("ancillary_studies"),
                    "differential_diagnosis": rec.get("differential_diagnosis"),
                    "reporting_category": rec.get("reporting_category"),
                    "staging": rec.get("staging"),
                    "prognosis_and_prediction": rec.get("prognosis_and_prediction"),
                    "tags": tags,
                    "timestamp": rec.get("timestamp"),
                    "gcs_video_path": normalize_gcs_path(rec.get("gcs_video_path")) if rec.get("gcs_video_path") else None,
                    "media": rec.get("media"),
                    "html_gcs_path": normalize_gcs_path(rec.get("html_gcs_path")) if rec.get("html_gcs_path") else None,
                    "created_at_utc": created_at,
                }
                entities_out.append(ent)

                chunk_text = build_entity_chunk_text(ent)
                if chunk_text:
                    chunks_out.append({
                        "schema_version": "lecture_chunk.v0_proposed",
                        "lecture_chunk_id": make_chunk_id(entity_id, 0),
                        "lecture_source_id": lecture_source_id,
                        "lecture_family_id": lecture_family_id,
                        "parent_id": entity_id,
                        "parent_type": "entity",
                        "source_gcs_uri": source_gcs_uri,
                        "adapter": adapter,
                        "subspecialty_guess": subspecialty,
                        "title": entity_name,
                        "entity_name": entity_name,
                        "start_time_sec": parse_time_to_seconds(rec.get("timestamp")),
                        "end_time_sec": None,
                        "image_gcs_uri": None,
                        "tags": tags,
                        "text": chunk_text,
                        "created_at_utc": created_at,
                    })

                for tag_ix, tag in enumerate(tags):
                    tags_out.append({
                        "schema_version": "lecture_tag_catalog.v0_proposed",
                        "lecture_source_id": lecture_source_id,
                        "lecture_family_id": lecture_family_id,
                        "parent_id": entity_id,
                        "parent_type": "entity",
                        "tag": str(tag),
                        "tag_source": "existing",
                        "tag_confidence": None,
                        "diagnostic_truth": False,
                        "created_at_utc": created_at,
                    })

                # Related figures from entity master records
                for fig_ix, fig in enumerate(safe_list(rec.get("related_figures"))):
                    if not isinstance(fig, dict):
                        continue

                    original = get_first(fig, ["gcs_path", "src", "image_path", "path", "figure_gcs_path"])
                    norm = normalize_gcs_path(original)

                    if original:
                        image_stats["image_ref_total"] += 1
                        if str(original).startswith("gs://"):
                            image_stats["image_ref_original_gcs"] += 1
                        elif str(original).startswith("_asset_library/") or str(original).startswith("/_asset_library/"):
                            image_stats["image_ref_original_relative_asset"] += 1
                        if is_gcs_uri(norm):
                            image_stats["image_ref_normalized_gcs"] += 1
                        else:
                            image_stats["image_ref_unresolved_or_non_gcs"] += 1

                    fig_label = get_first(fig, ["id", "figure_id", "label"], fig_ix)
                    fig_id = make_figure_id(lecture_source_id, norm or original, fallback_id=fig_label)
                    fig_key = (fig_id, entity_id)

                    if fig_key not in seen_figure_rows:
                        seen_figure_rows.add(fig_key)
                        figures_out.append({
                            "schema_version": "lecture_figure.v0_proposed",
                            "lecture_figure_id": fig_id,
                            "lecture_source_id": lecture_source_id,
                            "lecture_family_id": lecture_family_id,
                            "parent_id": entity_id,
                            "parent_type": "entity",
                            "source_gcs_uri": source_gcs_uri,
                            "source_record_id": rec_ix,
                            "figure_label": fig_label,
                            "caption": get_first(fig, ["caption", "legend", "diagnosis"]),
                            "legend": fig.get("legend"),
                            "diagnosis": fig.get("diagnosis"),
                            "image_path_original": original,
                            "image_gcs_uri": norm if is_gcs_uri(norm) else None,
                            "image_path_unresolved": norm if norm and not is_gcs_uri(norm) else None,
                            "figure_url": None,
                            "image_url": None,
                            "public_url_verified": False,
                            "created_at_utc": created_at,
                        })

    except Exception as e:
        source_status_counts["error"] += 1
        unsupported.append({
            "source_gcs_uri": source_gcs_uri,
            "lecture_source_id": lecture_source_id,
            "error": repr(e),
        })
        sources_out.append({
            **source_base,
            "normalization_status": "error",
            "error": repr(e),
        })

# ----------------------------
# Write local JSONLs + audit
# ----------------------------

local_paths = {
    "sources": LOCAL_OUT_DIR / "lecture_sources.jsonl",
    "segments": LOCAL_OUT_DIR / "lecture_segments.jsonl",
    "entities": LOCAL_OUT_DIR / "lecture_entities.jsonl",
    "chunks": LOCAL_OUT_DIR / "lecture_chunks.jsonl",
    "figures": LOCAL_OUT_DIR / "lecture_figures.jsonl",
    "tags": LOCAL_OUT_DIR / "lecture_tag_catalog.jsonl",
    "audit": LOCAL_OUT_DIR / f"lecture_normalization_audit_{LECTURE_WORKSTREAM_DATE}.json",
}

write_jsonl_local(sources_out, local_paths["sources"])
write_jsonl_local(segments_out, local_paths["segments"])
write_jsonl_local(entities_out, local_paths["entities"])
write_jsonl_local(chunks_out, local_paths["chunks"])
write_jsonl_local(figures_out, local_paths["figures"])
write_jsonl_local(tags_out, local_paths["tags"])

audit = {
    "schema_version": "lecture_normalization_audit.v0_proposed",
    "created_at_utc": created_at,
    "inventory_source": INVENTORY_JSONL_GCS,
    "source_bucket": SOURCE_BUCKET,
    "dest_bucket": DEST_BUCKET,
    "normalization_only": True,
    "indexed": False,
    "vectorized": False,
    "api_exposed": False,
    "public_figures_enabled": LECTURE_PUBLIC_FIGURES_ENABLED,
    "input_file_count": len(inventory),
    "source_status_counts": dict(source_status_counts),
    "adapter_counts": dict(adapter_counts),
    "skipped_exact_duplicate_count": len(skipped_duplicates),
    "output_counts": {
        "lecture_sources": len(sources_out),
        "lecture_segments": len(segments_out),
        "lecture_entities": len(entities_out),
        "lecture_chunks": len(chunks_out),
        "lecture_figures": len(figures_out),
        "lecture_tag_catalog": len(tags_out),
    },
    "image_stats": dict(image_stats),
    "unsupported_count": len(unsupported),
    "unsupported_examples": unsupported[:50],
    "output_gcs_paths": {
        "lecture_sources": f"gs://{DEST_BUCKET}/{OUT_SOURCES}",
        "lecture_segments": f"gs://{DEST_BUCKET}/{OUT_SEGMENTS}",
        "lecture_entities": f"gs://{DEST_BUCKET}/{OUT_ENTITIES}",
        "lecture_chunks": f"gs://{DEST_BUCKET}/{OUT_CHUNKS}",
        "lecture_figures": f"gs://{DEST_BUCKET}/{OUT_FIGURES}",
        "lecture_tag_catalog": f"gs://{DEST_BUCKET}/{OUT_TAGS}",
        "lecture_normalization_audit": f"gs://{DEST_BUCKET}/{OUT_AUDIT}",
    },
    "known_limitations": [
        "Lecture normalized artifacts are proposed v0 schemas and should be registered before API exposure.",
        "No FTS index was built by this cell.",
        "No vector index was built by this cell.",
        "No lecture source was added to searchEvidence by this cell.",
        "Figure rows store source GCS paths only; browser-safe public URLs require a later derivative/proxy audit.",
        "Tags/labels are preserved as metadata only and are not diagnostic truth.",
    ],
}

with open(local_paths["audit"], "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

# ----------------------------
# Upload
# ----------------------------

uploaded = {}

if NORMALIZER_WRITE_OUTPUTS:
    uploaded["lecture_sources"] = upload_file(local_paths["sources"], OUT_SOURCES)
    uploaded["lecture_segments"] = upload_file(local_paths["segments"], OUT_SEGMENTS)
    uploaded["lecture_entities"] = upload_file(local_paths["entities"], OUT_ENTITIES)
    uploaded["lecture_chunks"] = upload_file(local_paths["chunks"], OUT_CHUNKS)
    uploaded["lecture_figures"] = upload_file(local_paths["figures"], OUT_FIGURES)
    uploaded["lecture_tag_catalog"] = upload_file(local_paths["tags"], OUT_TAGS)
    uploaded["lecture_normalization_audit"] = upload_file(local_paths["audit"], OUT_AUDIT)

print("=== LECTURE NORMALIZATION COMPLETE ===")
print(json.dumps({
    "created_at_utc": created_at,
    "input_file_count": len(inventory),
    "source_status_counts": dict(source_status_counts),
    "adapter_counts": dict(adapter_counts),
    "skipped_exact_duplicate_count": len(skipped_duplicates),
    "output_counts": audit["output_counts"],
    "image_stats": dict(image_stats),
    "unsupported_count": len(unsupported),
    "uploaded": uploaded,
    "flags": {
        "normalization_only": True,
        "indexed": False,
        "vectorized": False,
        "api_exposed": False,
        "public_figures_enabled": False,
    }
}, indent=2))

if unsupported:
    print("\n=== UNSUPPORTED EXAMPLES FIRST 10 ===")
    for u in unsupported[:10]:
        print("\n---")
        print(u.get("source_gcs_uri"))
        print("keys:", u.get("key_union_first_25"))
        if u.get("error"):
            print("error:", u.get("error"))

In [ ]:
# ============================================================
# Pathology Hub — Lecture Normalization QA / Audit v0
# ============================================================
# Purpose:
# - Validate normalized lecture artifacts.
# - Identify empty/unusable source(s), duplicate IDs, short/empty chunks,
#   unresolved image paths, and basic corpus distribution.
# - Upload QA audit.
#
# This does NOT build FTS, vectors, public figures, or API exposure.
# ============================================================

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
from pathlib import Path
from datetime import datetime, timezone
import json, collections, statistics, re, math

PROJECT_ID = "pathology-annotation-project"
DEST_BUCKET = "pathology_hub"

NORMALIZED_PREFIX = "02_normalized/lectures/"
AUDIT_PREFIX = "06_audits/lectures/"

PATHS = {
    "sources": f"gs://{DEST_BUCKET}/{NORMALIZED_PREFIX}lecture_sources.jsonl",
    "segments": f"gs://{DEST_BUCKET}/{NORMALIZED_PREFIX}lecture_segments.jsonl",
    "entities": f"gs://{DEST_BUCKET}/{NORMALIZED_PREFIX}lecture_entities.jsonl",
    "chunks": f"gs://{DEST_BUCKET}/{NORMALIZED_PREFIX}lecture_chunks.jsonl",
    "figures": f"gs://{DEST_BUCKET}/{NORMALIZED_PREFIX}lecture_figures.jsonl",
    "tags": f"gs://{DEST_BUCKET}/{NORMALIZED_PREFIX}lecture_tag_catalog.jsonl",
}

OUT_LOCAL = "/content/lecture_normalization_qc_20260623.json"
OUT_GCS = f"{AUDIT_PREFIX}lecture_normalization_qc_20260623.json"

client = storage.Client(project=PROJECT_ID)

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def parse_gcs_uri(uri):
    assert uri.startswith("gs://"), uri
    rest = uri[5:]
    bucket, name = rest.split("/", 1)
    return bucket, name

def iter_jsonl_gcs(uri):
    bucket_name, blob_name = parse_gcs_uri(uri)
    blob = client.bucket(bucket_name).blob(blob_name)
    with blob.open("rt", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                try:
                    yield json.loads(line)
                except Exception as e:
                    yield {
                        "_json_parse_error": repr(e),
                        "_line_no": line_no,
                        "_raw_preview": line[:300],
                    }

def short_counter(counter, n=25):
    return dict(counter.most_common(n))

def duplicate_id_summary(rows_iter, id_field, sample_n=20):
    seen = set()
    dupes = []
    count = 0
    for r in rows_iter:
        val = r.get(id_field)
        if val is None:
            continue
        count += 1
        if val in seen and len(dupes) < sample_n:
            dupes.append(val)
        seen.add(val)
    return {
        "id_field": id_field,
        "rows_with_id": count,
        "unique_ids": len(seen),
        "duplicate_count_estimate": count - len(seen),
        "duplicate_examples": dupes,
    }

created_at = utc_now_iso()

# ----------------------------
# Sources QA
# ----------------------------

source_counts = collections.Counter()
source_adapter_counts = collections.Counter()
source_subspecialty_counts = collections.Counter()
empty_or_unusable_sources = []
skipped_duplicate_sources = []
source_errors = []
all_sources = []

for r in iter_jsonl_gcs(PATHS["sources"]):
    all_sources.append(r)
    source_counts[r.get("normalization_status", "missing")] += 1
    source_adapter_counts[r.get("adapter", "missing")] += 1
    source_subspecialty_counts[r.get("source_subspecialty_guess", "missing")] += 1

    if r.get("adapter") == "empty_or_unusable" or r.get("records_after_flatten") == 0:
        empty_or_unusable_sources.append(r)

    if r.get("normalization_status") == "skipped_exact_duplicate":
        skipped_duplicate_sources.append(r)

    if r.get("normalization_status") == "error":
        source_errors.append(r)

# ----------------------------
# Segments QA
# ----------------------------

segment_adapter_counts = collections.Counter()
segment_subspecialty_counts = collections.Counter()
segments_missing_text = []
segments_missing_time = 0
segments_missing_image_any = 0
segments_unresolved_image = []
segments_with_gcs_image = 0
segment_count = 0

for r in iter_jsonl_gcs(PATHS["segments"]):
    segment_count += 1
    segment_adapter_counts[r.get("adapter", "missing")] += 1
    segment_subspecialty_counts[r.get("subspecialty_guess", "missing")] += 1

    txt_fields = [
        r.get("title"),
        r.get("summary"),
        r.get("visual_desc"),
        r.get("key_points"),
        r.get("cleaned_transcript"),
        r.get("raw_transcript"),
    ]
    if not any(str(x).strip() for x in txt_fields if x is not None):
        if len(segments_missing_text) < 50:
            segments_missing_text.append(r)

    if r.get("start_time_sec") is None and r.get("end_time_sec") is None:
        segments_missing_time += 1

    if not r.get("image_gcs_uri") and not r.get("image_path_unresolved"):
        segments_missing_image_any += 1

    if r.get("image_gcs_uri"):
        segments_with_gcs_image += 1

    if r.get("image_path_unresolved"):
        if len(segments_unresolved_image) < 100:
            segments_unresolved_image.append({
                "lecture_segment_id": r.get("lecture_segment_id"),
                "source_gcs_uri": r.get("source_gcs_uri"),
                "image_path_original": r.get("image_path_original"),
                "image_path_unresolved": r.get("image_path_unresolved"),
                "title": r.get("title"),
            })

# ----------------------------
# Entities QA
# ----------------------------

entity_count = 0
entity_subspecialty_counts = collections.Counter()
entities_missing_name = []
entities_missing_definition = []
entity_tagged_count = 0

for r in iter_jsonl_gcs(PATHS["entities"]):
    entity_count += 1
    entity_subspecialty_counts[r.get("subspecialty_guess", "missing")] += 1

    if r.get("tags"):
        entity_tagged_count += 1

    if not str(r.get("entity_name") or "").strip() and len(entities_missing_name) < 50:
        entities_missing_name.append(r)

    if not str(r.get("definition") or "").strip() and len(entities_missing_definition) < 50:
        entities_missing_definition.append({
            "lecture_entity_id": r.get("lecture_entity_id"),
            "source_gcs_uri": r.get("source_gcs_uri"),
            "entity_name": r.get("entity_name"),
        })

# ----------------------------
# Chunks QA
# ----------------------------

chunk_count = 0
chunk_parent_counts = collections.Counter()
chunk_adapter_counts = collections.Counter()
chunk_subspecialty_counts = collections.Counter()
chunk_lengths = []
empty_or_short_chunks = []
chunk_parse_errors = 0

for r in iter_jsonl_gcs(PATHS["chunks"]):
    if r.get("_json_parse_error"):
        chunk_parse_errors += 1
        continue

    chunk_count += 1
    chunk_parent_counts[r.get("parent_type", "missing")] += 1
    chunk_adapter_counts[r.get("adapter", "missing")] += 1
    chunk_subspecialty_counts[r.get("subspecialty_guess", "missing")] += 1

    text = r.get("text") or ""
    L = len(text)
    chunk_lengths.append(L)

    if L < 80 and len(empty_or_short_chunks) < 100:
        empty_or_short_chunks.append({
            "lecture_chunk_id": r.get("lecture_chunk_id"),
            "parent_type": r.get("parent_type"),
            "adapter": r.get("adapter"),
            "subspecialty_guess": r.get("subspecialty_guess"),
            "title": r.get("title"),
            "text_len": L,
            "text_preview": text[:250],
            "source_gcs_uri": r.get("source_gcs_uri"),
        })

def length_stats(vals):
    if not vals:
        return {}
    vals_sorted = sorted(vals)
    def pct(p):
        ix = int(round((len(vals_sorted) - 1) * p))
        return vals_sorted[ix]
    return {
        "min": min(vals),
        "p01": pct(0.01),
        "p05": pct(0.05),
        "median": statistics.median(vals),
        "p95": pct(0.95),
        "p99": pct(0.99),
        "max": max(vals),
        "mean": round(statistics.mean(vals), 1),
    }

# ----------------------------
# Figures QA
# ----------------------------

figure_count = 0
figure_parent_counts = collections.Counter()
figures_with_gcs = 0
figures_with_unresolved = 0
figures_with_public_url = 0
unresolved_figure_examples = []
figure_source_counts = collections.Counter()

for r in iter_jsonl_gcs(PATHS["figures"]):
    figure_count += 1
    figure_parent_counts[r.get("parent_type", "missing")] += 1
    figure_source_counts[r.get("lecture_family_id", "missing")] += 1

    if r.get("image_gcs_uri"):
        figures_with_gcs += 1

    if r.get("image_path_unresolved"):
        figures_with_unresolved += 1
        if len(unresolved_figure_examples) < 100:
            unresolved_figure_examples.append({
                "lecture_figure_id": r.get("lecture_figure_id"),
                "parent_id": r.get("parent_id"),
                "source_gcs_uri": r.get("source_gcs_uri"),
                "image_path_original": r.get("image_path_original"),
                "image_path_unresolved": r.get("image_path_unresolved"),
                "caption": r.get("caption"),
            })

    if r.get("figure_url") or r.get("image_url") or r.get("public_url_verified"):
        figures_with_public_url += 1

# ----------------------------
# Tags QA
# ----------------------------

tag_count = 0
tag_source_counts = collections.Counter()
top_tags = collections.Counter()
tag_parent_counts = collections.Counter()

for r in iter_jsonl_gcs(PATHS["tags"]):
    tag_count += 1
    tag_source_counts[r.get("tag_source", "missing")] += 1
    tag_parent_counts[r.get("parent_type", "missing")] += 1
    tag = r.get("tag")
    if tag:
        top_tags[tag] += 1

# ----------------------------
# Duplicate IDs
# ----------------------------

dupe_summaries = {
    "sources": duplicate_id_summary(iter_jsonl_gcs(PATHS["sources"]), "lecture_source_id"),
    "segments": duplicate_id_summary(iter_jsonl_gcs(PATHS["segments"]), "lecture_segment_id"),
    "entities": duplicate_id_summary(iter_jsonl_gcs(PATHS["entities"]), "lecture_entity_id"),
    "chunks": duplicate_id_summary(iter_jsonl_gcs(PATHS["chunks"]), "lecture_chunk_id"),
    "figures": duplicate_id_summary(iter_jsonl_gcs(PATHS["figures"]), "lecture_figure_id"),
}

# ----------------------------
# Build QA audit
# ----------------------------

qc = {
    "schema_version": "lecture_normalization_qc.v0_proposed",
    "created_at_utc": created_at,
    "normalization_only": True,
    "indexed": False,
    "vectorized": False,
    "api_exposed": False,
    "input_paths": PATHS,

    "source_summary": {
        "source_count": len(all_sources),
        "normalization_status_counts": short_counter(source_counts),
        "adapter_counts": short_counter(source_adapter_counts),
        "subspecialty_counts": short_counter(source_subspecialty_counts),
        "empty_or_unusable_count": len(empty_or_unusable_sources),
        "empty_or_unusable_sources": empty_or_unusable_sources[:25],
        "skipped_duplicate_count": len(skipped_duplicate_sources),
        "skipped_duplicate_sources": skipped_duplicate_sources[:25],
        "source_error_count": len(source_errors),
        "source_errors": source_errors[:25],
    },

    "segment_summary": {
        "segment_count": segment_count,
        "adapter_counts": short_counter(segment_adapter_counts),
        "subspecialty_counts": short_counter(segment_subspecialty_counts),
        "segments_missing_text_count": len(segments_missing_text),
        "segments_missing_text_examples": segments_missing_text[:25],
        "segments_missing_time_count": segments_missing_time,
        "segments_missing_image_any_count": segments_missing_image_any,
        "segments_with_gcs_image_count": segments_with_gcs_image,
        "segments_unresolved_image_example_count_capped": len(segments_unresolved_image),
        "segments_unresolved_image_examples": segments_unresolved_image[:50],
    },

    "entity_summary": {
        "entity_count": entity_count,
        "subspecialty_counts": short_counter(entity_subspecialty_counts),
        "entity_tagged_count": entity_tagged_count,
        "entities_missing_name_count": len(entities_missing_name),
        "entities_missing_name_examples": entities_missing_name[:25],
        "entities_missing_definition_count": len(entities_missing_definition),
        "entities_missing_definition_examples": entities_missing_definition[:25],
    },

    "chunk_summary": {
        "chunk_count": chunk_count,
        "json_parse_error_count": chunk_parse_errors,
        "parent_type_counts": short_counter(chunk_parent_counts),
        "adapter_counts": short_counter(chunk_adapter_counts),
        "subspecialty_counts": short_counter(chunk_subspecialty_counts),
        "text_length_stats": length_stats(chunk_lengths),
        "empty_or_short_chunk_count_capped": len(empty_or_short_chunks),
        "empty_or_short_chunk_examples": empty_or_short_chunks[:50],
    },

    "figure_summary": {
        "figure_count": figure_count,
        "parent_type_counts": short_counter(figure_parent_counts),
        "figures_with_gcs_count": figures_with_gcs,
        "figures_with_unresolved_path_count": figures_with_unresolved,
        "figures_with_public_url_count": figures_with_public_url,
        "top_lecture_families_by_figure_count": short_counter(figure_source_counts, 30),
        "unresolved_figure_examples": unresolved_figure_examples[:50],
    },

    "tag_summary": {
        "tag_count": tag_count,
        "tag_source_counts": short_counter(tag_source_counts),
        "tag_parent_type_counts": short_counter(tag_parent_counts),
        "top_tags": short_counter(top_tags, 50),
    },

    "duplicate_id_summaries": dupe_summaries,

    "recommended_next_step": (
        "If duplicate chunk IDs are 0 and empty/short chunks are acceptable, build SQLite FTS next. "
        "Do not build public figure serving until unresolved image paths are categorized."
    )
}

with open(OUT_LOCAL, "w", encoding="utf-8") as f:
    json.dump(qc, f, indent=2, ensure_ascii=False)

bucket_name, blob_name = DEST_BUCKET, OUT_GCS
client.bucket(bucket_name).blob(blob_name).upload_from_filename(OUT_LOCAL)

print("=== LECTURE NORMALIZATION QC COMPLETE ===")
print(json.dumps({
    "created_at_utc": created_at,
    "source_summary": {
        "source_count": qc["source_summary"]["source_count"],
        "normalization_status_counts": qc["source_summary"]["normalization_status_counts"],
        "adapter_counts": qc["source_summary"]["adapter_counts"],
        "empty_or_unusable_count": qc["source_summary"]["empty_or_unusable_count"],
        "skipped_duplicate_count": qc["source_summary"]["skipped_duplicate_count"],
        "source_error_count": qc["source_summary"]["source_error_count"],
    },
    "segment_summary": {
        "segment_count": segment_count,
        "segments_missing_text_count": len(segments_missing_text),
        "segments_missing_time_count": segments_missing_time,
        "segments_missing_image_any_count": segments_missing_image_any,
        "segments_with_gcs_image_count": segments_with_gcs_image,
        "segments_unresolved_image_example_count_capped": len(segments_unresolved_image),
    },
    "entity_summary": {
        "entity_count": entity_count,
        "entity_tagged_count": entity_tagged_count,
        "entities_missing_name_count": len(entities_missing_name),
        "entities_missing_definition_count": len(entities_missing_definition),
    },
    "chunk_summary": {
        "chunk_count": chunk_count,
        "parent_type_counts": qc["chunk_summary"]["parent_type_counts"],
        "text_length_stats": qc["chunk_summary"]["text_length_stats"],
        "empty_or_short_chunk_count_capped": len(empty_or_short_chunks),
    },
    "figure_summary": {
        "figure_count": figure_count,
        "figures_with_gcs_count": figures_with_gcs,
        "figures_with_unresolved_path_count": figures_with_unresolved,
        "figures_with_public_url_count": figures_with_public_url,
    },
    "tag_summary": {
        "tag_count": tag_count,
        "tag_source_counts": qc["tag_summary"]["tag_source_counts"],
    },
    "duplicate_id_summaries": dupe_summaries,
    "uploaded_qc": f"gs://{DEST_BUCKET}/{OUT_GCS}",
    "flags": {
        "normalization_only": True,
        "indexed": False,
        "vectorized": False,
        "api_exposed": False,
    }
}, indent=2))

print("\nEmpty/unusable source examples:")
for r in empty_or_unusable_sources[:10]:
    print("-", r.get("source_gcs_uri"), "| records_after_flatten:", r.get("records_after_flatten"))

print("\nUnresolved figure path examples:")
for r in unresolved_figure_examples[:10]:
    print("-", r.get("image_path_unresolved"), "| source:", r.get("source_gcs_uri"))

In [ ]:
# ============================================================
# Pathology Hub — Purge Generic Duplicate Lecture JSON Artifacts
# ============================================================
# Purpose:
# - Remove junk duplicate sources like final_ENHANCED_data.json from
#   normalized lecture artifacts.
# - Also remove final_STRUCTURED_data.json by default because QC showed
#   the same duplicate-ID pattern.
# - Back up current normalized artifacts before overwrite.
#
# This does NOT build FTS/vector/API exposure.
# Source GCS deletion is optional and OFF by default.
# ============================================================

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
from pathlib import Path
from datetime import datetime, timezone
import json, re, collections

PROJECT_ID = "pathology-annotation-project"
SOURCE_BUCKET = "pathology-hub-0"
DEST_BUCKET = "pathology_hub"

NORMALIZED_PREFIX = "02_normalized/lectures/"
AUDIT_PREFIX = "06_audits/lectures/"

PATHS = {
    "sources": NORMALIZED_PREFIX + "lecture_sources.jsonl",
    "segments": NORMALIZED_PREFIX + "lecture_segments.jsonl",
    "entities": NORMALIZED_PREFIX + "lecture_entities.jsonl",
    "chunks": NORMALIZED_PREFIX + "lecture_chunks.jsonl",
    "figures": NORMALIZED_PREFIX + "lecture_figures.jsonl",
    "tags": NORMALIZED_PREFIX + "lecture_tag_catalog.jsonl",
}

# Main purge rule
INCLUDE_FINAL_STRUCTURED_TOO = True

BAD_STEMS = {
    "final_enhanced",
    "final_enhanced_data",
}

if INCLUDE_FINAL_STRUCTURED_TOO:
    BAD_STEMS |= {
        "final_structured",
        "final_structured_data",
    }

# This removes them from normalized artifacts.
FILTER_NORMALIZED_ARTIFACTS = True

# This permanently deletes original source JSONs from pathology-hub-0 after archiving.
# Leave False unless you really want source GCS objects removed.
ARCHIVE_AND_DELETE_SOURCE_JSONS = False

ARCHIVE_SOURCE_DELETE_PREFIX = "01_staged/lectures/excluded_source_jsons/final_enhanced_structured_purge_20260623/"
BACKUP_PREFIX = "06_audits/lectures/backups_before_final_enhanced_structured_purge_20260623/"

OUT_AUDIT = AUDIT_PREFIX + "lecture_final_enhanced_structured_purge_audit_20260623.json"

client = storage.Client(project=PROJECT_ID)

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def parse_gcs_uri(uri):
    assert uri.startswith("gs://"), uri
    rest = uri[5:]
    bucket, name = rest.split("/", 1)
    return bucket, name

def iter_jsonl_blob(blob_name):
    blob = client.bucket(DEST_BUCKET).blob(blob_name)
    with blob.open("rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def write_jsonl(rows, local_path):
    with open(local_path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def upload(local_path, blob_name):
    blob = client.bucket(DEST_BUCKET).blob(blob_name)
    blob.upload_from_filename(str(local_path))
    return f"gs://{DEST_BUCKET}/{blob_name}"

def source_filename(row):
    if row.get("source_filename"):
        return row["source_filename"]
    if row.get("source_blob_name"):
        return Path(row["source_blob_name"]).name
    if row.get("source_gcs_uri"):
        return Path(parse_gcs_uri(row["source_gcs_uri"])[1]).name
    return ""

def source_stem(row):
    return Path(source_filename(row)).stem.lower()

def is_bad_source_row(row):
    return source_stem(row) in BAD_STEMS

def row_source_uri(row):
    return row.get("source_gcs_uri")

def duplicate_summary(rows, field):
    vals = [r.get(field) for r in rows if r.get(field)]
    c = collections.Counter(vals)
    dupes = [k for k, v in c.items() if v > 1]
    return {
        "field": field,
        "rows_with_id": len(vals),
        "unique_ids": len(c),
        "duplicate_count": len(vals) - len(c),
        "duplicate_examples": dupes[:20],
    }

created_at = utc_now_iso()

# ----------------------------
# Load current normalized artifacts
# ----------------------------

data = {k: list(iter_jsonl_blob(v)) for k, v in PATHS.items()}

print("Loaded current normalized artifacts:")
print(json.dumps({k: len(v) for k, v in data.items()}, indent=2))

# ----------------------------
# Identify bad sources
# ----------------------------

bad_source_rows = [r for r in data["sources"] if is_bad_source_row(r)]
bad_source_uris = {r.get("source_gcs_uri") for r in bad_source_rows if r.get("source_gcs_uri")}
bad_source_ids = {r.get("lecture_source_id") for r in bad_source_rows if r.get("lecture_source_id")}
bad_family_ids = {r.get("lecture_family_id") for r in bad_source_rows if r.get("lecture_family_id")}

print("\nBad/generic source rows identified:")
print(json.dumps({
    "bad_source_count": len(bad_source_rows),
    "bad_stems": sorted(BAD_STEMS),
    "example_bad_sources": [r.get("source_gcs_uri") for r in bad_source_rows[:20]],
}, indent=2))

# ----------------------------
# Back up current normalized artifacts before overwrite
# ----------------------------

bucket = client.bucket(DEST_BUCKET)
backup_uploaded = {}

for key, blob_name in PATHS.items():
    src_blob = bucket.blob(blob_name)
    backup_blob_name = BACKUP_PREFIX + Path(blob_name).name
    bucket.copy_blob(src_blob, bucket, backup_blob_name)
    backup_uploaded[key] = f"gs://{DEST_BUCKET}/{backup_blob_name}"

# ----------------------------
# Filter rows
# ----------------------------

def keep_by_source(row):
    uri = row.get("source_gcs_uri")
    sid = row.get("lecture_source_id")
    fam = row.get("lecture_family_id")

    if uri and uri in bad_source_uris:
        return False
    if sid and sid in bad_source_ids:
        return False

    # Family-level removal is intentionally NOT used by default because
    # family_id may point to the real renamed lecture too.
    return True

filtered = {}

filtered["sources"] = [r for r in data["sources"] if not is_bad_source_row(r)]
filtered["segments"] = [r for r in data["segments"] if keep_by_source(r)]
filtered["entities"] = [r for r in data["entities"] if keep_by_source(r)]
filtered["chunks"] = [r for r in data["chunks"] if keep_by_source(r)]
filtered["figures"] = [r for r in data["figures"] if keep_by_source(r)]

# Tags may not always have source_gcs_uri, so use source id too.
filtered["tags"] = [r for r in data["tags"] if keep_by_source(r)]

removed_counts = {
    k: len(data[k]) - len(filtered[k])
    for k in data.keys()
}

# ----------------------------
# Duplicate summaries after purge
# ----------------------------

dupes_after = {
    "sources": duplicate_summary(filtered["sources"], "lecture_source_id"),
    "segments": duplicate_summary(filtered["segments"], "lecture_segment_id"),
    "entities": duplicate_summary(filtered["entities"], "lecture_entity_id"),
    "chunks": duplicate_summary(filtered["chunks"], "lecture_chunk_id"),
    "figures": duplicate_summary(filtered["figures"], "lecture_figure_id"),
}

# ----------------------------
# Write + upload filtered normalized artifacts
# ----------------------------

local_dir = Path("/content/lecture_purged_final_enhanced_structured")
local_dir.mkdir(parents=True, exist_ok=True)

uploaded_filtered = {}

if FILTER_NORMALIZED_ARTIFACTS:
    for key, rows in filtered.items():
        local_path = local_dir / Path(PATHS[key]).name
        write_jsonl(rows, local_path)
        uploaded_filtered[key] = upload(local_path, PATHS[key])

# ----------------------------
# Optional source GCS archive + delete
# ----------------------------

archived_and_deleted = []
archive_errors = []

if ARCHIVE_AND_DELETE_SOURCE_JSONS:
    for uri in sorted(bad_source_uris):
        try:
            src_bucket_name, src_blob_name = parse_gcs_uri(uri)
            src_bucket = client.bucket(src_bucket_name)
            src_blob = src_bucket.blob(src_blob_name)

            archive_blob_name = ARCHIVE_SOURCE_DELETE_PREFIX + src_blob_name
            client.bucket(src_bucket_name).copy_blob(src_blob, client.bucket(DEST_BUCKET), archive_blob_name)

            src_blob.delete()

            archived_and_deleted.append({
                "source_gcs_uri_deleted": uri,
                "archived_to": f"gs://{DEST_BUCKET}/{archive_blob_name}",
            })
        except Exception as e:
            archive_errors.append({
                "source_gcs_uri": uri,
                "error": repr(e),
            })

# ----------------------------
# Audit
# ----------------------------

audit = {
    "schema_version": "lecture_final_enhanced_structured_purge_audit.v0",
    "created_at_utc": created_at,
    "normalization_only": True,
    "indexed": False,
    "vectorized": False,
    "api_exposed": False,
    "public_figures_enabled": False,
    "bad_stems": sorted(BAD_STEMS),
    "include_final_structured_too": INCLUDE_FINAL_STRUCTURED_TOO,
    "bad_source_count": len(bad_source_rows),
    "bad_source_uris": sorted(bad_source_uris),
    "bad_source_ids": sorted(bad_source_ids),
    "counts_before": {k: len(v) for k, v in data.items()},
    "counts_after": {k: len(v) for k, v in filtered.items()},
    "removed_counts": removed_counts,
    "duplicate_id_summaries_after_purge": dupes_after,
    "backups": backup_uploaded,
    "uploaded_filtered": uploaded_filtered,
    "archive_and_delete_source_jsons": ARCHIVE_AND_DELETE_SOURCE_JSONS,
    "archived_and_deleted_count": len(archived_and_deleted),
    "archived_and_deleted": archived_and_deleted[:50],
    "archive_errors": archive_errors[:50],
    "known_limitations": [
        "This purges generic duplicate final_ENHANCED/final_STRUCTURED artifacts from normalized lecture outputs.",
        "It does not rebuild FTS or vector indexes.",
        "It does not expose lectures through searchEvidence.",
        "Original source JSONs are only deleted if ARCHIVE_AND_DELETE_SOURCE_JSONS=True.",
    ],
}

audit_local = local_dir / "lecture_final_enhanced_structured_purge_audit_20260623.json"
with open(audit_local, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

uploaded_audit = upload(audit_local, OUT_AUDIT)

print("\n=== LECTURE FINAL_ENHANCED / FINAL_STRUCTURED PURGE COMPLETE ===")
print(json.dumps({
    "created_at_utc": created_at,
    "bad_source_count": len(bad_source_rows),
    "bad_stems": sorted(BAD_STEMS),
    "counts_before": audit["counts_before"],
    "counts_after": audit["counts_after"],
    "removed_counts": removed_counts,
    "duplicate_id_summaries_after_purge": dupes_after,
    "backups": backup_uploaded,
    "uploaded_filtered": uploaded_filtered,
    "uploaded_audit": uploaded_audit,
    "archive_and_delete_source_jsons": ARCHIVE_AND_DELETE_SOURCE_JSONS,
    "archived_and_deleted_count": len(archived_and_deleted),
    "archive_errors_count": len(archive_errors),
    "flags": {
        "normalization_only": True,
        "indexed": False,
        "vectorized": False,
        "api_exposed": False,
        "public_figures_enabled": False,
    }
}, indent=2))

In [ ]:
# ============================================================
# Pathology Hub — Lecture Post-Purge QC Cell
# ============================================================
# Run after purging final_ENHANCED / final_STRUCTURED duplicate junk.
# Checks normalized artifact counts, duplicate IDs, unresolved figures.
# ============================================================

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
from datetime import datetime, timezone
import json, collections

PROJECT_ID = "pathology-annotation-project"
DEST_BUCKET = "pathology_hub"
NORMALIZED_PREFIX = "02_normalized/lectures/"
AUDIT_PREFIX = "06_audits/lectures/"

PATHS = {
    "sources": NORMALIZED_PREFIX + "lecture_sources.jsonl",
    "segments": NORMALIZED_PREFIX + "lecture_segments.jsonl",
    "entities": NORMALIZED_PREFIX + "lecture_entities.jsonl",
    "chunks": NORMALIZED_PREFIX + "lecture_chunks.jsonl",
    "figures": NORMALIZED_PREFIX + "lecture_figures.jsonl",
    "tags": NORMALIZED_PREFIX + "lecture_tag_catalog.jsonl",
}

OUT_AUDIT = AUDIT_PREFIX + "lecture_post_purge_qc_20260623.json"

client = storage.Client(project=PROJECT_ID)

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def iter_jsonl(blob_name):
    blob = client.bucket(DEST_BUCKET).blob(blob_name)
    with blob.open("rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

def dup_summary(rows, field):
    vals = [r.get(field) for r in rows if r.get(field)]
    c = collections.Counter(vals)
    dupes = [k for k, v in c.items() if v > 1]
    return {
        "field": field,
        "rows_with_id": len(vals),
        "unique_ids": len(c),
        "duplicate_count": len(vals) - len(c),
        "duplicate_examples": dupes[:20],
    }

def count_by(rows, field):
    return dict(collections.Counter(r.get(field, "missing") for r in rows).most_common())

def source_stem(row):
    import pathlib
    fn = row.get("source_filename") or row.get("source_blob_name") or row.get("source_gcs_uri") or ""
    return pathlib.Path(fn).stem.lower()

rows = {k: list(iter_jsonl(v)) for k, v in PATHS.items()}

sources = rows["sources"]
segments = rows["segments"]
entities = rows["entities"]
chunks = rows["chunks"]
figures = rows["figures"]
tags = rows["tags"]

bad_stems_remaining = [
    r.get("source_gcs_uri")
    for r in sources
    if source_stem(r) in {
        "final_enhanced",
        "final_enhanced_data",
        "final_structured",
        "final_structured_data",
    }
]

chunk_text_lengths = [len(r.get("text") or "") for r in chunks]
short_chunks = [
    {
        "lecture_chunk_id": r.get("lecture_chunk_id"),
        "title": r.get("title"),
        "text_len": len(r.get("text") or ""),
        "source_gcs_uri": r.get("source_gcs_uri"),
        "text_preview": (r.get("text") or "")[:160],
    }
    for r in chunks
    if len(r.get("text") or "") < 80
][:50]

unresolved_figures = [
    {
        "lecture_figure_id": r.get("lecture_figure_id"),
        "image_path_unresolved": r.get("image_path_unresolved"),
        "source_gcs_uri": r.get("source_gcs_uri"),
        "caption": r.get("caption"),
    }
    for r in figures
    if r.get("image_path_unresolved")
][:50]

qc = {
    "schema_version": "lecture_post_purge_qc.v0",
    "created_at_utc": utc_now_iso(),
    "normalization_only": True,
    "indexed": False,
    "vectorized": False,
    "api_exposed": False,
    "counts": {k: len(v) for k, v in rows.items()},
    "source_status_counts": count_by(sources, "normalization_status"),
    "source_adapter_counts": count_by(sources, "adapter"),
    "source_subspecialty_counts": count_by(sources, "source_subspecialty_guess"),
    "bad_generic_sources_remaining_count": len(bad_stems_remaining),
    "bad_generic_sources_remaining_examples": bad_stems_remaining[:50],
    "duplicate_id_summaries": {
        "sources": dup_summary(sources, "lecture_source_id"),
        "segments": dup_summary(segments, "lecture_segment_id"),
        "entities": dup_summary(entities, "lecture_entity_id"),
        "chunks": dup_summary(chunks, "lecture_chunk_id"),
        "figures": dup_summary(figures, "lecture_figure_id"),
    },
    "figure_summary": {
        "figure_count": len(figures),
        "figures_with_gcs_count": sum(1 for r in figures if r.get("image_gcs_uri")),
        "figures_with_unresolved_path_count": sum(1 for r in figures if r.get("image_path_unresolved")),
        "figures_with_public_url_count": sum(1 for r in figures if r.get("figure_url") or r.get("image_url")),
        "unresolved_figure_examples": unresolved_figures,
    },
    "chunk_summary": {
        "chunk_count": len(chunks),
        "short_chunk_count_under_80": sum(1 for r in chunks if len(r.get("text") or "") < 80),
        "short_chunk_examples": short_chunks,
        "text_length_min": min(chunk_text_lengths) if chunk_text_lengths else None,
        "text_length_max": max(chunk_text_lengths) if chunk_text_lengths else None,
    },
}

local_path = "/content/lecture_post_purge_qc_20260623.json"
with open(local_path, "w", encoding="utf-8") as f:
    json.dump(qc, f, indent=2, ensure_ascii=False)

client.bucket(DEST_BUCKET).blob(OUT_AUDIT).upload_from_filename(local_path)

print("=== LECTURE POST-PURGE QC COMPLETE ===")
print(json.dumps({
    "created_at_utc": qc["created_at_utc"],
    "counts": qc["counts"],
    "source_status_counts": qc["source_status_counts"],
    "source_adapter_counts": qc["source_adapter_counts"],
    "bad_generic_sources_remaining_count": qc["bad_generic_sources_remaining_count"],
    "duplicate_id_summaries": qc["duplicate_id_summaries"],
    "figure_summary": {
        "figure_count": qc["figure_summary"]["figure_count"],
        "figures_with_gcs_count": qc["figure_summary"]["figures_with_gcs_count"],
        "figures_with_unresolved_path_count": qc["figure_summary"]["figures_with_unresolved_path_count"],
        "figures_with_public_url_count": qc["figure_summary"]["figures_with_public_url_count"],
    },
    "chunk_summary": {
        "chunk_count": qc["chunk_summary"]["chunk_count"],
        "short_chunk_count_under_80": qc["chunk_summary"]["short_chunk_count_under_80"],
        "text_length_min": qc["chunk_summary"]["text_length_min"],
        "text_length_max": qc["chunk_summary"]["text_length_max"],
    },
    "uploaded_qc": f"gs://{DEST_BUCKET}/{OUT_AUDIT}",
    "flags": {
        "normalization_only": True,
        "indexed": False,
        "vectorized": False,
        "api_exposed": False,
    }
}, indent=2))

In [ ]:
# ============================================================
# Pathology Hub — Lecture SQLite FTS Index Builder v0
# ============================================================
# Builds SQLite FTS5 index over normalized lecture_chunks.jsonl.
#
# Inputs:
#   gs://pathology_hub/02_normalized/lectures/lecture_chunks.jsonl
#
# Outputs:
#   gs://pathology_hub/03_indexes/lectures/fts/lecture_fts.sqlite
#   gs://pathology_hub/03_indexes/lectures/fts/lecture_fts_index_manifest.json
#   gs://pathology_hub/06_audits/lectures/lecture_fts_build_audit_20260623.json
#
# Important:
# - This creates a keyword-searchable FTS artifact.
# - This does NOT vectorize lectures.
# - This does NOT expose lectures through searchEvidence/API.
# ============================================================

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
from datetime import datetime, timezone
from pathlib import Path
import sqlite3, json, os, re, hashlib, collections, time

PROJECT_ID = "pathology-annotation-project"
DEST_BUCKET = "pathology_hub"

LECTURE_CHUNKS_GCS = "gs://pathology_hub/02_normalized/lectures/lecture_chunks.jsonl"

OUT_DB_GCS = "03_indexes/lectures/fts/lecture_fts.sqlite"
OUT_MANIFEST_GCS = "03_indexes/lectures/fts/lecture_fts_index_manifest.json"
OUT_AUDIT_GCS = "06_audits/lectures/lecture_fts_build_audit_20260623.json"

LOCAL_DB = "/content/lecture_fts.sqlite"
LOCAL_MANIFEST = "/content/lecture_fts_index_manifest.json"
LOCAL_AUDIT = "/content/lecture_fts_build_audit_20260623.json"

OVERWRITE = True

client = storage.Client(project=PROJECT_ID)

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def parse_gcs_uri(uri):
    assert uri.startswith("gs://"), uri
    rest = uri[5:]
    bucket, name = rest.split("/", 1)
    return bucket, name

def iter_jsonl_gcs(uri):
    bucket_name, blob_name = parse_gcs_uri(uri)
    blob = client.bucket(bucket_name).blob(blob_name)
    with blob.open("rt", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                try:
                    yield line_no, json.loads(line)
                except Exception as e:
                    yield line_no, {"_json_parse_error": repr(e), "_raw_preview": line[:300]}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def upload_file(local_path, gcs_blob_name):
    blob = client.bucket(DEST_BUCKET).blob(gcs_blob_name)
    if blob.exists() and not OVERWRITE:
        raise FileExistsError(f"Refusing to overwrite gs://{DEST_BUCKET}/{gcs_blob_name}")
    blob.upload_from_filename(local_path)
    return f"gs://{DEST_BUCKET}/{gcs_blob_name}"

def clean_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return re.sub(r"\s+", " ", x).strip()
    if isinstance(x, list):
        return " ".join(clean_text(v) for v in x)
    if isinstance(x, dict):
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    return str(x)

def tags_to_text(tags):
    if tags is None:
        return ""
    if isinstance(tags, list):
        return " ".join(str(t) for t in tags if t is not None)
    if isinstance(tags, str):
        return tags
    return json.dumps(tags, ensure_ascii=False)

def safe_fts_query(q):
    """
    Simple broad OR query for smoke testing.
    API layer can use a better query parser later.
    """
    tokens = re.findall(r"[A-Za-z0-9_]+", q)
    tokens = [t for t in tokens if len(t) > 1]
    if not tokens:
        return ""
    return " OR ".join(tokens)

# ----------------------------
# Confirm FTS5 support
# ----------------------------

tmp = sqlite3.connect(":memory:")
try:
    tmp.execute("CREATE VIRTUAL TABLE fts_test USING fts5(x)")
finally:
    tmp.close()

# ----------------------------
# Build DB
# ----------------------------

if Path(LOCAL_DB).exists():
    os.remove(LOCAL_DB)

created_at = utc_now_iso()
start = time.time()

conn = sqlite3.connect(LOCAL_DB)
cur = conn.cursor()

cur.execute("PRAGMA journal_mode=OFF")
cur.execute("PRAGMA synchronous=OFF")
cur.execute("PRAGMA temp_store=MEMORY")
cur.execute("PRAGMA cache_size=-200000")

cur.execute("""
CREATE TABLE lecture_chunks (
    lecture_chunk_id TEXT PRIMARY KEY,
    lecture_source_id TEXT,
    lecture_family_id TEXT,
    parent_id TEXT,
    parent_type TEXT,
    source_gcs_uri TEXT,
    adapter TEXT,
    subspecialty_guess TEXT,
    title TEXT,
    entity_name TEXT,
    start_time_sec REAL,
    end_time_sec REAL,
    image_gcs_uri TEXT,
    tags_json TEXT,
    tags_text TEXT,
    text TEXT,
    text_len INTEGER,
    created_at_utc TEXT
)
""")

cur.execute("""
CREATE VIRTUAL TABLE lecture_chunks_fts USING fts5(
    title,
    entity_name,
    tags_text,
    text,
    content='lecture_chunks',
    content_rowid='rowid',
    tokenize='unicode61 remove_diacritics 2'
)
""")

cur.execute("CREATE INDEX idx_lecture_chunks_source ON lecture_chunks(lecture_source_id)")
cur.execute("CREATE INDEX idx_lecture_chunks_family ON lecture_chunks(lecture_family_id)")
cur.execute("CREATE INDEX idx_lecture_chunks_subspecialty ON lecture_chunks(subspecialty_guess)")
cur.execute("CREATE INDEX idx_lecture_chunks_parent ON lecture_chunks(parent_type, parent_id)")
cur.execute("CREATE INDEX idx_lecture_chunks_entity ON lecture_chunks(entity_name)")

row_count = 0
parse_error_count = 0
duplicate_chunk_ids = []
seen_ids = set()
parent_counts = collections.Counter()
adapter_counts = collections.Counter()
subspecialty_counts = collections.Counter()
text_lengths = []
max_text_len = 0
max_text_chunk_id = None

insert_rows = []

for line_no, r in iter_jsonl_gcs(LECTURE_CHUNKS_GCS):
    if r.get("_json_parse_error"):
        parse_error_count += 1
        continue

    chunk_id = r.get("lecture_chunk_id")
    if not chunk_id:
        parse_error_count += 1
        continue

    if chunk_id in seen_ids:
        duplicate_chunk_ids.append(chunk_id)
        continue
    seen_ids.add(chunk_id)

    tags = r.get("tags")
    tags_text = tags_to_text(tags)
    tags_json = json.dumps(tags, ensure_ascii=False) if tags is not None else None

    text = clean_text(r.get("text"))
    title = clean_text(r.get("title"))
    entity_name = clean_text(r.get("entity_name"))

    L = len(text)
    text_lengths.append(L)

    if L > max_text_len:
        max_text_len = L
        max_text_chunk_id = chunk_id

    parent_counts[r.get("parent_type") or "missing"] += 1
    adapter_counts[r.get("adapter") or "missing"] += 1
    subspecialty_counts[r.get("subspecialty_guess") or "missing"] += 1

    insert_rows.append((
        chunk_id,
        r.get("lecture_source_id"),
        r.get("lecture_family_id"),
        r.get("parent_id"),
        r.get("parent_type"),
        r.get("source_gcs_uri"),
        r.get("adapter"),
        r.get("subspecialty_guess"),
        title,
        entity_name,
        r.get("start_time_sec"),
        r.get("end_time_sec"),
        r.get("image_gcs_uri"),
        tags_json,
        tags_text,
        text,
        L,
        created_at,
    ))

    row_count += 1

    if len(insert_rows) >= 5000:
        cur.executemany("""
        INSERT INTO lecture_chunks (
            lecture_chunk_id, lecture_source_id, lecture_family_id, parent_id, parent_type,
            source_gcs_uri, adapter, subspecialty_guess, title, entity_name,
            start_time_sec, end_time_sec, image_gcs_uri,
            tags_json, tags_text, text, text_len, created_at_utc
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, insert_rows)
        conn.commit()
        print(f"Inserted metadata rows: {row_count}")
        insert_rows = []

if insert_rows:
    cur.executemany("""
    INSERT INTO lecture_chunks (
        lecture_chunk_id, lecture_source_id, lecture_family_id, parent_id, parent_type,
        source_gcs_uri, adapter, subspecialty_guess, title, entity_name,
        start_time_sec, end_time_sec, image_gcs_uri,
        tags_json, tags_text, text, text_len, created_at_utc
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, insert_rows)
    conn.commit()

print("Metadata insert complete:", row_count)

# Populate FTS from content table
cur.execute("""
INSERT INTO lecture_chunks_fts(rowid, title, entity_name, tags_text, text)
SELECT rowid, title, entity_name, tags_text, text
FROM lecture_chunks
""")
conn.commit()

cur.execute("INSERT INTO lecture_chunks_fts(lecture_chunks_fts) VALUES('optimize')")
conn.commit()

# ----------------------------
# Smoke search
# ----------------------------

def smoke_search(query, limit=3):
    fts_q = safe_fts_query(query)
    if not fts_q:
        return []

    sql = """
    SELECT
        c.lecture_chunk_id,
        c.title,
        c.entity_name,
        c.subspecialty_guess,
        c.parent_type,
        c.source_gcs_uri,
        c.start_time_sec,
        c.end_time_sec,
        c.image_gcs_uri,
        snippet(lecture_chunks_fts, 3, '[', ']', ' ... ', 32) AS snippet,
        bm25(lecture_chunks_fts) AS rank
    FROM lecture_chunks_fts
    JOIN lecture_chunks c ON c.rowid = lecture_chunks_fts.rowid
    WHERE lecture_chunks_fts MATCH ?
    ORDER BY rank
    LIMIT ?
    """
    rows = []
    for rec in cur.execute(sql, (fts_q, limit)).fetchall():
        rows.append({
            "lecture_chunk_id": rec[0],
            "title": rec[1],
            "entity_name": rec[2],
            "subspecialty_guess": rec[3],
            "parent_type": rec[4],
            "source_gcs_uri": rec[5],
            "start_time_sec": rec[6],
            "end_time_sec": rec[7],
            "image_gcs_uri": rec[8],
            "snippet": rec[9],
            "rank": rec[10],
        })
    return rows

smoke_queries = [
    "prostate grading Gleason pattern 4 cribriform",
    "FH deficient RCC CK7 fumarate hydratase",
    "metanephric adenoma WT1 CD57",
    "folliculitis demodex rosacea",
    "CSF cytology",
    "breast fibroepithelial lesion",
    "thyroid cytology Bethesda",
]

smoke_results = {}
for q in smoke_queries:
    smoke_results[q] = smoke_search(q, limit=3)

# ----------------------------
# Counts and manifest
# ----------------------------

db_row_count = cur.execute("SELECT COUNT(*) FROM lecture_chunks").fetchone()[0]
fts_row_count = cur.execute("SELECT COUNT(*) FROM lecture_chunks_fts").fetchone()[0]
source_count = cur.execute("SELECT COUNT(DISTINCT lecture_source_id) FROM lecture_chunks").fetchone()[0]
family_count = cur.execute("SELECT COUNT(DISTINCT lecture_family_id) FROM lecture_chunks").fetchone()[0]

conn.commit()
conn.close()

elapsed_sec = round(time.time() - start, 1)
db_size_bytes = os.path.getsize(LOCAL_DB)
db_sha256 = sha256_file(LOCAL_DB)

def len_stats(vals):
    if not vals:
        return {}
    vals_sorted = sorted(vals)
    def pct(p):
        ix = int(round((len(vals_sorted) - 1) * p))
        return vals_sorted[ix]
    return {
        "min": min(vals),
        "p01": pct(0.01),
        "p05": pct(0.05),
        "median": vals_sorted[len(vals_sorted)//2],
        "p95": pct(0.95),
        "p99": pct(0.99),
        "max": max(vals),
        "mean": round(sum(vals) / len(vals), 1),
    }

manifest = {
    "schema_version": "lecture_fts_index_manifest.v0_proposed",
    "created_at_utc": created_at,
    "corpus": "lectures",
    "index_type": "sqlite_fts5",
    "source_chunks": LECTURE_CHUNKS_GCS,
    "index_gcs_uri": f"gs://{DEST_BUCKET}/{OUT_DB_GCS}",
    "manifest_gcs_uri": f"gs://{DEST_BUCKET}/{OUT_MANIFEST_GCS}",
    "row_count": db_row_count,
    "fts_row_count": fts_row_count,
    "source_count": source_count,
    "lecture_family_count": family_count,
    "db_size_bytes": db_size_bytes,
    "db_sha256": db_sha256,
    "parent_type_counts": dict(parent_counts),
    "adapter_counts": dict(adapter_counts),
    "subspecialty_counts": dict(subspecialty_counts),
    "text_length_stats": len_stats(text_lengths),
    "max_text_chunk_id": max_text_chunk_id,
    "max_text_len": max_text_len,
    "parse_error_count": parse_error_count,
    "duplicate_chunk_id_count": len(duplicate_chunk_ids),
    "duplicate_chunk_id_examples": duplicate_chunk_ids[:20],
    "searchable": True,
    "indexed": True,
    "vectorized": False,
    "api_exposed": False,
    "public_figures_enabled": False,
    "notes": [
        "Lectures are keyword-searchable via this SQLite FTS artifact.",
        "This does not expose lectures through searchEvidence yet.",
        "This does not build FAISS/vector artifacts.",
        "Figures have source GCS paths only; no public figure URLs are included."
    ],
}

audit = {
    "schema_version": "lecture_fts_build_audit.v0_proposed",
    "created_at_utc": created_at,
    "elapsed_sec": elapsed_sec,
    "manifest": manifest,
    "smoke_queries": smoke_queries,
    "smoke_results": smoke_results,
    "smoke_status": {
        q: "ok" if smoke_results.get(q) else "no_hits"
        for q in smoke_queries
    },
}

with open(LOCAL_MANIFEST, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

with open(LOCAL_AUDIT, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

uploaded = {
    "lecture_fts_sqlite": upload_file(LOCAL_DB, OUT_DB_GCS),
    "lecture_fts_index_manifest": upload_file(LOCAL_MANIFEST, OUT_MANIFEST_GCS),
    "lecture_fts_build_audit": upload_file(LOCAL_AUDIT, OUT_AUDIT_GCS),
}

print("=== LECTURE SQLITE FTS BUILD COMPLETE ===")
print(json.dumps({
    "created_at_utc": created_at,
    "elapsed_sec": elapsed_sec,
    "row_count": db_row_count,
    "fts_row_count": fts_row_count,
    "source_count": source_count,
    "lecture_family_count": family_count,
    "db_size_bytes": db_size_bytes,
    "db_sha256": db_sha256,
    "parent_type_counts": manifest["parent_type_counts"],
    "adapter_counts": manifest["adapter_counts"],
    "subspecialty_counts": manifest["subspecialty_counts"],
    "text_length_stats": manifest["text_length_stats"],
    "parse_error_count": parse_error_count,
    "duplicate_chunk_id_count": len(duplicate_chunk_ids),
    "smoke_status": audit["smoke_status"],
    "uploaded": uploaded,
    "flags": {
        "indexed": True,
        "searchable": True,
        "vectorized": False,
        "api_exposed": False,
        "public_figures_enabled": False,
    }
}, indent=2))

print("\n=== SMOKE RESULT PREVIEW ===")
for q, hits in smoke_results.items():
    print("\nQUERY:", q)
    if not hits:
        print("  NO HITS")
    else:
        for h in hits[:2]:
            print(" -", h["title"], "|", h["subspecialty_guess"], "|", h["source_gcs_uri"])
            print("   ", h["snippet"][:300].replace("\n", " "))

In [ ]:
# ============================================================
# Pathology Hub — Lecture FAISS Vector Index Builder v0
# ============================================================
# Builds vector artifacts over normalized lecture_chunks.jsonl.
#
# Inputs:
#   gs://pathology_hub/02_normalized/lectures/lecture_chunks.jsonl
#
# Outputs:
#   gs://pathology_hub/03_indexes/lectures/vector/lecture_embeddings.npy
#   gs://pathology_hub/03_indexes/lectures/vector/lecture_faiss.index
#   gs://pathology_hub/03_indexes/lectures/vector/lecture_vector_docstore.jsonl
#   gs://pathology_hub/03_indexes/lectures/vector/lecture_vector_manifest.json
#   gs://pathology_hub/06_audits/lectures/lecture_vector_build_audit_20260623.json
#
# IMPORTANT:
# - This vectorizes lectures.
# - This does NOT expose lectures through searchEvidence/API.
# - This does NOT create public figure URLs.
# ============================================================

from google.colab import auth
auth.authenticate_user()

# Install dependencies if needed
import sys, subprocess, importlib.util

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

ensure_package("openai", "openai")
ensure_package("faiss", "faiss-cpu")

from google.cloud import storage, secretmanager
from openai import OpenAI
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import faiss
import json, os, re, time, hashlib, collections, math, random

# ----------------------------
# Config
# ----------------------------

PROJECT_ID = "pathology-annotation-project"
DEST_BUCKET = "pathology_hub"

LECTURE_CHUNKS_GCS = "gs://pathology_hub/02_normalized/lectures/lecture_chunks.jsonl"
LECTURE_FTS_MANIFEST_GCS = "gs://pathology_hub/03_indexes/lectures/fts/lecture_fts_index_manifest.json"

VECTOR_PREFIX = "03_indexes/lectures/vector/"
AUDIT_PREFIX = "06_audits/lectures/"

OUT_EMBEDDINGS_GCS = VECTOR_PREFIX + "lecture_embeddings.npy"
OUT_FAISS_GCS = VECTOR_PREFIX + "lecture_faiss.index"
OUT_DOCSTORE_GCS = VECTOR_PREFIX + "lecture_vector_docstore.jsonl"
OUT_MANIFEST_GCS = VECTOR_PREFIX + "lecture_vector_manifest.json"
OUT_AUDIT_GCS = AUDIT_PREFIX + "lecture_vector_build_audit_20260623.json"

LOCAL_DIR = Path("/content/lecture_vector_v0")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_EMBEDDINGS = LOCAL_DIR / "lecture_embeddings.npy"
LOCAL_FAISS = LOCAL_DIR / "lecture_faiss.index"
LOCAL_DOCSTORE = LOCAL_DIR / "lecture_vector_docstore.jsonl"
LOCAL_MANIFEST = LOCAL_DIR / "lecture_vector_manifest.json"
LOCAL_AUDIT = LOCAL_DIR / "lecture_vector_build_audit_20260623.json"

CHECKPOINT_EMBEDDINGS = LOCAL_DIR / "lecture_embeddings_checkpoint.npy"
CHECKPOINT_DONE = LOCAL_DIR / "lecture_embeddings_done.npy"

CHECKPOINT_PREFIX = VECTOR_PREFIX + "_checkpoints/"
CHECKPOINT_EMBEDDINGS_GCS = CHECKPOINT_PREFIX + "lecture_embeddings_checkpoint.npy"
CHECKPOINT_DONE_GCS = CHECKPOINT_PREFIX + "lecture_embeddings_done.npy"

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536
BATCH_SIZE = 128
CHECKPOINT_EVERY_N_RECORDS = 5000

# Keep None for full build. Use e.g. 500 for a quick pilot.
MAX_RECORDS = None

# Conservative char truncation for embedding endpoint token limits.
EMBED_TEXT_MAX_CHARS = 24000

# Docstore text cap keeps docstore manageable; original length is retained.
DOCSTORE_TEXT_MAX_CHARS = 12000

OVERWRITE = True
RESUME_FROM_CHECKPOINT = True

client = storage.Client(project=PROJECT_ID)

# ----------------------------
# Helpers
# ----------------------------

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def parse_gcs_uri(uri):
    assert uri.startswith("gs://"), uri
    rest = uri[5:]
    bucket, name = rest.split("/", 1)
    return bucket, name

def iter_jsonl_gcs(uri):
    bucket_name, blob_name = parse_gcs_uri(uri)
    blob = client.bucket(bucket_name).blob(blob_name)
    with blob.open("rt", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                yield line_no, json.loads(line)

def download_gcs_if_exists(gcs_blob_name, local_path):
    blob = client.bucket(DEST_BUCKET).blob(gcs_blob_name)
    if blob.exists():
        blob.download_to_filename(str(local_path))
        return True
    return False

def upload_file(local_path, gcs_blob_name):
    blob = client.bucket(DEST_BUCKET).blob(gcs_blob_name)
    if blob.exists() and not OVERWRITE:
        raise FileExistsError(f"Refusing to overwrite gs://{DEST_BUCKET}/{gcs_blob_name}")
    blob.upload_from_filename(str(local_path))
    return f"gs://{DEST_BUCKET}/{gcs_blob_name}"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def clean_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return re.sub(r"\s+", " ", x).strip()
    if isinstance(x, list):
        return " ".join(clean_text(v) for v in x)
    if isinstance(x, dict):
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    return str(x)

def build_embedding_text(row):
    parts = []
    for label, key in [
        ("Title", "title"),
        ("Entity", "entity_name"),
        ("Subspecialty", "subspecialty_guess"),
        ("Tags", "tags"),
        ("Text", "text"),
    ]:
        val = row.get(key)
        if val is None or val == "":
            continue
        if isinstance(val, list):
            val = "; ".join(str(v) for v in val)
        val = clean_text(val)
        if val:
            parts.append(f"{label}: {val}")
    txt = "\n".join(parts).strip()
    return txt[:EMBED_TEXT_MAX_CHARS], len(txt), len(txt) > EMBED_TEXT_MAX_CHARS

def normalize_matrix(mat):
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return mat / norms

def get_openai_api_key():
    # Prefer env var if present.
    key = os.environ.get("OPENAI_API_KEY")
    if key:
        return key

    # Pathology Hub prior vector builds used Secret Manager secret OPEN_AI_KEY_01.
    sm = secretmanager.SecretManagerServiceClient()
    for secret_name in ["OPEN_AI_KEY_01", "OPENAI_API_KEY"]:
        try:
            secret_path = f"projects/{PROJECT_ID}/secrets/{secret_name}/versions/latest"
            resp = sm.access_secret_version(request={"name": secret_path})
            key = resp.payload.data.decode("utf-8").strip()
            if key:
                return key
        except Exception:
            pass

    raise RuntimeError(
        "No OpenAI API key found. Set OPENAI_API_KEY env var or Secret Manager secret OPEN_AI_KEY_01."
    )

def embed_batch(openai_client, texts, max_retries=6):
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = openai_client.embeddings.create(
                model=EMBEDDING_MODEL,
                input=texts,
            )
            vectors = [d.embedding for d in resp.data]
            arr = np.asarray(vectors, dtype=np.float32)
            if arr.shape[1] != EMBEDDING_DIM:
                raise ValueError(f"Embedding dim mismatch: got {arr.shape[1]}, expected {EMBEDDING_DIM}")
            return arr
        except Exception as e:
            last_err = e
            sleep_s = min(60, (2 ** attempt) + random.random())
            print(f"Embedding batch failed attempt {attempt+1}/{max_retries}: {repr(e)}; sleeping {sleep_s:.1f}s")
            time.sleep(sleep_s)
    raise RuntimeError(f"Embedding batch failed after retries: {repr(last_err)}")

# ----------------------------
# Load chunks into memory
# ----------------------------

created_at = utc_now_iso()
start_time = time.time()

chunks = []
parse_error_count = 0

for line_no, row in iter_jsonl_gcs(LECTURE_CHUNKS_GCS):
    if not row.get("lecture_chunk_id") or not row.get("text"):
        parse_error_count += 1
        continue

    emb_text, original_emb_text_len, truncated = build_embedding_text(row)

    chunks.append({
        "ix": len(chunks),
        "lecture_chunk_id": row.get("lecture_chunk_id"),
        "lecture_source_id": row.get("lecture_source_id"),
        "lecture_family_id": row.get("lecture_family_id"),
        "parent_id": row.get("parent_id"),
        "parent_type": row.get("parent_type"),
        "source_gcs_uri": row.get("source_gcs_uri"),
        "adapter": row.get("adapter"),
        "subspecialty_guess": row.get("subspecialty_guess"),
        "title": row.get("title"),
        "entity_name": row.get("entity_name"),
        "start_time_sec": row.get("start_time_sec"),
        "end_time_sec": row.get("end_time_sec"),
        "image_gcs_uri": row.get("image_gcs_uri"),
        "tags": row.get("tags"),
        "text": clean_text(row.get("text"))[:DOCSTORE_TEXT_MAX_CHARS],
        "text_full_len": len(clean_text(row.get("text"))),
        "embedding_text": emb_text,
        "embedding_text_len_original": original_emb_text_len,
        "embedding_text_len_used": len(emb_text),
        "embedding_text_truncated": truncated,
    })

    if MAX_RECORDS is not None and len(chunks) >= MAX_RECORDS:
        break

N = len(chunks)
print(f"Loaded chunks for embedding: {N}; parse_error_count={parse_error_count}")

if N == 0:
    raise RuntimeError("No chunks loaded for embedding.")

# Check unique IDs
chunk_ids = [r["lecture_chunk_id"] for r in chunks]
dup_count = len(chunk_ids) - len(set(chunk_ids))
if dup_count:
    raise RuntimeError(f"Duplicate lecture_chunk_id count before vector build: {dup_count}")

# ----------------------------
# Init or resume embeddings
# ----------------------------

embeddings = None
done = None

if RESUME_FROM_CHECKPOINT:
    got_emb = download_gcs_if_exists(CHECKPOINT_EMBEDDINGS_GCS, CHECKPOINT_EMBEDDINGS)
    got_done = download_gcs_if_exists(CHECKPOINT_DONE_GCS, CHECKPOINT_DONE)
    if got_emb and got_done:
        emb0 = np.load(CHECKPOINT_EMBEDDINGS)
        done0 = np.load(CHECKPOINT_DONE)
        if emb0.shape == (N, EMBEDDING_DIM) and done0.shape == (N,):
            embeddings = emb0.astype(np.float32, copy=False)
            done = done0.astype(bool, copy=False)
            print(f"Resumed checkpoint: {int(done.sum())}/{N} embedded")
        else:
            print("Checkpoint shape mismatch; starting fresh.")

if embeddings is None:
    embeddings = np.zeros((N, EMBEDDING_DIM), dtype=np.float32)
    done = np.zeros((N,), dtype=bool)

# ----------------------------
# Embed
# ----------------------------

api_key = get_openai_api_key()
oai = OpenAI(api_key=api_key)

embedded_before = int(done.sum())
last_checkpoint_done = embedded_before

for start_ix in range(0, N, BATCH_SIZE):
    end_ix = min(N, start_ix + BATCH_SIZE)
    batch_indices = [i for i in range(start_ix, end_ix) if not done[i]]

    if not batch_indices:
        continue

    texts = [chunks[i]["embedding_text"] for i in batch_indices]

    arr = embed_batch(oai, texts)
    embeddings[batch_indices, :] = arr
    done[batch_indices] = True

    total_done = int(done.sum())

    if total_done % 1000 < BATCH_SIZE:
        print(f"Embedded {total_done}/{N}")

    if total_done - last_checkpoint_done >= CHECKPOINT_EVERY_N_RECORDS or total_done == N:
        np.save(CHECKPOINT_EMBEDDINGS, embeddings)
        np.save(CHECKPOINT_DONE, done)
        upload_file(CHECKPOINT_EMBEDDINGS, CHECKPOINT_EMBEDDINGS_GCS)
        upload_file(CHECKPOINT_DONE, CHECKPOINT_DONE_GCS)
        last_checkpoint_done = total_done
        print(f"Checkpoint uploaded at {total_done}/{N}")

if int(done.sum()) != N:
    raise RuntimeError(f"Embedding incomplete: {int(done.sum())}/{N}")

# Normalize embeddings for cosine/IP search
embeddings = normalize_matrix(embeddings.astype(np.float32, copy=False))

np.save(LOCAL_EMBEDDINGS, embeddings)

# ----------------------------
# Build FAISS index
# ----------------------------

index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(embeddings)

faiss.write_index(index, str(LOCAL_FAISS))

# ----------------------------
# Write docstore
# ----------------------------

with open(LOCAL_DOCSTORE, "w", encoding="utf-8") as f:
    for r in chunks:
        doc = {
            "ix": r["ix"],
            "lecture_chunk_id": r["lecture_chunk_id"],
            "lecture_source_id": r["lecture_source_id"],
            "lecture_family_id": r["lecture_family_id"],
            "parent_id": r["parent_id"],
            "parent_type": r["parent_type"],
            "source_gcs_uri": r["source_gcs_uri"],
            "adapter": r["adapter"],
            "subspecialty_guess": r["subspecialty_guess"],
            "title": r["title"],
            "entity_name": r["entity_name"],
            "start_time_sec": r["start_time_sec"],
            "end_time_sec": r["end_time_sec"],
            "image_gcs_uri": r["image_gcs_uri"],
            "tags": r["tags"],
            "text": r["text"],
            "text_full_len": r["text_full_len"],
            "embedding_text_len_original": r["embedding_text_len_original"],
            "embedding_text_len_used": r["embedding_text_len_used"],
            "embedding_text_truncated": r["embedding_text_truncated"],
        }
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

# ----------------------------
# Vector smoke search
# ----------------------------

def vector_search(query, k=5):
    q_arr = embed_batch(oai, [query])
    q_arr = normalize_matrix(q_arr)
    scores, idxs = index.search(q_arr.astype(np.float32), k)

    hits = []
    for score, ix in zip(scores[0], idxs[0]):
        if ix < 0:
            continue
        d = chunks[int(ix)]
        hits.append({
            "score": float(score),
            "ix": int(ix),
            "lecture_chunk_id": d["lecture_chunk_id"],
            "title": d["title"],
            "entity_name": d["entity_name"],
            "subspecialty_guess": d["subspecialty_guess"],
            "source_gcs_uri": d["source_gcs_uri"],
            "start_time_sec": d["start_time_sec"],
            "end_time_sec": d["end_time_sec"],
            "image_gcs_uri": d["image_gcs_uri"],
            "text_preview": d["text"][:600],
        })
    return hits

smoke_queries = [
    "prostate grading cribriform pattern four",
    "FH deficient renal cell carcinoma fumarate hydratase immunostains",
    "metanephric adenoma WT1 CD57 racemase",
    "rosacea demodex folliculitis dermatology",
    "CSF cytology lymphoma meningitis",
    "breast fibroepithelial lesion phyllodes",
    "thyroid Bethesda cytology categories",
]

smoke_results = {}
for q in smoke_queries:
    smoke_results[q] = vector_search(q, k=5)

smoke_status = {
    q: "ok" if smoke_results[q] else "no_hits"
    for q in smoke_queries
}

# ----------------------------
# Manifest + audit
# ----------------------------

elapsed_sec = round(time.time() - start_time, 1)

parent_counts = collections.Counter(r["parent_type"] for r in chunks)
adapter_counts = collections.Counter(r["adapter"] for r in chunks)
subspecialty_counts = collections.Counter(r["subspecialty_guess"] for r in chunks)
truncated_count = sum(1 for r in chunks if r["embedding_text_truncated"])

embedding_sha = sha256_file(LOCAL_EMBEDDINGS)
faiss_sha = sha256_file(LOCAL_FAISS)
docstore_sha = sha256_file(LOCAL_DOCSTORE)

manifest = {
    "schema_version": "lecture_vector_manifest.v0_proposed",
    "created_at_utc": created_at,
    "corpus": "lectures",
    "source_chunks": LECTURE_CHUNKS_GCS,
    "related_fts_manifest": LECTURE_FTS_MANIFEST_GCS,
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dim": EMBEDDING_DIM,
    "record_count": N,
    "faiss_ntotal": int(index.ntotal),
    "index_type": "faiss_IndexFlatIP_cosine_normalized",
    "embeddings_normalized": True,
    "embedding_text_max_chars": EMB_TEXT_MAX_CHARS if False else EMBED_TEXT_MAX_CHARS,
    "docstore_text_max_chars": DOCSTORE_TEXT_MAX_CHARS,
    "embedding_text_truncated_count": truncated_count,
    "parent_type_counts": dict(parent_counts),
    "adapter_counts": dict(adapter_counts),
    "subspecialty_counts": dict(subspecialty_counts),
    "artifacts": {
        "embeddings_npy": f"gs://{DEST_BUCKET}/{OUT_EMBEDDINGS_GCS}",
        "faiss_index": f"gs://{DEST_BUCKET}/{OUT_FAISS_GCS}",
        "docstore_jsonl": f"gs://{DEST_BUCKET}/{OUT_DOCSTORE_GCS}",
        "manifest_json": f"gs://{DEST_BUCKET}/{OUT_MANIFEST_GCS}",
        "audit_json": f"gs://{DEST_BUCKET}/{OUT_AUDIT_GCS}",
    },
    "artifact_sha256": {
        "embeddings_npy": embedding_sha,
        "faiss_index": faiss_sha,
        "docstore_jsonl": docstore_sha,
    },
    "indexed": True,
    "searchable": True,
    "vectorized": True,
    "api_exposed": False,
    "public_figures_enabled": False,
    "notes": [
        "Lecture vector artifacts are built but not yet exposed through searchEvidence.",
        "Use hybrid FTS + vector RRF in backend integration.",
        "Figures remain source GCS paths only; no public figure URLs are created by this build."
    ],
}

audit = {
    "schema_version": "lecture_vector_build_audit.v0_proposed",
    "created_at_utc": created_at,
    "elapsed_sec": elapsed_sec,
    "manifest": manifest,
    "parse_error_count": parse_error_count,
    "duplicate_chunk_id_count": dup_count,
    "embedded_before_resume": embedded_before,
    "embedded_final": int(done.sum()),
    "smoke_queries": smoke_queries,
    "smoke_status": smoke_status,
    "smoke_results": smoke_results,
}

with open(LOCAL_MANIFEST, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

with open(LOCAL_AUDIT, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

uploaded = {
    "lecture_embeddings": upload_file(LOCAL_EMBEDDINGS, OUT_EMBEDDINGS_GCS),
    "lecture_faiss": upload_file(LOCAL_FAISS, OUT_FAISS_GCS),
    "lecture_vector_docstore": upload_file(LOCAL_DOCSTORE, OUT_DOCSTORE_GCS),
    "lecture_vector_manifest": upload_file(LOCAL_MANIFEST, OUT_MANIFEST_GCS),
    "lecture_vector_build_audit": upload_file(LOCAL_AUDIT, OUT_AUDIT_GCS),
}

print("=== LECTURE VECTOR BUILD COMPLETE ===")
print(json.dumps({
    "created_at_utc": created_at,
    "elapsed_sec": elapsed_sec,
    "record_count": N,
    "faiss_ntotal": int(index.ntotal),
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dim": EMBEDDING_DIM,
    "embedding_text_truncated_count": truncated_count,
    "parent_type_counts": manifest["parent_type_counts"],
    "adapter_counts": manifest["adapter_counts"],
    "subspecialty_counts": manifest["subspecialty_counts"],
    "artifact_sha256": manifest["artifact_sha256"],
    "smoke_status": smoke_status,
    "uploaded": uploaded,
    "flags": {
        "indexed": True,
        "searchable": True,
        "vectorized": True,
        "api_exposed": False,
        "public_figures_enabled": False,
    }
}, indent=2))

print("\n=== VECTOR SMOKE PREVIEW ===")
for q, hits in smoke_results.items():
    print("\nQUERY:", q)
    for h in hits[:2]:
        print(" -", h["title"], "|", h["subspecialty_guess"], "| score:", round(h["score"], 4))
        print("   ", h["source_gcs_uri"])
        print("   ", h["text_preview"][:260].replace("\n", " "))

In [ ]:
# ============================================================
# Pathology Hub — Lecture Hybrid FTS + FAISS Smoke Harness v0
# ============================================================
# Purpose:
# - Load lecture SQLite FTS + FAISS vector artifacts from GCS.
# - Run local hybrid retrieval using Reciprocal Rank Fusion.
# - Upload smoke audit/results.
#
# This does NOT expose lectures through searchEvidence/API.
# ============================================================

from google.colab import auth
auth.authenticate_user()

import sys, subprocess, importlib.util

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

ensure_package("openai", "openai")
ensure_package("faiss", "faiss-cpu")

from google.cloud import storage, secretmanager
from openai import OpenAI
from datetime import datetime, timezone
from pathlib import Path
import sqlite3, json, re, os, hashlib, collections, time, random
import numpy as np
import faiss

PROJECT_ID = "pathology-annotation-project"
DEST_BUCKET = "pathology_hub"

FTS_DB_GCS = "03_indexes/lectures/fts/lecture_fts.sqlite"
FTS_MANIFEST_GCS = "03_indexes/lectures/fts/lecture_fts_index_manifest.json"

VECTOR_FAISS_GCS = "03_indexes/lectures/vector/lecture_faiss.index"
VECTOR_DOCSTORE_GCS = "03_indexes/lectures/vector/lecture_vector_docstore.jsonl"
VECTOR_MANIFEST_GCS = "03_indexes/lectures/vector/lecture_vector_manifest.json"

OUT_AUDIT_GCS = "06_audits/lectures/lecture_hybrid_smoke_audit_20260623.json"
OUT_RESULTS_GCS = "04_api_artifacts/lectures/lecture_hybrid_smoke_results_20260623.json"

LOCAL_DIR = Path("/content/lecture_hybrid_smoke_v0")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_FTS_DB = LOCAL_DIR / "lecture_fts.sqlite"
LOCAL_FAISS = LOCAL_DIR / "lecture_faiss.index"
LOCAL_DOCSTORE = LOCAL_DIR / "lecture_vector_docstore.jsonl"
LOCAL_FTS_MANIFEST = LOCAL_DIR / "lecture_fts_index_manifest.json"
LOCAL_VECTOR_MANIFEST = LOCAL_DIR / "lecture_vector_manifest.json"
LOCAL_AUDIT = LOCAL_DIR / "lecture_hybrid_smoke_audit_20260623.json"
LOCAL_RESULTS = LOCAL_DIR / "lecture_hybrid_smoke_results_20260623.json"

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536

FTS_LIMIT = 20
VECTOR_LIMIT = 20
FINAL_LIMIT = 8
RRF_K = 60

OVERWRITE = True

client = storage.Client(project=PROJECT_ID)

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def download_blob(blob_name, local_path):
    client.bucket(DEST_BUCKET).blob(blob_name).download_to_filename(str(local_path))
    return str(local_path)

def upload_file(local_path, blob_name):
    blob = client.bucket(DEST_BUCKET).blob(blob_name)
    if blob.exists() and not OVERWRITE:
        raise FileExistsError(f"Refusing to overwrite gs://{DEST_BUCKET}/{blob_name}")
    blob.upload_from_filename(str(local_path))
    return f"gs://{DEST_BUCKET}/{blob_name}"

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def safe_fts_query(q):
    tokens = re.findall(r"[A-Za-z0-9_]+", q)
    tokens = [t for t in tokens if len(t) > 1]
    if not tokens:
        return ""
    return " OR ".join(tokens)

def normalize_matrix(mat):
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return mat / norms

def get_openai_api_key():
    key = os.environ.get("OPENAI_API_KEY")
    if key:
        return key

    sm = secretmanager.SecretManagerServiceClient()
    for secret_name in ["OPEN_AI_KEY_01", "OPENAI_API_KEY"]:
        try:
            secret_path = f"projects/{PROJECT_ID}/secrets/{secret_name}/versions/latest"
            resp = sm.access_secret_version(request={"name": secret_path})
            key = resp.payload.data.decode("utf-8").strip()
            if key:
                return key
        except Exception:
            pass

    raise RuntimeError("No OpenAI API key found in env or Secret Manager.")

def embed_query(oai, query, max_retries=5):
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = oai.embeddings.create(
                model=EMBEDDING_MODEL,
                input=[query],
            )
            arr = np.asarray([resp.data[0].embedding], dtype=np.float32)
            if arr.shape[1] != EMBEDDING_DIM:
                raise ValueError(f"Embedding dim mismatch: {arr.shape[1]}")
            return normalize_matrix(arr)
        except Exception as e:
            last_err = e
            sleep_s = min(30, (2 ** attempt) + random.random())
            print(f"Query embedding failed attempt {attempt+1}/{max_retries}: {repr(e)}; sleeping {sleep_s:.1f}s")
            time.sleep(sleep_s)

    raise RuntimeError(f"Query embedding failed: {repr(last_err)}")

# ----------------------------
# Download artifacts
# ----------------------------

created_at = utc_now_iso()
start = time.time()

download_blob(FTS_DB_GCS, LOCAL_FTS_DB)
download_blob(VECTOR_FAISS_GCS, LOCAL_FAISS)
download_blob(VECTOR_DOCSTORE_GCS, LOCAL_DOCSTORE)
download_blob(FTS_MANIFEST_GCS, LOCAL_FTS_MANIFEST)
download_blob(VECTOR_MANIFEST_GCS, LOCAL_VECTOR_MANIFEST)

fts_manifest = load_json(LOCAL_FTS_MANIFEST)
vector_manifest = load_json(LOCAL_VECTOR_MANIFEST)

print("Loaded manifests:")
print(json.dumps({
    "fts_row_count": fts_manifest.get("row_count"),
    "fts_indexed": fts_manifest.get("indexed"),
    "vector_record_count": vector_manifest.get("record_count"),
    "faiss_ntotal_manifest": vector_manifest.get("faiss_ntotal"),
    "vectorized": vector_manifest.get("vectorized"),
    "api_exposed": vector_manifest.get("api_exposed"),
}, indent=2))

# ----------------------------
# Load FAISS + docstore
# ----------------------------

index = faiss.read_index(str(LOCAL_FAISS))

docstore = []
with open(LOCAL_DOCSTORE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            docstore.append(json.loads(line))

if index.ntotal != len(docstore):
    raise RuntimeError(f"FAISS/docstore mismatch: ntotal={index.ntotal}, docstore={len(docstore)}")

doc_by_chunk_id = {d["lecture_chunk_id"]: d for d in docstore}

# ----------------------------
# Search functions
# ----------------------------

conn = sqlite3.connect(str(LOCAL_FTS_DB))
conn.row_factory = sqlite3.Row

def fts_search(query, limit=FTS_LIMIT):
    fts_q = safe_fts_query(query)
    if not fts_q:
        return []

    sql = """
    SELECT
        c.lecture_chunk_id,
        c.title,
        c.entity_name,
        c.subspecialty_guess,
        c.parent_type,
        c.source_gcs_uri,
        c.start_time_sec,
        c.end_time_sec,
        c.image_gcs_uri,
        snippet(lecture_chunks_fts, 3, '[', ']', ' ... ', 40) AS snippet,
        bm25(lecture_chunks_fts) AS bm25_rank
    FROM lecture_chunks_fts
    JOIN lecture_chunks c ON c.rowid = lecture_chunks_fts.rowid
    WHERE lecture_chunks_fts MATCH ?
    ORDER BY bm25_rank
    LIMIT ?
    """
    rows = []
    for rank_ix, rec in enumerate(conn.execute(sql, (fts_q, limit)).fetchall(), start=1):
        rows.append({
            "retrieval": "fts",
            "rank": rank_ix,
            "lecture_chunk_id": rec["lecture_chunk_id"],
            "title": rec["title"],
            "entity_name": rec["entity_name"],
            "subspecialty_guess": rec["subspecialty_guess"],
            "parent_type": rec["parent_type"],
            "source_gcs_uri": rec["source_gcs_uri"],
            "start_time_sec": rec["start_time_sec"],
            "end_time_sec": rec["end_time_sec"],
            "image_gcs_uri": rec["image_gcs_uri"],
            "snippet": rec["snippet"],
            "bm25_rank": rec["bm25_rank"],
        })
    return rows

oai = OpenAI(api_key=get_openai_api_key())

def vector_search(query, limit=VECTOR_LIMIT):
    q = embed_query(oai, query)
    scores, idxs = index.search(q.astype(np.float32), limit)

    rows = []
    for rank_ix, (score, ix) in enumerate(zip(scores[0], idxs[0]), start=1):
        if ix < 0:
            continue
        d = docstore[int(ix)]
        rows.append({
            "retrieval": "vector",
            "rank": rank_ix,
            "score": float(score),
            "lecture_chunk_id": d["lecture_chunk_id"],
            "title": d.get("title"),
            "entity_name": d.get("entity_name"),
            "subspecialty_guess": d.get("subspecialty_guess"),
            "parent_type": d.get("parent_type"),
            "source_gcs_uri": d.get("source_gcs_uri"),
            "start_time_sec": d.get("start_time_sec"),
            "end_time_sec": d.get("end_time_sec"),
            "image_gcs_uri": d.get("image_gcs_uri"),
            "snippet": (d.get("text") or "")[:800],
        })
    return rows

def hybrid_rrf(query, final_limit=FINAL_LIMIT):
    fts_hits = fts_search(query, FTS_LIMIT)
    vec_hits = vector_search(query, VECTOR_LIMIT)

    fused = {}

    def add(hit, source_name):
        cid = hit["lecture_chunk_id"]
        if cid not in fused:
            fused[cid] = {
                **hit,
                "rrf_score": 0.0,
                "retrieval_sources": [],
                "fts_rank": None,
                "vector_rank": None,
                "vector_score": None,
            }

        fused[cid]["rrf_score"] += 1.0 / (RRF_K + hit["rank"])
        fused[cid]["retrieval_sources"].append(source_name)

        if source_name == "fts":
            fused[cid]["fts_rank"] = hit["rank"]
            fused[cid]["fts_snippet"] = hit.get("snippet")
        elif source_name == "vector":
            fused[cid]["vector_rank"] = hit["rank"]
            fused[cid]["vector_score"] = hit.get("score")
            fused[cid]["vector_snippet"] = hit.get("snippet")

    for h in fts_hits:
        add(h, "fts")

    for h in vec_hits:
        add(h, "vector")

    ranked = sorted(
        fused.values(),
        key=lambda x: (-x["rrf_score"], x.get("fts_rank") or 9999, x.get("vector_rank") or 9999)
    )

    # Prefer compact result shape
    final = []
    for h in ranked[:final_limit]:
        final.append({
            "lecture_chunk_id": h["lecture_chunk_id"],
            "title": h.get("title"),
            "entity_name": h.get("entity_name"),
            "subspecialty_guess": h.get("subspecialty_guess"),
            "parent_type": h.get("parent_type"),
            "source_gcs_uri": h.get("source_gcs_uri"),
            "start_time_sec": h.get("start_time_sec"),
            "end_time_sec": h.get("end_time_sec"),
            "image_gcs_uri": h.get("image_gcs_uri"),
            "rrf_score": h["rrf_score"],
            "retrieval_sources": sorted(set(h["retrieval_sources"])),
            "fts_rank": h.get("fts_rank"),
            "vector_rank": h.get("vector_rank"),
            "vector_score": h.get("vector_score"),
            "snippet": h.get("fts_snippet") or h.get("vector_snippet") or h.get("snippet"),
        })

    return {
        "query": query,
        "search_mode": "hybrid_fts_faiss_vector_rrf",
        "fts_hit_count": len(fts_hits),
        "vector_hit_count": len(vec_hits),
        "hybrid_hit_count": len(final),
        "results": final,
    }

# ----------------------------
# Smoke queries
# ----------------------------

smoke_queries = [
    "prostate grading cribriform pattern 4",
    "FH deficient RCC CK7 fumarate hydratase",
    "metanephric adenoma WT1 CD57 racemase",
    "demodex rosacea folliculitis",
    "CSF cytology lymphoma",
    "breast fibroepithelial phyllodes",
    "thyroid Bethesda cytology",
    "soft tissue grossing sarcoma margin",
    "alopecia lichen planopilaris perifollicular fibrosis",
    "pancreas IPMN cytology mucin",
]

hybrid_results = []
for q in smoke_queries:
    print("Hybrid query:", q)
    hybrid_results.append(hybrid_rrf(q))

smoke_status = {
    r["query"]: "ok" if r["hybrid_hit_count"] > 0 else "no_hits"
    for r in hybrid_results
}

elapsed_sec = round(time.time() - start, 1)

audit = {
    "schema_version": "lecture_hybrid_smoke_audit.v0_proposed",
    "created_at_utc": created_at,
    "elapsed_sec": elapsed_sec,
    "normalization_only": False,
    "indexed": True,
    "searchable": True,
    "vectorized": True,
    "api_exposed": False,
    "public_figures_enabled": False,
    "search_mode": "hybrid_fts_faiss_vector_rrf",
    "rrf_k": RRF_K,
    "fts_limit": FTS_LIMIT,
    "vector_limit": VECTOR_LIMIT,
    "final_limit": FINAL_LIMIT,
    "artifact_inputs": {
        "fts_sqlite": f"gs://{DEST_BUCKET}/{FTS_DB_GCS}",
        "fts_manifest": f"gs://{DEST_BUCKET}/{FTS_MANIFEST_GCS}",
        "vector_faiss": f"gs://{DEST_BUCKET}/{VECTOR_FAISS_GCS}",
        "vector_docstore": f"gs://{DEST_BUCKET}/{VECTOR_DOCSTORE_GCS}",
        "vector_manifest": f"gs://{DEST_BUCKET}/{VECTOR_MANIFEST_GCS}",
    },
    "manifest_status": {
        "fts_row_count": fts_manifest.get("row_count"),
        "vector_record_count": vector_manifest.get("record_count"),
        "faiss_ntotal_runtime": int(index.ntotal),
        "docstore_records_runtime": len(docstore),
        "embedding_model": vector_manifest.get("embedding_model"),
        "embedding_dim": vector_manifest.get("embedding_dim"),
    },
    "smoke_queries": smoke_queries,
    "smoke_status": smoke_status,
    "known_limitations": [
        "Hybrid search is local Colab smoke only.",
        "Lectures are not yet exposed through searchEvidence/API.",
        "Figure records have source GCS paths only; no public/browser-safe figure URLs are emitted.",
        "Subspecialty inference still has Unknown records that may need later metadata cleanup.",
    ],
}

results_packet = {
    "schema_version": "lecture_hybrid_smoke_results.v0_proposed",
    "created_at_utc": created_at,
    "search_mode": "hybrid_fts_faiss_vector_rrf",
    "api_exposed": False,
    "results": hybrid_results,
}

with open(LOCAL_AUDIT, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

with open(LOCAL_RESULTS, "w", encoding="utf-8") as f:
    json.dump(results_packet, f, indent=2, ensure_ascii=False)

uploaded = {
    "lecture_hybrid_smoke_audit": upload_file(LOCAL_AUDIT, OUT_AUDIT_GCS),
    "lecture_hybrid_smoke_results": upload_file(LOCAL_RESULTS, OUT_RESULTS_GCS),
}

print("=== LECTURE HYBRID SMOKE COMPLETE ===")
print(json.dumps({
    "created_at_utc": created_at,
    "elapsed_sec": elapsed_sec,
    "search_mode": "hybrid_fts_faiss_vector_rrf",
    "manifest_status": audit["manifest_status"],
    "smoke_status": smoke_status,
    "uploaded": uploaded,
    "flags": {
        "indexed": True,
        "searchable": True,
        "vectorized": True,
        "api_exposed": False,
        "public_figures_enabled": False,
    }
}, indent=2))

print("\n=== HYBRID SMOKE PREVIEW ===")
for r in hybrid_results:
    print("\nQUERY:", r["query"])
    for h in r["results"][:3]:
        print(
            " -",
            h["title"],
            "|",
            h["subspecialty_guess"],
            "| sources:",
            ",".join(h["retrieval_sources"]),
            "| fts:",
            h["fts_rank"],
            "| vec:",
            h["vector_rank"],
        )
        print("   ", h["source_gcs_uri"])
        print("   ", (h["snippet"] or "")[:280].replace("\n", " "))

In [ ]:
# ============================================================
# v04.6 SOURCE LOCATOR REGISTRY BUILD
#
# Builds a registry for:
# - textbook source_id -> public PDF URL
# - video source_id -> public MP4 URL
#
# Later API patch uses this to add:
# - source_pdf_url
# - source_page_url
# - video_url
# - video_time_url
# - reference_links[]
# ============================================================

import json, re, subprocess, datetime, urllib.parse
from pathlib import Path
from collections import Counter

PROJECT_ID = "pathology-annotation-project"

TEXTBOOK_SOURCES_GCS = "gs://pathology_hub/02_normalized/textbooks/lean/textbook_lean_sources.jsonl"

PDF_PREFIXES = [
    "gs://pathology-hub-0/source_pdfs/",
    "gs://pathology_hub/source_pdfs/",
    "gs://pathology_hub/00_sources/source_pdfs/",
]

VIDEO_PREFIXES = [
    "gs://pathology-hub-0/source_videos/",
    "gs://pathology_hub/source_videos/",
    "gs://pathology_hub/00_sources/source_videos/",
]

OUT_REGISTRY_GCS = "gs://pathology_hub/02_normalized/source_registry/source_locator_registry_v1.jsonl"
OUT_AUDIT_GCS = "gs://pathology_hub/06_audits/source_registry/source_locator_registry_audit_v1.json"

WORK = Path("/content/pathology_hub_source_locator")
WORK.mkdir(parents=True, exist_ok=True)

LOCAL_TEXTBOOK_SOURCES = WORK / "textbook_lean_sources.jsonl"
LOCAL_REGISTRY = WORK / "source_locator_registry_v1.jsonl"
LOCAL_AUDIT = WORK / "source_locator_registry_audit_v1.json"

# Manual seeds for known examples / important overrides.
# Add page offsets later if audited, e.g. "pdf_page_offset": 12
MANUAL_SOURCE_OVERRIDES = {
    "bst_horvai": {
        "source_family": "textbooks",
        "source_id": "bst_horvai",
        "title": "BST Horvai",
        "pdf_url": "https://storage.googleapis.com/pathology-hub-0/source_pdfs/BST_Horvai.pdf",
        "pdf_gcs_uri": "gs://pathology-hub-0/source_pdfs/BST_Horvai.pdf",
        "pdf_page_offset": 0,
        "page_link_verified": False,
        "note": "Manual seed from user-provided URL; page offset not audited."
    },
}

MANUAL_VIDEO_SOURCES = {
    "asc_global_12_2025_scope_session": {
        "source_family": "videos",
        "source_id": "asc_global_12_2025_scope_session",
        "title": "ASC Global 12 2025 Scope Session",
        "video_url": "https://storage.googleapis.com/pathology-hub-0/source_videos/ASC_Global_12_2025_Scope_Session.mp4",
        "video_gcs_uri": "gs://pathology-hub-0/source_videos/ASC_Global_12_2025_Scope_Session.mp4",
        "timestamp_link_verified": False,
        "note": "Manual seed from user-provided URL; exact timestamps require transcript/timecoded chunks."
    },
}

def run(cmd, check=True, quiet=False):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout and not quiet:
        print(p.stdout[-6000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

def gcs_exists(uri):
    return run(["gcloud", "storage", "ls", uri], check=False, quiet=True).returncode == 0

def list_gcs(prefix):
    p = run(["gcloud", "storage", "ls", "--recursive", prefix], check=False, quiet=True)
    if p.returncode != 0:
        return []
    return [x.strip() for x in p.stdout.splitlines() if x.strip().startswith("gs://")]

def parse_gs(gs_uri):
    rest = gs_uri[5:]
    bucket, key = rest.split("/", 1)
    return bucket, key

def gs_to_https(gs_uri):
    bucket, key = parse_gs(gs_uri)
    return f"https://storage.googleapis.com/{bucket}/{urllib.parse.quote(key, safe='/')}"

def norm_id(s):
    s = str(s or "").strip()
    s = re.sub(r"\.[A-Za-z0-9]+$", "", s)
    s = s.replace("&", " and ")
    s = re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_").lower()
    s = re.sub(r"_+", "_", s)
    return s

def compact(s):
    return re.sub(r"[^a-z0-9]+", "", str(s or "").lower())

def get_first(obj, *keys):
    for k in keys:
        v = obj.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
        if v not in (None, "", []):
            return v
    return None

run(["gcloud", "config", "set", "project", PROJECT_ID])

# Load textbook source records if present.
if gcs_exists(TEXTBOOK_SOURCES_GCS):
    run(["gcloud", "storage", "cp", TEXTBOOK_SOURCES_GCS, str(LOCAL_TEXTBOOK_SOURCES)])
else:
    print("WARNING: textbook sources file not found:", TEXTBOOK_SOURCES_GCS)

textbook_sources = []
if LOCAL_TEXTBOOK_SOURCES.exists():
    with LOCAL_TEXTBOOK_SOURCES.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    textbook_sources.append(json.loads(line))
                except Exception:
                    pass

print("Textbook sources loaded:", len(textbook_sources))

# List public/source PDFs.
pdf_uris = []
for prefix in PDF_PREFIXES:
    pdf_uris.extend(list_gcs(prefix))
pdf_uris = sorted(set([u for u in pdf_uris if u.lower().endswith(".pdf")]))

print("PDF objects discovered:", len(pdf_uris))
for u in pdf_uris[:20]:
    print(" -", u)

pdf_candidates = []
for uri in pdf_uris:
    name = Path(uri).name
    pdf_candidates.append({
        "pdf_gcs_uri": uri,
        "pdf_url": gs_to_https(uri),
        "filename": name,
        "norm_filename": norm_id(name),
        "compact_filename": compact(name),
    })

# Match textbook source_id/title to PDFs.
registry = []
matched_source_ids = set()

for obj in textbook_sources:
    source_id = str(get_first(obj, "source_id", "id", "book_id") or "").strip()
    if not source_id:
        continue

    title = str(get_first(obj, "title", "source_title", "book_title", "display_name") or source_id).strip()
    source_candidates = [
        source_id,
        title,
        get_first(obj, "filename", "pdf_filename", "source_filename", "original_filename") or "",
    ]
    source_norms = [norm_id(x) for x in source_candidates if x]
    source_compacts = [compact(x) for x in source_candidates if x]

    best = None
    best_score = 0

    for pdf in pdf_candidates:
        score = 0
        pn = pdf["norm_filename"]
        pc = pdf["compact_filename"]

        for sn in source_norms:
            if sn and sn == pn:
                score = max(score, 100)
            elif sn and (sn in pn or pn in sn):
                score = max(score, 85)

        for sc in source_compacts:
            if sc and sc == pc:
                score = max(score, 100)
            elif sc and (sc in pc or pc in sc):
                score = max(score, 80)

        if score > best_score:
            best_score = score
            best = pdf

    rec = {
        "schema_version": "source_locator_registry.v1",
        "source_family": "textbooks",
        "source_id": source_id,
        "title": title,
        "pdf_page_offset": 0,
        "page_link_method": "pdf_fragment_page_from_chunk_page_plus_offset",
        "page_link_verified": False,
        "matched_score": best_score,
        "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    }

    if best and best_score >= 80:
        rec.update({
            "pdf_gcs_uri": best["pdf_gcs_uri"],
            "pdf_url": best["pdf_url"],
            "pdf_filename": best["filename"],
        })
        matched_source_ids.add(source_id)

    registry.append(rec)

# Apply manual textbook overrides.
by_key = {(r.get("source_family"), r.get("source_id")): r for r in registry}
for sid, override in MANUAL_SOURCE_OVERRIDES.items():
    key = ("textbooks", sid)
    if key in by_key:
        by_key[key].update(override)
    else:
        override = {
            "schema_version": "source_locator_registry.v1",
            "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
            **override,
        }
        registry.append(override)
        by_key[key] = override

# List videos.
video_uris = []
for prefix in VIDEO_PREFIXES:
    video_uris.extend(list_gcs(prefix))
video_uris = sorted(set([u for u in video_uris if u.lower().endswith((".mp4", ".mov", ".m4v", ".webm"))]))

print("Video objects discovered:", len(video_uris))
for u in video_uris[:20]:
    print(" -", u)

for uri in video_uris:
    name = Path(uri).name
    sid = norm_id(name)
    key = ("videos", sid)
    rec = {
        "schema_version": "source_locator_registry.v1",
        "source_family": "videos",
        "source_id": sid,
        "title": re.sub(r"[_-]+", " ", Path(name).stem).strip(),
        "video_gcs_uri": uri,
        "video_url": gs_to_https(uri),
        "timestamp_link_method": "mp4_media_fragment_t_start_end",
        "timestamp_link_verified": False,
        "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    }
    if key in by_key:
        by_key[key].update(rec)
    else:
        registry.append(rec)
        by_key[key] = rec

# Apply manual videos.
for sid, override in MANUAL_VIDEO_SOURCES.items():
    key = ("videos", sid)
    if key in by_key:
        by_key[key].update(override)
    else:
        override = {
            "schema_version": "source_locator_registry.v1",
            "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
            **override,
        }
        registry.append(override)
        by_key[key] = override

# Write registry.
with LOCAL_REGISTRY.open("w", encoding="utf-8") as f:
    for r in registry:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

counts = Counter((r.get("source_family"), bool(r.get("pdf_url")), bool(r.get("video_url"))) for r in registry)
missing_textbook_pdf = [r for r in registry if r.get("source_family") == "textbooks" and not r.get("pdf_url")]

audit = {
    "schema_version": "source_locator_registry_audit.v1",
    "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "input_textbook_sources_gcs": TEXTBOOK_SOURCES_GCS,
    "pdf_prefixes": PDF_PREFIXES,
    "video_prefixes": VIDEO_PREFIXES,
    "output_registry_gcs": OUT_REGISTRY_GCS,
    "counts": {
        "registry_records": len(registry),
        "textbook_sources_loaded": len(textbook_sources),
        "pdf_objects_discovered": len(pdf_uris),
        "video_objects_discovered": len(video_uris),
        "textbook_records_missing_pdf_url": len(missing_textbook_pdf),
    },
    "counter": {str(k): v for k, v in counts.items()},
    "known_limitations": [
        "PDF page links use chunk page plus pdf_page_offset; offsets are not audited unless page_link_verified=true.",
        "Video timestamp links require timecoded transcript/chunk start_sec fields; full-video links can be provided without timestamps.",
        "Registry links are not evidence by themselves; API must enrich returned evidence hits."
    ],
    "sample_missing_textbook_pdf": [
        {"source_id": r.get("source_id"), "title": r.get("title"), "matched_score": r.get("matched_score")}
        for r in missing_textbook_pdf[:30]
    ],
}

LOCAL_AUDIT.write_text(json.dumps(audit, indent=2), encoding="utf-8")

run(["gcloud", "storage", "cp", str(LOCAL_REGISTRY), OUT_REGISTRY_GCS])
run(["gcloud", "storage", "cp", str(LOCAL_AUDIT), OUT_AUDIT_GCS])

print("\n✅ SOURCE LOCATOR REGISTRY BUILT")
print("Registry:", OUT_REGISTRY_GCS)
print("Audit:", OUT_AUDIT_GCS)
print(json.dumps(audit["counts"], indent=2))

In [ ]:
# ============================================================
# DEPLOY PATHOLOGY HUB v04.6 SOURCE LOCATOR PATCH
#
# Adds reference_links to searchEvidence results:
# - textbook PDF source_page_url using source_id + page
# - future video timestamp links using source_id + start_sec/end_sec
#
# Does NOT add a new GPT Action.
# ============================================================

import os, re, json, tarfile, subprocess, datetime, textwrap
from pathlib import Path
import requests

PROJECT_ID = "pathology-annotation-project"
REGION = "us-central1"
SERVICE_NAME = "pathology-hub-v04"
AR_REPO = "pathology-hub"
IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/{SERVICE_NAME}:latest"

APP_DIR = Path("/content/pathology_hub_v04_textbook_api")
APP_DIR.mkdir(parents=True, exist_ok=True)
APP_PATH = APP_DIR / "app.py"

SOURCE_LOCATOR_REGISTRY_GCS = "gs://pathology_hub/02_normalized/source_registry/source_locator_registry_v1.jsonl"
OPENAI_SECRET_NAME = "OPEN_AI_KEY_01"

def run(cmd, cwd=None, check=True, quiet=False):
    print("\nRUN:", " ".join(str(x) for x in cmd))
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if p.stdout and not quiet:
        print(p.stdout[-12000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(map(str, cmd))}")
    return p

try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Colab authenticated.")
except Exception as e:
    print("Auth note:", repr(e))

run(["gcloud", "config", "set", "project", PROJECT_ID])
run(["gcloud", "storage", "ls", SOURCE_LOCATOR_REGISTRY_GCS])
run(["gcloud", "secrets", "describe", "pathology-hub-api-key"])
run(["gcloud", "secrets", "describe", OPENAI_SECRET_NAME], check=False)

def recover_latest_source_from_cloudbuild():
    print("\nRecovering latest Cloud Build source tarball for pathology-hub-v04...")
    builds_raw = subprocess.check_output([
        "gcloud", "builds", "list",
        "--sort-by=~createTime",
        "--limit=30",
        "--format=json",
    ], text=True)
    builds = json.loads(builds_raw)

    chosen = None
    for b in builds:
        images = b.get("images") or []
        status = b.get("status")
        src = b.get("source", {}).get("storageSource", {})
        if status == "SUCCESS" and any("pathology-hub-v04" in x for x in images) and src.get("bucket") and src.get("object"):
            chosen = b
            break

    if not chosen:
        raise RuntimeError("Could not find successful Cloud Build source for pathology-hub-v04.")

    src = chosen["source"]["storageSource"]
    src_uri = f"gs://{src['bucket']}/{src['object']}"
    local_tgz = APP_DIR / "latest_cloudbuild_source.tgz"
    run(["gcloud", "storage", "cp", src_uri, str(local_tgz)])

    extract_dir = APP_DIR / "_source_extract"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(local_tgz, "r:gz") as tf:
        tf.extractall(extract_dir)

    candidates = list(extract_dir.rglob("app.py"))
    if not candidates:
        raise RuntimeError("No app.py in Cloud Build source.")
    root = candidates[0].parent

    for name in ["app.py", "requirements.txt", "Dockerfile"]:
        p = root / name
        if p.exists():
            (APP_DIR / name).write_text(p.read_text(encoding="utf-8"), encoding="utf-8")

    print("Recovered source from:", src_uri)

if not APP_PATH.exists():
    recover_latest_source_from_cloudbuild()

app_text = APP_PATH.read_text(encoding="utf-8")

backup = APP_DIR / f"app_pre_v046_{datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')}.py"
backup.write_text(app_text, encoding="utf-8")
print("Backup:", backup)

# Remove previous v04.6 patch if rerunning.
marker = "# ============================================================\n# v04.6 SOURCE LOCATOR PATCH"
pos = app_text.find(marker)
if pos >= 0:
    app_text = app_text[:pos].rstrip() + "\n"

# Patch app version if present.
app_text = re.sub(
    r'APP_VERSION\s*=\s*"[^"]+"',
    'APP_VERSION = "1.5.6-source-locator-v04"',
    app_text,
    count=1,
)

v046_patch = r'''
# ============================================================
# v04.6 SOURCE LOCATOR PATCH
# Enriches searchEvidence results with source/reference links.
# No new GPT Action.
# ============================================================

SOURCE_LOCATOR_REGISTRY_GCS = os.environ.get("SOURCE_LOCATOR_REGISTRY_GCS")
SOURCE_LOCATOR_PATH = DATA_DIR / "source_locator_registry_v1.jsonl"
_SOURCE_LOCATOR = None

def ensure_source_locator_registry():
    if SOURCE_LOCATOR_REGISTRY_GCS and (not SOURCE_LOCATOR_PATH.exists() or SOURCE_LOCATOR_PATH.stat().st_size < 100):
        _download_gcs(SOURCE_LOCATOR_REGISTRY_GCS, SOURCE_LOCATOR_PATH)

def load_source_locator_registry():
    global _SOURCE_LOCATOR
    ensure_source_locator_registry()
    if _SOURCE_LOCATOR is not None:
        return _SOURCE_LOCATOR

    by_family_id = {}
    by_source_id = {}

    if SOURCE_LOCATOR_PATH.exists():
        with SOURCE_LOCATOR_PATH.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    r = json.loads(line)
                except Exception:
                    continue
                fam = str(r.get("source_family") or "").lower()
                sid = str(r.get("source_id") or "").strip()
                if sid:
                    by_family_id[(fam, sid)] = r
                    by_source_id.setdefault(sid, r)

    _SOURCE_LOCATOR = {
        "by_family_id": by_family_id,
        "by_source_id": by_source_id,
        "count": len(by_family_id),
    }
    return _SOURCE_LOCATOR

def _safe_int(x):
    try:
        if x is None or x == "":
            return None
        return int(float(x))
    except Exception:
        return None

def _safe_float(x):
    try:
        if x is None or x == "":
            return None
        return float(x)
    except Exception:
        return None

def _append_fragment(url, fragment):
    if not url:
        return None
    base = str(url).split("#", 1)[0]
    return base + fragment

def make_pdf_page_url(pdf_url, page, offset=0):
    p = _safe_int(page)
    off = _safe_int(offset) or 0
    if not pdf_url or p is None:
        return None
    # Browser PDF viewers commonly use #page=N. This is document-page based.
    return _append_fragment(pdf_url, f"#page={max(1, p + off)}")

def make_video_time_url(video_url, start_sec=None, end_sec=None):
    s = _safe_float(start_sec)
    e = _safe_float(end_sec)
    if not video_url or s is None:
        return None
    if e is not None and e > s:
        return _append_fragment(video_url, f"#t={s:g},{e:g}")
    return _append_fragment(video_url, f"#t={s:g}")

def enrich_hit_with_locator(hit: dict, family_hint: str = None):
    if not isinstance(hit, dict):
        return hit

    loc = load_source_locator_registry()
    by_family_id = loc.get("by_family_id", {})
    by_source_id = loc.get("by_source_id", {})

    source_id = str(hit.get("source_id") or hit.get("source") or hit.get("article_id") or "").strip()
    family = (family_hint or hit.get("source") or hit.get("source_name") or "").lower()

    rec = None
    if source_id:
        rec = by_family_id.get((family, source_id)) or by_family_id.get(("textbooks", source_id)) or by_family_id.get(("videos", source_id)) or by_source_id.get(source_id)

    reference_links = list(hit.get("reference_links") or [])

    # Existing source URL from WHO/PathOut/Journals.
    existing_url = hit.get("source_url") or hit.get("url")
    if existing_url:
        reference_links.append({
            "kind": "source_url",
            "label": "Source URL",
            "url": existing_url,
            "source_family": family_hint or hit.get("source") or hit.get("source_name"),
        })

    if rec:
        # Textbook PDF page links.
        pdf_url = rec.get("pdf_url")
        page = hit.get("page")
        page_url = make_pdf_page_url(pdf_url, page, rec.get("pdf_page_offset", 0))

        if pdf_url:
            hit["source_pdf_url"] = pdf_url
            hit["source_pdf_gcs_uri"] = rec.get("pdf_gcs_uri")
        if page_url:
            hit["source_page_url"] = page_url
            hit["pdf_page_link_verified"] = bool(rec.get("page_link_verified"))
            reference_links.append({
                "kind": "textbook_pdf_page",
                "label": f"PDF page {page}",
                "url": page_url,
                "source_family": "textbooks",
                "source_id": rec.get("source_id"),
                "page": page,
                "pdf_page_offset": rec.get("pdf_page_offset", 0),
                "verified": bool(rec.get("page_link_verified")),
            })

        # Video timestamp links.
        video_url = rec.get("video_url")
        start_sec = hit.get("start_sec") or hit.get("timestamp_start_sec") or hit.get("start")
        end_sec = hit.get("end_sec") or hit.get("timestamp_end_sec") or hit.get("end")
        time_url = make_video_time_url(video_url, start_sec, end_sec)

        if video_url:
            hit["video_url"] = video_url
            hit["video_gcs_uri"] = rec.get("video_gcs_uri")
            reference_links.append({
                "kind": "video",
                "label": "Source video",
                "url": video_url,
                "source_family": "videos",
                "source_id": rec.get("source_id"),
            })
        if time_url:
            hit["video_time_url"] = time_url
            hit["timestamp_link_verified"] = bool(rec.get("timestamp_link_verified"))
            reference_links.append({
                "kind": "video_timestamp",
                "label": f"Video timestamp {start_sec}",
                "url": time_url,
                "source_family": "videos",
                "source_id": rec.get("source_id"),
                "start_sec": start_sec,
                "end_sec": end_sec,
                "verified": bool(rec.get("timestamp_link_verified")),
            })

        hit["source_locator"] = {
            "source_family": rec.get("source_family"),
            "source_id": rec.get("source_id"),
            "title": rec.get("title"),
            "page_link_method": rec.get("page_link_method"),
            "timestamp_link_method": rec.get("timestamp_link_method"),
        }

    # Deduplicate reference_links by URL/kind.
    seen = set()
    clean_links = []
    for link in reference_links:
        key = (link.get("kind"), link.get("url"))
        if not link.get("url") or key in seen:
            continue
        seen.add(key)
        clean_links.append(link)

    if clean_links:
        hit["reference_links"] = clean_links

    return hit

def enrich_evidence_response_with_locators(resp):
    if not isinstance(resp, dict):
        return resp

    for group, fam in [
        ("textbook_results", "textbooks"),
        ("journal_results", "journals"),
        ("pathout_results", "pathout"),
        ("who_results", "who"),
        ("video_results", "videos"),
    ]:
        if isinstance(resp.get(group), list):
            resp[group] = [enrich_hit_with_locator(h, fam) for h in resp[group]]

    if isinstance(resp.get("figures"), list):
        for f in resp["figures"]:
            if isinstance(f, dict):
                # Figures already should have figure_url/image_url. Add source URL link if present.
                links = list(f.get("reference_links") or [])
                for url_key in ["figure_url", "image_url", "source_url", "url"]:
                    url = f.get(url_key)
                    if url:
                        links.append({
                            "kind": url_key,
                            "label": url_key,
                            "url": url,
                            "source_family": f.get("source") or f.get("source_name"),
                            "source_id": f.get("source_id"),
                        })
                seen = set()
                out = []
                for link in links:
                    key = (link.get("kind"), link.get("url"))
                    if not link.get("url") or key in seen:
                        continue
                    seen.add(key)
                    out.append(link)
                if out:
                    f["reference_links"] = out

    resp.setdefault("warnings", [])
    try:
        count = load_source_locator_registry().get("count", 0)
        resp["source_locator_status"] = {
            "enabled": True,
            "registry_records_loaded": count,
            "note": "Textbook page links use #page=N from chunk page plus optional offset; timestamp links require start_sec/end_sec."
        }
    except Exception as e:
        resp["source_locator_status"] = {"enabled": False, "error": repr(e)}
        resp["warnings"].append(f"source_locator_error: {repr(e)}")

    return resp

# Capture existing routes before replacement.
_OLD_HEALTH_ENDPOINT_V046 = None
_OLD_SEARCH_ENDPOINT_V046 = None

for _r in list(app.router.routes):
    if getattr(_r, "path", None) == "/health" and "GET" in getattr(_r, "methods", set()):
        _OLD_HEALTH_ENDPOINT_V046 = getattr(_r, "endpoint", None)
    if getattr(_r, "path", None) == "/evidence/search" and "POST" in getattr(_r, "methods", set()):
        _OLD_SEARCH_ENDPOINT_V046 = getattr(_r, "endpoint", None)

# Remove old /health and /evidence/search routes.
app.router.routes = [
    r for r in app.router.routes
    if not (
        (getattr(r, "path", None) == "/evidence/search" and "POST" in getattr(r, "methods", set()))
        or (getattr(r, "path", None) == "/health" and "GET" in getattr(r, "methods", set()))
    )
]

@app.on_event("startup")
def startup_event_v046():
    try:
        ensure_artifacts()
    except Exception as e:
        print(f"Startup artifact warning: {e}")
    try:
        ensure_source_locator_registry()
    except Exception as e:
        print(f"Startup source locator warning: {e}")

@app.get("/health")
def health_v046():
    base = {}
    if _OLD_HEALTH_ENDPOINT_V046:
        try:
            base = _OLD_HEALTH_ENDPOINT_V046()
        except Exception as e:
            base = {"old_health_error": repr(e)}
    if not isinstance(base, dict):
        base = {"old_health": str(base)}

    try:
        locator = load_source_locator_registry()
        base["source_locator_enabled"] = True
        base["source_locator_records_loaded"] = locator.get("count", 0)
        base["source_locator_registry_gcs"] = SOURCE_LOCATOR_REGISTRY_GCS
    except Exception as e:
        base["source_locator_enabled"] = False
        base["source_locator_error"] = repr(e)

    base["version"] = "1.5.6-source-locator-v04"
    base["source_locator_mode"] = "textbook_pdf_page_links_and_video_timestamp_links"
    return base

@app.post("/evidence/search")
def search_evidence_v046(req: EvidenceSearchRequest, request: Request, x_api_key: Optional[str] = Header(None, alias="X-API-Key")):
    if not _OLD_SEARCH_ENDPOINT_V046:
        raise HTTPException(status_code=500, detail="Previous searchEvidence endpoint not found for v04.6 wrapper.")

    try:
        resp = _OLD_SEARCH_ENDPOINT_V046(req, request, x_api_key)
    except TypeError:
        # Fallback for older function arg order/names.
        resp = _OLD_SEARCH_ENDPOINT_V046(req=req, request=request, x_api_key=x_api_key)

    return enrich_evidence_response_with_locators(resp)
'''

APP_PATH.write_text(app_text.rstrip() + "\n\n" + v046_patch + "\n", encoding="utf-8")

print("Patched app.py to v04.6")
run(["python", "-m", "py_compile", str(APP_PATH)])

# Requirements check
req_path = APP_DIR / "requirements.txt"
if req_path.exists():
    req = req_path.read_text(encoding="utf-8")
else:
    req = ""
if "fastapi" not in req:
    req += "\nfastapi==0.115.6\nuvicorn[standard]==0.34.0\ngoogle-cloud-storage==2.19.0\nrequests==2.32.3\npydantic==2.10.4\nnumpy==1.26.4\nfaiss-cpu==1.8.0.post1\nopenai>=1.0.0\npillow>=10.0.0\n"
req_path.write_text(req.strip() + "\n", encoding="utf-8")

dockerfile = APP_DIR / "Dockerfile"
if not dockerfile.exists():
    dockerfile.write_text("""\
FROM python:3.11-slim
ENV PYTHONUNBUFFERED=1
WORKDIR /app
RUN apt-get update && apt-get install -y --no-install-recommends ca-certificates libopenjp2-7 libtiff6 && rm -rf /var/lib/apt/lists/*
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8080"]
""", encoding="utf-8")

# Build/deploy
for api in ["artifactregistry.googleapis.com", "cloudbuild.googleapis.com", "run.googleapis.com", "secretmanager.googleapis.com", "storage.googleapis.com"]:
    run(["gcloud", "services", "enable", api])

run(["gcloud", "auth", "configure-docker", f"{REGION}-docker.pkg.dev", "--quiet"])

print("\nBuilding v04.6 image...")
run(["gcloud", "builds", "submit", "--tag", IMAGE, "."], cwd=APP_DIR)

print("\nDeploying v04.6...")
# Preserve existing service env and add SOURCE_LOCATOR_REGISTRY_GCS.
# This deploy assumes previous app env vars are baked in existing code/defaults where needed.
# To be safer, we only add/update source locator env var.
run([
    "gcloud", "run", "services", "update", SERVICE_NAME,
    "--region", REGION,
    "--image", IMAGE,
    "--memory", "12Gi",
    "--cpu", "4",
    "--timeout", "300",
    "--min-instances", "1",
    "--update-env-vars", f"SOURCE_LOCATOR_REGISTRY_GCS={SOURCE_LOCATOR_REGISTRY_GCS}",
    "--quiet",
])

service_url = subprocess.check_output([
    "gcloud", "run", "services", "describe", SERVICE_NAME,
    "--region", REGION,
    "--format", "value(status.url)"
], text=True).strip()

print("\nSERVICE URL:", service_url)

api_key = subprocess.check_output([
    "gcloud", "secrets", "versions", "access", "latest",
    "--secret", "pathology-hub-api-key"
], text=True).strip()

print("\nHealth check...")
hr = requests.get(f"{service_url}/health", timeout=300)
print("health:", hr.status_code)
health = hr.json()
print(json.dumps(health, indent=2)[:6000])
assert hr.status_code == 200
assert health.get("source_locator_enabled") is True

def post_search(payload):
    r = requests.post(
        f"{service_url}/evidence/search",
        headers={"X-API-Key": api_key, "Content-Type": "application/json"},
        json=payload,
        timeout=300,
    )
    print("\nQUERY:", payload["query"])
    print("status:", r.status_code)
    data = r.json()
    print("source_status:", data.get("source_status"))
    print("source_locator_status:", data.get("source_locator_status"))
    print(json.dumps(data, indent=2)[:9000])
    assert r.status_code == 200
    return data

print("\nTextbook page-link smoke test...")
tb = post_search({
    "query": "soft tissue tumor immunohistochemistry",
    "sources": ["textbooks"],
    "max_results": 3,
    "include_figures": False,
    "max_figures": 0,
    "compact": True,
    "excerpt_char_limit": 700,
})

# Verify at least any textbook hit has reference_links if its source_id mapped.
has_locator = any(h.get("reference_links") for h in tb.get("textbook_results", []))
print("Any textbook reference_links:", has_locator)

print("\nManual BST Horvai page-link smoke test...")
tb2 = post_search({
    "query": "BST Horvai bone tumor",
    "sources": ["textbooks"],
    "max_results": 3,
    "include_figures": False,
    "max_figures": 0,
    "compact": True,
    "excerpt_char_limit": 700,
})

# Write handoff
handoff = {
    "created_at_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "workstream": "Backend API / Source Locator Link Layer",
    "purpose": "Enrich searchEvidence results with exact source links: textbook PDF page URLs and future video timestamp URLs.",
    "service": SERVICE_NAME,
    "service_url": service_url,
    "version": "1.5.6-source-locator-v04",
    "source_locator_registry_gcs": SOURCE_LOCATOR_REGISTRY_GCS,
    "provided_action": "POST /evidence/search",
    "new_response_fields": [
        "source_pdf_url",
        "source_page_url",
        "video_url",
        "video_time_url",
        "reference_links",
        "source_locator",
        "source_locator_status"
    ],
    "known_limitations": [
        "PDF page links use chunk page plus optional offset; offsets require per-book audit for perfect page fidelity.",
        "Video timestamp links require timecoded transcript chunks with start_sec/end_sec.",
        "No separate GPT Action was added."
    ],
    "next_steps": [
        "Audit PDF page offsets for high-use textbooks.",
        "Ingest VTT/SRT/JSON video transcripts into normalized timestamped chunks.",
        "Teach GPT to display reference_links when asked for sources, pages, or video timestamps."
    ]
}

handoff_path = APP_DIR / "HANDOFF_PATHOLOGY_HUB_V04_6_SOURCE_LOCATOR_API.json"
handoff_path.write_text(json.dumps(handoff, indent=2), encoding="utf-8")
run(["gcloud", "storage", "cp", str(handoff_path), "gs://pathology_hub/06_audits/handoff_packets/"])

print("\n✅ DEPLOYED v04.6 SOURCE LOCATOR API")
print("Service:", service_url)
print("Handoff:", handoff_path)